# Leakage-Safe Sparse Hierarchical Two-RC LPV Identification

## Purpose

This notebook develops the first parameter-varying equivalent-circuit model after the fixed one-RC and two-RC baselines were rejected in Notebook 11.

The model retains the two-RC Thévenin structure while allowing the physical parameter maps to vary with the frozen scheduling coordinates. The hierarchy is

$$
\text{global} \rightarrow \text{chemistry} \rightarrow \text{cell}.
$$

The notebook performs:

- exact Notebook 11 handoff verification;
- chronological split and trajectory-identity auditing;
- strict-test quarantine during development;
- training-only scheduler normalization;
- second-order strong-heredity basis construction;
- physically positive and ordered two-RC parameterization;
- identifiable global–chemistry–cell decomposition;
- recursive free-run optimization;
- calibration-only model and regularization selection;
- final ordinary-test and strict-test evaluation after freezing;
- diagnostics, decisions, and export of the frozen model bundle.

Positive current denotes discharge throughout.


# Hypotheses

**H1 — LPV adequacy.** A two-RC model with parameter maps scheduled by SOC and temperature reduces held-out recursive voltage error relative to the rejected fixed two-RC baseline.

**H2 — Hierarchical transfer.** Shared global structure plus chemistry- and cell-level deviations improves calibration performance relative to a global-only LPV model.

**H3 — Sparse sufficiency.** A strong-heredity sparse basis attains predictive performance comparable to or better than the dense hierarchical basis with fewer active groups.

**H4 — Physical consistency.** The transformed parameterization preserves

$$
R_0>0,\quad R_1>0,\quad R_2>0,\quad 0<\tau_1<\tau_2
$$

throughout the evaluated scheduling support.

**H5 — Recursive stability.** For every positive sampling interval,

$$
0<\exp(-\Delta t/\tau_1)<1,\qquad
0<\exp(-\Delta t/\tau_2)<1.
$$

**H6 — Leakage safety.** Hyperparameters, support, normalization, and model selection are determined without ordinary-test or strict-test sample access.


# Data-to-model contract

The authoritative trajectory identifier is

$$
\texttt{chemistry::cell\_id::file\_id::segment\_index}.
$$

The inherited state-augmented sample table must provide measured voltage, discharge-positive current, temperature, reconstructed SOC, chemistry-specific OCV, positive sampling intervals, trajectory identity, and published split labels.

The active scheduling vector is frozen as

$$
z_k = \begin{bmatrix}
\mathrm{SOC}_k \\
T_k
\end{bmatrix}.
$$

Training-only affine normalization maps each scheduler to approximately $[-1,1]$. The candidate basis is

$$
\phi(z)=
\begin{bmatrix}
1,\ \xi_{\mathrm{SOC}},\ \xi_T,\ 
\xi_{\mathrm{SOC}}^2,\ \xi_T^2,\ 
\xi_{\mathrm{SOC}}\xi_T
\end{bmatrix}^{\!\top}.
$$

The transformed parameter maps are

$$
R_0=e^{\eta_{R_0}},\quad
R_1=e^{\eta_{R_1}},\quad
R_2=e^{\eta_{R_2}},\quad
\tau_1=e^{\eta_{\tau_1}},\quad
\tau_2=\tau_1+e^{\eta_{\Delta\tau}}.
$$

The terminal-voltage model is evaluated recursively without resetting polarization states inside a trajectory.


In [1]:
# ======================================================================================
# CELL 5 — BOOTSTRAP THE PROJECT ENVIRONMENT AND DEFINE CORE UTILITIES
# ======================================================================================

from __future__ import annotations

import contextlib
import hashlib
import importlib
import json
import math
import os
import random
import re
import sys
import time
import warnings

from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from scipy import linalg as scipy_linalg
from scipy import sparse as scipy_sparse

try:
    import jax
    import jax.numpy as jnp
except Exception as exc:
    raise RuntimeError(
        "JAX is required for Notebook 12. Run the Notebook 10 environment setup first."
    ) from exc

jax.config.update("jax_enable_x64", True)

PROJECT_ROOT_CANDIDATES = (
    Path("/home/xzkhu/Downloads/batter_lpv_project"),
    Path.cwd(),
    Path.cwd().parent,
)

PROJECT_ROOT = next(
    (
        candidate.expanduser().resolve()
        for candidate in PROJECT_ROOT_CANDIDATES
        if (candidate / "data").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "The battery LPV project root could not be located. "
        "Expected /home/xzkhu/Downloads/batter_lpv_project or a parent containing data/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ENVIRONMENT_IMPORTED = False
PROJECT_ENVIRONMENT_IMPORT_ERROR = None

try:
    import src.project_environment as project_environment

    project_environment = importlib.reload(project_environment)

    for shared_name in dir(project_environment):
        if shared_name.startswith("_"):
            continue
        globals().setdefault(shared_name, getattr(project_environment, shared_name))

    PROJECT_ENVIRONMENT_IMPORTED = True
except Exception as exc:
    PROJECT_ENVIRONMENT_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def ensure_directory(path: str | Path) -> Path:
    resolved = Path(path).expanduser().resolve()
    resolved.mkdir(parents=True, exist_ok=True)
    return resolved


def calculate_streaming_sha256(
    file_path: str | Path,
    *,
    chunk_bytes: int = 8 * 1024 * 1024,
) -> str:
    path = Path(file_path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Cannot hash a missing file: {path}")

    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(int(chunk_bytes))
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def write_json_atomic(
    file_path: str | Path,
    payload: Mapping[str, Any] | Sequence[Any],
) -> None:
    path = Path(file_path).expanduser().resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")

    def convert(value: Any) -> Any:
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, np.ndarray):
            return [convert(item) for item in value.tolist()]
        if isinstance(value, (np.integer,)):
            return int(value)
        if isinstance(value, (np.floating,)):
            number = float(value)
            return number if math.isfinite(number) else None
        if isinstance(value, (np.bool_,)):
            return bool(value)
        if isinstance(value, Mapping):
            return {str(key): convert(item) for key, item in value.items()}
        if isinstance(value, (list, tuple, set)):
            return [convert(item) for item in value]
        return value

    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(convert(payload), handle, indent=2, sort_keys=True)
        handle.write("\n")

    temporary.replace(path)


def normalize_column_name(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().casefold())


def resolve_column(
    available_columns: Sequence[str],
    candidates: Sequence[str],
    *,
    semantic_name: str,
    required: bool = True,
) -> str | None:
    available = tuple(str(column) for column in available_columns)

    for candidate in candidates:
        if candidate in available:
            return candidate

    casefold_lookup: dict[str, list[str]] = {}
    normalized_lookup: dict[str, list[str]] = {}

    for column in available:
        casefold_lookup.setdefault(column.casefold(), []).append(column)
        normalized_lookup.setdefault(normalize_column_name(column), []).append(column)

    for candidate in candidates:
        matches = casefold_lookup.get(str(candidate).casefold(), [])
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(
                f"Ambiguous case-insensitive columns for {semantic_name!r}: {matches}"
            )

    for candidate in candidates:
        matches = normalized_lookup.get(normalize_column_name(candidate), [])
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(
                f"Ambiguous normalized columns for {semantic_name!r}: {matches}"
            )

    if required:
        raise KeyError(
            f"No column could be resolved for {semantic_name!r}. "
            f"Candidates: {tuple(candidates)}"
        )
    return None


def canonical_string_series(
    series: pd.Series,
    *,
    uppercase: bool = False,
) -> pd.Series:
    output = series.astype("string").str.strip()
    if uppercase:
        output = output.str.upper()
    return output


def canonical_nullable_integer_series(
    series: pd.Series,
    *,
    column_name: str,
    allow_missing: bool,
) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    nonmissing = numeric.dropna()

    if not np.allclose(
        nonmissing.to_numpy(dtype=np.float64),
        np.round(nonmissing.to_numpy(dtype=np.float64)),
        rtol=0.0,
        atol=0.0,
    ):
        raise ValueError(f"{column_name} contains nonintegral values.")

    output = pd.Series(pd.array(np.round(numeric), dtype="Int64"), index=series.index)

    if not allow_missing and output.isna().any():
        raise ValueError(f"{column_name} contains missing values.")

    return output


def make_dataframe_parquet_safe(
    dataframe: pd.DataFrame,
    *,
    dataframe_name: str = "dataframe",
) -> pd.DataFrame:
    output = dataframe.copy()

    for column in output.columns:
        series = output[column]

        if pd.api.types.is_object_dtype(series.dtype):
            nonmissing = series.dropna()

            if nonmissing.empty:
                output[column] = series.astype("string")
                continue

            value_types = {type(value) for value in nonmissing.head(1000)}

            if value_types.issubset({str}):
                output[column] = series.astype("string")
            elif value_types.issubset({bool, np.bool_}):
                output[column] = series.astype("boolean")
            elif all(
                isinstance(value, (int, np.integer))
                and not isinstance(value, (bool, np.bool_))
                for value in nonmissing.head(1000)
            ):
                output[column] = pd.array(series, dtype="Int64")
            elif all(
                isinstance(value, (int, float, np.integer, np.floating))
                and not isinstance(value, (bool, np.bool_))
                for value in nonmissing.head(1000)
            ):
                output[column] = pd.to_numeric(series, errors="coerce").astype("Float64")
            else:
                output[column] = series.map(
                    lambda value: None
                    if value is None or (isinstance(value, float) and np.isnan(value))
                    else json.dumps(value, sort_keys=True)
                    if isinstance(value, (dict, list, tuple, set))
                    else str(value)
                ).astype("string")

    try:
        pa.Table.from_pandas(output, preserve_index=False, safe=True)
    except Exception as exc:
        raise RuntimeError(
            f"{dataframe_name} could not be converted to a deterministic Arrow table."
        ) from exc

    return output


def write_dataframe_parquet_atomic(
    dataframe: pd.DataFrame,
    file_path: str | Path,
) -> None:
    path = Path(file_path).expanduser().resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    safe_df = make_dataframe_parquet_safe(dataframe, dataframe_name=path.name)
    table = pa.Table.from_pandas(safe_df, preserve_index=False, safe=True)
    pq.write_table(
        table,
        temporary,
        compression="zstd",
        use_dictionary=True,
        write_statistics=True,
    )
    temporary.replace(path)


print("=" * 100)
print("PROJECT BOOTSTRAP")
print("=" * 100)
print(f"Project root                  : {PROJECT_ROOT}")
print(f"Shared environment imported  : {PROJECT_ENVIRONMENT_IMPORTED}")
if PROJECT_ENVIRONMENT_IMPORT_ERROR:
    print(f"Shared import note            : {PROJECT_ENVIRONMENT_IMPORT_ERROR}")
print(f"JAX float64 enabled          : {jax.config.read('jax_enable_x64')}")
print("=" * 100)


PROJECT BOOTSTRAP
Project root                  : /home/xzkhu/Downloads/batter_lpv_project
Shared environment imported  : True
JAX float64 enabled          : True


In [2]:
# ======================================================================================
# CELL 6 — FREEZE NOTEBOOK 12 CONFIGURATION
# ======================================================================================

NOTEBOOK_NUMBER: int = 12
NOTEBOOK_NAME: str = "Leakage-Safe Sparse Hierarchical Two-RC LPV Identification"
NOTEBOOK_VERSION: str = "1.0.0"

RANDOM_SEED: int = 20260701
FAST_DEMO: bool = False

PRIMARY_MODEL_ORDER: int = 2
CURRENT_POSITIVE_IS_DISCHARGE: bool = True
COULOMBIC_EFFICIENCY: float = 1.0

TRAJECTORY_UID_COLUMN: str = "trajectory_uid"
TRAJECTORY_ID_SEPARATOR: str = "::"

PRIMARY_SPLIT_LABELS: tuple[str, ...] = (
    "train",
    "calibration",
    "test",
    "strict_test",
    "embargo",
)

REST_CURRENT_A: float = 0.05
INITIAL_REST_SAMPLES: int = 5
LOSS_START_SAMPLE_INDEX: int = 1

NOTEBOOK_12_SCHEDULING_VARIABLES: tuple[str, ...] = (
    "soc",
    "temperature",
)

NOTEBOOK_12_SCHEDULER_DIMENSION: int = len(
    NOTEBOOK_12_SCHEDULING_VARIABLES
)

LPV_TRANSFORMED_PARAMETERS: tuple[str, ...] = (
    "eta_R0",
    "eta_R1",
    "eta_R2",
    "eta_tau1",
    "eta_tau_gap",
)

NOTEBOOK_12_PARAMETER_COUNT: int = len(
    LPV_TRANSFORMED_PARAMETERS
)

# Physical numerical guardrails used during optimization and final auditing.
R_MIN_OHM: float = 1.0e-5
R_MAX_OHM: float = 1.0
TAU1_MIN_S: float = 1.0e-2
TAU1_MAX_S: float = 2.0e3
TAU_GAP_MIN_S: float = 1.0e-2
TAU_GAP_MAX_S: float = 2.0e4

# Optimization settings.
TRAINING_BATCH_TRAJECTORIES: int = 8 if not FAST_DEMO else 2
TRAINING_STEPS_DENSE: int = 1800 if not FAST_DEMO else 60
TRAINING_STEPS_SPARSE: int = 2400 if not FAST_DEMO else 80
REFIT_STEPS: int = 1200 if not FAST_DEMO else 40
LEARNING_RATE: float = 2.0e-3
GRADIENT_NORM_CLIP: float = 10.0
ADAM_BETA1: float = 0.9
ADAM_BETA2: float = 0.999
ADAM_EPSILON: float = 1.0e-8
RIDGE_WEIGHT: float = 1.0e-7
PSEUDO_HUBER_DELTA_V: float = 0.020
GROUP_SUPPORT_THRESHOLD: float = 1.0e-4

SPARSE_LAMBDA_GRID: tuple[float, ...] = (
    (1.0e-6, 3.0e-6, 1.0e-5, 3.0e-5, 1.0e-4)
    if not FAST_DEMO
    else (1.0e-5, 1.0e-4)
)

CALIBRATION_COMPLEXITY_TOLERANCE: float = 0.0025
CHECKPOINT_INTERVAL_STEPS: int = 100
EVALUATION_TRAJECTORY_LIMIT: int | None = None if not FAST_DEMO else 30

NOTEBOOK_12_HASH_CHUNK_BYTES: int = 8 * 1024 * 1024

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
NOTEBOOK_12_RANDOM_GENERATOR = np.random.default_rng(RANDOM_SEED)
JAX_RANDOM_KEY = jax.random.PRNGKey(RANDOM_SEED)

if PRIMARY_MODEL_ORDER != 2:
    raise RuntimeError("Notebook 12 requires a two-RC model.")

if not CURRENT_POSITIVE_IS_DISCHARGE:
    raise RuntimeError("Notebook 12 requires discharge-positive current.")

if NOTEBOOK_12_SCHEDULER_DIMENSION != 2:
    raise RuntimeError("The frozen Notebook 12 scheduler dimension must be two.")

if NOTEBOOK_12_PARAMETER_COUNT != 5:
    raise RuntimeError("The two-RC transformed parameter count must be five.")

if not (0.0 < PSEUDO_HUBER_DELTA_V):
    raise ValueError("PSEUDO_HUBER_DELTA_V must be positive.")

if not all(value > 0.0 for value in SPARSE_LAMBDA_GRID):
    raise ValueError("Every sparse regularization value must be positive.")

NOTEBOOK_12_CONFIGURATION = {
    "notebook": {
        "number": NOTEBOOK_NUMBER,
        "name": NOTEBOOK_NAME,
        "version": NOTEBOOK_VERSION,
    },
    "reproducibility": {
        "random_seed": RANDOM_SEED,
        "fast_demo": FAST_DEMO,
    },
    "model": {
        "order": PRIMARY_MODEL_ORDER,
        "current_positive_is_discharge": CURRENT_POSITIVE_IS_DISCHARGE,
        "transformed_parameters": list(LPV_TRANSFORMED_PARAMETERS),
        "scheduler_variables": list(NOTEBOOK_12_SCHEDULING_VARIABLES),
    },
    "optimization": {
        "batch_trajectories": TRAINING_BATCH_TRAJECTORIES,
        "dense_steps": TRAINING_STEPS_DENSE,
        "sparse_steps": TRAINING_STEPS_SPARSE,
        "refit_steps": REFIT_STEPS,
        "learning_rate": LEARNING_RATE,
        "pseudo_huber_delta_v": PSEUDO_HUBER_DELTA_V,
        "sparse_lambda_grid": list(SPARSE_LAMBDA_GRID),
    },
}

print("=" * 100)
print("NOTEBOOK 12 CONFIGURATION")
print("=" * 100)
print(f"Random seed                 : {RANDOM_SEED}")
print(f"Fast demo                   : {FAST_DEMO}")
print(f"Scheduler variables         : {NOTEBOOK_12_SCHEDULING_VARIABLES}")
print(f"Transformed parameter maps  : {LPV_TRANSFORMED_PARAMETERS}")
print(f"Sparse lambda grid          : {SPARSE_LAMBDA_GRID}")
print("=" * 100)


NOTEBOOK 12 CONFIGURATION
Random seed                 : 20260701
Fast demo                   : False
Scheduler variables         : ('soc', 'temperature')
Transformed parameter maps  : ('eta_R0', 'eta_R1', 'eta_R2', 'eta_tau1', 'eta_tau_gap')
Sparse lambda grid          : (1e-06, 3e-06, 1e-05, 3e-05, 0.0001)


In [3]:
# ======================================================================================
# CELL 7 — CREATE OUTPUT DIRECTORIES AND CORE MANIFESTS
# ======================================================================================

NOTEBOOK_12_ROOT = ensure_directory(
    PROJECT_ROOT / "data" / "processed" / "notebook_12"
)

NOTEBOOK_12_DATA_DIR = ensure_directory(
    NOTEBOOK_12_ROOT / "data"
)

NOTEBOOK_12_TABLE_DIR = ensure_directory(
    NOTEBOOK_12_ROOT / "tables"
)

NOTEBOOK_12_MODEL_DIR = ensure_directory(
    NOTEBOOK_12_ROOT / "models"
)

NOTEBOOK_12_MANIFEST_DIR = ensure_directory(
    NOTEBOOK_12_ROOT / "manifests"
)

NOTEBOOK_12_CONTRACT_DIR = ensure_directory(
    NOTEBOOK_12_ROOT / "contracts"
)

NOTEBOOK_12_CHECKPOINT_DIR = ensure_directory(
    NOTEBOOK_12_ROOT / "checkpoints"
)

NOTEBOOK_12_LOG_DIR = ensure_directory(
    NOTEBOOK_12_ROOT / "logs"
)

NOTEBOOK_12_FIGURE_DIR = ensure_directory(
    NOTEBOOK_12_ROOT / "figures"
)

NOTEBOOK_12_REPRODUCIBILITY_MANIFEST_PATH = (
    NOTEBOOK_12_MANIFEST_DIR
    / "notebook12_reproducibility_manifest.json"
)

NOTEBOOK_12_ARTIFACT_HASH_PATH = (
    NOTEBOOK_12_MANIFEST_DIR
    / "notebook12_artifact_hashes.json"
)

NOTEBOOK_12_STRICT_TEST_QUARANTINE_PATH = (
    NOTEBOOK_12_CONTRACT_DIR
    / "strict_test_quarantine.json"
)

NOTEBOOK_12_CONFIGURATION_PATH = (
    NOTEBOOK_12_CONTRACT_DIR
    / "notebook12_configuration.json"
)

write_json_atomic(
    NOTEBOOK_12_CONFIGURATION_PATH,
    NOTEBOOK_12_CONFIGURATION,
)

write_json_atomic(
    NOTEBOOK_12_STRICT_TEST_QUARANTINE_PATH,
    {
        "created_utc": utc_now_iso(),
        "status": "ACTIVE",
        "ordinary_test_arrays_opened": False,
        "strict_test_arrays_opened": False,
        "permitted_before_model_freeze": [
            "training metadata",
            "training sample arrays",
            "calibration metadata",
            "calibration sample arrays",
            "test trajectory identities only",
            "strict-test trajectory identities only",
        ],
        "prohibited_before_model_freeze": [
            "ordinary-test sample arrays",
            "strict-test sample arrays",
            "test-driven support selection",
            "test-driven regularization selection",
            "strict-test-driven threshold selection",
        ],
    },
)

write_json_atomic(
    NOTEBOOK_12_REPRODUCIBILITY_MANIFEST_PATH,
    {
        "created_utc": utc_now_iso(),
        "notebook_number": NOTEBOOK_NUMBER,
        "notebook_name": NOTEBOOK_NAME,
        "notebook_version": NOTEBOOK_VERSION,
        "project_root": str(PROJECT_ROOT),
        "configuration_path": str(NOTEBOOK_12_CONFIGURATION_PATH),
        "random_seed": RANDOM_SEED,
        "fast_demo": FAST_DEMO,
        "status": "INITIALIZED",
    },
)

write_json_atomic(
    NOTEBOOK_12_ARTIFACT_HASH_PATH,
    {
        "created_utc": utc_now_iso(),
        "files": {
            str(NOTEBOOK_12_CONFIGURATION_PATH): calculate_streaming_sha256(
                NOTEBOOK_12_CONFIGURATION_PATH,
                chunk_bytes=NOTEBOOK_12_HASH_CHUNK_BYTES,
            ),
            str(NOTEBOOK_12_STRICT_TEST_QUARANTINE_PATH): calculate_streaming_sha256(
                NOTEBOOK_12_STRICT_TEST_QUARANTINE_PATH,
                chunk_bytes=NOTEBOOK_12_HASH_CHUNK_BYTES,
            ),
        },
    },
)

NOTEBOOK_12_DIRECTORY_REGISTRY = {
    "root": NOTEBOOK_12_ROOT,
    "data": NOTEBOOK_12_DATA_DIR,
    "tables": NOTEBOOK_12_TABLE_DIR,
    "models": NOTEBOOK_12_MODEL_DIR,
    "manifests": NOTEBOOK_12_MANIFEST_DIR,
    "contracts": NOTEBOOK_12_CONTRACT_DIR,
    "checkpoints": NOTEBOOK_12_CHECKPOINT_DIR,
    "logs": NOTEBOOK_12_LOG_DIR,
    "figures": NOTEBOOK_12_FIGURE_DIR,
}

display(
    pd.DataFrame(
        [
            {
                "role": role,
                "path": str(path),
                "exists": path.is_dir(),
            }
            for role, path in NOTEBOOK_12_DIRECTORY_REGISTRY.items()
        ]
    )
)


,role,path,exists
0,root,/home/xzkhu/Downloads/batter_lpv_project/data/...,True
1,data,/home/xzkhu/Downloads/batter_lpv_project/data/...,True
2,tables,/home/xzkhu/Downloads/batter_lpv_project/data/...,True
3,models,/home/xzkhu/Downloads/batter_lpv_project/data/...,True
4,manifests,/home/xzkhu/Downloads/batter_lpv_project/data/...,True
5,contracts,/home/xzkhu/Downloads/batter_lpv_project/data/...,True
6,checkpoints,/home/xzkhu/Downloads/batter_lpv_project/data/...,True
7,logs,/home/xzkhu/Downloads/batter_lpv_project/data/...,True
8,figures,/home/xzkhu/Downloads/batter_lpv_project/data/...,True


In [4]:
# ======================================================================================
# CELL 8 — DISCOVER AND REGISTER NOTEBOOK 11 HANDOFF ARTIFACTS
# ======================================================================================

PUBLISHED_DATASET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "published"
    / "battery_lpv_dataset_v1"
)

NOTEBOOK11_TABLE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_11"
    / "tables"
)

NOTEBOOK11_METADATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_11"
    / "metadata"
)

NOTEBOOK11_STATE_DIR = (
    NOTEBOOK11_TABLE_DIR
    / "state_reconstruction"
)

NOTEBOOK11_FIXED_MODEL_DIR = (
    NOTEBOOK11_TABLE_DIR
    / "fixed_ecm_models"
)

MANIFESTS_DIR = (
    PUBLISHED_DATASET_ROOT
    / "manifests"
)

STATE_AUGMENTED_SAMPLES_PATH = (
    NOTEBOOK11_STATE_DIR
    / "state_augmented_model_ready_samples.parquet"
)

STATE_AUGMENTED_TRAJECTORY_REGISTRY_PATH = (
    NOTEBOOK11_STATE_DIR
    / "state_augmented_trajectory_registry.parquet"
)

STATE_RECONSTRUCTION_SUMMARY_PATH = (
    NOTEBOOK11_STATE_DIR
    / "trajectory_state_reconstruction_summary.parquet"
)

STATE_RECONSTRUCTION_VALIDATION_PATH = (
    NOTEBOOK11_STATE_DIR
    / "state_reconstruction_validation.parquet"
)

FIXED_ECM_FROZEN_PARAMETERS_PATH = (
    NOTEBOOK11_FIXED_MODEL_DIR
    / "fixed_ecm_frozen_parameters.parquet"
)

FIXED_ECM_FINAL_ACCEPTANCE_DECISION_PATH = (
    NOTEBOOK11_FIXED_MODEL_DIR
    / "fixed_ecm_final_acceptance_decisions.parquet"
)

FIXED_ECM_TEST_HEADLINE_METRICS_PATH = (
    NOTEBOOK11_FIXED_MODEL_DIR
    / "fixed_ecm_test_headline_metrics.parquet"
)

PRIMARY_CHRONOLOGICAL_MANIFEST_PATH = (
    MANIFESTS_DIR
    / "primary_chronological_manifest.parquet"
)

TRAIN_SEGMENTS_MANIFEST_PATH = (
    MANIFESTS_DIR
    / "train_segments.parquet"
)

CALIBRATION_SEGMENTS_MANIFEST_PATH = (
    MANIFESTS_DIR
    / "calibration_segments.parquet"
)

TEST_SEGMENTS_MANIFEST_PATH = (
    MANIFESTS_DIR
    / "test_segments.parquet"
)

EMBARGO_SEGMENTS_MANIFEST_PATH = (
    MANIFESTS_DIR
    / "embargo_segments.parquet"
)

STRICT_TEST_SEGMENTS_MANIFEST_PATH = (
    MANIFESTS_DIR
    / "strict_test_segments.parquet"
)

NOTEBOOK_11_HANDOFF_PATHS: dict[str, Path] = {
    "state_augmented_samples": STATE_AUGMENTED_SAMPLES_PATH,
    "state_augmented_trajectory_registry": STATE_AUGMENTED_TRAJECTORY_REGISTRY_PATH,
    "state_reconstruction_summary": STATE_RECONSTRUCTION_SUMMARY_PATH,
    "state_reconstruction_validation": STATE_RECONSTRUCTION_VALIDATION_PATH,
    "fixed_ecm_frozen_parameters": FIXED_ECM_FROZEN_PARAMETERS_PATH,
    "fixed_ecm_final_acceptance_decisions": FIXED_ECM_FINAL_ACCEPTANCE_DECISION_PATH,
    "fixed_ecm_test_headline_metrics": FIXED_ECM_TEST_HEADLINE_METRICS_PATH,
    "primary_chronological_manifest": PRIMARY_CHRONOLOGICAL_MANIFEST_PATH,
    "train_segments_manifest": TRAIN_SEGMENTS_MANIFEST_PATH,
    "calibration_segments_manifest": CALIBRATION_SEGMENTS_MANIFEST_PATH,
    "test_segments_manifest": TEST_SEGMENTS_MANIFEST_PATH,
    "embargo_segments_manifest": EMBARGO_SEGMENTS_MANIFEST_PATH,
    "strict_test_segments_manifest": STRICT_TEST_SEGMENTS_MANIFEST_PATH,
}

MANDATORY_HANDOFF_KEYS = {
    "state_augmented_samples",
    "state_augmented_trajectory_registry",
    "state_reconstruction_summary",
    "state_reconstruction_validation",
    "fixed_ecm_frozen_parameters",
    "fixed_ecm_final_acceptance_decisions",
    "primary_chronological_manifest",
    "train_segments_manifest",
    "calibration_segments_manifest",
    "test_segments_manifest",
    "embargo_segments_manifest",
}

handoff_records: list[dict[str, Any]] = []

for artifact_name, artifact_path in NOTEBOOK_11_HANDOFF_PATHS.items():
    exists = artifact_path.is_file()
    suffix = artifact_path.suffix.lower()

    row_count = None
    row_group_count = None
    column_count = None
    sha256 = None

    if exists:
        sha256 = calculate_streaming_sha256(
            artifact_path,
            chunk_bytes=NOTEBOOK_12_HASH_CHUNK_BYTES,
        )

        if suffix == ".parquet":
            parquet_file = pq.ParquetFile(artifact_path)
            row_count = int(parquet_file.metadata.num_rows)
            row_group_count = int(parquet_file.metadata.num_row_groups)
            column_count = len(parquet_file.schema_arrow)

    handoff_records.append(
        {
            "artifact_name": artifact_name,
            "path": str(artifact_path),
            "mandatory": artifact_name in MANDATORY_HANDOFF_KEYS,
            "exists": exists,
            "row_count": row_count,
            "row_group_count": row_group_count,
            "column_count": column_count,
            "sha256": sha256,
        }
    )

NOTEBOOK_11_HANDOFF_DF = pd.DataFrame(handoff_records)

missing_mandatory = NOTEBOOK_11_HANDOFF_DF.loc[
    NOTEBOOK_11_HANDOFF_DF["mandatory"]
    &
    ~NOTEBOOK_11_HANDOFF_DF["exists"]
]

if not missing_mandatory.empty:
    display(missing_mandatory)
    raise FileNotFoundError(
        "One or more mandatory Notebook 11 handoff artifacts are missing."
    )

NOTEBOOK_11_HANDOFF_REGISTRY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "notebook11_handoff_registry.parquet"
)

write_dataframe_parquet_atomic(
    NOTEBOOK_11_HANDOFF_DF,
    NOTEBOOK_11_HANDOFF_REGISTRY_PATH,
)

print("=" * 110)
print("NOTEBOOK 11 HANDOFF")
print("=" * 110)
print(f"Published dataset root : {PUBLISHED_DATASET_ROOT}")
print(f"Notebook 11 table root : {NOTEBOOK11_TABLE_DIR}")
print(f"Registered artifacts   : {len(NOTEBOOK_11_HANDOFF_DF):,}")
print(f"Mandatory missing      : {len(missing_mandatory):,}")
print("=" * 110)

display(NOTEBOOK_11_HANDOFF_DF)


NOTEBOOK 11 HANDOFF
Published dataset root : /home/xzkhu/Downloads/batter_lpv_project/data/published/battery_lpv_dataset_v1
Notebook 11 table root : /home/xzkhu/Downloads/batter_lpv_project/data/processed/notebook_11/tables
Registered artifacts   : 13
Mandatory missing      : 0


,artifact_name,path,mandatory,exists,row_count,row_group_count,column_count,sha256
0,state_augmented_samples,/home/xzkhu/Downloads/batter_lpv_project/data/...,True,True,18318873.0,4938.0,33.0,4a4664125cd3087f67ebb7483f84e38c78eac7fe05be17...
1,state_augmented_trajectory_registry,/home/xzkhu/Downloads/batter_lpv_project/data/...,True,True,4938.0,1.0,170.0,2377a6582d53204efe2d8abd0a6d0192357cc63deccade...
2,state_reconstruction_summary,/home/xzkhu/Downloads/batter_lpv_project/data/...,True,True,4938.0,1.0,59.0,d938bda0687b0a8db6b740c7c2c4ced1f02f2dbbcab14d...
3,state_reconstruction_validation,/home/xzkhu/Downloads/batter_lpv_project/data/...,True,True,9.0,1.0,4.0,d5afa9a6fbd40c471e169d832ee16431c6e6f2f776bd21...
4,fixed_ecm_frozen_parameters,/home/xzkhu/Downloads/batter_lpv_project/data/...,True,True,8.0,1.0,15.0,7951643d0f67f8584c72618f9b6a27ade5ace1634bda27...
5,fixed_ecm_final_acceptance_decisions,/home/xzkhu/Downloads/batter_lpv_project/data/...,True,True,8.0,1.0,56.0,0830f6b10b1d13c52278b8deaf7430216bf8b005305608...
6,fixed_ecm_test_headline_metrics,/home/xzkhu/Downloads/batter_lpv_project/data/...,False,True,63.0,1.0,20.0,0dc2952e159374e245b7bd2b41d66c23f7f2e785e3be8c...
7,primary_chronological_manifest,/home/xzkhu/Downloads/batter_lpv_project/data/...,True,True,4942.0,1.0,59.0,fa17cb1d38c25c05e125876ff02fb91f3525c3a475325b...
8,train_segments_manifest,/home/xzkhu/Downloads/batter_lpv_project/data/...,True,True,2835.0,1.0,59.0,e877e0ce3d074218e48ecc4093e66764610f206b21a0e2...
9,calibration_segments_manifest,/home/xzkhu/Downloads/batter_lpv_project/data/...,True,True,1018.0,1.0,59.0,6cfec647095d07aef6c06e419959c9aa81bfe7f22fa46e...


In [5]:
# ======================================================================================
# CELL 9 — LOAD THE AUTHORITATIVE REGISTRIES AND RESOLVE SAMPLE COLUMNS
# ======================================================================================

STATE_SAMPLE_PARQUET = pq.ParquetFile(
    STATE_AUGMENTED_SAMPLES_PATH
)

STATE_SAMPLE_SCHEMA = STATE_SAMPLE_PARQUET.schema_arrow
STATE_SAMPLE_COLUMNS = tuple(STATE_SAMPLE_SCHEMA.names)

STATE_SAMPLE_COLUMN_BINDINGS: dict[str, str | None] = {
    "trajectory_uid": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("trajectory_uid", "unique_trajectory_id", "trajectory_id"),
        semantic_name="trajectory UID",
    ),
    "chemistry": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("chemistry", "chemistry_id", "battery_chemistry"),
        semantic_name="chemistry",
    ),
    "cell_id": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("cell_id", "cell", "battery_cell_id"),
        semantic_name="cell ID",
    ),
    "file_id": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("file_id", "source_file_id", "source_file", "file_name"),
        semantic_name="source file",
    ),
    "segment_index": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("segment_index", "trajectory_index", "segment_id"),
        semantic_name="segment index",
    ),
    "sample_index": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("sample_index", "sample_number", "row_index"),
        semantic_name="sample index",
    ),
    "elapsed_time_s": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("elapsed_time_s", "time_s", "relative_time_s"),
        semantic_name="elapsed time",
    ),
    "delta_t_to_next_s": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("delta_t_to_next_s", "dt_to_next_s", "delta_t_s", "sampling_interval_s"),
        semantic_name="sampling interval",
    ),
    "voltage_v": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("measured_voltage_v", "voltage_v", "terminal_voltage_v", "Voltage"),
        semantic_name="measured voltage",
    ),
    "current_discharge_a": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("current_discharge_a", "current_a", "Current"),
        semantic_name="discharge-positive current",
    ),
    "temperature_c": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("cell_temperature_c", "temperature_c", "Temperature"),
        semantic_name="cell temperature",
    ),
    "primary_split": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("primary_split", "split", "split_label", "dataset_split"),
        semantic_name="primary split",
    ),
    "is_strict_test": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("is_strict_test", "strict_test", "strict_test_overlay"),
        semantic_name="strict-test indicator",
        required=False,
    ),
    "capacity_for_soc_ah": resolve_column(
        STATE_SAMPLE_COLUMNS,
        (
            "capacity_for_soc_ah",
            "baseline_capacity_train_only_ah",
            "capacity_train_only_ah",
            "effective_capacity_ah",
        ),
        semantic_name="training-safe capacity",
    ),
    "soc_for_ocv": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("soc_for_ocv", "soc_reconstructed_bounded", "soc_reconstructed_raw", "soc"),
        semantic_name="SOC scheduler",
    ),
    "ocv_v": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("ocv_v", "open_circuit_voltage_v", "estimated_ocv_v"),
        semantic_name="open-circuit voltage",
    ),
    "soc_fit_eligible": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("soc_fit_eligible", "fit_eligible", "model_fit_eligible"),
        semantic_name="fit eligibility",
    ),
    "state_reconstruction_valid": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("state_reconstruction_valid", "state_valid", "soc_reconstruction_valid"),
        semantic_name="state validity",
    ),
    "soh_fraction": resolve_column(
        STATE_SAMPLE_COLUMNS,
        ("soh_fraction", "capacity_retention", "soh", "relative_capacity"),
        semantic_name="SOH fraction",
        required=False,
    ),
}

STATE_AUGMENTED_TRAJECTORY_REGISTRY_DF = pd.read_parquet(
    STATE_AUGMENTED_TRAJECTORY_REGISTRY_PATH,
    engine="pyarrow",
)

STATE_RECONSTRUCTION_SUMMARY_DF = pd.read_parquet(
    STATE_RECONSTRUCTION_SUMMARY_PATH,
    engine="pyarrow",
)

STATE_RECONSTRUCTION_VALIDATION_DF = pd.read_parquet(
    STATE_RECONSTRUCTION_VALIDATION_PATH,
    engine="pyarrow",
)

for dataframe_name, dataframe in (
    ("state trajectory registry", STATE_AUGMENTED_TRAJECTORY_REGISTRY_DF),
    ("state reconstruction summary", STATE_RECONSTRUCTION_SUMMARY_DF),
):
    if TRAJECTORY_UID_COLUMN not in dataframe.columns:
        trajectory_column = resolve_column(
            dataframe.columns,
            ("trajectory_uid", "unique_trajectory_id", "trajectory_id"),
            semantic_name=f"{dataframe_name} trajectory UID",
        )
        dataframe.rename(
            columns={trajectory_column: TRAJECTORY_UID_COLUMN},
            inplace=True,
        )

    dataframe[TRAJECTORY_UID_COLUMN] = canonical_string_series(
        dataframe[TRAJECTORY_UID_COLUMN]
    )

if "passed" in STATE_RECONSTRUCTION_VALIDATION_DF.columns:
    failed_state_checks = STATE_RECONSTRUCTION_VALIDATION_DF.loc[
        ~STATE_RECONSTRUCTION_VALIDATION_DF["passed"]
        .fillna(False)
        .astype(bool)
    ]

    if not failed_state_checks.empty:
        display(failed_state_checks)
        raise RuntimeError(
            "The Notebook 11 state-reconstruction validation contains failures."
        )

expected_sample_rows = int(
    STATE_AUGMENTED_TRAJECTORY_REGISTRY_DF[
        resolve_column(
            STATE_AUGMENTED_TRAJECTORY_REGISTRY_DF.columns,
            ("sample_count", "state_sample_count", "number_of_samples"),
            semantic_name="trajectory sample count",
        )
    ]
    .fillna(0)
    .sum()
)

observed_sample_rows = int(
    STATE_SAMPLE_PARQUET.metadata.num_rows
)

if expected_sample_rows != observed_sample_rows:
    raise RuntimeError(
        "The state-augmented sample count differs from the trajectory registry.\n"
        f"Expected: {expected_sample_rows:,}\n"
        f"Observed: {observed_sample_rows:,}"
    )

NOTEBOOK_12_COLUMN_BINDING_PATH = (
    NOTEBOOK_12_CONTRACT_DIR
    / "state_sample_column_bindings.json"
)

write_json_atomic(
    NOTEBOOK_12_COLUMN_BINDING_PATH,
    {
        "created_utc": utc_now_iso(),
        "state_sample_path": str(STATE_AUGMENTED_SAMPLES_PATH),
        "row_count": observed_sample_rows,
        "row_group_count": int(STATE_SAMPLE_PARQUET.metadata.num_row_groups),
        "bindings": STATE_SAMPLE_COLUMN_BINDINGS,
    },
)

display(
    pd.DataFrame(
        [
            {
                "semantic_role": semantic_role,
                "resolved_column": resolved_column,
                "arrow_type": (
                    str(
                        STATE_SAMPLE_SCHEMA.field(resolved_column).type
                    )
                    if resolved_column is not None
                    else None
                ),
            }
            for semantic_role, resolved_column in STATE_SAMPLE_COLUMN_BINDINGS.items()
        ]
    )
)


,semantic_role,resolved_column,arrow_type
0,trajectory_uid,trajectory_uid,string
1,chemistry,chemistry,string
2,cell_id,cell_id,string
3,file_id,file_id,string
4,segment_index,segment_index,int64
5,sample_index,sample_index,int64
6,elapsed_time_s,elapsed_time_s,double
7,delta_t_to_next_s,delta_t_to_next_s,double
8,voltage_v,voltage_v,double
9,current_discharge_a,current_discharge_a,double


In [6]:
# ======================================================================================
# CELL 10 — VERIFY TRAJECTORY IDENTITY, SPLIT EXCLUSIVITY, AND CHRONOLOGY
# ======================================================================================

def reconstruct_trajectory_uid(
    chemistry: Any,
    cell_id: Any,
    file_id: Any,
    segment_index: Any,
) -> str:
    chemistry_text = str(chemistry).strip().upper()
    cell_text = str(cell_id).strip()
    file_text = str(file_id).strip()

    numeric_segment = pd.to_numeric(
        pd.Series([segment_index]),
        errors="coerce",
    ).iloc[0]

    if pd.isna(numeric_segment):
        segment_text = str(segment_index).strip()
    else:
        segment_text = str(int(round(float(numeric_segment))))

    return TRAJECTORY_ID_SEPARATOR.join(
        (
            chemistry_text,
            cell_text,
            file_text,
            segment_text,
        )
    )


registry_df = STATE_AUGMENTED_TRAJECTORY_REGISTRY_DF.copy()

registry_bindings = {
    "chemistry": resolve_column(
        registry_df.columns,
        ("chemistry", "chemistry_id"),
        semantic_name="registry chemistry",
    ),
    "cell_id": resolve_column(
        registry_df.columns,
        ("cell_id", "cell"),
        semantic_name="registry cell",
    ),
    "file_id": resolve_column(
        registry_df.columns,
        ("file_id", "source_file_id", "source_file"),
        semantic_name="registry file",
    ),
    "segment_index": resolve_column(
        registry_df.columns,
        ("segment_index", "trajectory_index", "segment_id"),
        semantic_name="registry segment",
    ),
    "primary_split": resolve_column(
        registry_df.columns,
        ("primary_split", "split", "split_label"),
        semantic_name="registry split",
    ),
    "is_strict_test": resolve_column(
        registry_df.columns,
        ("is_strict_test", "strict_test", "strict_test_overlay"),
        semantic_name="registry strict-test overlay",
        required=False,
    ),
    "sample_count": resolve_column(
        registry_df.columns,
        ("sample_count", "state_sample_count", "number_of_samples"),
        semantic_name="registry sample count",
    ),
}

registry_df["chemistry"] = canonical_string_series(
    registry_df[registry_bindings["chemistry"]],
    uppercase=True,
)

registry_df["cell_id"] = canonical_string_series(
    registry_df[registry_bindings["cell_id"]]
)

registry_df["file_id"] = canonical_string_series(
    registry_df[registry_bindings["file_id"]]
)

registry_df["segment_index"] = canonical_nullable_integer_series(
    registry_df[registry_bindings["segment_index"]],
    column_name="registry.segment_index",
    allow_missing=False,
)

registry_df["primary_split"] = (
    canonical_string_series(
        registry_df[registry_bindings["primary_split"]]
    )
    .str.lower()
)

if registry_bindings["is_strict_test"] is None:
    registry_df["is_strict_test"] = False
else:
    registry_df["is_strict_test"] = (
        registry_df[registry_bindings["is_strict_test"]]
        .fillna(False)
        .astype(bool)
    )

registry_df["sample_count"] = canonical_nullable_integer_series(
    registry_df[registry_bindings["sample_count"]],
    column_name="registry.sample_count",
    allow_missing=False,
)

registry_df["reconstructed_trajectory_uid"] = [
    reconstruct_trajectory_uid(
        chemistry,
        cell_id,
        file_id,
        segment_index,
    )
    for chemistry, cell_id, file_id, segment_index in zip(
        registry_df["chemistry"],
        registry_df["cell_id"],
        registry_df["file_id"],
        registry_df["segment_index"],
        strict=True,
    )
]

registry_df[TRAJECTORY_UID_COLUMN] = canonical_string_series(
    registry_df[TRAJECTORY_UID_COLUMN]
)

uid_mismatch_df = registry_df.loc[
    registry_df[TRAJECTORY_UID_COLUMN]
    !=
    registry_df["reconstructed_trajectory_uid"]
].copy()

if not uid_mismatch_df.empty:
    display(uid_mismatch_df.head(20))
    raise RuntimeError(
        "The authoritative trajectory UID differs from the reconstructed UID."
    )

if registry_df[TRAJECTORY_UID_COLUMN].duplicated().any():
    duplicate_uid_df = registry_df.loc[
        registry_df[TRAJECTORY_UID_COLUMN].duplicated(keep=False)
    ]
    display(duplicate_uid_df)
    raise RuntimeError(
        "The state-augmented trajectory registry contains duplicate trajectory UIDs."
    )

if not registry_df["primary_split"].isin(
    ("train", "calibration", "test", "embargo")
).all():
    invalid_split_df = registry_df.loc[
        ~registry_df["primary_split"].isin(
            ("train", "calibration", "test", "embargo")
        )
    ]
    display(invalid_split_df)
    raise RuntimeError(
        "The registry contains an invalid primary split."
    )

if not registry_df.loc[
    registry_df["is_strict_test"],
    "primary_split",
].eq("test").all():
    raise RuntimeError(
        "Strict-test trajectories must be an overlay inside the primary test split."
    )

split_uid_sets = {
    split_name: set(
        registry_df.loc[
            registry_df["primary_split"].eq(split_name),
            TRAJECTORY_UID_COLUMN,
        ].astype(str)
    )
    for split_name in ("train", "calibration", "test", "embargo")
}

split_overlap_records: list[dict[str, Any]] = []

for first_index, first_split in enumerate(split_uid_sets):
    for second_split in tuple(split_uid_sets)[first_index + 1:]:
        overlap = split_uid_sets[first_split] & split_uid_sets[second_split]
        split_overlap_records.append(
            {
                "first_split": first_split,
                "second_split": second_split,
                "overlap_count": len(overlap),
                "passed": len(overlap) == 0,
            }
        )

NOTEBOOK_12_SPLIT_OVERLAP_DF = pd.DataFrame(split_overlap_records)

if not NOTEBOOK_12_SPLIT_OVERLAP_DF["passed"].all():
    display(NOTEBOOK_12_SPLIT_OVERLAP_DF)
    raise RuntimeError(
        "One or more primary split trajectory sets overlap."
    )

chronology_rank = {
    "train": 0,
    "calibration": 1,
    "test": 2,
}

chronology_issue_records: list[dict[str, Any]] = []

for group_key, group_df in registry_df.loc[
    registry_df["primary_split"].isin(chronology_rank)
].groupby(
    ["chemistry", "cell_id", "file_id"],
    sort=False,
    dropna=False,
):
    ordered = group_df.sort_values(
        by="segment_index",
        kind="mergesort",
    )

    ranks = ordered["primary_split"].map(chronology_rank).to_numpy(
        dtype=np.int64
    )

    if (np.diff(ranks) < 0).any():
        chronology_issue_records.append(
            {
                "chemistry": group_key[0],
                "cell_id": group_key[1],
                "file_id": group_key[2],
                "issue": "SPLIT_ORDER_REVERSAL",
                "split_sequence": json.dumps(
                    ordered["primary_split"].astype(str).tolist()
                ),
            }
        )

NOTEBOOK_12_CHRONOLOGY_ISSUES_DF = pd.DataFrame(
    chronology_issue_records,
    columns=(
        "chemistry",
        "cell_id",
        "file_id",
        "issue",
        "split_sequence",
    ),
)

if not NOTEBOOK_12_CHRONOLOGY_ISSUES_DF.empty:
    display(NOTEBOOK_12_CHRONOLOGY_ISSUES_DF)
    raise RuntimeError(
        "The published chronological split order is violated."
    )

split_summary_df = (
    registry_df
    .groupby("primary_split", as_index=False)
    .agg(
        trajectory_count=(TRAJECTORY_UID_COLUMN, "nunique"),
        sample_count=("sample_count", "sum"),
    )
)

strict_summary_row = pd.DataFrame(
    [
        {
            "primary_split": "strict_test_overlay",
            "trajectory_count": int(registry_df["is_strict_test"].sum()),
            "sample_count": int(
                registry_df.loc[
                    registry_df["is_strict_test"],
                    "sample_count",
                ].sum()
            ),
        }
    ]
)

NOTEBOOK_12_SPLIT_SUMMARY_DF = pd.concat(
    [split_summary_df, strict_summary_row],
    ignore_index=True,
)

NOTEBOOK_12_TRAJECTORY_REGISTRY_DF = registry_df[
    [
        TRAJECTORY_UID_COLUMN,
        "chemistry",
        "cell_id",
        "file_id",
        "segment_index",
        "primary_split",
        "is_strict_test",
        "sample_count",
    ]
].copy()

NOTEBOOK_12_TRAJECTORY_REGISTRY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "validated_trajectory_registry.parquet"
)

NOTEBOOK_12_SPLIT_SUMMARY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "split_summary.parquet"
)

NOTEBOOK_12_SPLIT_OVERLAP_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "split_overlap_audit.parquet"
)

NOTEBOOK_12_CHRONOLOGY_ISSUES_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "chronology_issues.parquet"
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_TRAJECTORY_REGISTRY_DF,
    NOTEBOOK_12_TRAJECTORY_REGISTRY_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_SPLIT_SUMMARY_DF,
    NOTEBOOK_12_SPLIT_SUMMARY_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_SPLIT_OVERLAP_DF,
    NOTEBOOK_12_SPLIT_OVERLAP_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_CHRONOLOGY_ISSUES_DF,
    NOTEBOOK_12_CHRONOLOGY_ISSUES_PATH,
)

print(
    "Chronological split integrity and whole-trajectory exclusivity "
    "were validated successfully."
)

display(NOTEBOOK_12_SPLIT_SUMMARY_DF)


Chronological split integrity and whole-trajectory exclusivity were validated successfully.


,primary_split,trajectory_count,sample_count
0,calibration,1017,4392705
1,test,1089,4953624
2,train,2832,8972544
3,strict_test_overlay,9,35330


In [7]:
# ======================================================================================
# CELL 11 — BUILD THE LEAKAGE-SAFE ACCESS PLAN AND MATERIALIZE DEVELOPMENT DATA
# ======================================================================================

NOTEBOOK_12_TRAIN_SAMPLE_PATH = (
    NOTEBOOK_12_DATA_DIR
    / "training_state_samples.parquet"
)

NOTEBOOK_12_CALIBRATION_SAMPLE_PATH = (
    NOTEBOOK_12_DATA_DIR
    / "calibration_state_samples.parquet"
)

NOTEBOOK_12_SAMPLE_ACCESS_PLAN_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "sample_access_plan.parquet"
)

identity_read_columns = [
    STATE_SAMPLE_COLUMN_BINDINGS["trajectory_uid"],
    STATE_SAMPLE_COLUMN_BINDINGS["primary_split"],
]

if STATE_SAMPLE_COLUMN_BINDINGS["is_strict_test"] is not None:
    identity_read_columns.append(
        STATE_SAMPLE_COLUMN_BINDINGS["is_strict_test"]
    )

identity_read_columns = list(dict.fromkeys(identity_read_columns))

row_group_identity_records: list[dict[str, Any]] = []

for row_group_index in range(
    int(STATE_SAMPLE_PARQUET.metadata.num_row_groups)
):
    identity_table = STATE_SAMPLE_PARQUET.read_row_group(
        row_group_index,
        columns=identity_read_columns,
        use_threads=True,
    )

    uid_values = [
        value
        for value in pc.unique(
            identity_table[
                STATE_SAMPLE_COLUMN_BINDINGS["trajectory_uid"]
            ]
        ).to_pylist()
        if value is not None
    ]

    split_values = [
        str(value).strip().lower()
        for value in pc.unique(
            identity_table[
                STATE_SAMPLE_COLUMN_BINDINGS["primary_split"]
            ]
        ).to_pylist()
        if value is not None
    ]

    if len(uid_values) != 1 or len(split_values) != 1:
        raise RuntimeError(
            "Every state-augmented row group must contain exactly one "
            "trajectory UID and one primary split."
        )

    if STATE_SAMPLE_COLUMN_BINDINGS["is_strict_test"] is None:
        strict_test = False
    else:
        strict_values = pc.fill_null(
            pc.cast(
                identity_table[
                    STATE_SAMPLE_COLUMN_BINDINGS["is_strict_test"]
                ],
                pa.bool_(),
                safe=False,
            ),
            False,
        )
        strict_test = bool(pc.any(strict_values).as_py())

    row_group_identity_records.append(
        {
            "source_row_group_index": int(row_group_index),
            TRAJECTORY_UID_COLUMN: str(uid_values[0]).strip(),
            "primary_split": split_values[0],
            "is_strict_test": strict_test,
            "sample_count": int(identity_table.num_rows),
        }
    )

ROW_GROUP_IDENTITY_DF = pd.DataFrame(
    row_group_identity_records
)

if ROW_GROUP_IDENTITY_DF[TRAJECTORY_UID_COLUMN].duplicated().any():
    raise RuntimeError(
        "The state-augmented file contains duplicate trajectory row groups."
    )

registry_identity_df = NOTEBOOK_12_TRAJECTORY_REGISTRY_DF[
    [
        TRAJECTORY_UID_COLUMN,
        "chemistry",
        "cell_id",
        "file_id",
        "segment_index",
        "primary_split",
        "is_strict_test",
        "sample_count",
    ]
].copy()

NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF = (
    registry_identity_df
    .merge(
        ROW_GROUP_IDENTITY_DF,
        on=TRAJECTORY_UID_COLUMN,
        how="outer",
        suffixes=("_registry", "_file"),
        validate="one_to_one",
        indicator=True,
    )
)

if not NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["_merge"].eq("both").all():
    display(
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF.loc[
            ~NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["_merge"].eq("both")
        ]
    )
    raise RuntimeError(
        "The trajectory registry and state-augmented row groups do not match."
    )

for column_name in ("primary_split", "is_strict_test", "sample_count"):
    registry_column = f"{column_name}_registry"
    file_column = f"{column_name}_file"

    mismatch = (
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF[registry_column]
        !=
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF[file_column]
    )

    if mismatch.any():
        display(
            NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF.loc[
                mismatch,
                [
                    TRAJECTORY_UID_COLUMN,
                    registry_column,
                    file_column,
                ],
            ]
        )
        raise RuntimeError(
            f"The registry and sample file disagree for {column_name!r}."
        )

NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["primary_split"] = (
    NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF[
        "primary_split_registry"
    ]
)

NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["is_strict_test"] = (
    NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF[
        "is_strict_test_registry"
    ]
    .fillna(False)
    .astype(bool)
)

NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["sample_count"] = (
    NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF[
        "sample_count_registry"
    ]
    .astype(np.int64)
)

NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["access_role"] = np.select(
    [
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["primary_split"].eq("train"),
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["primary_split"].eq("calibration"),
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["is_strict_test"],
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["primary_split"].eq("test"),
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["primary_split"].eq("embargo"),
    ],
    [
        "TRAIN_MODEL_FIT",
        "CALIBRATION_MODEL_SELECTION",
        "STRICT_TEST_QUARANTINE",
        "ORDINARY_TEST_HOLDOUT",
        "EMBARGO_EXCLUDED",
    ],
    default="UNRESOLVED",
)

NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["sample_arrays_opened_in_cell_11"] = False
NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["_merge"] = None

if NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["access_role"].eq("UNRESOLVED").any():
    raise RuntimeError(
        "At least one trajectory has an unresolved access role."
    )


def materialize_development_partition(
    *,
    access_role: str,
    output_path: Path,
) -> dict[str, Any]:
    selected = (
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF.loc[
            NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["access_role"].eq(access_role)
        ]
        .sort_values(
            by="source_row_group_index",
            kind="mergesort",
        )
    )

    if selected.empty:
        raise RuntimeError(
            f"No trajectories were assigned to access role {access_role!r}."
        )

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".tmp"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    writer: pq.ParquetWriter | None = None
    expected_schema: pa.Schema | None = None
    output_rows = 0
    output_row_groups = 0

    try:
        for source_row_group_index in selected[
            "source_row_group_index"
        ].astype(int):
            table = STATE_SAMPLE_PARQUET.read_row_group(
                int(source_row_group_index),
                use_threads=True,
            )

            if writer is None:
                expected_schema = table.schema
                writer = pq.ParquetWriter(
                    temporary_path,
                    expected_schema,
                    compression="zstd",
                    use_dictionary=True,
                    write_statistics=True,
                )
            elif not table.schema.equals(
                expected_schema,
                check_metadata=False,
            ):
                raise RuntimeError(
                    "The state sample schema changed between row groups."
                )

            writer.write_table(
                table,
                row_group_size=table.num_rows,
            )
            output_rows += int(table.num_rows)
            output_row_groups += 1

    except Exception:
        if writer is not None:
            writer.close()
        if temporary_path.exists():
            temporary_path.unlink()
        raise

    if writer is None:
        raise RuntimeError(
            f"No data were written for access role {access_role!r}."
        )

    writer.close()
    temporary_path.replace(output_path)

    saved = pq.ParquetFile(output_path)

    if int(saved.metadata.num_rows) != output_rows:
        raise RuntimeError(
            "The materialized development row count is incorrect."
        )

    if int(saved.metadata.num_row_groups) != output_row_groups:
        raise RuntimeError(
            "The materialized development row-group count is incorrect."
        )

    NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF.loc[
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["access_role"].eq(access_role),
        "sample_arrays_opened_in_cell_11",
    ] = True

    return {
        "access_role": access_role,
        "output_path": str(output_path),
        "sample_count": output_rows,
        "trajectory_count": output_row_groups,
        "row_group_count": output_row_groups,
        "sha256": calculate_streaming_sha256(
            output_path,
            chunk_bytes=NOTEBOOK_12_HASH_CHUNK_BYTES,
        ),
    }


TRAIN_MATERIALIZATION_RESULT = materialize_development_partition(
    access_role="TRAIN_MODEL_FIT",
    output_path=NOTEBOOK_12_TRAIN_SAMPLE_PATH,
)

CALIBRATION_MATERIALIZATION_RESULT = materialize_development_partition(
    access_role="CALIBRATION_MODEL_SELECTION",
    output_path=NOTEBOOK_12_CALIBRATION_SAMPLE_PATH,
)

protected_roles = (
    "ORDINARY_TEST_HOLDOUT",
    "STRICT_TEST_QUARANTINE",
    "EMBARGO_EXCLUDED",
)

if NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF.loc[
    NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF["access_role"].isin(protected_roles),
    "sample_arrays_opened_in_cell_11",
].fillna(False).astype(bool).any():
    raise RuntimeError(
        "A protected partition was opened during development materialization."
    )

write_dataframe_parquet_atomic(
    NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF,
    NOTEBOOK_12_SAMPLE_ACCESS_PLAN_PATH,
)

print("=" * 110)
print("LEAKAGE-SAFE DEVELOPMENT MATERIALIZATION")
print("=" * 110)
print(
    f"Training trajectories       : "
    f"{TRAIN_MATERIALIZATION_RESULT['trajectory_count']:,}"
)
print(
    f"Calibration trajectories    : "
    f"{CALIBRATION_MATERIALIZATION_RESULT['trajectory_count']:,}"
)
print("Ordinary-test arrays opened : False")
print("Strict-test arrays opened   : False")
print("Embargo arrays opened       : False")
print("=" * 110)


LEAKAGE-SAFE DEVELOPMENT MATERIALIZATION
Training trajectories       : 2,832
Calibration trajectories    : 1,017
Ordinary-test arrays opened : False
Strict-test arrays opened   : False
Embargo arrays opened       : False


In [8]:
# ======================================================================================
# CELL 12 — FREEZE TRAINING-ONLY SCHEDULER SUPPORT AND NORMALIZATION
# ======================================================================================

NOTEBOOK_12_ACTIVE_SCHEDULER_COLUMNS: dict[str, str] = {
    "soc": str(
        STATE_SAMPLE_COLUMN_BINDINGS["soc_for_ocv"]
    ),
    "temperature": str(
        STATE_SAMPLE_COLUMN_BINDINGS["temperature_c"]
    ),
}

TRAIN_DEVELOPMENT_PARQUET = pq.ParquetFile(
    NOTEBOOK_12_TRAIN_SAMPLE_PATH
)

scheduler_accumulators: dict[str, dict[str, float | int]] = {
    scheduler_name: {
        "minimum": np.inf,
        "maximum": -np.inf,
        "sum": 0.0,
        "sum_squares": 0.0,
        "count": 0,
    }
    for scheduler_name in NOTEBOOK_12_SCHEDULING_VARIABLES
}

for row_group_index in range(
    int(TRAIN_DEVELOPMENT_PARQUET.metadata.num_row_groups)
):
    table = TRAIN_DEVELOPMENT_PARQUET.read_row_group(
        row_group_index,
        columns=list(
            NOTEBOOK_12_ACTIVE_SCHEDULER_COLUMNS.values()
        ),
        use_threads=True,
    )

    for scheduler_name, source_column in (
        NOTEBOOK_12_ACTIVE_SCHEDULER_COLUMNS.items()
    ):
        values = np.asarray(
            pc.cast(
                table[source_column],
                pa.float64(),
                safe=False,
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.float64,
        )

        finite_values = values[
            np.isfinite(values)
        ]

        if finite_values.size == 0:
            raise RuntimeError(
                f"Training scheduler {scheduler_name!r} has no finite values."
            )

        accumulator = scheduler_accumulators[
            scheduler_name
        ]

        accumulator["minimum"] = min(
            float(accumulator["minimum"]),
            float(np.min(finite_values)),
        )

        accumulator["maximum"] = max(
            float(accumulator["maximum"]),
            float(np.max(finite_values)),
        )

        accumulator["sum"] = (
            float(accumulator["sum"])
            +
            float(np.sum(finite_values))
        )

        accumulator["sum_squares"] = (
            float(accumulator["sum_squares"])
            +
            float(np.sum(np.square(finite_values)))
        )

        accumulator["count"] = (
            int(accumulator["count"])
            +
            int(finite_values.size)
        )

NOTEBOOK_12_SCHEDULER_SUPPORT: dict[
    str,
    dict[str, float | int | str],
] = {}

scheduler_support_records: list[dict[str, Any]] = []

for scheduler_name in NOTEBOOK_12_SCHEDULING_VARIABLES:
    accumulator = scheduler_accumulators[
        scheduler_name
    ]

    minimum = float(
        accumulator["minimum"]
    )

    maximum = float(
        accumulator["maximum"]
    )

    count = int(
        accumulator["count"]
    )

    mean = float(
        accumulator["sum"]
    ) / count

    variance = max(
        float(
            accumulator["sum_squares"]
        ) / count
        -
        mean * mean,
        0.0,
    )

    standard_deviation = math.sqrt(
        variance
    )

    center = 0.5 * (
        minimum
        +
        maximum
    )

    scale = 0.5 * (
        maximum
        -
        minimum
    )

    if not np.isfinite(
        scale
    ) or scale <= 0.0:
        raise RuntimeError(
            f"Training scheduler {scheduler_name!r} has zero support width."
        )

    NOTEBOOK_12_SCHEDULER_SUPPORT[
        scheduler_name
    ] = {
        "source_column": (
            NOTEBOOK_12_ACTIVE_SCHEDULER_COLUMNS[
                scheduler_name
            ]
        ),
        "minimum": minimum,
        "maximum": maximum,
        "center": center,
        "scale": scale,
        "mean": mean,
        "standard_deviation": standard_deviation,
        "count": count,
        "fit_partition": "train",
    }

    scheduler_support_records.append(
        {
            "scheduler": scheduler_name,
            **NOTEBOOK_12_SCHEDULER_SUPPORT[
                scheduler_name
            ],
        }
    )


def normalize_scheduler_array(
    values: np.ndarray | Sequence[float],
    *,
    scheduler_name: str,
    clip_to_training_support: bool = False,
) -> np.ndarray:
    if scheduler_name not in (
        NOTEBOOK_12_SCHEDULER_SUPPORT
    ):
        raise KeyError(
            f"Unknown scheduler: {scheduler_name!r}"
        )

    numeric_values = np.asarray(
        values,
        dtype=np.float64,
    )

    support = NOTEBOOK_12_SCHEDULER_SUPPORT[
        scheduler_name
    ]

    normalized = (
        numeric_values
        -
        float(
            support["center"]
        )
    ) / float(
        support["scale"]
    )

    if clip_to_training_support:
        normalized = np.clip(
            normalized,
            -1.0,
            1.0,
        )

    return normalized


def denormalize_scheduler_array(
    normalized_values: np.ndarray | Sequence[float],
    *,
    scheduler_name: str,
) -> np.ndarray:
    support = NOTEBOOK_12_SCHEDULER_SUPPORT[
        scheduler_name
    ]

    normalized_array = np.asarray(
        normalized_values,
        dtype=np.float64,
    )

    return (
        float(
            support["center"]
        )
        +
        float(
            support["scale"]
        )
        *
        normalized_array
    )


NOTEBOOK_12_SCHEDULER_SUPPORT_DF = pd.DataFrame(
    scheduler_support_records
)

NOTEBOOK_12_SCHEDULER_SUPPORT_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "training_scheduler_support.parquet"
)

NOTEBOOK_12_SCHEDULER_CONTRACT_PATH = (
    NOTEBOOK_12_CONTRACT_DIR
    / "notebook12_scheduler_contract.json"
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_SCHEDULER_SUPPORT_DF,
    NOTEBOOK_12_SCHEDULER_SUPPORT_PATH,
)

write_json_atomic(
    NOTEBOOK_12_SCHEDULER_CONTRACT_PATH,
    {
        "created_utc": utc_now_iso(),
        "status": "FROZEN",
        "fit_partition": "train",
        "active_scheduler_sequence": list(
            NOTEBOOK_12_SCHEDULING_VARIABLES
        ),
        "scheduler_dimension": (
            NOTEBOOK_12_SCHEDULER_DIMENSION
        ),
        "active_scheduler_columns": (
            NOTEBOOK_12_ACTIVE_SCHEDULER_COLUMNS
        ),
        "support": (
            NOTEBOOK_12_SCHEDULER_SUPPORT
        ),
        "normalization": (
            "xi = (value - center) / scale"
        ),
        "clipping_during_training": False,
        "clipping_during_calibration": False,
        "ordinary_test_arrays_used": False,
        "strict_test_arrays_used": False,
    },
)

print(
    "Training-only scheduler normalization was frozen successfully."
)

display(
    NOTEBOOK_12_SCHEDULER_SUPPORT_DF
)


Training-only scheduler normalization was frozen successfully.


,scheduler,source_column,minimum,maximum,center,scale,mean,standard_deviation,count,fit_partition
0,soc,soc_for_ocv,0.0000,1.0000,0.50000,0.50000,0.695010,0.328604,8972544,train
1,temperature,cell_temperature_c,23.5929,57.9206,40.75675,17.16385,32.473613,4.627993,8972544,train


In [9]:
# ======================================================================================
# CELL 13 — FREEZE THE LPV BASIS, PHYSICAL PARAMETER MAPS, AND HIERARCHY
# ======================================================================================

NOTEBOOK_12_PARAMETER_TRANSFORM_DF = pd.DataFrame(
    [
        {
            "parameter_position": 0,
            "transformed_parameter": "eta_R0",
            "physical_parameter": "R0_ohm",
            "forward_transform": "R0_ohm = exp(eta_R0)",
            "physical_unit": "ohm",
        },
        {
            "parameter_position": 1,
            "transformed_parameter": "eta_R1",
            "physical_parameter": "R1_ohm",
            "forward_transform": "R1_ohm = exp(eta_R1)",
            "physical_unit": "ohm",
        },
        {
            "parameter_position": 2,
            "transformed_parameter": "eta_R2",
            "physical_parameter": "R2_ohm",
            "forward_transform": "R2_ohm = exp(eta_R2)",
            "physical_unit": "ohm",
        },
        {
            "parameter_position": 3,
            "transformed_parameter": "eta_tau1",
            "physical_parameter": "tau1_s",
            "forward_transform": "tau1_s = exp(eta_tau1)",
            "physical_unit": "second",
        },
        {
            "parameter_position": 4,
            "transformed_parameter": "eta_tau_gap",
            "physical_parameter": "tau2_s",
            "forward_transform": "tau2_s = tau1_s + exp(eta_tau_gap)",
            "physical_unit": "second",
        },
    ]
)

# The frozen two-dimensional second-order basis has six terms.
basis_records = [
    {
        "basis_position": 0,
        "basis_id": "basis__intercept",
        "basis_expression": "1",
        "term_type": "constant",
        "total_degree": 0,
        "is_intercept": True,
        "parent_basis_ids": json.dumps([]),
        "exponent_vector": json.dumps([0, 0]),
    },
    {
        "basis_position": 1,
        "basis_id": "basis__soc",
        "basis_expression": "xi_soc",
        "term_type": "linear",
        "total_degree": 1,
        "is_intercept": False,
        "parent_basis_ids": json.dumps([]),
        "exponent_vector": json.dumps([1, 0]),
    },
    {
        "basis_position": 2,
        "basis_id": "basis__temperature",
        "basis_expression": "xi_temperature",
        "term_type": "linear",
        "total_degree": 1,
        "is_intercept": False,
        "parent_basis_ids": json.dumps([]),
        "exponent_vector": json.dumps([0, 1]),
    },
    {
        "basis_position": 3,
        "basis_id": "basis__soc_squared",
        "basis_expression": "xi_soc^2",
        "term_type": "quadratic",
        "total_degree": 2,
        "is_intercept": False,
        "parent_basis_ids": json.dumps(["basis__soc"]),
        "exponent_vector": json.dumps([2, 0]),
    },
    {
        "basis_position": 4,
        "basis_id": "basis__temperature_squared",
        "basis_expression": "xi_temperature^2",
        "term_type": "quadratic",
        "total_degree": 2,
        "is_intercept": False,
        "parent_basis_ids": json.dumps(["basis__temperature"]),
        "exponent_vector": json.dumps([0, 2]),
    },
    {
        "basis_position": 5,
        "basis_id": "basis__soc_temperature",
        "basis_expression": "xi_soc * xi_temperature",
        "term_type": "interaction",
        "total_degree": 2,
        "is_intercept": False,
        "parent_basis_ids": json.dumps(
            ["basis__soc", "basis__temperature"]
        ),
        "exponent_vector": json.dumps([1, 1]),
    },
]

NOTEBOOK_12_BASIS_TERM_DF = pd.DataFrame(
    basis_records
)

NOTEBOOK_12_BASIS_TERM_COUNT: int = int(
    len(
        NOTEBOOK_12_BASIS_TERM_DF
    )
)

NOTEBOOK_12_BASIS_EXPONENT_MATRIX = np.asarray(
    [
        json.loads(
            exponent_text
        )
        for exponent_text in (
            NOTEBOOK_12_BASIS_TERM_DF[
                "exponent_vector"
            ].astype(str)
        )
    ],
    dtype=np.int64,
)

NOTEBOOK_12_BASIS_DESIGN_COLUMNS: tuple[str, ...] = tuple(
    f"phi_{basis_position:03d}"
    for basis_position in range(
        NOTEBOOK_12_BASIS_TERM_COUNT
    )
)

NORMALIZED_SCHEDULER_DESIGN_COLUMNS: tuple[str, ...] = (
    "xi_soc",
    "xi_temperature",
)


def evaluate_lpv_basis_numpy(
    normalized_scheduler_matrix: np.ndarray,
) -> np.ndarray:
    scheduler_matrix = np.asarray(
        normalized_scheduler_matrix,
        dtype=np.float64,
    )

    if scheduler_matrix.ndim != 2:
        raise ValueError(
            "The normalized scheduler matrix must be two-dimensional."
        )

    if scheduler_matrix.shape[1] != NOTEBOOK_12_SCHEDULER_DIMENSION:
        raise ValueError(
            "The normalized scheduler matrix has an incorrect column count."
        )

    if not np.isfinite(
        scheduler_matrix
    ).all():
        raise ValueError(
            "The normalized scheduler matrix contains nonfinite values."
        )

    basis_matrix = np.ones(
        (
            scheduler_matrix.shape[0],
            NOTEBOOK_12_BASIS_TERM_COUNT,
        ),
        dtype=np.float64,
    )

    for basis_position, exponent_vector in enumerate(
        NOTEBOOK_12_BASIS_EXPONENT_MATRIX
    ):
        for scheduler_position, exponent in enumerate(
            exponent_vector
        ):
            if exponent > 0:
                basis_matrix[:, basis_position] *= (
                    scheduler_matrix[:, scheduler_position]
                    ** int(
                        exponent
                    )
                )

    return basis_matrix


training_hierarchy_df = (
    NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF.loc[
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF[
            "access_role"
        ].eq(
            "TRAIN_MODEL_FIT"
        )
    ]
    .copy()
)

training_hierarchy_df["chemistry"] = (
    canonical_string_series(
        training_hierarchy_df[
            "chemistry"
        ],
        uppercase=True,
    )
)

training_hierarchy_df["cell_id"] = (
    canonical_string_series(
        training_hierarchy_df[
            "cell_id"
        ]
    )
)

training_hierarchy_df["cell_entity_id"] = (
    training_hierarchy_df[
        "chemistry"
    ].astype(str)
    +
    TRAJECTORY_ID_SEPARATOR
    +
    training_hierarchy_df[
        "cell_id"
    ].astype(str)
)

NOTEBOOK_12_CHEMISTRIES: tuple[str, ...] = tuple(
    sorted(
        training_hierarchy_df[
            "chemistry"
        ].astype(str).unique()
    )
)

NOTEBOOK_12_CELL_ENTITIES: tuple[str, ...] = tuple(
    sorted(
        training_hierarchy_df[
            "cell_entity_id"
        ].astype(str).unique()
    )
)

chemistry_weight_df = (
    training_hierarchy_df
    .groupby(
        "chemistry",
        as_index=False,
    )
    .agg(
        training_sample_count=(
            "sample_count",
            "sum",
        ),
        trajectory_count=(
            TRAJECTORY_UID_COLUMN,
            "nunique",
        ),
    )
    .sort_values(
        "chemistry",
        kind="mergesort",
        ignore_index=True,
    )
)

chemistry_weight_df["weight"] = (
    chemistry_weight_df[
        "training_sample_count"
    ]
    /
    chemistry_weight_df[
        "training_sample_count"
    ].sum()
)

cell_weight_df = (
    training_hierarchy_df
    .groupby(
        [
            "chemistry",
            "cell_entity_id",
            "cell_id",
        ],
        as_index=False,
    )
    .agg(
        training_sample_count=(
            "sample_count",
            "sum",
        ),
        trajectory_count=(
            TRAJECTORY_UID_COLUMN,
            "nunique",
        ),
    )
    .sort_values(
        [
            "chemistry",
            "cell_entity_id",
        ],
        kind="mergesort",
        ignore_index=True,
    )
)

cell_weight_df[
    "chemistry_training_sample_count"
] = (
    cell_weight_df
    .groupby(
        "chemistry"
    )[
        "training_sample_count"
    ]
    .transform(
        "sum"
    )
)

cell_weight_df["weight"] = (
    cell_weight_df[
        "training_sample_count"
    ]
    /
    cell_weight_df[
        "chemistry_training_sample_count"
    ]
)

hierarchy_entity_records: list[dict[str, Any]] = [
    {
        "scope_level": 0,
        "scope": "GLOBAL",
        "entity_id": "GLOBAL",
        "parent_entity_id": None,
        "chemistry": None,
        "cell_id": None,
        "training_sample_count": int(
            training_hierarchy_df[
                "sample_count"
            ].sum()
        ),
        "weight_within_parent": 1.0,
    }
]

for row in chemistry_weight_df.itertuples(
    index=False
):
    hierarchy_entity_records.append(
        {
            "scope_level": 1,
            "scope": "CHEMISTRY",
            "entity_id": str(
                row.chemistry
            ),
            "parent_entity_id": "GLOBAL",
            "chemistry": str(
                row.chemistry
            ),
            "cell_id": None,
            "training_sample_count": int(
                row.training_sample_count
            ),
            "weight_within_parent": float(
                row.weight
            ),
        }
    )

for row in cell_weight_df.itertuples(
    index=False
):
    hierarchy_entity_records.append(
        {
            "scope_level": 2,
            "scope": "CELL",
            "entity_id": str(
                row.cell_entity_id
            ),
            "parent_entity_id": str(
                row.chemistry
            ),
            "chemistry": str(
                row.chemistry
            ),
            "cell_id": str(
                row.cell_id
            ),
            "training_sample_count": int(
                row.training_sample_count
            ),
            "weight_within_parent": float(
                row.weight
            ),
        }
    )

NOTEBOOK_12_HIERARCHY_ENTITY_DF = pd.DataFrame(
    hierarchy_entity_records
)

NOTEBOOK_12_HIERARCHY_WEIGHT_DF = pd.concat(
    [
        chemistry_weight_df.assign(
            constraint_level="CHEMISTRY_WITHIN_GLOBAL",
            parent_entity_id="GLOBAL",
            entity_id=chemistry_weight_df[
                "chemistry"
            ],
        )[
            [
                "constraint_level",
                "parent_entity_id",
                "entity_id",
                "chemistry",
                "training_sample_count",
                "weight",
            ]
        ],
        cell_weight_df.assign(
            constraint_level="CELL_WITHIN_CHEMISTRY",
            parent_entity_id=cell_weight_df[
                "chemistry"
            ],
            entity_id=cell_weight_df[
                "cell_entity_id"
            ],
        )[
            [
                "constraint_level",
                "parent_entity_id",
                "entity_id",
                "chemistry",
                "cell_id",
                "training_sample_count",
                "weight",
            ]
        ],
    ],
    ignore_index=True,
)

NOTEBOOK_12_BASIS_TERM_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "lpv_candidate_basis_terms.parquet"
)

NOTEBOOK_12_PARAMETER_TRANSFORM_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "lpv_parameter_transform_registry.parquet"
)

NOTEBOOK_12_HIERARCHY_ENTITY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "lpv_hierarchy_entities.parquet"
)

NOTEBOOK_12_HIERARCHY_WEIGHT_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "lpv_hierarchy_weights.parquet"
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_BASIS_TERM_DF,
    NOTEBOOK_12_BASIS_TERM_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_PARAMETER_TRANSFORM_DF,
    NOTEBOOK_12_PARAMETER_TRANSFORM_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_HIERARCHY_ENTITY_DF,
    NOTEBOOK_12_HIERARCHY_ENTITY_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_HIERARCHY_WEIGHT_DF,
    NOTEBOOK_12_HIERARCHY_WEIGHT_PATH,
)

print("=" * 110)
print("FROZEN LPV ARCHITECTURE")
print("=" * 110)
print(f"Basis terms          : {NOTEBOOK_12_BASIS_TERM_COUNT}")
print(f"Chemistries          : {len(NOTEBOOK_12_CHEMISTRIES)}")
print(f"Cell entities        : {len(NOTEBOOK_12_CELL_ENTITIES)}")
print(f"Hierarchy entities   : {len(NOTEBOOK_12_HIERARCHY_ENTITY_DF)}")
print("=" * 110)

display(
    NOTEBOOK_12_BASIS_TERM_DF
)


FROZEN LPV ARCHITECTURE
Basis terms          : 6
Chemistries          : 3
Cell entities        : 8
Hierarchy entities   : 12


,basis_position,basis_id,basis_expression,term_type,total_degree,is_intercept,parent_basis_ids,exponent_vector
0,0,basis__intercept,1,constant,0,True,[],"[0, 0]"
1,1,basis__soc,xi_soc,linear,1,False,[],"[1, 0]"
2,2,basis__temperature,xi_temperature,linear,1,False,[],"[0, 1]"
3,3,basis__soc_squared,xi_soc^2,quadratic,2,False,"[""basis__soc""]","[2, 0]"
4,4,basis__temperature_squared,xi_temperature^2,quadratic,2,False,"[""basis__temperature""]","[0, 2]"
5,5,basis__soc_temperature,xi_soc * xi_temperature,interaction,2,False,"[""basis__soc"", ""basis__temperature""]","[1, 1]"


In [10]:
# ======================================================================================
# CELL 14 — CONSTRUCT THE IDENTIFIABLE FULL AND REDUCED COEFFICIENT SPACES
# ======================================================================================

def deterministic_weighted_nullspace(
    weights: Sequence[float],
) -> np.ndarray:
    weight_array = np.asarray(
        weights,
        dtype=np.float64,
    ).reshape(
        -1
    )

    if weight_array.size == 0:
        raise ValueError(
            "The weight vector is empty."
        )

    if not np.isfinite(
        weight_array
    ).all():
        raise ValueError(
            "The weight vector contains nonfinite values."
        )

    if not (
        weight_array
        >
        0.0
    ).all():
        raise ValueError(
            "Every hierarchy weight must be positive."
        )

    if weight_array.size == 1:
        return np.zeros(
            (
                1,
                0,
            ),
            dtype=np.float64,
        )

    normalized_weight = (
        weight_array
        /
        np.linalg.norm(
            weight_array
        )
    )

    complete_q, _ = scipy_linalg.qr(
        normalized_weight.reshape(
            -1,
            1,
        ),
        mode="full",
        pivoting=False,
        check_finite=True,
    )

    nullspace = complete_q[
        :,
        1:
    ]

    for column_index in range(
        nullspace.shape[
            1
        ]
    ):
        column = nullspace[
            :,
            column_index
        ]

        pivot_index = int(
            np.argmax(
                np.abs(
                    column
                )
            )
        )

        if column[
            pivot_index
        ] < 0.0:
            nullspace[
                :,
                column_index
            ] *= -1.0

    if np.max(
        np.abs(
            weight_array
            @
            nullspace
        ),
        initial=0.0,
    ) > 1.0e-12:
        raise RuntimeError(
            "The weighted null-space basis violates the centering constraint."
        )

    if np.max(
        np.abs(
            nullspace.T
            @
            nullspace
            -
            np.eye(
                nullspace.shape[
                    1
                ]
            )
        ),
        initial=0.0,
    ) > 1.0e-12:
        raise RuntimeError(
            "The weighted null-space basis is not orthonormal."
        )

    return nullspace


def stable_identifier(
    prefix: str,
    *components: Any,
) -> str:
    raw_text = "||".join(
        str(component)
        for component in components
    )

    digest = hashlib.sha256(
        raw_text.encode(
            "utf-8"
        )
    ).hexdigest()[
        :16
    ]

    return f"{prefix}__{digest}"


full_coefficient_records: list[
    dict[str, Any]
] = []

for parameter_row in (
    NOTEBOOK_12_PARAMETER_TRANSFORM_DF
    .sort_values(
        "parameter_position",
        kind="mergesort",
    )
    .itertuples(
        index=False
    )
):
    for entity_row in (
        NOTEBOOK_12_HIERARCHY_ENTITY_DF
        .sort_values(
            [
                "scope_level",
                "entity_id",
            ],
            kind="mergesort",
        )
        .itertuples(
            index=False
        )
    ):
        for basis_row in (
            NOTEBOOK_12_BASIS_TERM_DF
            .sort_values(
                "basis_position",
                kind="mergesort",
            )
            .itertuples(
                index=False
            )
        ):
            full_coefficient_records.append(
                {
                    "coefficient_id": stable_identifier(
                        "coef",
                        parameter_row.transformed_parameter,
                        entity_row.scope,
                        entity_row.entity_id,
                        basis_row.basis_id,
                    ),
                    "parameter_position": int(
                        parameter_row.parameter_position
                    ),
                    "transformed_parameter": str(
                        parameter_row.transformed_parameter
                    ),
                    "physical_parameter": str(
                        parameter_row.physical_parameter
                    ),
                    "scope_level": int(
                        entity_row.scope_level
                    ),
                    "scope": str(
                        entity_row.scope
                    ),
                    "entity_id": str(
                        entity_row.entity_id
                    ),
                    "parent_entity_id": (
                        entity_row.parent_entity_id
                    ),
                    "chemistry": (
                        entity_row.chemistry
                    ),
                    "cell_id": (
                        entity_row.cell_id
                    ),
                    "basis_position": int(
                        basis_row.basis_position
                    ),
                    "basis_id": str(
                        basis_row.basis_id
                    ),
                    "basis_expression": str(
                        basis_row.basis_expression
                    ),
                    "term_type": str(
                        basis_row.term_type
                    ),
                    "is_intercept": bool(
                        basis_row.is_intercept
                    ),
                }
            )

NOTEBOOK_12_FULL_COEFFICIENT_INDEX_DF = (
    pd.DataFrame(
        full_coefficient_records
    )
    .sort_values(
        [
            "parameter_position",
            "scope_level",
            "entity_id",
            "basis_position",
        ],
        kind="mergesort",
        ignore_index=True,
    )
)

NOTEBOOK_12_FULL_COEFFICIENT_INDEX_DF[
    "full_coefficient_index"
] = np.arange(
    len(
        NOTEBOOK_12_FULL_COEFFICIENT_INDEX_DF
    ),
    dtype=np.int64,
)

NOTEBOOK_12_FULL_COEFFICIENT_COUNT = int(
    len(
        NOTEBOOK_12_FULL_COEFFICIENT_INDEX_DF
    )
)

full_lookup: dict[
    tuple[str, str, str, str],
    tuple[int, str],
] = {
    (
        str(
            row.transformed_parameter
        ),
        str(
            row.scope
        ),
        str(
            row.entity_id
        ),
        str(
            row.basis_id
        ),
    ): (
        int(
            row.full_coefficient_index
        ),
        str(
            row.coefficient_id
        ),
    )
    for row in (
        NOTEBOOK_12_FULL_COEFFICIENT_INDEX_DF.itertuples(
            index=False
        )
    )
}

chemistry_weight_ordered_df = (
    chemistry_weight_df
    .sort_values(
        "chemistry",
        kind="mergesort",
        ignore_index=True,
    )
)

chemistry_entities = tuple(
    chemistry_weight_ordered_df[
        "chemistry"
    ].astype(str)
)

chemistry_weights = (
    chemistry_weight_ordered_df[
        "weight"
    ].to_numpy(
        dtype=np.float64
    )
)

CHEMISTRY_CONTRAST_BASIS = (
    deterministic_weighted_nullspace(
        chemistry_weights
    )
)

CELL_ENTITIES_BY_CHEMISTRY: dict[
    str,
    tuple[str, ...],
] = {}

CELL_WEIGHTS_BY_CHEMISTRY: dict[
    str,
    np.ndarray,
] = {}

CELL_CONTRAST_BASES: dict[
    str,
    np.ndarray,
] = {}

for chemistry in chemistry_entities:
    chemistry_cells_df = (
        cell_weight_df.loc[
            cell_weight_df[
                "chemistry"
            ].astype(str).eq(
                chemistry
            )
        ]
        .sort_values(
            "cell_entity_id",
            kind="mergesort",
            ignore_index=True,
        )
    )

    entities = tuple(
        chemistry_cells_df[
            "cell_entity_id"
        ].astype(str)
    )

    weights = chemistry_cells_df[
        "weight"
    ].to_numpy(
        dtype=np.float64
    )

    CELL_ENTITIES_BY_CHEMISTRY[
        chemistry
    ] = entities

    CELL_WEIGHTS_BY_CHEMISTRY[
        chemistry
    ] = weights

    CELL_CONTRAST_BASES[
        chemistry
    ] = deterministic_weighted_nullspace(
        weights
    )

reduced_records: list[
    dict[str, Any]
] = []

expansion_rows: list[int] = []
expansion_columns: list[int] = []
expansion_values: list[float] = []


def register_reduced_coordinate(
    *,
    transformed_parameter: str,
    parameter_position: int,
    basis_id: str,
    basis_position: int,
    basis_expression: str,
    term_type: str,
    is_intercept: bool,
    scope: str,
    scope_level: int,
    entity_block_id: str,
    chemistry: str | None,
    contrast_position: int,
    penalized: bool,
) -> tuple[int, str]:
    reduced_index = len(
        reduced_records
    )

    reduced_id = stable_identifier(
        "reduced",
        transformed_parameter,
        basis_id,
        scope,
        entity_block_id,
        contrast_position,
    )

    group_id = stable_identifier(
        "group",
        transformed_parameter,
        basis_id,
        scope,
        entity_block_id,
    )

    reduced_records.append(
        {
            "reduced_coefficient_index": int(
                reduced_index
            ),
            "reduced_coefficient_id": reduced_id,
            "parameter_position": int(
                parameter_position
            ),
            "transformed_parameter": (
                transformed_parameter
            ),
            "basis_id": basis_id,
            "basis_position": int(
                basis_position
            ),
            "basis_expression": (
                basis_expression
            ),
            "term_type": term_type,
            "is_intercept": bool(
                is_intercept
            ),
            "scope": scope,
            "scope_level": int(
                scope_level
            ),
            "entity_block_id": (
                entity_block_id
            ),
            "chemistry": chemistry,
            "contrast_position": int(
                contrast_position
            ),
            "penalized": bool(
                penalized
            ),
            "penalty_group_id": group_id,
        }
    )

    return reduced_index, reduced_id


def add_expansion_entry(
    full_index: int,
    reduced_index: int,
    value: float,
) -> None:
    numeric_value = float(
        value
    )

    if np.isclose(
        numeric_value,
        0.0,
        atol=1.0e-15,
        rtol=0.0,
    ):
        return

    expansion_rows.append(
        int(
            full_index
        )
    )

    expansion_columns.append(
        int(
            reduced_index
        )
    )

    expansion_values.append(
        numeric_value
    )


for parameter_row in (
    NOTEBOOK_12_PARAMETER_TRANSFORM_DF
    .sort_values(
        "parameter_position",
        kind="mergesort",
    )
    .itertuples(
        index=False
    )
):
    transformed_parameter = str(
        parameter_row.transformed_parameter
    )

    parameter_position = int(
        parameter_row.parameter_position
    )

    for basis_row in (
        NOTEBOOK_12_BASIS_TERM_DF
        .sort_values(
            "basis_position",
            kind="mergesort",
        )
        .itertuples(
            index=False
        )
    ):
        basis_id = str(
            basis_row.basis_id
        )

        global_reduced_index, _ = (
            register_reduced_coordinate(
                transformed_parameter=(
                    transformed_parameter
                ),
                parameter_position=(
                    parameter_position
                ),
                basis_id=basis_id,
                basis_position=int(
                    basis_row.basis_position
                ),
                basis_expression=str(
                    basis_row.basis_expression
                ),
                term_type=str(
                    basis_row.term_type
                ),
                is_intercept=bool(
                    basis_row.is_intercept
                ),
                scope="GLOBAL",
                scope_level=0,
                entity_block_id="GLOBAL",
                chemistry=None,
                contrast_position=0,
                penalized=not bool(
                    basis_row.is_intercept
                ),
            )
        )

        global_full_index, _ = full_lookup[
            (
                transformed_parameter,
                "GLOBAL",
                "GLOBAL",
                basis_id,
            )
        ]

        add_expansion_entry(
            global_full_index,
            global_reduced_index,
            1.0,
        )

        chemistry_reduced_indices: list[
            int
        ] = []

        for contrast_position in range(
            CHEMISTRY_CONTRAST_BASIS.shape[
                1
            ]
        ):
            reduced_index, _ = (
                register_reduced_coordinate(
                    transformed_parameter=(
                        transformed_parameter
                    ),
                    parameter_position=(
                        parameter_position
                    ),
                    basis_id=basis_id,
                    basis_position=int(
                        basis_row.basis_position
                    ),
                    basis_expression=str(
                        basis_row.basis_expression
                    ),
                    term_type=str(
                        basis_row.term_type
                    ),
                    is_intercept=bool(
                        basis_row.is_intercept
                    ),
                    scope="CHEMISTRY",
                    scope_level=1,
                    entity_block_id="GLOBAL",
                    chemistry=None,
                    contrast_position=(
                        contrast_position
                    ),
                    penalized=True,
                )
            )

            chemistry_reduced_indices.append(
                reduced_index
            )

        for chemistry_position, chemistry in enumerate(
            chemistry_entities
        ):
            chemistry_full_index, _ = full_lookup[
                (
                    transformed_parameter,
                    "CHEMISTRY",
                    chemistry,
                    basis_id,
                )
            ]

            for contrast_position, reduced_index in enumerate(
                chemistry_reduced_indices
            ):
                add_expansion_entry(
                    chemistry_full_index,
                    reduced_index,
                    CHEMISTRY_CONTRAST_BASIS[
                        chemistry_position,
                        contrast_position,
                    ],
                )

        for chemistry in chemistry_entities:
            cell_basis = CELL_CONTRAST_BASES[
                chemistry
            ]

            cell_reduced_indices: list[
                int
            ] = []

            for contrast_position in range(
                cell_basis.shape[
                    1
                ]
            ):
                reduced_index, _ = (
                    register_reduced_coordinate(
                        transformed_parameter=(
                            transformed_parameter
                        ),
                        parameter_position=(
                            parameter_position
                        ),
                        basis_id=basis_id,
                        basis_position=int(
                            basis_row.basis_position
                        ),
                        basis_expression=str(
                            basis_row.basis_expression
                        ),
                        term_type=str(
                            basis_row.term_type
                        ),
                        is_intercept=bool(
                            basis_row.is_intercept
                        ),
                        scope="CELL",
                        scope_level=2,
                        entity_block_id=(
                            chemistry
                        ),
                        chemistry=chemistry,
                        contrast_position=(
                            contrast_position
                        ),
                        penalized=True,
                    )
                )

                cell_reduced_indices.append(
                    reduced_index
                )

            for cell_position, cell_entity_id in enumerate(
                CELL_ENTITIES_BY_CHEMISTRY[
                    chemistry
                ]
            ):
                cell_full_index, _ = full_lookup[
                    (
                        transformed_parameter,
                        "CELL",
                        cell_entity_id,
                        basis_id,
                    )
                ]

                for contrast_position, reduced_index in enumerate(
                    cell_reduced_indices
                ):
                    add_expansion_entry(
                        cell_full_index,
                        reduced_index,
                        cell_basis[
                            cell_position,
                            contrast_position,
                        ],
                    )

NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF = (
    pd.DataFrame(
        reduced_records
    )
    .sort_values(
        "reduced_coefficient_index",
        kind="mergesort",
        ignore_index=True,
    )
)

NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT = int(
    len(
        NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF
    )
)

NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX = (
    scipy_sparse.coo_matrix(
        (
            np.asarray(
                expansion_values,
                dtype=np.float64,
            ),
            (
                np.asarray(
                    expansion_rows,
                    dtype=np.int64,
                ),
                np.asarray(
                    expansion_columns,
                    dtype=np.int64,
                ),
            ),
        ),
        shape=(
            NOTEBOOK_12_FULL_COEFFICIENT_COUNT,
            NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
        ),
        dtype=np.float64,
    )
    .tocsr()
)

NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX.eliminate_zeros()
NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX.sort_indices()

gram_residual = (
    NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX.T
    @
    NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX
    -
    scipy_sparse.eye(
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
        format="csr",
        dtype=np.float64,
    )
)

gram_residual.eliminate_zeros()

maximum_gram_residual = (
    0.0
    if gram_residual.nnz == 0
    else float(
        np.max(
            np.abs(
                gram_residual.data
            )
        )
    )
)

if maximum_gram_residual > 1.0e-12:
    raise RuntimeError(
        "The identifiable expansion matrix columns are not orthonormal."
    )

group_records: list[
    dict[str, Any]
] = []

for group_id, group_df in (
    NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF
    .groupby(
        "penalty_group_id",
        sort=False,
    )
):
    ordered_group = group_df.sort_values(
        "reduced_coefficient_index",
        kind="mergesort",
    )

    member_indices = ordered_group[
        "reduced_coefficient_index"
    ].astype(int).tolist()

    first = ordered_group.iloc[
        0
    ]

    parent_basis_ids = json.loads(
        NOTEBOOK_12_BASIS_TERM_DF.loc[
            NOTEBOOK_12_BASIS_TERM_DF[
                "basis_id"
            ].eq(
                first[
                    "basis_id"
                ]
            ),
            "parent_basis_ids",
        ].iloc[
            0
        ]
    )

    group_records.append(
        {
            "penalty_group_id": str(
                group_id
            ),
            "transformed_parameter": str(
                first[
                    "transformed_parameter"
                ]
            ),
            "basis_id": str(
                first[
                    "basis_id"
                ]
            ),
            "basis_position": int(
                first[
                    "basis_position"
                ]
            ),
            "scope": str(
                first[
                    "scope"
                ]
            ),
            "scope_level": int(
                first[
                    "scope_level"
                ]
            ),
            "entity_block_id": str(
                first[
                    "entity_block_id"
                ]
            ),
            "member_count": int(
                len(
                    member_indices
                )
            ),
            "member_reduced_indices": json.dumps(
                member_indices
            ),
            "penalized": bool(
                ordered_group[
                    "penalized"
                ].all()
            ),
            "group_lasso_weight": (
                math.sqrt(
                    len(
                        member_indices
                    )
                )
                if bool(
                    ordered_group[
                        "penalized"
                    ].all()
                )
                else 0.0
            ),
            "parent_basis_ids": json.dumps(
                parent_basis_ids
            ),
        }
    )

NOTEBOOK_12_PENALTY_GROUP_REGISTRY_DF = (
    pd.DataFrame(
        group_records
    )
)

NOTEBOOK_12_FULL_COEFFICIENT_INDEX_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "lpv_full_coefficient_index.parquet"
)

NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "lpv_reduced_coefficient_registry.parquet"
)

NOTEBOOK_12_PENALTY_GROUP_REGISTRY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "lpv_hierarchical_penalty_groups.parquet"
)

NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX_PATH = (
    NOTEBOOK_12_MODEL_DIR
    / "lpv_identifiable_expansion_matrix.npz"
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_FULL_COEFFICIENT_INDEX_DF,
    NOTEBOOK_12_FULL_COEFFICIENT_INDEX_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF,
    NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_PENALTY_GROUP_REGISTRY_DF,
    NOTEBOOK_12_PENALTY_GROUP_REGISTRY_PATH,
)

scipy_sparse.save_npz(
    NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX_PATH,
    NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX,
    compressed=True,
)

NOTEBOOK_12_REDUCED_PARAMETERIZATION_CONTRACT_PATH = (
    NOTEBOOK_12_CONTRACT_DIR
    / "notebook12_reduced_parameterization_contract.json"
)

write_json_atomic(
    NOTEBOOK_12_REDUCED_PARAMETERIZATION_CONTRACT_PATH,
    {
        "created_utc": utc_now_iso(),
        "status": "FROZEN",
        "full_coefficient_count": (
            NOTEBOOK_12_FULL_COEFFICIENT_COUNT
        ),
        "reduced_coefficient_count": (
            NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT
        ),
        "expansion_matrix_shape": list(
            NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX.shape
        ),
        "expansion_matrix_nonzero_count": int(
            NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX.nnz
        ),
        "maximum_orthogonality_residual": (
            maximum_gram_residual
        ),
        "weighted_sum_to_zero": True,
        "sample_arrays_read": False,
        "ordinary_test_arrays_read": False,
        "strict_test_arrays_read": False,
    },
)

print("=" * 110)
print("IDENTIFIABLE HIERARCHICAL PARAMETERIZATION")
print("=" * 110)
print(
    f"Full coefficient count    : "
    f"{NOTEBOOK_12_FULL_COEFFICIENT_COUNT:,}"
)
print(
    f"Reduced coefficient count : "
    f"{NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT:,}"
)
print(
    f"Expansion nonzeros        : "
    f"{NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX.nnz:,}"
)
print(
    f"Orthogonality residual    : "
    f"{maximum_gram_residual:.3e}"
)
print("=" * 110)


IDENTIFIABLE HIERARCHICAL PARAMETERIZATION
Full coefficient count    : 360
Reduced coefficient count : 240
Expansion nonzeros        : 690
Orthogonality residual    : 2.220e-16


In [11]:
# ======================================================================================
# CELL 15 — MATERIALIZE NORMALIZED SCHEDULERS AND FROZEN LPV BASIS COLUMNS
# ======================================================================================

CHEMISTRY_TO_INDEX: dict[
    str,
    int,
] = {
    chemistry: index
    for index, chemistry in enumerate(
        NOTEBOOK_12_CHEMISTRIES
    )
}

CELL_ENTITY_TO_INDEX: dict[
    str,
    int,
] = {
    cell_entity_id: index
    for index, cell_entity_id in enumerate(
        NOTEBOOK_12_CELL_ENTITIES
    )
}

STANDARDIZED_SOURCE_BINDINGS: tuple[
    tuple[str, str, pa.DataType],
    ...
] = (
    (
        "trajectory_uid",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "trajectory_uid"
            ]
        ),
        pa.string(),
    ),
    (
        "chemistry",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "chemistry"
            ]
        ),
        pa.string(),
    ),
    (
        "cell_id",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "cell_id"
            ]
        ),
        pa.string(),
    ),
    (
        "file_id",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "file_id"
            ]
        ),
        pa.string(),
    ),
    (
        "segment_index",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "segment_index"
            ]
        ),
        pa.int64(),
    ),
    (
        "sample_index",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "sample_index"
            ]
        ),
        pa.int64(),
    ),
    (
        "elapsed_time_s",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "elapsed_time_s"
            ]
        ),
        pa.float64(),
    ),
    (
        "delta_t_to_next_s",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "delta_t_to_next_s"
            ]
        ),
        pa.float64(),
    ),
    (
        "voltage_v",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "voltage_v"
            ]
        ),
        pa.float64(),
    ),
    (
        "current_discharge_a",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "current_discharge_a"
            ]
        ),
        pa.float64(),
    ),
    (
        "temperature_c",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "temperature_c"
            ]
        ),
        pa.float64(),
    ),
    (
        "capacity_for_soc_ah",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "capacity_for_soc_ah"
            ]
        ),
        pa.float64(),
    ),
    (
        "soc_for_ocv",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "soc_for_ocv"
            ]
        ),
        pa.float64(),
    ),
    (
        "ocv_v",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "ocv_v"
            ]
        ),
        pa.float64(),
    ),
    (
        "primary_split",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "primary_split"
            ]
        ),
        pa.string(),
    ),
    (
        "soc_fit_eligible",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "soc_fit_eligible"
            ]
        ),
        pa.bool_(),
    ),
    (
        "state_reconstruction_valid",
        str(
            STATE_SAMPLE_COLUMN_BINDINGS[
                "state_reconstruction_valid"
            ]
        ),
        pa.bool_(),
    ),
)

if STATE_SAMPLE_COLUMN_BINDINGS[
    "is_strict_test"
] is not None:
    STANDARDIZED_SOURCE_BINDINGS = (
        *STANDARDIZED_SOURCE_BINDINGS,
        (
            "is_strict_test",
            str(
                STATE_SAMPLE_COLUMN_BINDINGS[
                    "is_strict_test"
                ]
            ),
            pa.bool_(),
        ),
    )


def unique_nonmissing_arrow_values(
    array: pa.Array | pa.ChunkedArray,
) -> list[Any]:
    return [
        value
        for value in pc.unique(
            array
        ).to_pylist()
        if value is not None
    ]


def arrow_float64_numpy(
    array: pa.Array | pa.ChunkedArray,
    *,
    column_name: str,
) -> np.ndarray:
    values = np.asarray(
        pc.cast(
            array,
            pa.float64(),
            safe=False,
        ).to_numpy(
            zero_copy_only=False
        ),
        dtype=np.float64,
    )

    if not np.isfinite(
        values
    ).all():
        raise RuntimeError(
            f"Column {column_name!r} contains nonfinite values."
        )

    return values


def construct_design_table(
    source_table: pa.Table,
    *,
    partition: str,
    source_row_group_index: int,
) -> tuple[
    pa.Table,
    dict[str, Any],
]:
    row_count = int(
        source_table.num_rows
    )

    if row_count < 2:
        raise RuntimeError(
            "A design trajectory must contain at least two samples."
        )

    trajectory_uid_values = unique_nonmissing_arrow_values(
        source_table[
            str(
                STATE_SAMPLE_COLUMN_BINDINGS[
                    "trajectory_uid"
                ]
            )
        ]
    )

    chemistry_values = unique_nonmissing_arrow_values(
        source_table[
            str(
                STATE_SAMPLE_COLUMN_BINDINGS[
                    "chemistry"
                ]
            )
        ]
    )

    cell_values = unique_nonmissing_arrow_values(
        source_table[
            str(
                STATE_SAMPLE_COLUMN_BINDINGS[
                    "cell_id"
                ]
            )
        ]
    )

    if (
        len(
            trajectory_uid_values
        )
        !=
        1
        or
        len(
            chemistry_values
        )
        !=
        1
        or
        len(
            cell_values
        )
        !=
        1
    ):
        raise RuntimeError(
            "Each design row group must contain one trajectory, chemistry, and cell."
        )

    trajectory_uid = str(
        trajectory_uid_values[
            0
        ]
    ).strip()

    chemistry = str(
        chemistry_values[
            0
        ]
    ).strip().upper()

    cell_id = str(
        cell_values[
            0
        ]
    ).strip()

    cell_entity_id = (
        chemistry
        +
        TRAJECTORY_ID_SEPARATOR
        +
        cell_id
    )

    if chemistry not in CHEMISTRY_TO_INDEX:
        raise RuntimeError(
            f"Unknown chemistry {chemistry!r} in design materialization."
        )

    cell_seen_in_training = (
        cell_entity_id
        in
        CELL_ENTITY_TO_INDEX
    )

    if partition == "train" and not cell_seen_in_training:
        raise RuntimeError(
            "A training trajectory belongs to an unknown cell entity."
        )

    if STATE_SAMPLE_COLUMN_BINDINGS[
        "is_strict_test"
    ] is not None:
        strict_values = pc.fill_null(
            pc.cast(
                source_table[
                    str(
                        STATE_SAMPLE_COLUMN_BINDINGS[
                            "is_strict_test"
                        ]
                    )
                ],
                pa.bool_(),
                safe=False,
            ),
            False,
        )

        if bool(
            pc.any(
                strict_values
            ).as_py()
        ):
            raise RuntimeError(
                "A strict-test trajectory entered development design materialization."
            )

    soc_values = arrow_float64_numpy(
        source_table[
            NOTEBOOK_12_ACTIVE_SCHEDULER_COLUMNS[
                "soc"
            ]
        ],
        column_name=(
            NOTEBOOK_12_ACTIVE_SCHEDULER_COLUMNS[
                "soc"
            ]
        ),
    )

    temperature_values = arrow_float64_numpy(
        source_table[
            NOTEBOOK_12_ACTIVE_SCHEDULER_COLUMNS[
                "temperature"
            ]
        ],
        column_name=(
            NOTEBOOK_12_ACTIVE_SCHEDULER_COLUMNS[
                "temperature"
            ]
        ),
    )

    xi_soc = normalize_scheduler_array(
        soc_values,
        scheduler_name="soc",
        clip_to_training_support=False,
    )

    xi_temperature = normalize_scheduler_array(
        temperature_values,
        scheduler_name="temperature",
        clip_to_training_support=False,
    )

    normalized_scheduler_matrix = np.column_stack(
        [
            xi_soc,
            xi_temperature,
        ]
    )

    basis_matrix = evaluate_lpv_basis_numpy(
        normalized_scheduler_matrix
    )

    output_arrays: list[
        pa.Array | pa.ChunkedArray
    ] = []

    output_names: list[str] = []

    for target_name, source_name, target_type in (
        STANDARDIZED_SOURCE_BINDINGS
    ):
        output_arrays.append(
            pc.cast(
                source_table[
                    source_name
                ],
                target_type,
                safe=False,
            )
        )

        output_names.append(
            target_name
        )

    output_arrays.extend(
        [
            pa.array(
                [
                    cell_entity_id
                ]
                *
                row_count,
                type=pa.string(),
            ),
            pa.array(
                np.full(
                    row_count,
                    CHEMISTRY_TO_INDEX[
                        chemistry
                    ],
                    dtype=np.int32,
                ),
                type=pa.int32(),
            ),
            pa.array(
                np.full(
                    row_count,
                    CELL_ENTITY_TO_INDEX.get(
                        cell_entity_id,
                        -1,
                    ),
                    dtype=np.int32,
                ),
                type=pa.int32(),
            ),
            pa.array(
                np.full(
                    row_count,
                    cell_seen_in_training,
                    dtype=bool,
                ),
                type=pa.bool_(),
            ),
            pa.array(
                np.full(
                    row_count,
                    int(
                        source_row_group_index
                    ),
                    dtype=np.int32,
                ),
                type=pa.int32(),
            ),
            pa.array(
                xi_soc,
                type=pa.float64(),
            ),
            pa.array(
                xi_temperature,
                type=pa.float64(),
            ),
        ]
    )

    output_names.extend(
        [
            "cell_entity_id",
            "chemistry_index",
            "cell_index",
            "cell_seen_in_training",
            "source_row_group_index",
            "xi_soc",
            "xi_temperature",
        ]
    )

    for basis_position, design_column in enumerate(
        NOTEBOOK_12_BASIS_DESIGN_COLUMNS
    ):
        output_arrays.append(
            pa.array(
                basis_matrix[
                    :,
                    basis_position
                ],
                type=pa.float64(),
            )
        )

        output_names.append(
            design_column
        )

    output_table = pa.Table.from_arrays(
        output_arrays,
        names=output_names,
    )

    manifest_record = {
        "partition": partition,
        "trajectory_uid": trajectory_uid,
        "chemistry": chemistry,
        "cell_id": cell_id,
        "cell_entity_id": cell_entity_id,
        "cell_seen_in_training": cell_seen_in_training,
        "source_row_group_index": int(
            source_row_group_index
        ),
        "sample_count": row_count,
        "xi_soc_minimum": float(
            np.min(
                xi_soc
            )
        ),
        "xi_soc_maximum": float(
            np.max(
                xi_soc
            )
        ),
        "xi_temperature_minimum": float(
            np.min(
                xi_temperature
            )
        ),
        "xi_temperature_maximum": float(
            np.max(
                xi_temperature
            )
        ),
        "passed": True,
    }

    return output_table, manifest_record


def materialize_design_partition(
    source_path: Path,
    output_path: Path,
    *,
    partition: str,
) -> dict[str, Any]:
    source_parquet = pq.ParquetFile(
        source_path
    )

    temporary_path = output_path.with_suffix(
        output_path.suffix
        +
        ".tmp"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    writer: pq.ParquetWriter | None = None
    output_schema: pa.Schema | None = None
    manifest_records: list[
        dict[str, Any]
    ] = []

    try:
        for row_group_index in range(
            int(
                source_parquet.metadata.num_row_groups
            )
        ):
            source_table = source_parquet.read_row_group(
                row_group_index,
                use_threads=True,
            )

            output_table, manifest_record = (
                construct_design_table(
                    source_table,
                    partition=partition,
                    source_row_group_index=(
                        row_group_index
                    ),
                )
            )

            if writer is None:
                output_schema = output_table.schema
                writer = pq.ParquetWriter(
                    temporary_path,
                    output_schema,
                    compression="zstd",
                    use_dictionary=True,
                    write_statistics=True,
                )

            elif not output_table.schema.equals(
                output_schema,
                check_metadata=False,
            ):
                raise RuntimeError(
                    "The LPV design schema changed between trajectories."
                )

            writer.write_table(
                output_table,
                row_group_size=(
                    output_table.num_rows
                ),
            )

            manifest_records.append(
                manifest_record
            )

    except Exception:
        if writer is not None:
            writer.close()
        if temporary_path.exists():
            temporary_path.unlink()
        raise

    if writer is None:
        raise RuntimeError(
            f"No {partition} design trajectories were written."
        )

    writer.close()
    temporary_path.replace(
        output_path
    )

    saved = pq.ParquetFile(
        output_path
    )

    return {
        "partition": partition,
        "path": str(
            output_path
        ),
        "sample_count": int(
            saved.metadata.num_rows
        ),
        "trajectory_count": int(
            saved.metadata.num_row_groups
        ),
        "schema": (
            saved.schema_arrow
        ),
        "sha256": calculate_streaming_sha256(
            output_path,
            chunk_bytes=(
                NOTEBOOK_12_HASH_CHUNK_BYTES
            ),
        ),
        "manifest_records": (
            manifest_records
        ),
    }


NOTEBOOK_12_TRAIN_DESIGN_SAMPLE_PATH = (
    NOTEBOOK_12_DATA_DIR
    /
    "lpv_training_design_samples.parquet"
)

NOTEBOOK_12_CALIBRATION_DESIGN_SAMPLE_PATH = (
    NOTEBOOK_12_DATA_DIR
    /
    "lpv_calibration_design_samples.parquet"
)

TRAIN_DESIGN_RESULT = materialize_design_partition(
    NOTEBOOK_12_TRAIN_SAMPLE_PATH,
    NOTEBOOK_12_TRAIN_DESIGN_SAMPLE_PATH,
    partition="train",
)

CALIBRATION_DESIGN_RESULT = (
    materialize_design_partition(
        NOTEBOOK_12_CALIBRATION_SAMPLE_PATH,
        NOTEBOOK_12_CALIBRATION_DESIGN_SAMPLE_PATH,
        partition="calibration",
    )
)

if not TRAIN_DESIGN_RESULT[
    "schema"
].equals(
    CALIBRATION_DESIGN_RESULT[
        "schema"
    ],
    check_metadata=False,
):
    raise RuntimeError(
        "Training and calibration design schemas do not match."
    )

NOTEBOOK_12_LPV_DESIGN_SCHEMA = (
    TRAIN_DESIGN_RESULT[
        "schema"
    ]
)

NOTEBOOK_12_DESIGN_TRAJECTORY_MANIFEST_DF = pd.DataFrame(
    TRAIN_DESIGN_RESULT[
        "manifest_records"
    ]
    +
    CALIBRATION_DESIGN_RESULT[
        "manifest_records"
    ]
)

NOTEBOOK_12_DESIGN_TRAJECTORY_MANIFEST_DF[
    "design_row_group_index"
] = (
    NOTEBOOK_12_DESIGN_TRAJECTORY_MANIFEST_DF
    .groupby(
        "partition"
    )
    .cumcount()
    .astype(
        np.int64
    )
)

NOTEBOOK_12_DESIGN_TRAJECTORY_MANIFEST_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_design_trajectory_manifest.parquet"
)

NOTEBOOK_12_DESIGN_DATA_CONTRACT_PATH = (
    NOTEBOOK_12_CONTRACT_DIR
    /
    "notebook12_lpv_design_data_contract.json"
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_DESIGN_TRAJECTORY_MANIFEST_DF,
    NOTEBOOK_12_DESIGN_TRAJECTORY_MANIFEST_PATH,
)

write_json_atomic(
    NOTEBOOK_12_DESIGN_DATA_CONTRACT_PATH,
    {
        "created_utc": utc_now_iso(),
        "status": "FROZEN",
        "training_design_path": str(
            NOTEBOOK_12_TRAIN_DESIGN_SAMPLE_PATH
        ),
        "calibration_design_path": str(
            NOTEBOOK_12_CALIBRATION_DESIGN_SAMPLE_PATH
        ),
        "training_sample_count": (
            TRAIN_DESIGN_RESULT[
                "sample_count"
            ]
        ),
        "calibration_sample_count": (
            CALIBRATION_DESIGN_RESULT[
                "sample_count"
            ]
        ),
        "training_trajectory_count": (
            TRAIN_DESIGN_RESULT[
                "trajectory_count"
            ]
        ),
        "calibration_trajectory_count": (
            CALIBRATION_DESIGN_RESULT[
                "trajectory_count"
            ]
        ),
        "basis_columns": list(
            NOTEBOOK_12_BASIS_DESIGN_COLUMNS
        ),
        "normalized_scheduler_columns": list(
            NORMALIZED_SCHEDULER_DESIGN_COLUMNS
        ),
        "ordinary_test_arrays_read": False,
        "strict_test_arrays_read": False,
    },
)

print(
    "Normalized scheduling coordinates and the frozen LPV basis "
    "were materialized successfully."
)


Normalized scheduling coordinates and the frozen LPV basis were materialized successfully.


# Candidate models and leakage-safe optimization

Five nested candidates are evaluated:

| Candidate | Structure | Sparsity |
|---|---|---|
| M0 | fixed global two-RC baseline inside the LPV parameterization | intercepts only |
| M1 | global LPV | dense |
| M2 | global LPV | group sparse |
| M3 | global + chemistry + cell LPV | dense |
| M4 | global + chemistry + cell LPV | group sparse with heredity closure |

All coefficients are estimated using training trajectories only. The calibration partition selects the model family and regularization strength. Test and strict-test arrays remain inaccessible until the selected support and coefficient vector are frozen.


In [13]:
# ======================================================================================
# CELL 17 — FREEZE CANDIDATE FAMILIES, ACTIVE MASKS, AND PENALTY GROUPS
# ======================================================================================

reduced_registry_df = (
    NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF
    .sort_values(
        "reduced_coefficient_index",
        kind="mergesort",
        ignore_index=True,
    )
    .copy()
)

if reduced_registry_df[
    "reduced_coefficient_index"
].astype(int).tolist() != list(
    range(
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT
    )
):
    raise RuntimeError(
        "The reduced coefficient indices are not contiguous."
    )

MODEL_CANDIDATE_SPECIFICATIONS: dict[
    str,
    dict[str, Any],
] = {
    "M0_FIXED_2RC": {
        "description": (
            "Global intercepts only; reproduces a fixed two-RC structure "
            "inside the hierarchical parameterization."
        ),
        "scope_policy": "GLOBAL_INTERCEPT_ONLY",
        "sparse": False,
        "regularization_grid": (0.0,),
    },
    "M1_GLOBAL_DENSE_LPV": {
        "description": (
            "Dense global SOC-temperature LPV parameter maps."
        ),
        "scope_policy": "GLOBAL_ALL_BASIS",
        "sparse": False,
        "regularization_grid": (0.0,),
    },
    "M2_GLOBAL_SPARSE_LPV": {
        "description": (
            "Sparse global SOC-temperature LPV parameter maps."
        ),
        "scope_policy": "GLOBAL_ALL_BASIS",
        "sparse": True,
        "regularization_grid": SPARSE_LAMBDA_GRID,
    },
    "M3_HIERARCHICAL_DENSE_LPV": {
        "description": (
            "Dense global, chemistry, and cell LPV parameter maps."
        ),
        "scope_policy": "ALL_HIERARCHY",
        "sparse": False,
        "regularization_grid": (0.0,),
    },
    "M4_HIERARCHICAL_SPARSE_LPV": {
        "description": (
            "Sparse global, chemistry, and cell LPV parameter maps "
            "with basis and scope heredity closure."
        ),
        "scope_policy": "ALL_HIERARCHY",
        "sparse": True,
        "regularization_grid": SPARSE_LAMBDA_GRID,
    },
}


def candidate_active_mask(
    candidate_id: str,
) -> np.ndarray:
    if candidate_id not in MODEL_CANDIDATE_SPECIFICATIONS:
        raise KeyError(
            f"Unknown candidate ID: {candidate_id!r}"
        )

    policy = MODEL_CANDIDATE_SPECIFICATIONS[
        candidate_id
    ][
        "scope_policy"
    ]

    if policy == "GLOBAL_INTERCEPT_ONLY":
        mask = (
            reduced_registry_df[
                "scope"
            ].eq(
                "GLOBAL"
            )
            &
            reduced_registry_df[
                "is_intercept"
            ].fillna(False).astype(bool)
        )

    elif policy == "GLOBAL_ALL_BASIS":
        mask = reduced_registry_df[
            "scope"
        ].eq(
            "GLOBAL"
        )

    elif policy == "ALL_HIERARCHY":
        mask = pd.Series(
            True,
            index=reduced_registry_df.index,
        )

    else:
        raise RuntimeError(
            f"Unsupported candidate scope policy: {policy!r}"
        )

    return mask.to_numpy(
        dtype=bool
    )


CANDIDATE_ACTIVE_MASKS: dict[
    str,
    np.ndarray,
] = {
    candidate_id: candidate_active_mask(
        candidate_id
    )
    for candidate_id in (
        MODEL_CANDIDATE_SPECIFICATIONS
    )
}

for candidate_id, mask in (
    CANDIDATE_ACTIVE_MASKS.items()
):
    if mask.shape != (
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
    ):
        raise RuntimeError(
            f"Candidate {candidate_id} has an invalid active-mask shape."
        )

    if not mask.any():
        raise RuntimeError(
            f"Candidate {candidate_id} has no active coefficients."
        )


PENALTY_GROUP_MEMBERS: dict[
    str,
    np.ndarray,
] = {}

PENALTY_GROUP_WEIGHTS: dict[
    str,
    float,
] = {}

PENALTY_GROUP_METADATA: dict[
    str,
    dict[str, Any],
] = {}

for row in (
    NOTEBOOK_12_PENALTY_GROUP_REGISTRY_DF.itertuples(
        index=False
    )
):
    group_id = str(
        row.penalty_group_id
    )

    members = np.asarray(
        json.loads(
            str(
                row.member_reduced_indices
            )
        ),
        dtype=np.int32,
    )

    PENALTY_GROUP_MEMBERS[
        group_id
    ] = members

    PENALTY_GROUP_WEIGHTS[
        group_id
    ] = float(
        row.group_lasso_weight
    )

    PENALTY_GROUP_METADATA[
        group_id
    ] = {
        "transformed_parameter": str(
            row.transformed_parameter
        ),
        "basis_id": str(
            row.basis_id
        ),
        "basis_position": int(
            row.basis_position
        ),
        "scope": str(
            row.scope
        ),
        "scope_level": int(
            row.scope_level
        ),
        "entity_block_id": str(
            row.entity_block_id
        ),
        "penalized": bool(
            row.penalized
        ),
        "parent_basis_ids": tuple(
            json.loads(
                str(
                    row.parent_basis_ids
                )
            )
        ),
    }


def group_ids_for_candidate(
    candidate_id: str,
) -> tuple[str, ...]:
    active_mask = CANDIDATE_ACTIVE_MASKS[
        candidate_id
    ]

    selected_group_ids: list[
        str
    ] = []

    for group_id, members in (
        PENALTY_GROUP_MEMBERS.items()
    ):
        if not PENALTY_GROUP_METADATA[
            group_id
        ][
            "penalized"
        ]:
            continue

        if np.any(
            active_mask[
                members
            ]
        ):
            selected_group_ids.append(
                group_id
            )

    return tuple(
        selected_group_ids
    )


CANDIDATE_PENALTY_GROUP_IDS: dict[
    str,
    tuple[str, ...],
] = {
    candidate_id: (
        group_ids_for_candidate(
            candidate_id
        )
        if MODEL_CANDIDATE_SPECIFICATIONS[
            candidate_id
        ][
            "sparse"
        ]
        else tuple()
    )
    for candidate_id in (
        MODEL_CANDIDATE_SPECIFICATIONS
    )
}


def group_lasso_penalty_numpy(
    coefficients: np.ndarray,
    *,
    candidate_id: str,
) -> float:
    coefficient_array = np.asarray(
        coefficients,
        dtype=np.float64,
    )

    penalty = 0.0

    for group_id in (
        CANDIDATE_PENALTY_GROUP_IDS[
            candidate_id
        ]
    ):
        members = PENALTY_GROUP_MEMBERS[
            group_id
        ]

        penalty += (
            PENALTY_GROUP_WEIGHTS[
                group_id
            ]
            *
            float(
                np.linalg.norm(
                    coefficient_array[
                        members
                    ]
                )
            )
        )

    return float(
        penalty
    )


candidate_records: list[
    dict[str, Any]
] = []

for candidate_id, specification in (
    MODEL_CANDIDATE_SPECIFICATIONS.items()
):
    active_mask = CANDIDATE_ACTIVE_MASKS[
        candidate_id
    ]

    candidate_records.append(
        {
            "candidate_id": candidate_id,
            "description": (
                specification[
                    "description"
                ]
            ),
            "scope_policy": (
                specification[
                    "scope_policy"
                ]
            ),
            "sparse": bool(
                specification[
                    "sparse"
                ]
            ),
            "active_reduced_coefficient_count": int(
                active_mask.sum()
            ),
            "inactive_reduced_coefficient_count": int(
                (
                    ~active_mask
                ).sum()
            ),
            "penalty_group_count": int(
                len(
                    CANDIDATE_PENALTY_GROUP_IDS[
                        candidate_id
                    ]
                )
            ),
            "regularization_grid": json.dumps(
                list(
                    specification[
                        "regularization_grid"
                    ]
                )
            ),
        }
    )

NOTEBOOK_12_CANDIDATE_REGISTRY_DF = pd.DataFrame(
    candidate_records
)

NOTEBOOK_12_CANDIDATE_REGISTRY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_candidate_model_registry.parquet"
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_CANDIDATE_REGISTRY_DF,
    NOTEBOOK_12_CANDIDATE_REGISTRY_PATH,
)

display(
    NOTEBOOK_12_CANDIDATE_REGISTRY_DF
)


,candidate_id,description,scope_policy,sparse,active_reduced_coefficient_count,inactive_reduced_coefficient_count,penalty_group_count,regularization_grid
0,M0_FIXED_2RC,Global intercepts only; reproduces a fixed two...,GLOBAL_INTERCEPT_ONLY,False,5,235,0,[0.0]
1,M1_GLOBAL_DENSE_LPV,Dense global SOC-temperature LPV parameter maps.,GLOBAL_ALL_BASIS,False,30,210,0,[0.0]
2,M2_GLOBAL_SPARSE_LPV,Sparse global SOC-temperature LPV parameter maps.,GLOBAL_ALL_BASIS,True,30,210,25,"[1e-06, 3e-06, 1e-05, 3e-05, 0.0001]"
3,M3_HIERARCHICAL_DENSE_LPV,"Dense global, chemistry, and cell LPV paramete...",ALL_HIERARCHY,False,240,0,0,[0.0]
4,M4_HIERARCHICAL_SPARSE_LPV,"Sparse global, chemistry, and cell LPV paramet...",ALL_HIERARCHY,True,240,0,145,"[1e-06, 3e-06, 1e-05, 3e-05, 0.0001]"


In [16]:
# ======================================================================================
# CELL 18 — BUILD MEMORY-SAFE PADDED TRAJECTORY BATCHES
# ======================================================================================
#
# Purpose
# -------
# Build fixed-shape trajectory batches for JAX optimization while preserving
# complete trajectory boundaries.
#
# The stored source sample index is treated as trajectory metadata. It is not
# required to start at zero or to use consecutive integer values. The loader
# verifies that it is strictly increasing and then constructs a local,
# zero-based sample index:
#
#     local_sample_index = 0, 1, ..., number_of_samples - 1
#
# The dynamic recursion continues to use elapsed_time_s and delta_t_to_next_s.
#
# This cell:
#
# 1. verifies the development design files and trajectory manifest;
# 2. reconstructs the global, chemistry, and cell coefficient selectors;
# 3. defines a self-contained trajectory loader;
# 4. preserves the original source sample indices for auditing;
# 5. constructs local zero-based indices for numerical batching;
# 6. creates padded NumPy batches and valid/loss masks;
# 7. converts padded batches to JAX arrays;
# 8. runs a training-batch smoke test.
#
# Ordinary-test, strict-test, and embargo arrays remain unopened.
# ======================================================================================

from __future__ import annotations

import math

from pathlib import Path
from typing import Any, Mapping, Sequence

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from IPython.display import display


# ======================================================================================
# 1. ENABLE JAX FLOAT64
# ======================================================================================

jax.config.update(
    "jax_enable_x64",
    True,
)


# ======================================================================================
# 2. VERIFY REQUIRED UPSTREAM OBJECTS
# ======================================================================================

REQUIRED_CELL_18_OBJECTS: tuple[str, ...] = (
    "NOTEBOOK_12_TRAIN_DESIGN_SAMPLE_PATH",
    "NOTEBOOK_12_CALIBRATION_DESIGN_SAMPLE_PATH",
    "NOTEBOOK_12_DESIGN_TRAJECTORY_MANIFEST_DF",
    "NOTEBOOK_12_FULL_COEFFICIENT_INDEX_DF",
    "NOTEBOOK_12_BASIS_TERM_DF",
    "NOTEBOOK_12_BASIS_DESIGN_COLUMNS",
    "NOTEBOOK_12_PARAMETER_COUNT",
    "NOTEBOOK_12_BASIS_TERM_COUNT",
    "LPV_TRANSFORMED_PARAMETERS",
    "TRAINING_BATCH_TRAJECTORIES",
)


missing_cell_18_objects = [
    object_name
    for object_name in REQUIRED_CELL_18_OBJECTS
    if object_name not in globals()
]


if missing_cell_18_objects:
    raise RuntimeError(
        "Cell 18 requires the completed design-data, architecture, and "
        "coefficient-registry cells.\n\n"
        "Missing objects:\n  - "
        + "\n  - ".join(
            missing_cell_18_objects
        )
    )


# ======================================================================================
# 3. NORMALIZE CONFIGURATION
# ======================================================================================

NOTEBOOK_12_TRAIN_DESIGN_SAMPLE_PATH = Path(
    NOTEBOOK_12_TRAIN_DESIGN_SAMPLE_PATH
).expanduser().resolve()

NOTEBOOK_12_CALIBRATION_DESIGN_SAMPLE_PATH = Path(
    NOTEBOOK_12_CALIBRATION_DESIGN_SAMPLE_PATH
).expanduser().resolve()


for required_path in (
    NOTEBOOK_12_TRAIN_DESIGN_SAMPLE_PATH,
    NOTEBOOK_12_CALIBRATION_DESIGN_SAMPLE_PATH,
):
    if not required_path.is_file():
        raise FileNotFoundError(
            "A required design-data file was not found:\n"
            f"  {required_path}"
        )


NOTEBOOK_12_PARAMETER_COUNT = int(
    NOTEBOOK_12_PARAMETER_COUNT
)

NOTEBOOK_12_BASIS_TERM_COUNT = int(
    NOTEBOOK_12_BASIS_TERM_COUNT
)

TRAINING_BATCH_TRAJECTORIES = int(
    TRAINING_BATCH_TRAJECTORIES
)

PADDED_TRAJECTORY_MULTIPLE: int = int(
    globals().get(
        "PADDED_TRAJECTORY_MULTIPLE",
        32,
    )
)


if NOTEBOOK_12_PARAMETER_COUNT <= 0:
    raise ValueError(
        "NOTEBOOK_12_PARAMETER_COUNT must be positive."
    )


if NOTEBOOK_12_BASIS_TERM_COUNT <= 0:
    raise ValueError(
        "NOTEBOOK_12_BASIS_TERM_COUNT must be positive."
    )


if TRAINING_BATCH_TRAJECTORIES <= 0:
    raise ValueError(
        "TRAINING_BATCH_TRAJECTORIES must be positive."
    )


if PADDED_TRAJECTORY_MULTIPLE <= 0:
    raise ValueError(
        "PADDED_TRAJECTORY_MULTIPLE must be positive."
    )


if len(
    NOTEBOOK_12_BASIS_DESIGN_COLUMNS
) != NOTEBOOK_12_BASIS_TERM_COUNT:
    raise RuntimeError(
        "The basis design-column count differs from the frozen basis count."
    )


if len(
    LPV_TRANSFORMED_PARAMETERS
) != NOTEBOOK_12_PARAMETER_COUNT:
    raise RuntimeError(
        "The transformed-parameter sequence differs from the "
        "parameter count."
    )


# ======================================================================================
# 4. OPEN AND VERIFY THE DEVELOPMENT DESIGN FILES
# ======================================================================================

TRAIN_DESIGN_PARQUET = pq.ParquetFile(
    str(
        NOTEBOOK_12_TRAIN_DESIGN_SAMPLE_PATH
    )
)

CALIBRATION_DESIGN_PARQUET = pq.ParquetFile(
    str(
        NOTEBOOK_12_CALIBRATION_DESIGN_SAMPLE_PATH
    )
)


if not TRAIN_DESIGN_PARQUET.schema_arrow.equals(
    CALIBRATION_DESIGN_PARQUET.schema_arrow,
    check_metadata=False,
):
    raise RuntimeError(
        "The training and calibration design schemas do not match."
    )


TRAIN_ROW_GROUP_COUNT: int = int(
    TRAIN_DESIGN_PARQUET.metadata.num_row_groups
)

CALIBRATION_ROW_GROUP_COUNT: int = int(
    CALIBRATION_DESIGN_PARQUET.metadata.num_row_groups
)

TRAIN_DESIGN_SAMPLE_COUNT: int = int(
    TRAIN_DESIGN_PARQUET.metadata.num_rows
)

CALIBRATION_DESIGN_SAMPLE_COUNT: int = int(
    CALIBRATION_DESIGN_PARQUET.metadata.num_rows
)


if TRAIN_ROW_GROUP_COUNT <= 0:
    raise RuntimeError(
        "The training design file contains no row groups."
    )


if CALIBRATION_ROW_GROUP_COUNT <= 0:
    raise RuntimeError(
        "The calibration design file contains no row groups."
    )


BASIS_DESIGN_COLUMNS: tuple[str, ...] = tuple(
    str(
        column_name
    )
    for column_name in NOTEBOOK_12_BASIS_DESIGN_COLUMNS
)


REQUIRED_TRAJECTORY_COLUMNS: tuple[str, ...] = tuple(
    dict.fromkeys(
        [
            "trajectory_uid",
            "chemistry",
            "cell_entity_id",
            "sample_index",
            "elapsed_time_s",
            "delta_t_to_next_s",
            "voltage_v",
            "current_discharge_a",
            "ocv_v",
            *BASIS_DESIGN_COLUMNS,
        ]
    )
)


missing_design_columns = [
    column_name
    for column_name in REQUIRED_TRAJECTORY_COLUMNS
    if TRAIN_DESIGN_PARQUET.schema_arrow.get_field_index(
        column_name
    )
    <
    0
]


if missing_design_columns:
    raise RuntimeError(
        "The design schema is missing required batching columns:\n  - "
        + "\n  - ".join(
            missing_design_columns
        )
    )


# ======================================================================================
# 5. NORMALIZE AND VERIFY THE TRAJECTORY MANIFEST
# ======================================================================================

REQUIRED_MANIFEST_COLUMNS: tuple[str, ...] = (
    "partition",
    "design_row_group_index",
    "trajectory_uid",
    "chemistry",
    "cell_entity_id",
    "cell_seen_in_training",
    "sample_count",
)


missing_manifest_columns = [
    column_name
    for column_name in REQUIRED_MANIFEST_COLUMNS
    if column_name not in (
        NOTEBOOK_12_DESIGN_TRAJECTORY_MANIFEST_DF.columns
    )
]


if missing_manifest_columns:
    raise RuntimeError(
        "The design trajectory manifest is missing required columns:\n  - "
        + "\n  - ".join(
            missing_manifest_columns
        )
    )


DEVELOPMENT_TRAJECTORY_MANIFEST_DF = (
    NOTEBOOK_12_DESIGN_TRAJECTORY_MANIFEST_DF[
        list(
            REQUIRED_MANIFEST_COLUMNS
        )
    ]
    .copy()
)


DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
    "partition"
] = (
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
        "partition"
    ]
    .astype("string")
    .str.strip()
    .str.lower()
)


DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
    "trajectory_uid"
] = (
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
        "trajectory_uid"
    ]
    .astype("string")
    .str.strip()
)


DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
    "chemistry"
] = (
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
        "chemistry"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)


DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
    "cell_entity_id"
] = (
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
        "cell_entity_id"
    ]
    .astype("string")
    .str.strip()
)


DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
    "design_row_group_index"
] = pd.to_numeric(
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
        "design_row_group_index"
    ],
    errors="raise",
).astype(
    np.int64
)


DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
    "sample_count"
] = pd.to_numeric(
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
        "sample_count"
    ],
    errors="raise",
).astype(
    np.int64
)


DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
    "cell_seen_in_training"
] = (
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
        "cell_seen_in_training"
    ]
    .fillna(False)
    .astype(bool)
)


if not DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
    "partition"
].isin(
    (
        "train",
        "calibration",
    )
).all():
    unexpected_partitions = sorted(
        DEVELOPMENT_TRAJECTORY_MANIFEST_DF.loc[
            ~DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
                "partition"
            ].isin(
                (
                    "train",
                    "calibration",
                )
            ),
            "partition",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "The trajectory manifest contains unexpected partitions:\n  - "
        + "\n  - ".join(
            unexpected_partitions
        )
    )


if DEVELOPMENT_TRAJECTORY_MANIFEST_DF.duplicated(
    subset=[
        "partition",
        "design_row_group_index",
    ],
    keep=False,
).any():
    raise RuntimeError(
        "Duplicate partition–row-group assignments exist in the "
        "trajectory manifest."
    )


if DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
    "sample_count"
].lt(
    2
).any():
    raise RuntimeError(
        "At least one dynamic trajectory contains fewer than two samples."
    )


DEVELOPMENT_TRAJECTORY_MANIFEST_DF = (
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF
    .sort_values(
        by=[
            "partition",
            "design_row_group_index",
        ],
        kind="mergesort",
        ignore_index=True,
    )
)


TRAIN_MANIFEST_DF = (
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF.loc[
        DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
            "partition"
        ].eq(
            "train"
        )
    ]
    .sort_values(
        by="design_row_group_index",
        kind="mergesort",
        ignore_index=True,
    )
)


CALIBRATION_MANIFEST_DF = (
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF.loc[
        DEVELOPMENT_TRAJECTORY_MANIFEST_DF[
            "partition"
        ].eq(
            "calibration"
        )
    ]
    .sort_values(
        by="design_row_group_index",
        kind="mergesort",
        ignore_index=True,
    )
)


if len(
    TRAIN_MANIFEST_DF
) != TRAIN_ROW_GROUP_COUNT:
    raise RuntimeError(
        "Training manifest and Parquet row-group counts differ.\n"
        f"Manifest: {len(TRAIN_MANIFEST_DF):,}\n"
        f"Parquet: {TRAIN_ROW_GROUP_COUNT:,}"
    )


if len(
    CALIBRATION_MANIFEST_DF
) != CALIBRATION_ROW_GROUP_COUNT:
    raise RuntimeError(
        "Calibration manifest and Parquet row-group counts differ.\n"
        f"Manifest: {len(CALIBRATION_MANIFEST_DF):,}\n"
        f"Parquet: {CALIBRATION_ROW_GROUP_COUNT:,}"
    )


training_manifest_row_groups = (
    TRAIN_MANIFEST_DF[
        "design_row_group_index"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


calibration_manifest_row_groups = (
    CALIBRATION_MANIFEST_DF[
        "design_row_group_index"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


if not np.array_equal(
    training_manifest_row_groups,
    np.arange(
        TRAIN_ROW_GROUP_COUNT,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Training row-group indices are not contiguous from zero."
    )


if not np.array_equal(
    calibration_manifest_row_groups,
    np.arange(
        CALIBRATION_ROW_GROUP_COUNT,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Calibration row-group indices are not contiguous from zero."
    )


if int(
    TRAIN_MANIFEST_DF[
        "sample_count"
    ].sum()
) != TRAIN_DESIGN_SAMPLE_COUNT:
    raise RuntimeError(
        "Training manifest and Parquet sample counts differ."
    )


if int(
    CALIBRATION_MANIFEST_DF[
        "sample_count"
    ].sum()
) != CALIBRATION_DESIGN_SAMPLE_COUNT:
    raise RuntimeError(
        "Calibration manifest and Parquet sample counts differ."
    )


# ======================================================================================
# 6. REBUILD THE TRAJECTORY COEFFICIENT SELECTORS
# ======================================================================================

REQUIRED_FULL_INDEX_COLUMNS: tuple[str, ...] = (
    "full_coefficient_index",
    "transformed_parameter",
    "scope",
    "entity_id",
    "basis_id",
)


missing_full_index_columns = [
    column_name
    for column_name in REQUIRED_FULL_INDEX_COLUMNS
    if column_name not in (
        NOTEBOOK_12_FULL_COEFFICIENT_INDEX_DF.columns
    )
]


if missing_full_index_columns:
    raise RuntimeError(
        "The full coefficient index is missing selector columns:\n  - "
        + "\n  - ".join(
            missing_full_index_columns
        )
    )


full_coefficient_index_df = (
    NOTEBOOK_12_FULL_COEFFICIENT_INDEX_DF[
        list(
            REQUIRED_FULL_INDEX_COLUMNS
        )
    ]
    .copy()
)


full_coefficient_index_df[
    "full_coefficient_index"
] = pd.to_numeric(
    full_coefficient_index_df[
        "full_coefficient_index"
    ],
    errors="raise",
).astype(
    np.int64
)


FULL_COEFFICIENT_COUNT_CELL_18: int = int(
    len(
        full_coefficient_index_df
    )
)


if FULL_COEFFICIENT_COUNT_CELL_18 <= 0:
    raise RuntimeError(
        "The full coefficient index is empty."
    )


full_index_values = np.sort(
    full_coefficient_index_df[
        "full_coefficient_index"
    ].to_numpy(
        dtype=np.int64
    )
)


if not np.array_equal(
    full_index_values,
    np.arange(
        FULL_COEFFICIENT_COUNT_CELL_18,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "The full coefficient indices are not contiguous from zero."
    )


FULL_COEFFICIENT_INDEX_LOOKUP: dict[
    tuple[str, str, str, str],
    int,
] = {
    (
        str(
            row.transformed_parameter
        ),
        str(
            row.scope
        ),
        str(
            row.entity_id
        ),
        str(
            row.basis_id
        ),
    ): int(
        row.full_coefficient_index
    )
    for row in full_coefficient_index_df.itertuples(
        index=False
    )
}


if len(
    FULL_COEFFICIENT_INDEX_LOOKUP
) != len(
    full_coefficient_index_df
):
    raise RuntimeError(
        "The full coefficient selector lookup is not one-to-one."
    )


for required_column in (
    "basis_position",
    "basis_id",
):
    if required_column not in (
        NOTEBOOK_12_BASIS_TERM_DF.columns
    ):
        raise RuntimeError(
            f"The basis registry is missing {required_column!r}."
        )


BASIS_ID_BY_POSITION: tuple[str, ...] = tuple(
    NOTEBOOK_12_BASIS_TERM_DF
    .sort_values(
        by="basis_position",
        kind="mergesort",
    )[
        "basis_id"
    ]
    .astype(str)
)


if len(
    BASIS_ID_BY_POSITION
) != NOTEBOOK_12_BASIS_TERM_COUNT:
    raise RuntimeError(
        "The basis-ID sequence has an incorrect length."
    )


if len(
    set(
        BASIS_ID_BY_POSITION
    )
) != len(
    BASIS_ID_BY_POSITION
):
    raise RuntimeError(
        "The basis-ID sequence contains duplicates."
    )


trajectory_count: int = int(
    len(
        DEVELOPMENT_TRAJECTORY_MANIFEST_DF
    )
)


selector_shape = (
    trajectory_count,
    NOTEBOOK_12_PARAMETER_COUNT,
    NOTEBOOK_12_BASIS_TERM_COUNT,
)


TRAJECTORY_GLOBAL_FULL_INDICES = np.empty(
    selector_shape,
    dtype=np.int32,
)

TRAJECTORY_CHEMISTRY_FULL_INDICES = np.empty(
    selector_shape,
    dtype=np.int32,
)

TRAJECTORY_CELL_FULL_INDICES = np.full(
    selector_shape,
    fill_value=-1,
    dtype=np.int32,
)

TRAJECTORY_HAS_CELL_EFFECT = np.zeros(
    trajectory_count,
    dtype=bool,
)


TRAJECTORY_POSITION_LOOKUP: dict[
    tuple[str, int],
    int,
] = {}


trajectory_uid_by_position: list[str] = []
trajectory_partition_by_position: list[str] = []
trajectory_chemistry_by_position: list[str] = []
trajectory_cell_by_position: list[str] = []
trajectory_sample_count_by_position: list[int] = []


for trajectory_position, manifest_row in enumerate(
    DEVELOPMENT_TRAJECTORY_MANIFEST_DF.itertuples(
        index=False
    )
):
    partition = str(
        manifest_row.partition
    )

    row_group_index = int(
        manifest_row.design_row_group_index
    )

    chemistry = str(
        manifest_row.chemistry
    )

    cell_entity_id = str(
        manifest_row.cell_entity_id
    )

    has_cell_effect = bool(
        manifest_row.cell_seen_in_training
    )


    trajectory_lookup_key = (
        partition,
        row_group_index,
    )


    if trajectory_lookup_key in TRAJECTORY_POSITION_LOOKUP:
        raise RuntimeError(
            "A duplicate trajectory selector key was encountered:\n"
            f"  {trajectory_lookup_key}"
        )


    TRAJECTORY_POSITION_LOOKUP[
        trajectory_lookup_key
    ] = int(
        trajectory_position
    )


    TRAJECTORY_HAS_CELL_EFFECT[
        trajectory_position
    ] = has_cell_effect


    trajectory_uid_by_position.append(
        str(
            manifest_row.trajectory_uid
        )
    )

    trajectory_partition_by_position.append(
        partition
    )

    trajectory_chemistry_by_position.append(
        chemistry
    )

    trajectory_cell_by_position.append(
        cell_entity_id
    )

    trajectory_sample_count_by_position.append(
        int(
            manifest_row.sample_count
        )
    )


    for parameter_position, transformed_parameter in enumerate(
        LPV_TRANSFORMED_PARAMETERS
    ):
        transformed_parameter = str(
            transformed_parameter
        )


        for basis_position, basis_id in enumerate(
            BASIS_ID_BY_POSITION
        ):
            global_key = (
                transformed_parameter,
                "GLOBAL",
                "GLOBAL",
                basis_id,
            )

            chemistry_key = (
                transformed_parameter,
                "CHEMISTRY",
                chemistry,
                basis_id,
            )


            global_index = FULL_COEFFICIENT_INDEX_LOOKUP.get(
                global_key
            )

            chemistry_index = FULL_COEFFICIENT_INDEX_LOOKUP.get(
                chemistry_key
            )


            if global_index is None:
                raise RuntimeError(
                    "A global coefficient selector is missing:\n"
                    f"  {global_key}"
                )


            if chemistry_index is None:
                raise RuntimeError(
                    "A chemistry coefficient selector is missing:\n"
                    f"  {chemistry_key}"
                )


            if has_cell_effect:
                cell_key = (
                    transformed_parameter,
                    "CELL",
                    cell_entity_id,
                    basis_id,
                )

                cell_index = FULL_COEFFICIENT_INDEX_LOOKUP.get(
                    cell_key
                )

                if cell_index is None:
                    raise RuntimeError(
                        "A cell coefficient selector is missing:\n"
                        f"  {cell_key}"
                    )

            else:
                cell_index = -1


            TRAJECTORY_GLOBAL_FULL_INDICES[
                trajectory_position,
                parameter_position,
                basis_position,
            ] = int(
                global_index
            )

            TRAJECTORY_CHEMISTRY_FULL_INDICES[
                trajectory_position,
                parameter_position,
                basis_position,
            ] = int(
                chemistry_index
            )

            TRAJECTORY_CELL_FULL_INDICES[
                trajectory_position,
                parameter_position,
                basis_position,
            ] = int(
                cell_index
            )


TRAJECTORY_UID_BY_POSITION: tuple[str, ...] = tuple(
    trajectory_uid_by_position
)

TRAJECTORY_PARTITION_BY_POSITION: tuple[str, ...] = tuple(
    trajectory_partition_by_position
)

TRAJECTORY_CHEMISTRY_BY_POSITION: tuple[str, ...] = tuple(
    trajectory_chemistry_by_position
)

TRAJECTORY_CELL_ENTITY_BY_POSITION: tuple[str, ...] = tuple(
    trajectory_cell_by_position
)

TRAJECTORY_SAMPLE_COUNT_BY_POSITION: tuple[int, ...] = tuple(
    trajectory_sample_count_by_position
)


if len(
    TRAJECTORY_POSITION_LOOKUP
) != trajectory_count:
    raise RuntimeError(
        "The reconstructed trajectory lookup is incomplete."
    )


if (
    TRAJECTORY_GLOBAL_FULL_INDICES
    <
    0
).any():
    raise RuntimeError(
        "A global coefficient selector is negative."
    )


if (
    TRAJECTORY_CHEMISTRY_FULL_INDICES
    <
    0
).any():
    raise RuntimeError(
        "A chemistry coefficient selector is negative."
    )


if (
    TRAJECTORY_GLOBAL_FULL_INDICES
    >=
    FULL_COEFFICIENT_COUNT_CELL_18
).any():
    raise RuntimeError(
        "A global selector exceeds the full coefficient dimension."
    )


if (
    TRAJECTORY_CHEMISTRY_FULL_INDICES
    >=
    FULL_COEFFICIENT_COUNT_CELL_18
).any():
    raise RuntimeError(
        "A chemistry selector exceeds the full coefficient dimension."
    )


valid_cell_selector_mask = (
    TRAJECTORY_CELL_FULL_INDICES
    >=
    0
)


if (
    TRAJECTORY_CELL_FULL_INDICES[
        valid_cell_selector_mask
    ]
    >=
    FULL_COEFFICIENT_COUNT_CELL_18
).any():
    raise RuntimeError(
        "A cell selector exceeds the full coefficient dimension."
    )


for trajectory_position in range(
    trajectory_count
):
    if TRAJECTORY_HAS_CELL_EFFECT[
        trajectory_position
    ]:
        if (
            TRAJECTORY_CELL_FULL_INDICES[
                trajectory_position
            ]
            <
            0
        ).any():
            raise RuntimeError(
                "A known training cell has a missing cell selector."
            )

    else:
        if not (
            TRAJECTORY_CELL_FULL_INDICES[
                trajectory_position
            ]
            ==
            -1
        ).all():
            raise RuntimeError(
                "An unseen cell contains an active cell selector."
            )


# ======================================================================================
# 7. DEFINE ARROW CONVERSION HELPERS
# ======================================================================================

def unique_nonmissing_arrow_values(
    array: pa.Array | pa.ChunkedArray,
) -> list[Any]:
    """
    Return unique nonmissing values from one Arrow array.
    """

    return [
        value
        for value in pc.unique(
            array
        ).to_pylist()
        if value is not None
    ]


def integer_column_to_numpy(
    table: pa.Table,
    column_name: str,
) -> np.ndarray:
    """
    Convert an Arrow column to a validated one-dimensional int64 array.
    """

    column = table[
        column_name
    ]


    if int(
        column.null_count
    ) > 0:
        raise RuntimeError(
            f"Column {column_name!r} contains missing values."
        )


    try:
        cast_column = pc.cast(
            column,
            pa.int64(),
            safe=False,
        )

    except Exception as exc:
        raise RuntimeError(
            f"Column {column_name!r} could not be converted to int64."
        ) from exc


    values = np.asarray(
        cast_column.to_numpy(
            zero_copy_only=False
        ),
        dtype=np.int64,
    )


    if values.ndim != 1:
        raise RuntimeError(
            f"Column {column_name!r} is not one-dimensional."
        )


    return values


def float_column_to_numpy(
    table: pa.Table,
    column_name: str,
    *,
    allow_terminal_nonfinite: bool = False,
) -> np.ndarray:
    """
    Convert an Arrow column to a validated one-dimensional float64 array.
    """

    column = table[
        column_name
    ]


    try:
        cast_column = pc.cast(
            column,
            pa.float64(),
            safe=False,
        )

    except Exception as exc:
        raise RuntimeError(
            f"Column {column_name!r} could not be converted to float64."
        ) from exc


    values = np.asarray(
        cast_column.to_numpy(
            zero_copy_only=False
        ),
        dtype=np.float64,
    )


    if values.ndim != 1:
        raise RuntimeError(
            f"Column {column_name!r} is not one-dimensional."
        )


    if allow_terminal_nonfinite:
        if values.size < 2:
            raise RuntimeError(
                f"Column {column_name!r} contains fewer than two values."
            )


        if not np.isfinite(
            values[
                :-1
            ]
        ).all():
            nonfinite_count = int(
                (
                    ~np.isfinite(
                        values[
                            :-1
                        ]
                    )
                ).sum()
            )

            raise RuntimeError(
                f"Column {column_name!r} contains "
                f"{nonfinite_count:,} nonfinite nonterminal values."
            )

    else:
        if not np.isfinite(
            values
        ).all():
            nonfinite_count = int(
                (
                    ~np.isfinite(
                        values
                    )
                ).sum()
            )

            raise RuntimeError(
                f"Column {column_name!r} contains "
                f"{nonfinite_count:,} nonfinite values."
            )


    return values


# ======================================================================================
# 8. DEFINE THE LOCAL TRAJECTORY LOADER
# ======================================================================================

def get_design_parquet(
    partition: str,
) -> pq.ParquetFile:
    """
    Return the approved Parquet object for one development partition.
    """

    normalized_partition = str(
        partition
    ).strip().lower()


    if normalized_partition == "train":
        return TRAIN_DESIGN_PARQUET


    if normalized_partition == "calibration":
        return CALIBRATION_DESIGN_PARQUET


    raise ValueError(
        "Only training and calibration design partitions may be loaded."
    )


def load_design_trajectory(
    partition: str,
    row_group_index: int,
) -> dict[str, Any]:
    """
    Load and validate one complete training or calibration trajectory.

    The source sample index is allowed to:

    - begin at zero;
    - begin at one;
    - begin at another inherited source offset;
    - contain positive gaps created by upstream source indexing.

    It must remain strictly increasing. A local zero-based sample index is
    constructed for all numerical operations.
    """

    normalized_partition = str(
        partition
    ).strip().lower()

    row_group_index = int(
        row_group_index
    )


    parquet_file = get_design_parquet(
        normalized_partition
    )


    row_group_count = int(
        parquet_file.metadata.num_row_groups
    )


    if row_group_index < 0:
        raise IndexError(
            "The requested row-group index must be nonnegative."
        )


    if row_group_index >= row_group_count:
        raise IndexError(
            f"Row group {row_group_index} is outside "
            f"{normalized_partition!r}; available count is "
            f"{row_group_count:,}."
        )


    trajectory_lookup_key = (
        normalized_partition,
        row_group_index,
    )


    if trajectory_lookup_key not in (
        TRAJECTORY_POSITION_LOOKUP
    ):
        raise KeyError(
            "No trajectory selector is registered for:\n"
            f"  partition={normalized_partition!r}\n"
            f"  row_group_index={row_group_index}"
        )


    trajectory_position = int(
        TRAJECTORY_POSITION_LOOKUP[
            trajectory_lookup_key
        ]
    )


    table = parquet_file.read_row_group(
        row_group_index,
        columns=list(
            REQUIRED_TRAJECTORY_COLUMNS
        ),
        use_threads=True,
    )


    sample_count = int(
        table.num_rows
    )


    expected_sample_count = int(
        TRAJECTORY_SAMPLE_COUNT_BY_POSITION[
            trajectory_position
        ]
    )


    if sample_count < 2:
        raise RuntimeError(
            "A dynamic trajectory must contain at least two samples."
        )


    if sample_count != expected_sample_count:
        raise RuntimeError(
            "The row-group sample count differs from the manifest.\n"
            f"Partition: {normalized_partition}\n"
            f"Row group: {row_group_index}\n"
            f"Expected: {expected_sample_count:,}\n"
            f"Observed: {sample_count:,}"
        )


    trajectory_uid_values = unique_nonmissing_arrow_values(
        table[
            "trajectory_uid"
        ]
    )

    chemistry_values = unique_nonmissing_arrow_values(
        table[
            "chemistry"
        ]
    )

    cell_entity_values = unique_nonmissing_arrow_values(
        table[
            "cell_entity_id"
        ]
    )


    if len(
        trajectory_uid_values
    ) != 1:
        raise RuntimeError(
            "A design row group does not contain exactly one "
            "trajectory UID."
        )


    if len(
        chemistry_values
    ) != 1:
        raise RuntimeError(
            "A design row group does not contain exactly one chemistry."
        )


    if len(
        cell_entity_values
    ) != 1:
        raise RuntimeError(
            "A design row group does not contain exactly one cell entity."
        )


    trajectory_uid = str(
        trajectory_uid_values[
            0
        ]
    ).strip()

    chemistry = str(
        chemistry_values[
            0
        ]
    ).strip().upper()

    cell_entity_id = str(
        cell_entity_values[
            0
        ]
    ).strip()


    expected_trajectory_uid = str(
        TRAJECTORY_UID_BY_POSITION[
            trajectory_position
        ]
    )

    expected_partition = str(
        TRAJECTORY_PARTITION_BY_POSITION[
            trajectory_position
        ]
    )

    expected_chemistry = str(
        TRAJECTORY_CHEMISTRY_BY_POSITION[
            trajectory_position
        ]
    )

    expected_cell_entity_id = str(
        TRAJECTORY_CELL_ENTITY_BY_POSITION[
            trajectory_position
        ]
    )


    if trajectory_uid != expected_trajectory_uid:
        raise RuntimeError(
            "The row-group trajectory UID differs from the manifest.\n"
            f"Expected: {expected_trajectory_uid}\n"
            f"Observed: {trajectory_uid}"
        )


    if normalized_partition != expected_partition:
        raise RuntimeError(
            "The row-group partition differs from the manifest.\n"
            f"Expected: {expected_partition}\n"
            f"Observed: {normalized_partition}"
        )


    if chemistry != expected_chemistry:
        raise RuntimeError(
            "The row-group chemistry differs from the manifest.\n"
            f"Expected: {expected_chemistry}\n"
            f"Observed: {chemistry}"
        )


    if cell_entity_id != expected_cell_entity_id:
        raise RuntimeError(
            "The row-group cell entity differs from the manifest.\n"
            f"Expected: {expected_cell_entity_id}\n"
            f"Observed: {cell_entity_id}"
        )


    # ------------------------------------------------------------------
    # Source and local sample indices
    # ------------------------------------------------------------------

    source_sample_index = integer_column_to_numpy(
        table,
        "sample_index",
    )


    if source_sample_index.shape != (
        sample_count,
    ):
        raise RuntimeError(
            "The source sample-index array has an incorrect shape."
        )


    source_sample_index_difference = np.diff(
        source_sample_index
    )


    if (
        source_sample_index_difference
        <=
        0
    ).any():
        invalid_positions = np.flatnonzero(
            source_sample_index_difference
            <=
            0
        )


        preview_positions = invalid_positions[
            :
            10
        ].tolist()


        raise RuntimeError(
            "The source sample index is not strictly increasing.\n"
            f"Partition: {normalized_partition}\n"
            f"Row group: {row_group_index}\n"
            f"Trajectory: {trajectory_uid}\n"
            f"First invalid difference positions: {preview_positions}"
        )


    local_sample_index = np.arange(
        sample_count,
        dtype=np.int64,
    )


    source_sample_index_is_unit_step = bool(
        np.all(
            source_sample_index_difference
            ==
            1
        )
    )


    source_sample_index_gap_count = int(
        (
            source_sample_index_difference
            >
            1
        ).sum()
    )


    source_sample_index_maximum_gap = int(
        source_sample_index_difference.max(
            initial=1
        )
    )


    # ------------------------------------------------------------------
    # Time variables
    # ------------------------------------------------------------------

    elapsed_time_s = float_column_to_numpy(
        table,
        "elapsed_time_s",
    )


    elapsed_time_difference = np.diff(
        elapsed_time_s
    )


    if (
        elapsed_time_difference
        <=
        0.0
    ).any():
        invalid_positions = np.flatnonzero(
            elapsed_time_difference
            <=
            0.0
        )


        raise RuntimeError(
            "The trajectory elapsed time is not strictly increasing.\n"
            f"Partition: {normalized_partition}\n"
            f"Row group: {row_group_index}\n"
            f"Trajectory: {trajectory_uid}\n"
            f"First invalid positions: "
            f"{invalid_positions[:10].tolist()}"
        )


    delta_t_s = float_column_to_numpy(
        table,
        "delta_t_to_next_s",
        allow_terminal_nonfinite=True,
    )


    if delta_t_s.shape != (
        sample_count,
    ):
        raise RuntimeError(
            "The sampling-interval array has an incorrect shape."
        )


    if (
        delta_t_s[
            :-1
        ]
        <=
        0.0
    ).any():
        invalid_positions = np.flatnonzero(
            delta_t_s[
                :-1
            ]
            <=
            0.0
        )


        raise RuntimeError(
            "A nonterminal sampling interval is not strictly positive.\n"
            f"Partition: {normalized_partition}\n"
            f"Row group: {row_group_index}\n"
            f"Trajectory: {trajectory_uid}\n"
            f"First invalid positions: "
            f"{invalid_positions[:10].tolist()}"
        )


    if (
        not np.isfinite(
            delta_t_s[
                -1
            ]
        )
        or
        delta_t_s[
            -1
        ]
        <
        0.0
    ):
        delta_t_s[
            -1
        ] = 0.0


    # ------------------------------------------------------------------
    # Dynamic model inputs
    # ------------------------------------------------------------------

    current_discharge_a = float_column_to_numpy(
        table,
        "current_discharge_a",
    )

    voltage_v = float_column_to_numpy(
        table,
        "voltage_v",
    )

    ocv_v = float_column_to_numpy(
        table,
        "ocv_v",
    )


    basis_matrix = np.column_stack(
        [
            float_column_to_numpy(
                table,
                column_name,
            )
            for column_name in BASIS_DESIGN_COLUMNS
        ]
    ).astype(
        np.float64,
        copy=False,
    )


    if basis_matrix.shape != (
        sample_count,
        NOTEBOOK_12_BASIS_TERM_COUNT,
    ):
        raise RuntimeError(
            "The loaded basis matrix has an incorrect shape.\n"
            f"Expected: "
            f"({sample_count}, {NOTEBOOK_12_BASIS_TERM_COUNT})\n"
            f"Observed: {basis_matrix.shape}"
        )


    if not np.isfinite(
        basis_matrix
    ).all():
        raise RuntimeError(
            "The loaded basis matrix contains nonfinite values."
        )


    if not np.allclose(
        basis_matrix[
            :,
            0
        ],
        1.0,
        rtol=0.0,
        atol=1.0e-12,
    ):
        raise RuntimeError(
            "The first LPV basis column is not the constant intercept."
        )


    return {
        "partition": normalized_partition,
        "row_group_index": int(
            row_group_index
        ),
        "trajectory_position": int(
            trajectory_position
        ),
        "trajectory_uid": trajectory_uid,
        "chemistry": chemistry,
        "cell_entity_id": cell_entity_id,
        "sample_count": int(
            sample_count
        ),

        # Source metadata retained for auditing.
        "source_sample_index": (
            source_sample_index
        ),
        "source_sample_index_start": int(
            source_sample_index[
                0
            ]
        ),
        "source_sample_index_end": int(
            source_sample_index[
                -1
            ]
        ),
        "source_sample_index_is_unit_step": bool(
            source_sample_index_is_unit_step
        ),
        "source_sample_index_gap_count": int(
            source_sample_index_gap_count
        ),
        "source_sample_index_maximum_gap": int(
            source_sample_index_maximum_gap
        ),

        # Numerical index used by the local trajectory representation.
        "sample_index": local_sample_index,

        "elapsed_time_s": elapsed_time_s,
        "delta_t_s": delta_t_s,
        "current_discharge_a": current_discharge_a,
        "voltage_v": voltage_v,
        "ocv_v": ocv_v,
        "basis_matrix": basis_matrix,

        "global_full_indices": (
            TRAJECTORY_GLOBAL_FULL_INDICES[
                trajectory_position
            ].copy()
        ),

        "chemistry_full_indices": (
            TRAJECTORY_CHEMISTRY_FULL_INDICES[
                trajectory_position
            ].copy()
        ),

        "cell_full_indices": (
            TRAJECTORY_CELL_FULL_INDICES[
                trajectory_position
            ].copy()
        ),

        "has_cell_effect": bool(
            TRAJECTORY_HAS_CELL_EFFECT[
                trajectory_position
            ]
        ),
    }


# ======================================================================================
# 9. DEFINE PADDED BATCH CONSTRUCTION
# ======================================================================================

def next_padding_length(
    maximum_length: int,
    *,
    multiple: int = PADDED_TRAJECTORY_MULTIPLE,
) -> int:
    """
    Round a sequence length upward to the selected padding multiple.
    """

    maximum_length = int(
        maximum_length
    )

    multiple = int(
        multiple
    )


    if maximum_length < 2:
        raise ValueError(
            "The padded sequence length must be at least two."
        )


    if multiple <= 0:
        raise ValueError(
            "The padding multiple must be positive."
        )


    return int(
        math.ceil(
            maximum_length
            /
            multiple
        )
        *
        multiple
    )


def load_padded_trajectory_batch(
    *,
    partition: str,
    row_group_indices: Sequence[int] | np.ndarray,
) -> dict[str, Any]:
    """
    Load complete trajectories and construct one fixed-shape NumPy batch.

    Repeated row-group indices are allowed. This supports sampling with
    replacement during stochastic optimization.
    """

    normalized_partition = str(
        partition
    ).strip().lower()


    if normalized_partition not in (
        "train",
        "calibration",
    ):
        raise ValueError(
            "Only training and calibration batches may be constructed."
        )


    requested_row_groups = np.asarray(
        row_group_indices,
        dtype=np.int64,
    ).reshape(
        -1
    )


    if requested_row_groups.size == 0:
        raise ValueError(
            "The requested trajectory batch is empty."
        )


    available_row_group_count = (
        TRAIN_ROW_GROUP_COUNT
        if normalized_partition == "train"
        else CALIBRATION_ROW_GROUP_COUNT
    )


    if (
        requested_row_groups
        <
        0
    ).any():
        raise IndexError(
            "A requested row-group index is negative."
        )


    if (
        requested_row_groups
        >=
        available_row_group_count
    ).any():
        invalid_row_groups = requested_row_groups[
            requested_row_groups
            >=
            available_row_group_count
        ]


        raise IndexError(
            f"A requested row group is outside "
            f"{normalized_partition!r}.\n"
            f"Available count: {available_row_group_count:,}\n"
            f"First invalid indices: "
            f"{invalid_row_groups[:10].tolist()}"
        )


    trajectories = [
        load_design_trajectory(
            normalized_partition,
            int(
                row_group_index
            ),
        )
        for row_group_index in requested_row_groups
    ]


    batch_size = int(
        len(
            trajectories
        )
    )


    maximum_trajectory_length = max(
        int(
            trajectory[
                "sample_count"
            ]
        )
        for trajectory in trajectories
    )


    padded_length = next_padding_length(
        maximum_trajectory_length
    )


    basis = np.zeros(
        (
            batch_size,
            padded_length,
            NOTEBOOK_12_BASIS_TERM_COUNT,
        ),
        dtype=np.float64,
    )


    current = np.zeros(
        (
            batch_size,
            padded_length,
        ),
        dtype=np.float64,
    )

    voltage = np.zeros_like(
        current
    )

    ocv = np.zeros_like(
        current
    )

    delta_t = np.zeros_like(
        current
    )


    valid_mask = np.zeros(
        (
            batch_size,
            padded_length,
        ),
        dtype=bool,
    )


    selector_batch_shape = (
        batch_size,
        NOTEBOOK_12_PARAMETER_COUNT,
        NOTEBOOK_12_BASIS_TERM_COUNT,
    )


    global_indices = np.empty(
        selector_batch_shape,
        dtype=np.int32,
    )

    chemistry_indices = np.empty(
        selector_batch_shape,
        dtype=np.int32,
    )

    cell_indices = np.empty(
        selector_batch_shape,
        dtype=np.int32,
    )


    has_cell_effect = np.zeros(
        batch_size,
        dtype=bool,
    )


    sample_counts = np.empty(
        batch_size,
        dtype=np.int32,
    )

    trajectory_positions = np.empty(
        batch_size,
        dtype=np.int32,
    )


    source_sample_index_start = np.empty(
        batch_size,
        dtype=np.int64,
    )

    source_sample_index_end = np.empty(
        batch_size,
        dtype=np.int64,
    )

    source_sample_index_gap_count = np.empty(
        batch_size,
        dtype=np.int32,
    )

    source_sample_index_maximum_gap = np.empty(
        batch_size,
        dtype=np.int64,
    )

    source_sample_index_is_unit_step = np.zeros(
        batch_size,
        dtype=bool,
    )


    trajectory_uids: list[str] = []


    for batch_position, trajectory in enumerate(
        trajectories
    ):
        sample_count = int(
            trajectory[
                "sample_count"
            ]
        )


        sample_counts[
            batch_position
        ] = sample_count


        trajectory_positions[
            batch_position
        ] = int(
            trajectory[
                "trajectory_position"
            ]
        )


        source_sample_index_start[
            batch_position
        ] = int(
            trajectory[
                "source_sample_index_start"
            ]
        )


        source_sample_index_end[
            batch_position
        ] = int(
            trajectory[
                "source_sample_index_end"
            ]
        )


        source_sample_index_gap_count[
            batch_position
        ] = int(
            trajectory[
                "source_sample_index_gap_count"
            ]
        )


        source_sample_index_maximum_gap[
            batch_position
        ] = int(
            trajectory[
                "source_sample_index_maximum_gap"
            ]
        )


        source_sample_index_is_unit_step[
            batch_position
        ] = bool(
            trajectory[
                "source_sample_index_is_unit_step"
            ]
        )


        trajectory_uids.append(
            str(
                trajectory[
                    "trajectory_uid"
                ]
            )
        )


        basis[
            batch_position,
            :
            sample_count,
            :,
        ] = trajectory[
            "basis_matrix"
        ]


        current[
            batch_position,
            :
            sample_count,
        ] = trajectory[
            "current_discharge_a"
        ]


        voltage[
            batch_position,
            :
            sample_count,
        ] = trajectory[
            "voltage_v"
        ]


        ocv[
            batch_position,
            :
            sample_count,
        ] = trajectory[
            "ocv_v"
        ]


        delta_t[
            batch_position,
            :
            sample_count,
        ] = trajectory[
            "delta_t_s"
        ]


        valid_mask[
            batch_position,
            :
            sample_count,
        ] = True


        global_indices[
            batch_position
        ] = trajectory[
            "global_full_indices"
        ]


        chemistry_indices[
            batch_position
        ] = trajectory[
            "chemistry_full_indices"
        ]


        cell_indices[
            batch_position
        ] = trajectory[
            "cell_full_indices"
        ]


        has_cell_effect[
            batch_position
        ] = bool(
            trajectory[
                "has_cell_effect"
            ]
        )


    loss_mask = valid_mask.copy()


    # Sample zero may participate in the initial-state policy.
    # It is excluded from the optimization loss.
    loss_mask[
        :,
        0,
    ] = False


    valid_sample_count = int(
        valid_mask.sum()
    )

    loss_sample_count = int(
        loss_mask.sum()
    )


    expected_valid_sample_count = int(
        sample_counts.astype(
            np.int64
        ).sum()
    )


    expected_loss_sample_count = int(
        expected_valid_sample_count
        -
        batch_size
    )


    if valid_sample_count != expected_valid_sample_count:
        raise RuntimeError(
            "The padded valid-mask count is incorrect.\n"
            f"Expected: {expected_valid_sample_count:,}\n"
            f"Observed: {valid_sample_count:,}"
        )


    if loss_sample_count != expected_loss_sample_count:
        raise RuntimeError(
            "The padded loss-mask count is incorrect.\n"
            f"Expected: {expected_loss_sample_count:,}\n"
            f"Observed: {loss_sample_count:,}"
        )


    if not loss_mask.any():
        raise RuntimeError(
            "The padded batch contains no optimization samples."
        )


    for array_name, array in (
        (
            "basis",
            basis,
        ),
        (
            "current",
            current,
        ),
        (
            "voltage",
            voltage,
        ),
        (
            "ocv",
            ocv,
        ),
        (
            "delta_t",
            delta_t,
        ),
    ):
        if not np.isfinite(
            array
        ).all():
            nonfinite_count = int(
                (
                    ~np.isfinite(
                        array
                    )
                ).sum()
            )


            raise RuntimeError(
                f"The padded {array_name} array contains "
                f"{nonfinite_count:,} nonfinite values."
            )


    return {
        "partition": normalized_partition,

        "row_group_indices": (
            requested_row_groups.astype(
                np.int32,
                copy=False,
            )
        ),

        "trajectory_uids": tuple(
            trajectory_uids
        ),

        "basis": basis,
        "current": current,
        "voltage": voltage,
        "ocv": ocv,
        "delta_t": delta_t,

        "valid_mask": valid_mask,
        "loss_mask": loss_mask,

        "global_indices": global_indices,
        "chemistry_indices": chemistry_indices,
        "cell_indices": cell_indices,
        "has_cell_effect": has_cell_effect,

        "sample_counts": sample_counts,
        "trajectory_positions": trajectory_positions,

        "source_sample_index_start": (
            source_sample_index_start
        ),
        "source_sample_index_end": (
            source_sample_index_end
        ),
        "source_sample_index_gap_count": (
            source_sample_index_gap_count
        ),
        "source_sample_index_maximum_gap": (
            source_sample_index_maximum_gap
        ),
        "source_sample_index_is_unit_step": (
            source_sample_index_is_unit_step
        ),

        "batch_size": int(
            batch_size
        ),
        "padded_length": int(
            padded_length
        ),
        "valid_sample_count": int(
            valid_sample_count
        ),
        "loss_sample_count": int(
            loss_sample_count
        ),
    }


# ======================================================================================
# 10. DEFINE NUMPY-TO-JAX CONVERSION
# ======================================================================================

def batch_to_jax(
    batch: Mapping[str, Any],
) -> dict[str, jax.Array]:
    """
    Convert one padded NumPy batch to JAX arrays.
    """

    required_batch_keys = (
        "basis",
        "current",
        "voltage",
        "ocv",
        "delta_t",
        "valid_mask",
        "loss_mask",
        "global_indices",
        "chemistry_indices",
        "cell_indices",
        "has_cell_effect",
    )


    missing_batch_keys = [
        key
        for key in required_batch_keys
        if key not in batch
    ]


    if missing_batch_keys:
        raise KeyError(
            "The padded batch is missing required keys:\n  - "
            + "\n  - ".join(
                missing_batch_keys
            )
        )


    jax_batch = {
        "basis": jnp.asarray(
            batch[
                "basis"
            ],
            dtype=jnp.float64,
        ),

        "current": jnp.asarray(
            batch[
                "current"
            ],
            dtype=jnp.float64,
        ),

        "voltage": jnp.asarray(
            batch[
                "voltage"
            ],
            dtype=jnp.float64,
        ),

        "ocv": jnp.asarray(
            batch[
                "ocv"
            ],
            dtype=jnp.float64,
        ),

        "delta_t": jnp.asarray(
            batch[
                "delta_t"
            ],
            dtype=jnp.float64,
        ),

        "valid_mask": jnp.asarray(
            batch[
                "valid_mask"
            ],
            dtype=jnp.bool_,
        ),

        "loss_mask": jnp.asarray(
            batch[
                "loss_mask"
            ],
            dtype=jnp.bool_,
        ),

        "global_indices": jnp.asarray(
            batch[
                "global_indices"
            ],
            dtype=jnp.int32,
        ),

        "chemistry_indices": jnp.asarray(
            batch[
                "chemistry_indices"
            ],
            dtype=jnp.int32,
        ),

        "cell_indices": jnp.asarray(
            batch[
                "cell_indices"
            ],
            dtype=jnp.int32,
        ),

        "has_cell_effect": jnp.asarray(
            batch[
                "has_cell_effect"
            ],
            dtype=jnp.bool_,
        ),
    }


    if jax_batch[
        "basis"
    ].shape != batch[
        "basis"
    ].shape:
        raise RuntimeError(
            "The JAX and NumPy basis-batch shapes differ."
        )


    if jax_batch[
        "loss_mask"
    ].shape != batch[
        "loss_mask"
    ].shape:
        raise RuntimeError(
            "The JAX and NumPy loss-mask shapes differ."
        )


    return jax_batch


# ======================================================================================
# 11. FREEZE ROW-GROUP ARRAYS
# ======================================================================================

training_row_group_indices = np.arange(
    TRAIN_ROW_GROUP_COUNT,
    dtype=np.int32,
)


calibration_row_group_indices = np.arange(
    CALIBRATION_ROW_GROUP_COUNT,
    dtype=np.int32,
)


if training_row_group_indices.size != (
    TRAIN_ROW_GROUP_COUNT
):
    raise RuntimeError(
        "The training row-group array has an incorrect length."
    )


if calibration_row_group_indices.size != (
    CALIBRATION_ROW_GROUP_COUNT
):
    raise RuntimeError(
        "The calibration row-group array has an incorrect length."
    )


# ======================================================================================
# 12. RUN THE PADDED-BATCH SMOKE TEST
# ======================================================================================

smoke_batch_row_group_count = min(
    TRAINING_BATCH_TRAJECTORIES,
    TRAIN_ROW_GROUP_COUNT,
)


smoke_batch = load_padded_trajectory_batch(
    partition="train",

    row_group_indices=training_row_group_indices[
        :
        smoke_batch_row_group_count
    ],
)


smoke_jax_batch = batch_to_jax(
    smoke_batch
)


if not bool(
    np.asarray(
        jnp.any(
            smoke_jax_batch[
                "loss_mask"
            ]
        )
    )
):
    raise RuntimeError(
        "The JAX smoke batch contains no optimization samples."
    )


estimated_smoke_batch_bytes = int(
    sum(
        smoke_batch[
            key
        ].nbytes
        for key in (
            "basis",
            "current",
            "voltage",
            "ocv",
            "delta_t",
            "valid_mask",
            "loss_mask",
            "global_indices",
            "chemistry_indices",
            "cell_indices",
            "has_cell_effect",
            "sample_counts",
            "trajectory_positions",
            "source_sample_index_start",
            "source_sample_index_end",
            "source_sample_index_gap_count",
            "source_sample_index_maximum_gap",
            "source_sample_index_is_unit_step",
        )
    )
)


NOTEBOOK_12_SAMPLE_INDEX_DIAGNOSTIC_DF = pd.DataFrame(
    {
        "batch_position": np.arange(
            smoke_batch[
                "batch_size"
            ],
            dtype=np.int32,
        ),

        "row_group_index": smoke_batch[
            "row_group_indices"
        ],

        "trajectory_uid": list(
            smoke_batch[
                "trajectory_uids"
            ]
        ),

        "sample_count": smoke_batch[
            "sample_counts"
        ],

        "source_sample_index_start": smoke_batch[
            "source_sample_index_start"
        ],

        "source_sample_index_end": smoke_batch[
            "source_sample_index_end"
        ],

        "source_sample_index_is_unit_step": smoke_batch[
            "source_sample_index_is_unit_step"
        ],

        "source_sample_index_gap_count": smoke_batch[
            "source_sample_index_gap_count"
        ],

        "source_sample_index_maximum_gap": smoke_batch[
            "source_sample_index_maximum_gap"
        ],
    }
)


nonzero_source_index_start_count = int(
    (
        NOTEBOOK_12_SAMPLE_INDEX_DIAGNOSTIC_DF[
            "source_sample_index_start"
        ]
        !=
        0
    ).sum()
)


nonunit_source_index_trajectory_count = int(
    (
        ~NOTEBOOK_12_SAMPLE_INDEX_DIAGNOSTIC_DF[
            "source_sample_index_is_unit_step"
        ]
        .astype(bool)
    ).sum()
)


# ======================================================================================
# 13. FINAL EXECUTION SUMMARY
# ======================================================================================

print("=" * 116)
print("MEMORY-SAFE PADDED TRAJECTORY BATCHING")
print("=" * 116)

print(
    f"Training row groups                     : "
    f"{TRAIN_ROW_GROUP_COUNT:,}"
)

print(
    f"Calibration row groups                  : "
    f"{CALIBRATION_ROW_GROUP_COUNT:,}"
)

print(
    f"Training design samples                 : "
    f"{TRAIN_DESIGN_SAMPLE_COUNT:,}"
)

print(
    f"Calibration design samples              : "
    f"{CALIBRATION_DESIGN_SAMPLE_COUNT:,}"
)

print(
    f"Reconstructed selector trajectories     : "
    f"{trajectory_count:,}"
)

print(
    f"Smoke-batch trajectories                : "
    f"{smoke_batch['batch_size']:,}"
)

print(
    f"Smoke-batch padded length               : "
    f"{smoke_batch['padded_length']:,}"
)

print(
    f"Smoke-batch basis shape                 : "
    f"{smoke_batch['basis'].shape}"
)

print(
    f"Smoke-batch valid samples               : "
    f"{smoke_batch['valid_sample_count']:,}"
)

print(
    f"Smoke-batch optimization samples        : "
    f"{smoke_batch['loss_sample_count']:,}"
)

print(
    f"Source indices not beginning at zero    : "
    f"{nonzero_source_index_start_count:,}"
)

print(
    f"Source indices with non-unit increments : "
    f"{nonunit_source_index_trajectory_count:,}"
)

print(
    f"Smoke-batch estimated memory            : "
    f"{estimated_smoke_batch_bytes / (1024 ** 2):.3f} MiB"
)

print(
    f"Padding multiple                        : "
    f"{PADDED_TRAJECTORY_MULTIPLE:,}"
)

print(
    f"JAX float64 enabled                     : "
    f"{bool(jax.config.read('jax_enable_x64'))}"
)

print("-" * 116)

print(
    "The source sample indices were validated as ordered metadata, "
    "local zero-based indices were constructed, and the padded trajectory "
    "batch was created successfully."
)

print("=" * 116)


display(
    NOTEBOOK_12_SAMPLE_INDEX_DIAGNOSTIC_DF
)

MEMORY-SAFE PADDED TRAJECTORY BATCHING
Training row groups                     : 2,832
Calibration row groups                  : 1,017
Training design samples                 : 8,972,544
Calibration design samples              : 4,392,705
Reconstructed selector trajectories     : 3,849
Smoke-batch trajectories                : 8
Smoke-batch padded length               : 1,344
Smoke-batch basis shape                 : (8, 1344, 6)
Smoke-batch valid samples               : 10,140
Smoke-batch optimization samples        : 10,132
Source indices not beginning at zero    : 8
Source indices with non-unit increments : 8
Smoke-batch estimated memory            : 0.844 MiB
Padding multiple                        : 32
JAX float64 enabled                     : True
--------------------------------------------------------------------------------------------------------------------
The source sample indices were validated as ordered metadata, local zero-based indices were constructed, and the padded

,batch_position,row_group_index,trajectory_uid,sample_count,source_sample_index_start,source_sample_index_end,source_sample_index_is_unit_step,source_sample_index_gap_count,source_sample_index_maximum_gap
0,0,0,LFP::LFP_CELL_1::LFP_CELL_1::1,1320,1,5619,False,2,3114
1,1,1,LFP::LFP_CELL_1::LFP_CELL_1::2,1260,1237,5627,False,1,3132
2,2,2,LFP::LFP_CELL_1::LFP_CELL_1::3,1260,1240,5635,False,1,3137
3,3,3,LFP::LFP_CELL_1::LFP_CELL_1::4,1260,1241,5638,False,1,3139
4,4,4,LFP::LFP_CELL_1::LFP_CELL_1::5,1260,1242,5641,False,1,3141
5,5,5,LFP::LFP_CELL_1::LFP_CELL_1::6,1260,1243,5644,False,1,3143
6,6,6,LFP::LFP_CELL_1::LFP_CELL_1::7,1260,1243,5645,False,1,3144
7,7,7,LFP::LFP_CELL_1::LFP_CELL_1::8,1260,1244,5648,False,1,3146


MEMORY-SAFE PADDED TRAJECTORY BATCHING
Training row groups                     : 2,832
Calibration row groups                  : 1,017
Training design samples                 : 8,972,544
Calibration design samples              : 4,392,705
Reconstructed selector trajectories     : 3,849
Smoke-batch trajectories                : 8
Smoke-batch padded length               : 1,344
Smoke-batch basis shape                 : (8, 1344, 6)
Smoke-batch valid samples               : 10,140
Smoke-batch optimization samples        : 10,132
Source indices not beginning at zero    : 8
Source indices with non-unit increments : 8
Smoke-batch estimated memory            : 0.844 MiB
Padding multiple                        : 32
JAX float64 enabled                     : True
--------------------------------------------------------------------------------------------------------------------
The source sample indices were validated as ordered metadata, local zero-based indices were constructed, and the padded

,batch_position,row_group_index,trajectory_uid,sample_count,source_sample_index_start,source_sample_index_end,source_sample_index_is_unit_step,source_sample_index_gap_count,source_sample_index_maximum_gap
0,0,0,LFP::LFP_CELL_1::LFP_CELL_1::1,1320,1,5619,False,2,3114
1,1,1,LFP::LFP_CELL_1::LFP_CELL_1::2,1260,1237,5627,False,1,3132
2,2,2,LFP::LFP_CELL_1::LFP_CELL_1::3,1260,1240,5635,False,1,3137
3,3,3,LFP::LFP_CELL_1::LFP_CELL_1::4,1260,1241,5638,False,1,3139
4,4,4,LFP::LFP_CELL_1::LFP_CELL_1::5,1260,1242,5641,False,1,3141
5,5,5,LFP::LFP_CELL_1::LFP_CELL_1::6,1260,1243,5644,False,1,3143
6,6,6,LFP::LFP_CELL_1::LFP_CELL_1::7,1260,1243,5645,False,1,3144
7,7,7,LFP::LFP_CELL_1::LFP_CELL_1::8,1260,1244,5648,False,1,3146


In [20]:
# ======================================================================================
# CELL 19 — DEFINE THE BATCHED RECURSIVE OBJECTIVE AND GROUP PENALTY
# ======================================================================================
#
# Purpose
# -------
# Define the batched differentiable two-RC LPV simulator and the optimization
# objective used for candidate fitting.
#
# This corrected cell resolves the nonfinite gradient at zero initialization.
# The ordinary Euclidean group norm has an undefined derivative at the zero
# vector. JAX therefore produces NaN values when every penalized coefficient
# starts at zero.
#
# This cell uses the exact group-Lasso value:
#
#     ||theta_g||_2
#
# while explicitly selecting the valid zero subgradient:
#
#     partial ||theta_g||_2 at theta_g = 0  ->  0.
#
# Therefore:
#
# - the group penalty remains exactly zero for an inactive group;
# - the group-Lasso value is not replaced by a biased positive constant;
# - the gradient is finite at zero initialization;
# - the five unpenalized global intercepts remain unaffected;
# - all 240 reduced coordinates receive finite gradients.
# ======================================================================================

from __future__ import annotations

from pathlib import Path
from typing import Any, Mapping, Sequence

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

from scipy import sparse as scipy_sparse


# ======================================================================================
# 1. ENABLE JAX FLOAT64
# ======================================================================================

jax.config.update(
    "jax_enable_x64",
    True,
)


# ======================================================================================
# 2. VERIFY REQUIRED OBJECTS
# ======================================================================================

REQUIRED_CELL_19_OBJECTS: tuple[str, ...] = (
    "NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF",
    "CANDIDATE_ACTIVE_MASKS",
    "CANDIDATE_PENALTY_GROUP_IDS",
    "PENALTY_GROUP_MEMBERS",
    "PENALTY_GROUP_WEIGHTS",
    "MODEL_CANDIDATE_SPECIFICATIONS",
    "batch_to_jax",
    "smoke_batch",
    "R_MIN_OHM",
    "R_MAX_OHM",
    "TAU1_MIN_S",
    "TAU1_MAX_S",
    "TAU_GAP_MIN_S",
    "TAU_GAP_MAX_S",
    "INITIAL_REST_SAMPLES",
    "REST_CURRENT_A",
    "PSEUDO_HUBER_DELTA_V",
    "RIDGE_WEIGHT",
    "SPARSE_LAMBDA_GRID",
)


missing_cell_19_objects = [
    object_name
    for object_name in REQUIRED_CELL_19_OBJECTS
    if object_name not in globals()
]


if missing_cell_19_objects:
    raise RuntimeError(
        "Cell 19 requires the simulator, candidate-family, and padded-batch "
        "cells to be executed first.\n\n"
        "Missing objects:\n  - "
        + "\n  - ".join(
            missing_cell_19_objects
        )
    )


# ======================================================================================
# 3. RECONSTRUCT THE REDUCED COEFFICIENT DIMENSION
# ======================================================================================

required_reduced_registry_columns: tuple[str, ...] = (
    "reduced_coefficient_index",
    "reduced_coefficient_id",
)


missing_reduced_registry_columns = [
    column_name
    for column_name in required_reduced_registry_columns
    if column_name not in (
        NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF.columns
    )
]


if missing_reduced_registry_columns:
    raise RuntimeError(
        "The reduced coefficient registry is missing required columns:\n  - "
        + "\n  - ".join(
            missing_reduced_registry_columns
        )
    )


NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT: int = int(
    len(
        NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF
    )
)


if NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT <= 0:
    raise RuntimeError(
        "The reduced coefficient registry is empty."
    )


reduced_coefficient_indices = np.sort(
    pd.to_numeric(
        NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF[
            "reduced_coefficient_index"
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.int64
    )
)


expected_reduced_coefficient_indices = np.arange(
    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
    dtype=np.int64,
)


if not np.array_equal(
    reduced_coefficient_indices,
    expected_reduced_coefficient_indices,
):
    raise RuntimeError(
        "The reduced coefficient indices are not contiguous from zero."
    )


if NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF[
    "reduced_coefficient_id"
].astype(str).duplicated().any():
    raise RuntimeError(
        "The reduced coefficient registry contains duplicate identifiers."
    )


# ======================================================================================
# 4. RESTORE OR VERIFY THE IDENTIFIABLE EXPANSION MATRIX
# ======================================================================================

if "JAX_IDENTIFIABLE_EXPANSION_MATRIX" not in globals():
    if (
        "NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX_PATH"
        not in globals()
    ):
        raise RuntimeError(
            "The identifiable expansion matrix is unavailable, and its "
            "saved path is not defined."
        )


    expansion_matrix_path = Path(
        NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX_PATH
    ).expanduser().resolve()


    if not expansion_matrix_path.is_file():
        raise FileNotFoundError(
            "The identifiable expansion matrix was not found:\n"
            f"  {expansion_matrix_path}"
        )


    restored_expansion_matrix = (
        scipy_sparse.load_npz(
            expansion_matrix_path
        )
        .tocsr()
        .astype(
            np.float64
        )
    )


    restored_expansion_matrix.eliminate_zeros()
    restored_expansion_matrix.sort_indices()


    if restored_expansion_matrix.nnz <= 0:
        raise RuntimeError(
            "The restored expansion matrix contains no nonzero entries."
        )


    if not np.isfinite(
        restored_expansion_matrix.data
    ).all():
        raise RuntimeError(
            "The restored expansion matrix contains nonfinite entries."
        )


    JAX_IDENTIFIABLE_EXPANSION_MATRIX = jnp.asarray(
        restored_expansion_matrix.toarray(),
        dtype=jnp.float64,
    )


expansion_matrix_shape: tuple[int, int] = tuple(
    int(
        dimension
    )
    for dimension in (
        JAX_IDENTIFIABLE_EXPANSION_MATRIX.shape
    )
)


if len(
    expansion_matrix_shape
) != 2:
    raise RuntimeError(
        "The identifiable expansion matrix must be two-dimensional."
    )


NOTEBOOK_12_FULL_COEFFICIENT_COUNT: int = int(
    expansion_matrix_shape[
        0
    ]
)


if expansion_matrix_shape[
    1
] != NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT:
    raise RuntimeError(
        "The reduced expansion-matrix dimension differs from the registry.\n"
        f"Expansion matrix: {expansion_matrix_shape[1]:,}\n"
        f"Registry: {NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT:,}"
    )


expansion_matrix_is_finite = bool(
    np.asarray(
        jnp.all(
            jnp.isfinite(
                JAX_IDENTIFIABLE_EXPANSION_MATRIX
            )
        )
    )
)


if not expansion_matrix_is_finite:
    raise RuntimeError(
        "The identifiable expansion matrix contains nonfinite values."
    )


# ======================================================================================
# 5. CREATE THE LOCAL ZERO INITIALIZATION
# ======================================================================================

zero_reduced_coefficients = np.zeros(
    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
    dtype=np.float64,
)


if zero_reduced_coefficients.shape != (
    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
):
    raise RuntimeError(
        "The zero reduced-coefficient vector has an incorrect shape."
    )


# ======================================================================================
# 6. VERIFY THE PADDED SMOKE BATCH
# ======================================================================================

required_smoke_batch_keys: tuple[str, ...] = (
    "basis",
    "current",
    "voltage",
    "ocv",
    "delta_t",
    "valid_mask",
    "loss_mask",
    "global_indices",
    "chemistry_indices",
    "cell_indices",
    "has_cell_effect",
)


missing_smoke_batch_keys = [
    key
    for key in required_smoke_batch_keys
    if key not in smoke_batch
]


if missing_smoke_batch_keys:
    raise RuntimeError(
        "The padded smoke batch is missing required arrays:\n  - "
        + "\n  - ".join(
            missing_smoke_batch_keys
        )
    )


if int(
    np.asarray(
        smoke_batch[
            "loss_mask"
        ],
        dtype=bool,
    ).sum()
) <= 0:
    raise RuntimeError(
        "The padded smoke batch contains no optimization samples."
    )


# ======================================================================================
# 7. VERIFY CANDIDATE ACTIVE MASKS
# ======================================================================================

if not isinstance(
    CANDIDATE_ACTIVE_MASKS,
    Mapping,
):
    raise TypeError(
        "CANDIDATE_ACTIVE_MASKS must be a mapping."
    )


if not isinstance(
    CANDIDATE_PENALTY_GROUP_IDS,
    Mapping,
):
    raise TypeError(
        "CANDIDATE_PENALTY_GROUP_IDS must be a mapping."
    )


if not isinstance(
    PENALTY_GROUP_MEMBERS,
    Mapping,
):
    raise TypeError(
        "PENALTY_GROUP_MEMBERS must be a mapping."
    )


if not isinstance(
    PENALTY_GROUP_WEIGHTS,
    Mapping,
):
    raise TypeError(
        "PENALTY_GROUP_WEIGHTS must be a mapping."
    )


for candidate_id, candidate_active_mask in (
    CANDIDATE_ACTIVE_MASKS.items()
):
    active_mask_array = np.asarray(
        candidate_active_mask,
        dtype=bool,
    )


    if active_mask_array.shape != (
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
    ):
        raise RuntimeError(
            f"Candidate {candidate_id!r} has an invalid active-mask shape.\n"
            f"Expected: "
            f"({NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT},)\n"
            f"Observed: {active_mask_array.shape}"
        )


    if not active_mask_array.any():
        raise RuntimeError(
            f"Candidate {candidate_id!r} has no active coefficients."
        )


# ======================================================================================
# 8. VERIFY PENALTY-GROUP MEMBERS AND WEIGHTS
# ======================================================================================

for group_id, raw_member_indices in (
    PENALTY_GROUP_MEMBERS.items()
):
    member_indices = np.asarray(
        raw_member_indices,
        dtype=np.int64,
    ).reshape(
        -1
    )


    if member_indices.size == 0:
        raise RuntimeError(
            f"Penalty group {group_id!r} contains no coefficients."
        )


    if (
        member_indices
        <
        0
    ).any():
        raise RuntimeError(
            f"Penalty group {group_id!r} contains a negative index."
        )


    if (
        member_indices
        >=
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT
    ).any():
        raise RuntimeError(
            f"Penalty group {group_id!r} contains an index outside the "
            "reduced coefficient vector."
        )


    if group_id not in PENALTY_GROUP_WEIGHTS:
        raise RuntimeError(
            f"Penalty group {group_id!r} has no registered weight."
        )


    group_weight = float(
        PENALTY_GROUP_WEIGHTS[
            group_id
        ]
    )


    if not np.isfinite(
        group_weight
    ):
        raise RuntimeError(
            f"Penalty group {group_id!r} has a nonfinite weight."
        )


    if group_weight < 0.0:
        raise RuntimeError(
            f"Penalty group {group_id!r} has a negative weight."
        )


for candidate_id, candidate_group_ids in (
    CANDIDATE_PENALTY_GROUP_IDS.items()
):
    for group_id in candidate_group_ids:
        if group_id not in PENALTY_GROUP_MEMBERS:
            raise RuntimeError(
                f"Candidate {candidate_id!r} references an unknown "
                f"penalty group: {group_id!r}."
            )


# ======================================================================================
# 9. VERIFY PHYSICAL AND LOSS CONFIGURATION
# ======================================================================================

if not (
    0.0
    <
    float(
        R_MIN_OHM
    )
    <
    float(
        R_MAX_OHM
    )
):
    raise ValueError(
        "The resistance bounds are invalid."
    )


if not (
    0.0
    <
    float(
        TAU1_MIN_S
    )
    <
    float(
        TAU1_MAX_S
    )
):
    raise ValueError(
        "The tau1 bounds are invalid."
    )


if not (
    0.0
    <
    float(
        TAU_GAP_MIN_S
    )
    <
    float(
        TAU_GAP_MAX_S
    )
):
    raise ValueError(
        "The time-constant-gap bounds are invalid."
    )


if int(
    INITIAL_REST_SAMPLES
) < 1:
    raise ValueError(
        "INITIAL_REST_SAMPLES must be at least one."
    )


if (
    not np.isfinite(
        float(
            REST_CURRENT_A
        )
    )
    or
    float(
        REST_CURRENT_A
    )
    <
    0.0
):
    raise ValueError(
        "REST_CURRENT_A must be finite and nonnegative."
    )


if (
    not np.isfinite(
        float(
            PSEUDO_HUBER_DELTA_V
        )
    )
    or
    float(
        PSEUDO_HUBER_DELTA_V
    )
    <=
    0.0
):
    raise ValueError(
        "PSEUDO_HUBER_DELTA_V must be finite and positive."
    )


if (
    not np.isfinite(
        float(
            RIDGE_WEIGHT
        )
    )
    or
    float(
        RIDGE_WEIGHT
    )
    <
    0.0
):
    raise ValueError(
        "RIDGE_WEIGHT must be finite and nonnegative."
    )


sparse_lambda_values = np.asarray(
    SPARSE_LAMBDA_GRID,
    dtype=np.float64,
).reshape(
    -1
)


if sparse_lambda_values.size == 0:
    raise RuntimeError(
        "SPARSE_LAMBDA_GRID is empty."
    )


if not np.isfinite(
    sparse_lambda_values
).all():
    raise ValueError(
        "SPARSE_LAMBDA_GRID contains nonfinite values."
    )


if (
    sparse_lambda_values
    <=
    0.0
).any():
    raise ValueError(
        "Every sparse regularization value must be positive."
    )


# ======================================================================================
# 10. DEFINE PHYSICAL LOG BOUNDS
# ======================================================================================

LOG_R_MIN: float = float(
    np.log(
        float(
            R_MIN_OHM
        )
    )
)

LOG_R_MAX: float = float(
    np.log(
        float(
            R_MAX_OHM
        )
    )
)

LOG_TAU1_MIN: float = float(
    np.log(
        float(
            TAU1_MIN_S
        )
    )
)

LOG_TAU1_MAX: float = float(
    np.log(
        float(
            TAU1_MAX_S
        )
    )
)

LOG_TAU_GAP_MIN: float = float(
    np.log(
        float(
            TAU_GAP_MIN_S
        )
    )
)

LOG_TAU_GAP_MAX: float = float(
    np.log(
        float(
            TAU_GAP_MAX_S
        )
    )
)


PHYSICAL_BARRIER_WEIGHT: float = float(
    globals().get(
        "PHYSICAL_BARRIER_WEIGHT",
        1.0e-3,
    )
)


if (
    not np.isfinite(
        PHYSICAL_BARRIER_WEIGHT
    )
    or
    PHYSICAL_BARRIER_WEIGHT
    <
    0.0
):
    raise ValueError(
        "PHYSICAL_BARRIER_WEIGHT must be finite and nonnegative."
    )


# ======================================================================================
# 11. DEFINE THE EXACT GROUP NORM WITH A ZERO SUBGRADIENT
# ======================================================================================

@jax.custom_jvp
def group_l2_norm_zero_subgradient(
    group_coefficients: jax.Array,
) -> jax.Array:
    """
    Return the exact Euclidean group norm.

    At a nonzero group, the derivative is the ordinary Euclidean-norm
    derivative.

    At a zero group, the Euclidean norm is nondifferentiable. Zero is a valid
    member of its subdifferential, so this function explicitly selects the
    zero subgradient.
    """

    return jnp.sqrt(
        jnp.sum(
            jnp.square(
                group_coefficients
            )
        )
    )


@group_l2_norm_zero_subgradient.defjvp
def group_l2_norm_zero_subgradient_jvp(
    primals: tuple[jax.Array],
    tangents: tuple[jax.Array],
) -> tuple[
    jax.Array,
    jax.Array,
]:
    """
    Define the directional derivative of the group norm.
    """

    (
        group_coefficients,
    ) = primals

    (
        group_coefficient_tangent,
    ) = tangents


    group_norm = group_l2_norm_zero_subgradient(
        group_coefficients
    )


    safe_group_norm = jnp.where(
        group_norm
        >
        0.0,
        group_norm,
        1.0,
    )


    nonzero_group_directional_derivative = (
        jnp.sum(
            group_coefficients
            *
            group_coefficient_tangent
        )
        /
        safe_group_norm
    )


    group_norm_tangent = jnp.where(
        group_norm
        >
        0.0,
        nonzero_group_directional_derivative,
        0.0,
    )


    return (
        group_norm,
        group_norm_tangent,
    )


# Verify the selected zero subgradient directly.

group_norm_test_vector = jnp.zeros(
    7,
    dtype=jnp.float64,
)


group_norm_test_value = float(
    np.asarray(
        group_l2_norm_zero_subgradient(
            group_norm_test_vector
        )
    )
)


group_norm_test_gradient = np.asarray(
    jax.grad(
        group_l2_norm_zero_subgradient
    )(
        group_norm_test_vector
    ),
    dtype=np.float64,
)


if group_norm_test_value != 0.0:
    raise RuntimeError(
        "The exact group norm is not zero at the zero vector."
    )


if not np.isfinite(
    group_norm_test_gradient
).all():
    raise RuntimeError(
        "The selected group-norm zero subgradient is nonfinite."
    )


if not np.allclose(
    group_norm_test_gradient,
    0.0,
    rtol=0.0,
    atol=0.0,
):
    raise RuntimeError(
        "The selected group-norm subgradient is not zero at the origin."
    )


# ======================================================================================
# 12. DEFINE BOUNDED PHYSICAL PARAMETER TRANSFORMS
# ======================================================================================

def bounded_exponential(
    transformed_values: jax.Array,
    lower_log_bound: float,
    upper_log_bound: float,
) -> jax.Array:
    """
    Exponentiate transformed values within the frozen physical range.
    """

    return jnp.exp(
        jnp.clip(
            transformed_values,
            lower_log_bound,
            upper_log_bound,
        )
    )


# ======================================================================================
# 13. DEFINE THE BATCHED TWO-RC LPV SIMULATOR
# ======================================================================================

def simulate_lpv_batch_core(
    reduced_coefficients: jax.Array,
    batch: Mapping[
        str,
        jax.Array,
    ],
) -> tuple[jax.Array, ...]:
    """
    Simulate a padded batch of complete two-RC LPV trajectories.
    """

    full_coefficients = (
        JAX_IDENTIFIABLE_EXPANSION_MATRIX
        @
        reduced_coefficients
    )


    safe_cell_indices = jnp.maximum(
        batch[
            "cell_indices"
        ],
        0,
    )


    cell_effect_multiplier = (
        batch[
            "has_cell_effect"
        ][
            :,
            None,
            None,
        ]
        .astype(
            jnp.float64
        )
    )


    effective_parameter_basis_coefficients = (
        full_coefficients[
            batch[
                "global_indices"
            ]
        ]
        +
        full_coefficients[
            batch[
                "chemistry_indices"
            ]
        ]
        +
        cell_effect_multiplier
        *
        full_coefficients[
            safe_cell_indices
        ]
    )


    transformed_parameters = jnp.einsum(
        "btk,bpk->btp",
        batch[
            "basis"
        ],
        effective_parameter_basis_coefficients,
    )


    eta_R0 = transformed_parameters[
        :,
        :,
        0,
    ]

    eta_R1 = transformed_parameters[
        :,
        :,
        1,
    ]

    eta_R2 = transformed_parameters[
        :,
        :,
        2,
    ]

    eta_tau1 = transformed_parameters[
        :,
        :,
        3,
    ]

    eta_tau_gap = transformed_parameters[
        :,
        :,
        4,
    ]


    R0_ohm = bounded_exponential(
        eta_R0,
        LOG_R_MIN,
        LOG_R_MAX,
    )

    R1_ohm = bounded_exponential(
        eta_R1,
        LOG_R_MIN,
        LOG_R_MAX,
    )

    R2_ohm = bounded_exponential(
        eta_R2,
        LOG_R_MIN,
        LOG_R_MAX,
    )

    tau1_s = bounded_exponential(
        eta_tau1,
        LOG_TAU1_MIN,
        LOG_TAU1_MAX,
    )

    tau_gap_s = bounded_exponential(
        eta_tau_gap,
        LOG_TAU_GAP_MIN,
        LOG_TAU_GAP_MAX,
    )

    tau2_s = (
        tau1_s
        +
        tau_gap_s
    )


    rest_window_length = min(
        int(
            INITIAL_REST_SAMPLES
        ),
        int(
            batch[
                "current"
            ].shape[
                1
            ]
        ),
    )


    rest_start_detected = jnp.all(
        jnp.abs(
            batch[
                "current"
            ][
                :,
                :
                rest_window_length,
            ]
        )
        <=
        float(
            REST_CURRENT_A
        ),
        axis=1,
    )


    initial_total_polarization_v = (
        batch[
            "ocv"
        ][
            :,
            0,
        ]
        -
        batch[
            "voltage"
        ][
            :,
            0,
        ]
        -
        R0_ohm[
            :,
            0,
        ]
        *
        batch[
            "current"
        ][
            :,
            0,
        ]
    )


    initial_branch_resistance_sum = (
        R1_ohm[
            :,
            0,
        ]
        +
        R2_ohm[
            :,
            0,
        ]
    )


    v1_initial = jnp.where(
        rest_start_detected,
        0.0,
        initial_total_polarization_v
        *
        R1_ohm[
            :,
            0,
        ]
        /
        initial_branch_resistance_sum,
    )


    v2_initial = jnp.where(
        rest_start_detected,
        0.0,
        initial_total_polarization_v
        *
        R2_ohm[
            :,
            0,
        ]
        /
        initial_branch_resistance_sum,
    )


    alpha1 = jnp.exp(
        -
        batch[
            "delta_t"
        ]
        /
        tau1_s
    )


    alpha2 = jnp.exp(
        -
        batch[
            "delta_t"
        ]
        /
        tau2_s
    )


    one_minus_alpha1 = (
        -
        jnp.expm1(
            -
            batch[
                "delta_t"
            ]
            /
            tau1_s
        )
    )


    one_minus_alpha2 = (
        -
        jnp.expm1(
            -
            batch[
                "delta_t"
            ]
            /
            tau2_s
        )
    )


    transition_mask = batch[
        "valid_mask"
    ][
        :,
        1:,
    ]


    time_step_count = int(
        batch[
            "basis"
        ].shape[
            1
        ]
        -
        1
    )


    def recursion_step(
        carry: tuple[
            jax.Array,
            jax.Array,
        ],
        time_index: jax.Array,
    ) -> tuple[
        tuple[
            jax.Array,
            jax.Array,
        ],
        tuple[
            jax.Array,
            jax.Array,
        ],
    ]:
        v1_current, v2_current = carry


        active_transition = transition_mask[
            :,
            time_index,
        ]


        v1_candidate = (
            alpha1[
                :,
                time_index,
            ]
            *
            v1_current
            +
            R1_ohm[
                :,
                time_index,
            ]
            *
            one_minus_alpha1[
                :,
                time_index,
            ]
            *
            batch[
                "current"
            ][
                :,
                time_index,
            ]
        )


        v2_candidate = (
            alpha2[
                :,
                time_index,
            ]
            *
            v2_current
            +
            R2_ohm[
                :,
                time_index,
            ]
            *
            one_minus_alpha2[
                :,
                time_index,
            ]
            *
            batch[
                "current"
            ][
                :,
                time_index,
            ]
        )


        v1_next = jnp.where(
            active_transition,
            v1_candidate,
            v1_current,
        )


        v2_next = jnp.where(
            active_transition,
            v2_candidate,
            v2_current,
        )


        next_carry = (
            v1_next,
            v2_next,
        )


        return (
            next_carry,
            next_carry,
        )


    (
        _,
        scanned_state_sequences,
    ) = jax.lax.scan(
        recursion_step,
        (
            v1_initial,
            v2_initial,
        ),
        jnp.arange(
            time_step_count,
            dtype=jnp.int32,
        ),
    )


    scanned_v1_sequence, scanned_v2_sequence = (
        scanned_state_sequences
    )


    v1_state_v = jnp.concatenate(
        [
            v1_initial[
                None,
                :,
            ],
            scanned_v1_sequence,
        ],
        axis=0,
    ).T


    v2_state_v = jnp.concatenate(
        [
            v2_initial[
                None,
                :,
            ],
            scanned_v2_sequence,
        ],
        axis=0,
    ).T


    predicted_voltage_v = (
        batch[
            "ocv"
        ]
        -
        R0_ohm
        *
        batch[
            "current"
        ]
        -
        v1_state_v
        -
        v2_state_v
    )


    residual_v = (
        batch[
            "voltage"
        ]
        -
        predicted_voltage_v
    )


    return (
        predicted_voltage_v,
        residual_v,
        transformed_parameters,
        R0_ohm,
        R1_ohm,
        R2_ohm,
        tau1_s,
        tau2_s,
        alpha1,
        alpha2,
        v1_state_v,
        v2_state_v,
        full_coefficients,
        effective_parameter_basis_coefficients,
        rest_start_detected,
        initial_total_polarization_v,
    )


simulate_lpv_batch_jax = jax.jit(
    simulate_lpv_batch_core
)


# ======================================================================================
# 14. DEFINE THE MASKED PSEUDO-HUBER DATA LOSS
# ======================================================================================

def masked_pseudo_huber_mean(
    residual_v: jax.Array,
    loss_mask: jax.Array,
    delta_v: float,
) -> jax.Array:
    """
    Return the masked mean pseudo-Huber voltage loss.
    """

    delta = jnp.asarray(
        delta_v,
        dtype=jnp.float64,
    )


    scaled_residual = (
        residual_v
        /
        delta
    )


    pseudo_huber_values = (
        jnp.square(
            delta
        )
        *
        (
            jnp.sqrt(
                1.0
                +
                jnp.square(
                    scaled_residual
                )
            )
            -
            1.0
        )
    )


    floating_mask = loss_mask.astype(
        jnp.float64
    )


    denominator = jnp.maximum(
        jnp.sum(
            floating_mask
        ),
        1.0,
    )


    return (
        jnp.sum(
            pseudo_huber_values
            *
            floating_mask
        )
        /
        denominator
    )


# ======================================================================================
# 15. DEFINE THE PHYSICAL-RANGE PENALTY
# ======================================================================================

def physical_parameter_barrier(
    transformed_parameters: jax.Array,
    loss_mask: jax.Array,
) -> jax.Array:
    """
    Penalize transformed parameters outside the frozen physical ranges.
    """

    parameter_lower_bounds = jnp.asarray(
        [
            LOG_R_MIN,
            LOG_R_MIN,
            LOG_R_MIN,
            LOG_TAU1_MIN,
            LOG_TAU_GAP_MIN,
        ],
        dtype=jnp.float64,
    )


    parameter_upper_bounds = jnp.asarray(
        [
            LOG_R_MAX,
            LOG_R_MAX,
            LOG_R_MAX,
            LOG_TAU1_MAX,
            LOG_TAU_GAP_MAX,
        ],
        dtype=jnp.float64,
    )


    lower_violation = jnp.maximum(
        parameter_lower_bounds[
            None,
            None,
            :,
        ]
        -
        transformed_parameters,
        0.0,
    )


    upper_violation = jnp.maximum(
        transformed_parameters
        -
        parameter_upper_bounds[
            None,
            None,
            :,
        ],
        0.0,
    )


    parameter_loss_mask = (
        loss_mask[
            :,
            :,
            None,
        ]
        .astype(
            jnp.float64
        )
    )


    parameter_count = int(
        transformed_parameters.shape[
            -1
        ]
    )


    denominator = jnp.maximum(
        jnp.sum(
            parameter_loss_mask
        )
        *
        float(
            parameter_count
        ),
        1.0,
    )


    return (
        jnp.sum(
            (
                jnp.square(
                    lower_violation
                )
                +
                jnp.square(
                    upper_violation
                )
            )
            *
            parameter_loss_mask
        )
        /
        denominator
    )


# ======================================================================================
# 16. DEFINE THE CANDIDATE OBJECTIVE FACTORY
# ======================================================================================

def make_candidate_objective(
    *,
    candidate_id: str,
    regularization_lambda: float,
):
    """
    Build a compiled value-and-gradient objective for one candidate family.
    """

    normalized_candidate_id = str(
        candidate_id
    )


    if normalized_candidate_id not in (
        CANDIDATE_ACTIVE_MASKS
    ):
        raise KeyError(
            f"Unknown candidate ID: {normalized_candidate_id!r}"
        )


    if normalized_candidate_id not in (
        CANDIDATE_PENALTY_GROUP_IDS
    ):
        raise KeyError(
            "No candidate penalty-group definition exists for:\n"
            f"  {normalized_candidate_id}"
        )


    active_mask_numpy = np.asarray(
        CANDIDATE_ACTIVE_MASKS[
            normalized_candidate_id
        ],
        dtype=np.float64,
    )


    if active_mask_numpy.shape != (
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
    ):
        raise RuntimeError(
            f"Candidate {normalized_candidate_id!r} has an invalid "
            "active-mask shape."
        )


    active_mask = jnp.asarray(
        active_mask_numpy,
        dtype=jnp.float64,
    )


    candidate_group_ids = tuple(
        str(
            group_id
        )
        for group_id in (
            CANDIDATE_PENALTY_GROUP_IDS[
                normalized_candidate_id
            ]
        )
    )


    penalty_group_definitions = tuple(
        (
            group_id,

            jnp.asarray(
                PENALTY_GROUP_MEMBERS[
                    group_id
                ],
                dtype=jnp.int32,
            ),

            float(
                PENALTY_GROUP_WEIGHTS[
                    group_id
                ]
            ),
        )
        for group_id in candidate_group_ids
    )


    regularization_lambda = float(
        regularization_lambda
    )


    if not np.isfinite(
        regularization_lambda
    ):
        raise ValueError(
            "regularization_lambda must be finite."
        )


    if regularization_lambda < 0.0:
        raise ValueError(
            "regularization_lambda must be nonnegative."
        )


    def candidate_objective(
        unconstrained_reduced_coefficients: jax.Array,
        batch: Mapping[
            str,
            jax.Array,
        ],
    ) -> tuple[
        jax.Array,
        tuple[
            jax.Array,
            jax.Array,
            jax.Array,
            jax.Array,
        ],
    ]:
        active_reduced_coefficients = (
            unconstrained_reduced_coefficients
            *
            active_mask
        )


        simulation_outputs = simulate_lpv_batch_core(
            active_reduced_coefficients,
            batch,
        )


        residual_v = simulation_outputs[
            1
        ]


        transformed_parameters = simulation_outputs[
            2
        ]


        data_loss = masked_pseudo_huber_mean(
            residual_v,
            batch[
                "loss_mask"
            ],
            float(
                PSEUDO_HUBER_DELTA_V
            ),
        )


        ridge_penalty = (
            float(
                RIDGE_WEIGHT
            )
            *
            jnp.mean(
                jnp.square(
                    active_reduced_coefficients
                )
            )
        )


        group_penalty = jnp.asarray(
            0.0,
            dtype=jnp.float64,
        )


        for (
            _,
            group_member_indices,
            group_weight,
        ) in penalty_group_definitions:
            group_coefficients = (
                active_reduced_coefficients[
                    group_member_indices
                ]
            )


            group_penalty = (
                group_penalty
                +
                float(
                    group_weight
                )
                *
                group_l2_norm_zero_subgradient(
                    group_coefficients
                )
            )


        physical_penalty = (
            float(
                PHYSICAL_BARRIER_WEIGHT
            )
            *
            physical_parameter_barrier(
                transformed_parameters,
                batch[
                    "loss_mask"
                ],
            )
        )


        total_loss = (
            data_loss
            +
            ridge_penalty
            +
            regularization_lambda
            *
            group_penalty
            +
            physical_penalty
        )


        auxiliary_losses = (
            data_loss,
            ridge_penalty,
            group_penalty,
            physical_penalty,
        )


        return (
            total_loss,
            auxiliary_losses,
        )


    return jax.jit(
        jax.value_and_grad(
            candidate_objective,
            argnums=0,
            has_aux=True,
        )
    )


# ======================================================================================
# 17. SELECT THE SMOKE-TEST CANDIDATE
# ======================================================================================

preferred_smoke_test_candidate = (
    "M4_HIERARCHICAL_SPARSE_LPV"
)


if preferred_smoke_test_candidate in (
    CANDIDATE_ACTIVE_MASKS
):
    SMOKE_TEST_CANDIDATE: str = (
        preferred_smoke_test_candidate
    )

else:
    registered_candidate_ids = tuple(
        str(
            candidate_id
        )
        for candidate_id in (
            CANDIDATE_ACTIVE_MASKS.keys()
        )
    )


    if len(
        registered_candidate_ids
    ) == 0:
        raise RuntimeError(
            "No model candidate is available for smoke testing."
        )


    SMOKE_TEST_CANDIDATE = (
        registered_candidate_ids[
            -1
        ]
    )


smoke_test_lambda: float = float(
    sparse_lambda_values[
        0
    ]
)


# ======================================================================================
# 18. BUILD THE SMOKE-TEST OBJECTIVE
# ======================================================================================

smoke_objective = make_candidate_objective(
    candidate_id=(
        SMOKE_TEST_CANDIDATE
    ),
    regularization_lambda=(
        smoke_test_lambda
    ),
)


smoke_jax_batch = batch_to_jax(
    smoke_batch
)


# ======================================================================================
# 19. RUN THE OBJECTIVE AND GRADIENT SMOKE TEST
# ======================================================================================

(
    (
        smoke_total_loss,
        smoke_auxiliary_losses,
    ),
    smoke_gradient,
) = smoke_objective(
    jnp.asarray(
        zero_reduced_coefficients,
        dtype=jnp.float64,
    ),
    smoke_jax_batch,
)


(
    smoke_data_loss,
    smoke_ridge_penalty,
    smoke_group_penalty,
    smoke_physical_penalty,
) = smoke_auxiliary_losses


smoke_total_loss_value = float(
    np.asarray(
        smoke_total_loss
    )
)

smoke_data_loss_value = float(
    np.asarray(
        smoke_data_loss
    )
)

smoke_ridge_penalty_value = float(
    np.asarray(
        smoke_ridge_penalty
    )
)

smoke_group_penalty_value = float(
    np.asarray(
        smoke_group_penalty
    )
)

smoke_physical_penalty_value = float(
    np.asarray(
        smoke_physical_penalty
    )
)


smoke_gradient_numpy = np.asarray(
    smoke_gradient,
    dtype=np.float64,
)


if smoke_gradient_numpy.shape != (
    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
):
    raise RuntimeError(
        "The smoke-test gradient has an incorrect shape.\n"
        f"Expected: "
        f"({NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT},)\n"
        f"Observed: {smoke_gradient_numpy.shape}"
    )


smoke_nonfinite_gradient_mask = (
    ~np.isfinite(
        smoke_gradient_numpy
    )
)


smoke_nonfinite_gradient_count = int(
    smoke_nonfinite_gradient_mask.sum()
)


if smoke_nonfinite_gradient_count > 0:
    nonfinite_gradient_positions = np.flatnonzero(
        smoke_nonfinite_gradient_mask
    )


    raise RuntimeError(
        "The corrected smoke-test gradient still contains nonfinite values.\n"
        f"Nonfinite count: {smoke_nonfinite_gradient_count:,}\n"
        f"First positions: "
        f"{nonfinite_gradient_positions[:20].tolist()}"
    )


smoke_gradient_norm = float(
    np.linalg.norm(
        smoke_gradient_numpy
    )
)


for value_name, value in (
    (
        "total loss",
        smoke_total_loss_value,
    ),
    (
        "data loss",
        smoke_data_loss_value,
    ),
    (
        "ridge penalty",
        smoke_ridge_penalty_value,
    ),
    (
        "group penalty",
        smoke_group_penalty_value,
    ),
    (
        "physical penalty",
        smoke_physical_penalty_value,
    ),
    (
        "gradient norm",
        smoke_gradient_norm,
    ),
):
    if not np.isfinite(
        value
    ):
        raise RuntimeError(
            f"The smoke-test {value_name} is nonfinite."
        )


if smoke_group_penalty_value != 0.0:
    raise RuntimeError(
        "The group penalty must equal zero at zero coefficient "
        "initialization."
    )


# Preserve the original object structure for subsequent fitting cells.

smoke_value = (
    smoke_total_loss,
    smoke_auxiliary_losses,
)


# ======================================================================================
# 20. BUILD THE GRADIENT DIAGNOSTIC TABLE
# ======================================================================================

active_smoke_mask = np.asarray(
    CANDIDATE_ACTIVE_MASKS[
        SMOKE_TEST_CANDIDATE
    ],
    dtype=bool,
)


NOTEBOOK_12_SMOKE_GRADIENT_DIAGNOSTIC_DF = pd.DataFrame(
    {
        "reduced_coefficient_index": np.arange(
            NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
            dtype=np.int64,
        ),

        "active_for_smoke_candidate": (
            active_smoke_mask
        ),

        "gradient": (
            smoke_gradient_numpy
        ),

        "absolute_gradient": np.abs(
            smoke_gradient_numpy
        ),

        "finite": np.isfinite(
            smoke_gradient_numpy
        ),
    }
)


active_gradient_count = int(
    active_smoke_mask.sum()
)


inactive_gradient_count = int(
    (
        ~active_smoke_mask
    ).sum()
)


finite_gradient_count = int(
    NOTEBOOK_12_SMOKE_GRADIENT_DIAGNOSTIC_DF[
        "finite"
    ].sum()
)


# ======================================================================================
# 21. FINAL EXECUTION SUMMARY
# ======================================================================================

print("=" * 116)
print("BATCHED RECURSIVE OBJECTIVE AND HIERARCHICAL GROUP PENALTY")
print("=" * 116)

print(
    f"Full coefficient count          : "
    f"{NOTEBOOK_12_FULL_COEFFICIENT_COUNT:,}"
)

print(
    f"Reduced coefficient count       : "
    f"{NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT:,}"
)

print(
    f"Expansion matrix shape          : "
    f"{expansion_matrix_shape}"
)

print(
    f"Smoke-test candidate            : "
    f"{SMOKE_TEST_CANDIDATE}"
)

print(
    f"Active smoke-test coefficients  : "
    f"{active_gradient_count:,}"
)

print(
    f"Inactive smoke-test coefficients: "
    f"{inactive_gradient_count:,}"
)

print(
    f"Smoke-test lambda               : "
    f"{smoke_test_lambda:.8e}"
)

print(
    f"Smoke-test total loss           : "
    f"{smoke_total_loss_value:.8e}"
)

print(
    f"Smoke-test data loss            : "
    f"{smoke_data_loss_value:.8e}"
)

print(
    f"Smoke-test ridge penalty        : "
    f"{smoke_ridge_penalty_value:.8e}"
)

print(
    f"Smoke-test group penalty        : "
    f"{smoke_group_penalty_value:.8e}"
)

print(
    f"Smoke-test physical penalty     : "
    f"{smoke_physical_penalty_value:.8e}"
)

print(
    f"Finite gradient coordinates     : "
    f"{finite_gradient_count:,} / "
    f"{NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT:,}"
)

print(
    f"Nonfinite gradient coordinates  : "
    f"{smoke_nonfinite_gradient_count:,}"
)

print(
    f"Gradient norm                   : "
    f"{smoke_gradient_norm:.8e}"
)

print(
    f"Group norm at zero              : "
    f"{group_norm_test_value:.8e}"
)

print(
    f"Group-norm zero-gradient norm   : "
    f"{float(np.linalg.norm(group_norm_test_gradient)):.8e}"
)

print(
    f"JAX float64 enabled             : "
    f"{bool(jax.config.read('jax_enable_x64'))}"
)

print("-" * 116)

print(
    "The exact group-Lasso penalty now uses a finite zero subgradient, "
    "and the batched recursive objective was validated successfully."
)

print("=" * 116)

BATCHED RECURSIVE OBJECTIVE AND HIERARCHICAL GROUP PENALTY
Full coefficient count          : 360
Reduced coefficient count       : 240
Expansion matrix shape          : (360, 240)
Smoke-test candidate            : M4_HIERARCHICAL_SPARSE_LPV
Active smoke-test coefficients  : 240
Inactive smoke-test coefficients: 0
Smoke-test lambda               : 1.00000000e-06
Smoke-test total loss           : 1.45162227e-03
Smoke-test data loss            : 1.45162227e-03
Smoke-test ridge penalty        : 0.00000000e+00
Smoke-test group penalty        : 0.00000000e+00
Smoke-test physical penalty     : 0.00000000e+00
Finite gradient coordinates     : 240 / 240
Nonfinite gradient coordinates  : 0
Gradient norm                   : 0.00000000e+00
Group norm at zero              : 0.00000000e+00
Group-norm zero-gradient norm   : 0.00000000e+00
JAX float64 enabled             : True
--------------------------------------------------------------------------------------------------------------------
The exac

In [21]:
# ======================================================================================
# CELL 20 — INITIALIZE THE LPV MODEL FROM THE FROZEN FIXED TWO-RC BASELINE
# ======================================================================================

fixed_parameter_df = pd.read_parquet(
    FIXED_ECM_FROZEN_PARAMETERS_PATH,
    engine="pyarrow",
)

fixed_scope_column = resolve_column(
    fixed_parameter_df.columns,
    ("parameter_scope", "scope"),
    semantic_name="fixed-ECM parameter scope",
)

fixed_order_column = resolve_column(
    fixed_parameter_df.columns,
    ("model_order", "order"),
    semantic_name="fixed-ECM model order",
)

fixed_parameter_columns = {
    "R0_ohm": resolve_column(
        fixed_parameter_df.columns,
        ("R0_ohm", "r0_ohm", "R0"),
        semantic_name="fixed R0",
    ),
    "R1_ohm": resolve_column(
        fixed_parameter_df.columns,
        ("R1_ohm", "r1_ohm", "R1"),
        semantic_name="fixed R1",
    ),
    "R2_ohm": resolve_column(
        fixed_parameter_df.columns,
        ("R2_ohm", "r2_ohm", "R2"),
        semantic_name="fixed R2",
    ),
    "tau1_s": resolve_column(
        fixed_parameter_df.columns,
        ("tau1_s", "tau_1_s", "tau1"),
        semantic_name="fixed tau1",
    ),
    "tau2_s": resolve_column(
        fixed_parameter_df.columns,
        ("tau2_s", "tau_2_s", "tau2"),
        semantic_name="fixed tau2",
    ),
}

two_rc_df = fixed_parameter_df.loc[
    pd.to_numeric(
        fixed_parameter_df[
            fixed_order_column
        ],
        errors="coerce",
    ).eq(
        2
    )
].copy()

if two_rc_df.empty:
    raise RuntimeError(
        "Notebook 11 contains no frozen two-RC parameter row."
    )

global_two_rc_df = two_rc_df.loc[
    two_rc_df[
        fixed_scope_column
    ].astype(str).str.upper().eq(
        "GLOBAL"
    )
]

if not global_two_rc_df.empty:
    initialization_source_df = global_two_rc_df
    initialization_source = "GLOBAL_2RC"
else:
    initialization_source_df = two_rc_df
    initialization_source = "MEDIAN_ACROSS_FROZEN_2RC_SCOPES"

initial_physical_parameters = {
    parameter_name: float(
        pd.to_numeric(
            initialization_source_df[
                source_column
            ],
            errors="coerce",
        ).median()
    )
    for parameter_name, source_column in (
        fixed_parameter_columns.items()
    )
}

if not all(
    np.isfinite(
        value
    )
    and
    value
    >
    0.0
    for value in (
        initial_physical_parameters.values()
    )
):
    raise RuntimeError(
        "The frozen two-RC initialization contains a nonpositive "
        "or nonfinite physical parameter."
    )

if initial_physical_parameters[
    "tau2_s"
] <= initial_physical_parameters[
    "tau1_s"
]:
    raise RuntimeError(
        "The fixed two-RC initialization does not satisfy tau2 > tau1."
    )

initial_transformed_parameters = {
    "eta_R0": float(
        np.log(
            initial_physical_parameters[
                "R0_ohm"
            ]
        )
    ),
    "eta_R1": float(
        np.log(
            initial_physical_parameters[
                "R1_ohm"
            ]
        )
    ),
    "eta_R2": float(
        np.log(
            initial_physical_parameters[
                "R2_ohm"
            ]
        )
    ),
    "eta_tau1": float(
        np.log(
            initial_physical_parameters[
                "tau1_s"
            ]
        )
    ),
    "eta_tau_gap": float(
        np.log(
            initial_physical_parameters[
                "tau2_s"
            ]
            -
            initial_physical_parameters[
                "tau1_s"
            ]
        )
    ),
}

initial_full_coefficients = np.zeros(
    NOTEBOOK_12_FULL_COEFFICIENT_COUNT,
    dtype=np.float64,
)

intercept_basis_id = str(
    NOTEBOOK_12_BASIS_TERM_DF.loc[
        NOTEBOOK_12_BASIS_TERM_DF[
            "is_intercept"
        ].fillna(False).astype(bool),
        "basis_id",
    ].iloc[
        0
    ]
)

for transformed_parameter, initial_value in (
    initial_transformed_parameters.items()
):
    full_index = FULL_COEFFICIENT_INDEX_LOOKUP[
        (
            transformed_parameter,
            "GLOBAL",
            "GLOBAL",
            intercept_basis_id,
        )
    ]

    initial_full_coefficients[
        full_index
    ] = initial_value

INITIAL_REDUCED_COEFFICIENTS = np.asarray(
    NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX.T
    @
    initial_full_coefficients,
    dtype=np.float64,
)

if INITIAL_REDUCED_COEFFICIENTS.shape != (
    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
):
    raise RuntimeError(
        "The reduced initialization has an incorrect shape."
    )

initial_roundtrip_full = np.asarray(
    NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX
    @
    INITIAL_REDUCED_COEFFICIENTS,
    dtype=np.float64,
)

initial_roundtrip_error = float(
    np.max(
        np.abs(
            initial_roundtrip_full
            -
            initial_full_coefficients
        )
    )
)

if initial_roundtrip_error > 1.0e-12:
    raise RuntimeError(
        "The fixed-baseline initialization is not in the identifiable subspace."
    )

NOTEBOOK_12_INITIALIZATION_PATH = (
    NOTEBOOK_12_MODEL_DIR
    /
    "lpv_initial_reduced_coefficients.npz"
)

np.savez_compressed(
    NOTEBOOK_12_INITIALIZATION_PATH,
    reduced_coefficients=(
        INITIAL_REDUCED_COEFFICIENTS
    ),
    full_coefficients=(
        initial_full_coefficients
    ),
)

NOTEBOOK_12_INITIALIZATION_CONTRACT_PATH = (
    NOTEBOOK_12_CONTRACT_DIR
    /
    "notebook12_initialization_contract.json"
)

write_json_atomic(
    NOTEBOOK_12_INITIALIZATION_CONTRACT_PATH,
    {
        "created_utc": utc_now_iso(),
        "source": initialization_source,
        "source_path": str(
            FIXED_ECM_FROZEN_PARAMETERS_PATH
        ),
        "physical_parameters": (
            initial_physical_parameters
        ),
        "transformed_parameters": (
            initial_transformed_parameters
        ),
        "reduced_coefficient_count": (
            NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT
        ),
        "roundtrip_error": (
            initial_roundtrip_error
        ),
        "test_data_used": False,
        "strict_test_data_used": False,
    },
)

print("=" * 110)
print("FIXED TWO-RC INITIALIZATION")
print("=" * 110)
print(f"Initialization source : {initialization_source}")
for parameter_name, value in initial_physical_parameters.items():
    print(
        f"{parameter_name:22s}: {value:.8g}"
    )
print(
    f"Reduced dimension     : "
    f"{NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT:,}"
)
print(
    f"Roundtrip error       : "
    f"{initial_roundtrip_error:.3e}"
)
print("=" * 110)


FIXED TWO-RC INITIALIZATION
Initialization source : GLOBAL_2RC
R0_ohm                : 1e-05
R1_ohm                : 1.0000007e-05
R2_ohm                : 1.0000045e-05
tau1_s                : 19.770562
tau2_s                : 19.929958
Reduced dimension     : 240
Roundtrip error       : 0.000e+00


In [23]:
# ======================================================================================
# CELL 21 — FIT ALL CANDIDATE AND REGULARIZATION JOBS
# ======================================================================================
#
# Purpose
# -------
# Fit every registered model candidate and regularization job using training
# trajectories only.
#
#
# 1. converting JAX outputs with np.array(..., copy=True);
# 2. never applying in-place arithmetic to JAX-derived NumPy arrays;
# 3. applying candidate masks through newly allocated arrays;
# 4. maintaining writable Adam parameter and moment arrays;
# 5. validating every loss, gradient, and coefficient update;
# 6. preserving deterministic trajectory batching and job initialization.
#
# Calibration, ordinary-test, strict-test, and embargo trajectories are not
# used for parameter fitting in this cell.
# ======================================================================================

from __future__ import annotations

import gc
import hashlib
import json
import time

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping, Sequence

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

from IPython.display import display


# ======================================================================================
# 1. ENABLE JAX FLOAT64
# ======================================================================================

jax.config.update(
    "jax_enable_x64",
    True,
)


# ======================================================================================
# 2. VERIFY REQUIRED UPSTREAM OBJECTS
# ======================================================================================

REQUIRED_CELL_21_OBJECTS: tuple[str, ...] = (
    "NOTEBOOK_12_CANDIDATE_FIT_JOBS",
    "CANDIDATE_ACTIVE_MASKS",
    "make_candidate_objective",
    "load_padded_trajectory_batch",
    "batch_to_jax",
    "training_row_group_indices",
    "TRAINING_BATCH_TRAJECTORIES",
    "NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF",
    "NOTEBOOK_12_TABLE_DIR",
    "NOTEBOOK_12_MODEL_DIR",
)


missing_cell_21_objects = [
    object_name
    for object_name in REQUIRED_CELL_21_OBJECTS
    if object_name not in globals()
]


if missing_cell_21_objects:
    raise RuntimeError(
        "Cell 21 requires the candidate, objective, and padded-batch cells "
        "to be executed first.\n\n"
        "Missing objects:\n  - "
        + "\n  - ".join(
            missing_cell_21_objects
        )
    )


NOTEBOOK_12_TABLE_DIR = Path(
    NOTEBOOK_12_TABLE_DIR
).expanduser().resolve()

NOTEBOOK_12_MODEL_DIR = Path(
    NOTEBOOK_12_MODEL_DIR
).expanduser().resolve()


NOTEBOOK_12_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NOTEBOOK_12_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ======================================================================================
# 3. DEFINE OUTPUT PATHS
# ======================================================================================

NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "lpv_candidate_fit_summary.parquet"
)

NOTEBOOK_12_CANDIDATE_FIT_HISTORY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    / "lpv_candidate_fit_history.parquet"
)

NOTEBOOK_12_CANDIDATE_FIT_COEFFICIENT_PATH = (
    NOTEBOOK_12_MODEL_DIR
    / "lpv_candidate_fit_coefficients.npz"
)

NOTEBOOK_12_CANDIDATE_FIT_CONFIGURATION_PATH = (
    NOTEBOOK_12_MODEL_DIR
    / "lpv_candidate_fit_configuration.json"
)


# ======================================================================================
# 4. RECONSTRUCT THE AUTHORITATIVE REDUCED DIMENSION
# ======================================================================================

required_reduced_registry_columns: tuple[str, ...] = (
    "reduced_coefficient_index",
    "reduced_coefficient_id",
)


missing_reduced_registry_columns = [
    column_name
    for column_name in required_reduced_registry_columns
    if column_name not in (
        NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF.columns
    )
]


if missing_reduced_registry_columns:
    raise RuntimeError(
        "The reduced coefficient registry is missing required columns:\n  - "
        + "\n  - ".join(
            missing_reduced_registry_columns
        )
    )


NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT: int = int(
    len(
        NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF
    )
)


if NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT <= 0:
    raise RuntimeError(
        "The reduced coefficient registry is empty."
    )


reduced_coefficient_indices = np.sort(
    pd.to_numeric(
        NOTEBOOK_12_REDUCED_COEFFICIENT_REGISTRY_DF[
            "reduced_coefficient_index"
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.int64
    )
)


if not np.array_equal(
    reduced_coefficient_indices,
    np.arange(
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "The reduced coefficient indices are not contiguous from zero."
    )


# ======================================================================================
# 5. FREEZE THE TRAINING ROW-GROUP REGISTRY
# ======================================================================================

training_row_group_indices = np.asarray(
    training_row_group_indices,
    dtype=np.int32,
).reshape(
    -1
)


if training_row_group_indices.size == 0:
    raise RuntimeError(
        "No training row groups are available for candidate fitting."
    )


if (
    training_row_group_indices
    <
    0
).any():
    raise RuntimeError(
        "The training row-group registry contains a negative index."
    )


if len(
    np.unique(
        training_row_group_indices
    )
) != len(
    training_row_group_indices
):
    raise RuntimeError(
        "The training row-group registry contains duplicate indices."
    )


TRAINING_ROW_GROUP_COUNT: int = int(
    training_row_group_indices.size
)


TRAINING_BATCH_TRAJECTORIES = int(
    TRAINING_BATCH_TRAJECTORIES
)


if TRAINING_BATCH_TRAJECTORIES <= 0:
    raise ValueError(
        "TRAINING_BATCH_TRAJECTORIES must be positive."
    )


# ======================================================================================
# 6. RESOLVE FITTING CONFIGURATION
# ======================================================================================

FAST_DEMO_CELL_21: bool = bool(
    globals().get(
        "FAST_DEMO",
        False,
    )
)


FIT_RANDOM_SEED: int = int(
    globals().get(
        "RANDOM_SEED",
        20260701,
    )
)


FIT_MAX_EPOCHS: int = int(
    globals().get(
        "CANDIDATE_FIT_MAX_EPOCHS",
        globals().get(
            "TRAINING_EPOCHS",
            5
            if FAST_DEMO_CELL_21
            else 60,
        ),
    )
)


FIT_MIN_EPOCHS: int = int(
    globals().get(
        "CANDIDATE_FIT_MIN_EPOCHS",
        2
        if FAST_DEMO_CELL_21
        else 8,
    )
)


FIT_EARLY_STOPPING_PATIENCE: int = int(
    globals().get(
        "CANDIDATE_FIT_PATIENCE",
        2
        if FAST_DEMO_CELL_21
        else 10,
    )
)


FIT_EARLY_STOPPING_MIN_DELTA: float = float(
    globals().get(
        "CANDIDATE_FIT_MIN_DELTA",
        1.0e-9,
    )
)


FIT_LEARNING_RATE: float = float(
    globals().get(
        "CANDIDATE_FIT_LEARNING_RATE",
        globals().get(
            "ADAM_LEARNING_RATE",
            2.5e-3,
        ),
    )
)


FIT_ADAM_BETA1: float = float(
    globals().get(
        "ADAM_BETA1",
        0.9,
    )
)


FIT_ADAM_BETA2: float = float(
    globals().get(
        "ADAM_BETA2",
        0.999,
    )
)


FIT_ADAM_EPSILON: float = float(
    globals().get(
        "ADAM_EPSILON",
        1.0e-8,
    )
)


FIT_GRADIENT_CLIP_NORM: float = float(
    globals().get(
        "GRADIENT_CLIP_NORM",
        10.0,
    )
)


FIT_MAX_PARAMETER_ABS: float = float(
    globals().get(
        "MAX_REDUCED_COEFFICIENT_ABS",
        20.0,
    )
)


FIT_SHUFFLE_EACH_EPOCH: bool = bool(
    globals().get(
        "SHUFFLE_TRAINING_TRAJECTORIES",
        True,
    )
)


if FIT_MAX_EPOCHS < 1:
    raise ValueError(
        "FIT_MAX_EPOCHS must be at least one."
    )


if FIT_MIN_EPOCHS < 1:
    raise ValueError(
        "FIT_MIN_EPOCHS must be at least one."
    )


if FIT_MIN_EPOCHS > FIT_MAX_EPOCHS:
    raise ValueError(
        "FIT_MIN_EPOCHS cannot exceed FIT_MAX_EPOCHS."
    )


if FIT_EARLY_STOPPING_PATIENCE < 1:
    raise ValueError(
        "FIT_EARLY_STOPPING_PATIENCE must be at least one."
    )


if (
    not np.isfinite(
        FIT_LEARNING_RATE
    )
    or
    FIT_LEARNING_RATE
    <=
    0.0
):
    raise ValueError(
        "FIT_LEARNING_RATE must be finite and positive."
    )


if not (
    0.0
    <
    FIT_ADAM_BETA1
    <
    1.0
):
    raise ValueError(
        "FIT_ADAM_BETA1 must lie strictly between zero and one."
    )


if not (
    0.0
    <
    FIT_ADAM_BETA2
    <
    1.0
):
    raise ValueError(
        "FIT_ADAM_BETA2 must lie strictly between zero and one."
    )


if (
    not np.isfinite(
        FIT_ADAM_EPSILON
    )
    or
    FIT_ADAM_EPSILON
    <=
    0.0
):
    raise ValueError(
        "FIT_ADAM_EPSILON must be finite and positive."
    )


if (
    not np.isfinite(
        FIT_GRADIENT_CLIP_NORM
    )
    or
    FIT_GRADIENT_CLIP_NORM
    <=
    0.0
):
    raise ValueError(
        "FIT_GRADIENT_CLIP_NORM must be finite and positive."
    )


if (
    not np.isfinite(
        FIT_MAX_PARAMETER_ABS
    )
    or
    FIT_MAX_PARAMETER_ABS
    <=
    0.0
):
    raise ValueError(
        "FIT_MAX_PARAMETER_ABS must be finite and positive."
    )


# ======================================================================================
# 7. DEFINE GENERIC JOB-FIELD ACCESSORS
# ======================================================================================

_MISSING_JOB_VALUE = object()


def get_job_value(
    job: Any,
    names: Sequence[str],
    *,
    default: Any = _MISSING_JOB_VALUE,
) -> Any:
    """
    Read one value from a mapping, dataclass, named tuple, or object.
    """

    for name in names:
        if isinstance(
            job,
            Mapping,
        ):
            if name in job:
                return job[
                    name
                ]

        if hasattr(
            job,
            name,
        ):
            return getattr(
                job,
                name,
            )


    if default is not _MISSING_JOB_VALUE:
        return default


    raise AttributeError(
        "The fit job does not provide any of these fields:\n  - "
        + "\n  - ".join(
            names
        )
    )


def normalize_job_id(
    job: Any,
) -> str:
    """
    Return the deterministic job identifier.
    """

    return str(
        get_job_value(
            job,
            (
                "job_id",
                "fit_job_id",
                "id",
            ),
        )
    ).strip()


def normalize_candidate_id(
    job: Any,
) -> str:
    """
    Return the candidate identifier associated with one fit job.
    """

    candidate_id = get_job_value(
        job,
        (
            "candidate_id",
            "model_candidate_id",
            "model_id",
        ),
        default=None,
    )


    if candidate_id is not None:
        return str(
            candidate_id
        ).strip()


    job_id = normalize_job_id(
        job
    )


    lambda_marker = "__lambda_"


    if lambda_marker in job_id:
        return job_id.split(
            lambda_marker,
            maxsplit=1,
        )[
            0
        ]


    raise RuntimeError(
        "The candidate ID could not be recovered from the fit job:\n"
        f"  {job_id}"
    )


def normalize_regularization_lambda(
    job: Any,
) -> float:
    """
    Return the regularization strength associated with one fit job.
    """

    value = get_job_value(
        job,
        (
            "regularization_lambda",
            "lambda_value",
            "sparse_lambda",
            "penalty_lambda",
        ),
        default=None,
    )


    if value is None:
        job_id = normalize_job_id(
            job
        )


        lambda_marker = "__lambda_"


        if lambda_marker not in job_id:
            return 0.0


        encoded_lambda = job_id.split(
            lambda_marker,
            maxsplit=1,
        )[
            1
        ]


        encoded_lambda = encoded_lambda.replace(
            "p",
            ".",
        )


        try:
            value = float(
                encoded_lambda
            )

        except ValueError as exc:
            raise RuntimeError(
                "The regularization value could not be parsed from job ID:\n"
                f"  {job_id}"
            ) from exc


    value = float(
        value
    )


    if not np.isfinite(
        value
    ):
        raise ValueError(
            "A candidate regularization value is nonfinite."
        )


    if value < 0.0:
        raise ValueError(
            "A candidate regularization value is negative."
        )


    return value


# ======================================================================================
# 8. NORMALIZE AND VERIFY FIT JOBS
# ======================================================================================

candidate_fit_jobs = list(
    NOTEBOOK_12_CANDIDATE_FIT_JOBS
)


if len(
    candidate_fit_jobs
) == 0:
    raise RuntimeError(
        "No candidate fit jobs were registered."
    )


normalized_job_records: list[
    dict[str, Any]
] = []


for job_position, job in enumerate(
    candidate_fit_jobs
):
    job_id = normalize_job_id(
        job
    )

    candidate_id = normalize_candidate_id(
        job
    )

    regularization_lambda = (
        normalize_regularization_lambda(
            job
        )
    )


    if not job_id:
        raise RuntimeError(
            "A candidate fit job has an empty job ID."
        )


    if candidate_id not in (
        CANDIDATE_ACTIVE_MASKS
    ):
        raise RuntimeError(
            "A fit job references an unknown candidate.\n"
            f"Job ID: {job_id}\n"
            f"Candidate ID: {candidate_id}"
        )


    active_mask = np.asarray(
        CANDIDATE_ACTIVE_MASKS[
            candidate_id
        ],
        dtype=bool,
    ).reshape(
        -1
    )


    if active_mask.shape != (
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
    ):
        raise RuntimeError(
            "A candidate active mask has an incorrect shape.\n"
            f"Candidate: {candidate_id}\n"
            f"Expected: "
            f"({NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT},)\n"
            f"Observed: {active_mask.shape}"
        )


    if not active_mask.any():
        raise RuntimeError(
            f"Candidate {candidate_id!r} has no active coefficients."
        )


    normalized_job_records.append(
        {
            "job_position": int(
                job_position
            ),
            "job": job,
            "job_id": job_id,
            "candidate_id": candidate_id,
            "regularization_lambda": float(
                regularization_lambda
            ),
            "active_coefficient_count": int(
                active_mask.sum()
            ),
        }
    )


job_ids = [
    record[
        "job_id"
    ]
    for record in normalized_job_records
]


if len(
    set(
        job_ids
    )
) != len(
    job_ids
):
    raise RuntimeError(
        "The candidate fit-job registry contains duplicate job IDs."
    )


# ======================================================================================
# 9. DEFINE DETERMINISTIC JOB SEEDS
# ======================================================================================

def deterministic_job_seed(
    job_id: str,
) -> int:
    """
    Generate one reproducible random seed from the global seed and job ID.
    """

    seed_text = (
        f"{FIT_RANDOM_SEED}::{job_id}"
    )


    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()


    seed_value = int.from_bytes(
        digest[
            :
            8
        ],
        byteorder="little",
        signed=False,
    )


    return int(
        seed_value
        %
        (
            2**32
            -
            1
        )
    )


# ======================================================================================
# 10. DEFINE WRITABLE NUMPY CONVERSION
# ======================================================================================

def writable_float64_array(
    value: Any,
    *,
    name: str,
    expected_shape: tuple[int, ...] | None = None,
) -> np.ndarray:
    """
    Convert an array-like value into an owned, writable float64 NumPy array.

    np.asarray() may expose a read-only JAX buffer. np.array(..., copy=True)
    guarantees that the optimizer receives an owned writable array.
    """

    output = np.array(
        value,
        dtype=np.float64,
        copy=True,
        order="C",
    )


    if expected_shape is not None:
        if output.shape != expected_shape:
            raise RuntimeError(
                f"{name} has an incorrect shape.\n"
                f"Expected: {expected_shape}\n"
                f"Observed: {output.shape}"
            )


    if not output.flags[
        "OWNDATA"
    ]:
        output = output.copy(
            order="C"
        )


    if not output.flags[
        "WRITEABLE"
    ]:
        output = output.copy(
            order="C"
        )


    if not output.flags[
        "WRITEABLE"
    ]:
        raise RuntimeError(
            f"{name} could not be converted to a writable NumPy array."
        )


    return output


# ======================================================================================
# 11. DEFINE THE ADAM UPDATE
# ======================================================================================

def adam_update(
    *,
    parameter_array: np.ndarray,
    gradient_array: np.ndarray,
    first_moment: np.ndarray,
    second_moment: np.ndarray,
    optimization_step: int,
    active_mask_float: np.ndarray,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    """
    Apply one writable, non-in-place Adam update.
    """

    optimization_step = int(
        optimization_step
    )


    if optimization_step < 1:
        raise ValueError(
            "optimization_step must be at least one."
        )


    # Every expression below allocates a new writable NumPy array.
    updated_first_moment = (
        FIT_ADAM_BETA1
        *
        first_moment
        +
        (
            1.0
            -
            FIT_ADAM_BETA1
        )
        *
        gradient_array
    )


    updated_second_moment = (
        FIT_ADAM_BETA2
        *
        second_moment
        +
        (
            1.0
            -
            FIT_ADAM_BETA2
        )
        *
        np.square(
            gradient_array
        )
    )


    bias_corrected_first_moment = (
        updated_first_moment
        /
        (
            1.0
            -
            FIT_ADAM_BETA1
            **
            optimization_step
        )
    )


    bias_corrected_second_moment = (
        updated_second_moment
        /
        (
            1.0
            -
            FIT_ADAM_BETA2
            **
            optimization_step
        )
    )


    adam_direction = (
        bias_corrected_first_moment
        /
        (
            np.sqrt(
                bias_corrected_second_moment
            )
            +
            FIT_ADAM_EPSILON
        )
    )


    updated_parameters = (
        parameter_array
        -
        FIT_LEARNING_RATE
        *
        adam_direction
    )


    updated_parameters = np.clip(
        updated_parameters,
        -
        FIT_MAX_PARAMETER_ABS,
        FIT_MAX_PARAMETER_ABS,
    )


    # Do not use updated_parameters *= active_mask_float.
    updated_parameters = (
        updated_parameters
        *
        active_mask_float
    )


    return (
        writable_float64_array(
            updated_parameters,
            name="updated parameter array",
            expected_shape=(
                NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
            ),
        ),

        writable_float64_array(
            updated_first_moment,
            name="updated Adam first moment",
            expected_shape=(
                NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
            ),
        ),

        writable_float64_array(
            updated_second_moment,
            name="updated Adam second moment",
            expected_shape=(
                NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
            ),
        ),
    )


# ======================================================================================
# 12. DEFINE ONE-CANDIDATE FITTING FUNCTION
# ======================================================================================

def fit_candidate_job(
    job: Any,
) -> dict[str, Any]:
    """
    Fit one candidate and regularization job on training trajectories only.
    """

    job_id = normalize_job_id(
        job
    )

    candidate_id = normalize_candidate_id(
        job
    )

    regularization_lambda = (
        normalize_regularization_lambda(
            job
        )
    )


    active_mask_boolean = np.asarray(
        CANDIDATE_ACTIVE_MASKS[
            candidate_id
        ],
        dtype=bool,
    ).reshape(
        -1
    )


    active_mask_float = np.array(
        active_mask_boolean,
        dtype=np.float64,
        copy=True,
    )


    if active_mask_float.shape != (
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
    ):
        raise RuntimeError(
            f"Candidate {candidate_id!r} has an invalid active mask."
        )


    active_coefficient_count = int(
        active_mask_boolean.sum()
    )


    # ------------------------------------------------------------------
    # Initialize writable optimizer arrays
    # ------------------------------------------------------------------

    parameter_array = np.zeros(
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
        dtype=np.float64,
    )


    initial_coefficients = get_job_value(
        job,
        (
            "initial_coefficients",
            "initial_reduced_coefficients",
        ),
        default=None,
    )


    if initial_coefficients is not None:
        parameter_array = writable_float64_array(
            initial_coefficients,
            name=(
                f"initial coefficient vector for {job_id}"
            ),
            expected_shape=(
                NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
            ),
        )


    parameter_array = (
        parameter_array
        *
        active_mask_float
    )


    parameter_array = writable_float64_array(
        parameter_array,
        name=(
            f"masked initial coefficients for {job_id}"
        ),
        expected_shape=(
            NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
        ),
    )


    first_moment = np.zeros_like(
        parameter_array
    )

    second_moment = np.zeros_like(
        parameter_array
    )


    best_parameter_array = parameter_array.copy()

    best_epoch: int = 0

    best_epoch_total_loss: float = float(
        "inf"
    )

    epochs_without_improvement: int = 0

    optimization_step: int = 0

    clipped_gradient_step_count: int = 0

    total_processed_trajectories: int = 0

    total_processed_loss_samples: int = 0


    job_seed = deterministic_job_seed(
        job_id
    )


    rng = np.random.default_rng(
        job_seed
    )


    objective = make_candidate_objective(
        candidate_id=candidate_id,
        regularization_lambda=float(
            regularization_lambda
        ),
    )


    history_records: list[
        dict[str, Any]
    ] = []


    job_start_time = time.perf_counter()


    stop_reason = "MAX_EPOCHS_REACHED"

    fit_status = "COMPLETED"


    for epoch_index in range(
        FIT_MAX_EPOCHS
    ):
        epoch_number = int(
            epoch_index
            +
            1
        )


        epoch_row_groups = np.array(
            training_row_group_indices,
            dtype=np.int32,
            copy=True,
        )


        if FIT_SHUFFLE_EACH_EPOCH:
            rng.shuffle(
                epoch_row_groups
            )


        epoch_total_loss_sum = 0.0
        epoch_data_loss_sum = 0.0
        epoch_ridge_penalty_sum = 0.0
        epoch_group_penalty_sum = 0.0
        epoch_physical_penalty_sum = 0.0

        epoch_weight_sum = 0

        epoch_gradient_norm_sum = 0.0
        epoch_gradient_norm_maximum = 0.0

        epoch_batch_count = 0
        epoch_trajectory_count = 0
        epoch_loss_sample_count = 0


        for batch_start in range(
            0,
            TRAINING_ROW_GROUP_COUNT,
            TRAINING_BATCH_TRAJECTORIES,
        ):
            batch_end = min(
                batch_start
                +
                TRAINING_BATCH_TRAJECTORIES,
                TRAINING_ROW_GROUP_COUNT,
            )


            batch_row_group_indices = (
                epoch_row_groups[
                    batch_start:
                    batch_end
                ]
            )


            numpy_batch = load_padded_trajectory_batch(
                partition="train",
                row_group_indices=(
                    batch_row_group_indices
                ),
            )


            jax_batch = batch_to_jax(
                numpy_batch
            )


            (
                (
                    total_loss,
                    auxiliary_losses,
                ),
                gradient,
            ) = objective(
                jnp.asarray(
                    parameter_array,
                    dtype=jnp.float64,
                ),
                jax_batch,
            )


            (
                data_loss,
                ridge_penalty,
                group_penalty,
                physical_penalty,
            ) = auxiliary_losses


            total_loss_value = float(
                np.asarray(
                    total_loss
                )
            )

            data_loss_value = float(
                np.asarray(
                    data_loss
                )
            )

            ridge_penalty_value = float(
                np.asarray(
                    ridge_penalty
                )
            )

            group_penalty_value = float(
                np.asarray(
                    group_penalty
                )
            )

            physical_penalty_value = float(
                np.asarray(
                    physical_penalty
                )
            )


            for value_name, value in (
                (
                    "total loss",
                    total_loss_value,
                ),
                (
                    "data loss",
                    data_loss_value,
                ),
                (
                    "ridge penalty",
                    ridge_penalty_value,
                ),
                (
                    "group penalty",
                    group_penalty_value,
                ),
                (
                    "physical penalty",
                    physical_penalty_value,
                ),
            ):
                if not np.isfinite(
                    value
                ):
                    raise RuntimeError(
                        f"Job {job_id!r} produced a nonfinite "
                        f"{value_name}.\n"
                        f"Epoch: {epoch_number}\n"
                        f"Batch row groups: "
                        f"{batch_row_group_indices.tolist()}"
                    )


            # ------------------------------------------------------------------
            # Critical correction:
            #
            # np.asarray(gradient) may produce a read-only JAX-backed view.
            # np.array(..., copy=True) guarantees an owned writable array.
            # ------------------------------------------------------------------

            gradient_array = writable_float64_array(
                gradient,
                name=(
                    f"gradient for {job_id}"
                ),
                expected_shape=(
                    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
                ),
            )


            if not np.isfinite(
                gradient_array
            ).all():
                nonfinite_positions = np.flatnonzero(
                    ~np.isfinite(
                        gradient_array
                    )
                )


                raise RuntimeError(
                    f"Job {job_id!r} produced a nonfinite gradient.\n"
                    f"Epoch: {epoch_number}\n"
                    f"Nonfinite count: {len(nonfinite_positions):,}\n"
                    f"First positions: "
                    f"{nonfinite_positions[:20].tolist()}"
                )


            # Do not use gradient_array *= active_mask.
            gradient_array = (
                gradient_array
                *
                active_mask_float
            )


            gradient_array = writable_float64_array(
                gradient_array,
                name=(
                    f"active gradient for {job_id}"
                ),
                expected_shape=(
                    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
                ),
            )


            raw_gradient_norm = float(
                np.linalg.norm(
                    gradient_array
                )
            )


            if not np.isfinite(
                raw_gradient_norm
            ):
                raise RuntimeError(
                    f"Job {job_id!r} produced a nonfinite gradient norm."
                )


            if raw_gradient_norm > FIT_GRADIENT_CLIP_NORM:
                gradient_scale = (
                    FIT_GRADIENT_CLIP_NORM
                    /
                    max(
                        raw_gradient_norm,
                        np.finfo(
                            np.float64
                        ).tiny,
                    )
                )


                clipped_gradient = (
                    gradient_array
                    *
                    gradient_scale
                )


                clipped_gradient_step_count += 1

            else:
                clipped_gradient = gradient_array.copy()


            clipped_gradient = writable_float64_array(
                clipped_gradient,
                name=(
                    f"clipped gradient for {job_id}"
                ),
                expected_shape=(
                    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
                ),
            )


            clipped_gradient_norm = float(
                np.linalg.norm(
                    clipped_gradient
                )
            )


            optimization_step += 1


            (
                parameter_array,
                first_moment,
                second_moment,
            ) = adam_update(
                parameter_array=(
                    parameter_array
                ),
                gradient_array=(
                    clipped_gradient
                ),
                first_moment=(
                    first_moment
                ),
                second_moment=(
                    second_moment
                ),
                optimization_step=(
                    optimization_step
                ),
                active_mask_float=(
                    active_mask_float
                ),
            )


            if not np.isfinite(
                parameter_array
            ).all():
                nonfinite_parameter_positions = np.flatnonzero(
                    ~np.isfinite(
                        parameter_array
                    )
                )


                raise RuntimeError(
                    f"Job {job_id!r} produced nonfinite coefficients.\n"
                    f"Epoch: {epoch_number}\n"
                    f"First positions: "
                    f"{nonfinite_parameter_positions[:20].tolist()}"
                )


            inactive_parameter_maximum = float(
                np.max(
                    np.abs(
                        parameter_array[
                            ~active_mask_boolean
                        ]
                    ),
                    initial=0.0,
                )
            )


            if inactive_parameter_maximum > 0.0:
                raise RuntimeError(
                    f"Job {job_id!r} activated a structurally inactive "
                    "coefficient."
                )


            batch_weight = int(
                numpy_batch[
                    "loss_sample_count"
                ]
            )


            if batch_weight <= 0:
                raise RuntimeError(
                    "A training batch contains no optimization samples."
                )


            epoch_total_loss_sum += (
                total_loss_value
                *
                batch_weight
            )

            epoch_data_loss_sum += (
                data_loss_value
                *
                batch_weight
            )

            epoch_ridge_penalty_sum += (
                ridge_penalty_value
                *
                batch_weight
            )

            epoch_group_penalty_sum += (
                group_penalty_value
                *
                batch_weight
            )

            epoch_physical_penalty_sum += (
                physical_penalty_value
                *
                batch_weight
            )


            epoch_weight_sum += batch_weight

            epoch_gradient_norm_sum += (
                clipped_gradient_norm
            )

            epoch_gradient_norm_maximum = max(
                epoch_gradient_norm_maximum,
                clipped_gradient_norm,
            )

            epoch_batch_count += 1

            epoch_trajectory_count += int(
                numpy_batch[
                    "batch_size"
                ]
            )

            epoch_loss_sample_count += int(
                numpy_batch[
                    "loss_sample_count"
                ]
            )


            total_processed_trajectories += int(
                numpy_batch[
                    "batch_size"
                ]
            )

            total_processed_loss_samples += int(
                numpy_batch[
                    "loss_sample_count"
                ]
            )


            del numpy_batch
            del jax_batch
            del gradient
            del gradient_array
            del clipped_gradient


        if epoch_weight_sum <= 0:
            raise RuntimeError(
                f"Job {job_id!r} produced no weighted epoch observations."
            )


        epoch_total_loss = float(
            epoch_total_loss_sum
            /
            epoch_weight_sum
        )

        epoch_data_loss = float(
            epoch_data_loss_sum
            /
            epoch_weight_sum
        )

        epoch_ridge_penalty = float(
            epoch_ridge_penalty_sum
            /
            epoch_weight_sum
        )

        epoch_group_penalty = float(
            epoch_group_penalty_sum
            /
            epoch_weight_sum
        )

        epoch_physical_penalty = float(
            epoch_physical_penalty_sum
            /
            epoch_weight_sum
        )

        epoch_mean_gradient_norm = float(
            epoch_gradient_norm_sum
            /
            max(
                epoch_batch_count,
                1,
            )
        )


        improved = bool(
            epoch_total_loss
            <
            (
                best_epoch_total_loss
                -
                FIT_EARLY_STOPPING_MIN_DELTA
            )
        )


        if improved:
            best_epoch_total_loss = float(
                epoch_total_loss
            )

            best_epoch = int(
                epoch_number
            )

            best_parameter_array = np.array(
                parameter_array,
                dtype=np.float64,
                copy=True,
            )

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1


        elapsed_seconds = float(
            time.perf_counter()
            -
            job_start_time
        )


        history_records.append(
            {
                "job_id": job_id,
                "candidate_id": candidate_id,
                "regularization_lambda": float(
                    regularization_lambda
                ),
                "epoch": int(
                    epoch_number
                ),
                "batch_count": int(
                    epoch_batch_count
                ),
                "trajectory_count": int(
                    epoch_trajectory_count
                ),
                "loss_sample_count": int(
                    epoch_loss_sample_count
                ),
                "total_loss": float(
                    epoch_total_loss
                ),
                "data_loss": float(
                    epoch_data_loss
                ),
                "ridge_penalty": float(
                    epoch_ridge_penalty
                ),
                "group_penalty": float(
                    epoch_group_penalty
                ),
                "physical_penalty": float(
                    epoch_physical_penalty
                ),
                "mean_gradient_norm": float(
                    epoch_mean_gradient_norm
                ),
                "maximum_gradient_norm": float(
                    epoch_gradient_norm_maximum
                ),
                "parameter_norm": float(
                    np.linalg.norm(
                        parameter_array
                    )
                ),
                "best_loss_so_far": float(
                    best_epoch_total_loss
                ),
                "best_epoch_so_far": int(
                    best_epoch
                ),
                "improved": bool(
                    improved
                ),
                "epochs_without_improvement": int(
                    epochs_without_improvement
                ),
                "elapsed_seconds": float(
                    elapsed_seconds
                ),
            }
        )


        print(
            f"    Epoch {epoch_number:03d} | "
            f"loss={epoch_total_loss:.8e} | "
            f"data={epoch_data_loss:.8e} | "
            f"grad={epoch_mean_gradient_norm:.4e} | "
            f"best={best_epoch_total_loss:.8e}"
        )


        if (
            epoch_number
            >=
            FIT_MIN_EPOCHS
            and
            epochs_without_improvement
            >=
            FIT_EARLY_STOPPING_PATIENCE
        ):
            stop_reason = "EARLY_STOPPING"

            break


    job_elapsed_seconds = float(
        time.perf_counter()
        -
        job_start_time
    )


    completed_epoch_count = int(
        len(
            history_records
        )
    )


    if completed_epoch_count == 0:
        raise RuntimeError(
            f"Job {job_id!r} completed no fitting epoch."
        )


    if not np.isfinite(
        best_parameter_array
    ).all():
        raise RuntimeError(
            f"Job {job_id!r} produced a nonfinite best coefficient vector."
        )


    best_parameter_array = (
        best_parameter_array
        *
        active_mask_float
    )


    best_parameter_array = writable_float64_array(
        best_parameter_array,
        name=(
            f"best coefficient vector for {job_id}"
        ),
        expected_shape=(
            NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
        ),
    )


    nonzero_coefficient_count = int(
        (
            np.abs(
                best_parameter_array
            )
            >
            1.0e-12
        ).sum()
    )


    final_history = history_records[
        -1
    ]


    result = {
        "job_id": job_id,
        "candidate_id": candidate_id,
        "regularization_lambda": float(
            regularization_lambda
        ),
        "job_seed": int(
            job_seed
        ),
        "status": fit_status,
        "stop_reason": stop_reason,
        "best_epoch": int(
            best_epoch
        ),
        "completed_epoch_count": int(
            completed_epoch_count
        ),
        "optimization_step_count": int(
            optimization_step
        ),
        "clipped_gradient_step_count": int(
            clipped_gradient_step_count
        ),
        "active_coefficient_count": int(
            active_coefficient_count
        ),
        "nonzero_coefficient_count": int(
            nonzero_coefficient_count
        ),
        "best_total_loss": float(
            best_epoch_total_loss
        ),
        "final_total_loss": float(
            final_history[
                "total_loss"
            ]
        ),
        "final_data_loss": float(
            final_history[
                "data_loss"
            ]
        ),
        "final_ridge_penalty": float(
            final_history[
                "ridge_penalty"
            ]
        ),
        "final_group_penalty": float(
            final_history[
                "group_penalty"
            ]
        ),
        "final_physical_penalty": float(
            final_history[
                "physical_penalty"
            ]
        ),
        "final_mean_gradient_norm": float(
            final_history[
                "mean_gradient_norm"
            ]
        ),
        "best_parameter_norm": float(
            np.linalg.norm(
                best_parameter_array
            )
        ),
        "total_processed_trajectories": int(
            total_processed_trajectories
        ),
        "total_processed_loss_samples": int(
            total_processed_loss_samples
        ),
        "elapsed_seconds": float(
            job_elapsed_seconds
        ),
        "best_reduced_coefficients": (
            best_parameter_array
        ),
        "final_reduced_coefficients": np.array(
            parameter_array,
            dtype=np.float64,
            copy=True,
        ),
        "history_records": history_records,
    }


    del objective

    gc.collect()


    return result


# ======================================================================================
# 13. FIT ALL REGISTERED JOBS
# ======================================================================================

NOTEBOOK_12_CANDIDATE_FIT_RESULT_BY_JOB_ID: dict[
    str,
    dict[str, Any],
] = {}


NOTEBOOK_12_CANDIDATE_COEFFICIENT_BY_JOB_ID: dict[
    str,
    np.ndarray,
] = {}


candidate_fit_summary_records: list[
    dict[str, Any]
] = []


candidate_fit_history_records: list[
    dict[str, Any]
] = []


fit_all_start_time = time.perf_counter()


for job_number, normalized_job_record in enumerate(
    normalized_job_records,
    start=1,
):
    job = normalized_job_record[
        "job"
    ]

    job_id = normalized_job_record[
        "job_id"
    ]


    print(
        f"[{job_number}/{len(normalized_job_records)}] "
        f"Fitting {job_id}"
    )


    result = fit_candidate_job(
        job
    )


    NOTEBOOK_12_CANDIDATE_FIT_RESULT_BY_JOB_ID[
        job_id
    ] = result


    NOTEBOOK_12_CANDIDATE_COEFFICIENT_BY_JOB_ID[
        job_id
    ] = np.array(
        result[
            "best_reduced_coefficients"
        ],
        dtype=np.float64,
        copy=True,
    )


    candidate_fit_summary_records.append(
        {
            key: value
            for key, value in result.items()
            if key not in (
                "best_reduced_coefficients",
                "final_reduced_coefficients",
                "history_records",
            )
        }
    )


    candidate_fit_history_records.extend(
        result[
            "history_records"
        ]
    )


    print(
        f"    Completed {job_id} | "
        f"best loss={result['best_total_loss']:.8e} | "
        f"best epoch={result['best_epoch']} | "
        f"stop={result['stop_reason']} | "
        f"time={result['elapsed_seconds']:.2f} s"
    )


total_fit_elapsed_seconds = float(
    time.perf_counter()
    -
    fit_all_start_time
)


# ======================================================================================
# 14. BUILD FITTING TABLES
# ======================================================================================

NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF = (
    pd.DataFrame(
        candidate_fit_summary_records
    )
    .sort_values(
        by=[
            "candidate_id",
            "regularization_lambda",
            "job_id",
        ],
        kind="mergesort",
        ignore_index=True,
    )
)


NOTEBOOK_12_CANDIDATE_FIT_HISTORY_DF = (
    pd.DataFrame(
        candidate_fit_history_records
    )
    .sort_values(
        by=[
            "job_id",
            "epoch",
        ],
        kind="mergesort",
        ignore_index=True,
    )
)


if len(
    NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF
) != len(
    normalized_job_records
):
    raise RuntimeError(
        "The candidate fit-summary row count is incorrect."
    )


if NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF[
    "job_id"
].duplicated().any():
    raise RuntimeError(
        "The candidate fit summary contains duplicate job IDs."
    )


if not NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF[
    "best_total_loss"
].map(
    np.isfinite
).all():
    raise RuntimeError(
        "The candidate fit summary contains a nonfinite best loss."
    )


# ======================================================================================
# 15. SAVE FITTING TABLES
# ======================================================================================

NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF.to_parquet(
    NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_PATH,
    index=False,
    compression="zstd",
)


NOTEBOOK_12_CANDIDATE_FIT_HISTORY_DF.to_parquet(
    NOTEBOOK_12_CANDIDATE_FIT_HISTORY_PATH,
    index=False,
    compression="zstd",
)


if not NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        "The candidate fit-summary table was not created."
    )


if not NOTEBOOK_12_CANDIDATE_FIT_HISTORY_PATH.is_file():
    raise FileNotFoundError(
        "The candidate fit-history table was not created."
    )


# ======================================================================================
# 16. SAVE ALL BEST COEFFICIENT VECTORS
# ======================================================================================

ordered_fit_job_ids: tuple[str, ...] = tuple(
    NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF[
        "job_id"
    ].astype(str)
)


ordered_candidate_ids: tuple[str, ...] = tuple(
    NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF[
        "candidate_id"
    ].astype(str)
)


ordered_regularization_lambdas = (
    NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF[
        "regularization_lambda"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


ordered_best_coefficient_matrix = np.stack(
    [
        NOTEBOOK_12_CANDIDATE_COEFFICIENT_BY_JOB_ID[
            job_id
        ]
        for job_id in ordered_fit_job_ids
    ],
    axis=0,
).astype(
    np.float64,
    copy=False,
)


expected_coefficient_matrix_shape = (
    len(
        ordered_fit_job_ids
    ),
    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
)


if ordered_best_coefficient_matrix.shape != (
    expected_coefficient_matrix_shape
):
    raise RuntimeError(
        "The fitted coefficient matrix has an incorrect shape.\n"
        f"Expected: {expected_coefficient_matrix_shape}\n"
        f"Observed: {ordered_best_coefficient_matrix.shape}"
    )


if not np.isfinite(
    ordered_best_coefficient_matrix
).all():
    raise RuntimeError(
        "The fitted coefficient matrix contains nonfinite values."
    )


temporary_coefficient_path = (
    NOTEBOOK_12_CANDIDATE_FIT_COEFFICIENT_PATH
    .with_name(
        NOTEBOOK_12_CANDIDATE_FIT_COEFFICIENT_PATH.stem
        +
        ".tmp.npz"
    )
)


if temporary_coefficient_path.exists():
    temporary_coefficient_path.unlink()


np.savez_compressed(
    temporary_coefficient_path,
    job_ids=np.asarray(
        ordered_fit_job_ids,
        dtype=np.str_,
    ),
    candidate_ids=np.asarray(
        ordered_candidate_ids,
        dtype=np.str_,
    ),
    regularization_lambdas=(
        ordered_regularization_lambdas
    ),
    best_reduced_coefficients=(
        ordered_best_coefficient_matrix
    ),
)


temporary_coefficient_path.replace(
    NOTEBOOK_12_CANDIDATE_FIT_COEFFICIENT_PATH
)


if not NOTEBOOK_12_CANDIDATE_FIT_COEFFICIENT_PATH.is_file():
    raise FileNotFoundError(
        "The candidate coefficient archive was not created."
    )


with np.load(
    NOTEBOOK_12_CANDIDATE_FIT_COEFFICIENT_PATH,
    allow_pickle=False,
) as reloaded_candidate_coefficients:
    reloaded_coefficient_matrix = np.asarray(
        reloaded_candidate_coefficients[
            "best_reduced_coefficients"
        ],
        dtype=np.float64,
    )


    if not np.array_equal(
        reloaded_coefficient_matrix,
        ordered_best_coefficient_matrix,
    ):
        raise RuntimeError(
            "The reloaded candidate coefficient matrix differs "
            "from the fitted matrix."
        )


# ======================================================================================
# 17. SAVE THE FITTING CONFIGURATION
# ======================================================================================

NOTEBOOK_12_CANDIDATE_FIT_CONFIGURATION: dict[
    str,
    Any,
] = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "status": "COMPLETED",

    "training_only": True,

    "candidate_fit_job_count": int(
        len(
            normalized_job_records
        )
    ),

    "training_row_group_count": int(
        TRAINING_ROW_GROUP_COUNT
    ),

    "training_batch_trajectories": int(
        TRAINING_BATCH_TRAJECTORIES
    ),

    "reduced_coefficient_count": int(
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT
    ),

    "maximum_epochs": int(
        FIT_MAX_EPOCHS
    ),

    "minimum_epochs": int(
        FIT_MIN_EPOCHS
    ),

    "early_stopping_patience": int(
        FIT_EARLY_STOPPING_PATIENCE
    ),

    "early_stopping_min_delta": float(
        FIT_EARLY_STOPPING_MIN_DELTA
    ),

    "learning_rate": float(
        FIT_LEARNING_RATE
    ),

    "adam_beta1": float(
        FIT_ADAM_BETA1
    ),

    "adam_beta2": float(
        FIT_ADAM_BETA2
    ),

    "adam_epsilon": float(
        FIT_ADAM_EPSILON
    ),

    "gradient_clip_norm": float(
        FIT_GRADIENT_CLIP_NORM
    ),

    "maximum_reduced_coefficient_absolute_value": float(
        FIT_MAX_PARAMETER_ABS
    ),

    "shuffle_each_epoch": bool(
        FIT_SHUFFLE_EACH_EPOCH
    ),

    "random_seed": int(
        FIT_RANDOM_SEED
    ),

    "ordinary_test_used": False,

    "strict_test_used": False,

    "embargo_used": False,

    "summary_path": str(
        NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_PATH
    ),

    "history_path": str(
        NOTEBOOK_12_CANDIDATE_FIT_HISTORY_PATH
    ),

    "coefficient_path": str(
        NOTEBOOK_12_CANDIDATE_FIT_COEFFICIENT_PATH
    ),

    "total_elapsed_seconds": float(
        total_fit_elapsed_seconds
    ),
}


temporary_configuration_path = (
    NOTEBOOK_12_CANDIDATE_FIT_CONFIGURATION_PATH
    .with_name(
        NOTEBOOK_12_CANDIDATE_FIT_CONFIGURATION_PATH.name
        +
        ".tmp"
    )
)


with temporary_configuration_path.open(
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        NOTEBOOK_12_CANDIDATE_FIT_CONFIGURATION,
        file_handle,
        indent=2,
        sort_keys=True,
    )

    file_handle.write(
        "\n"
    )


temporary_configuration_path.replace(
    NOTEBOOK_12_CANDIDATE_FIT_CONFIGURATION_PATH
)


# ======================================================================================
# 18. FINAL EXECUTION SUMMARY
# ======================================================================================

completed_job_count = int(
    len(
        NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF
    )
)


total_optimization_steps = int(
    NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF[
        "optimization_step_count"
    ].sum()
)


total_clipped_gradient_steps = int(
    NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF[
        "clipped_gradient_step_count"
    ].sum()
)


print("=" * 116)
print("TRAINING-ONLY CANDIDATE FITTING")
print("=" * 116)

print(
    f"Candidate and regularization jobs : "
    f"{completed_job_count:,}"
)

print(
    f"Training trajectory row groups    : "
    f"{TRAINING_ROW_GROUP_COUNT:,}"
)

print(
    f"Trajectories per training batch   : "
    f"{TRAINING_BATCH_TRAJECTORIES:,}"
)

print(
    f"Reduced coefficient dimension     : "
    f"{NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT:,}"
)

print(
    f"Maximum fitting epochs            : "
    f"{FIT_MAX_EPOCHS:,}"
)

print(
    f"Minimum fitting epochs            : "
    f"{FIT_MIN_EPOCHS:,}"
)

print(
    f"Early-stopping patience           : "
    f"{FIT_EARLY_STOPPING_PATIENCE:,}"
)

print(
    f"Adam learning rate                : "
    f"{FIT_LEARNING_RATE:.8e}"
)

print(
    f"Gradient clipping norm            : "
    f"{FIT_GRADIENT_CLIP_NORM:.8e}"
)

print(
    f"Total optimization steps          : "
    f"{total_optimization_steps:,}"
)

print(
    f"Clipped-gradient steps            : "
    f"{total_clipped_gradient_steps:,}"
)

print(
    f"Total fitting time                : "
    f"{total_fit_elapsed_seconds:.3f} s"
)

print(
    f"Fit summary                       : "
    f"{NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_PATH}"
)

print(
    f"Fit history                       : "
    f"{NOTEBOOK_12_CANDIDATE_FIT_HISTORY_PATH}"
)

print(
    f"Fitted coefficient archive        : "
    f"{NOTEBOOK_12_CANDIDATE_FIT_COEFFICIENT_PATH}"
)

print(
    f"Calibration trajectories used     : "
    f"False"
)

print(
    f"Ordinary-test trajectories used   : "
    f"False"
)

print(
    f"Strict-test trajectories used     : "
    f"False"
)

print(
    f"Embargo trajectories used         : "
    f"False"
)

print("-" * 116)

print(
    "All candidate and regularization jobs were fitted using "
    "training trajectories only."
)

print("=" * 116)


display(
    NOTEBOOK_12_CANDIDATE_FIT_SUMMARY_DF
)

[1/13] Fitting M0_FIXED_2RC__lambda_0p000e00
    Epoch 001 | loss=1.12997987e-02 | data=1.12997986e-02 | grad=5.2628e-03 | best=1.12997987e-02
    Epoch 002 | loss=8.81494042e-03 | data=8.81493969e-03 | grad=3.5147e-03 | best=8.81494042e-03
    Epoch 003 | loss=7.36929024e-03 | data=7.36928855e-03 | grad=2.4337e-03 | best=7.36929024e-03
    Epoch 004 | loss=6.45244248e-03 | data=6.45243952e-03 | grad=1.9619e-03 | best=6.45244248e-03
    Epoch 005 | loss=5.80396973e-03 | data=5.80396530e-03 | grad=1.4389e-03 | best=5.80396973e-03
    Epoch 006 | loss=5.35922149e-03 | data=5.35921542e-03 | grad=1.1972e-03 | best=5.35922149e-03
    Epoch 007 | loss=5.04804140e-03 | data=5.04803358e-03 | grad=9.0822e-04 | best=5.04804140e-03
    Epoch 008 | loss=4.80388451e-03 | data=4.80387486e-03 | grad=7.6351e-04 | best=4.80388451e-03
    Epoch 009 | loss=4.61884158e-03 | data=4.61882980e-03 | grad=5.7360e-04 | best=4.61884158e-03
    Epoch 010 | loss=4.48780306e-03 | data=4.48778879e-03 | grad=4.6658e-

,job_id,candidate_id,regularization_lambda,job_seed,status,stop_reason,best_epoch,completed_epoch_count,optimization_step_count,clipped_gradient_step_count,active_coefficient_count,nonzero_coefficient_count,best_total_loss,final_total_loss,final_data_loss,final_ridge_penalty,final_group_penalty,final_physical_penalty,final_mean_gradient_norm,best_parameter_norm,total_processed_trajectories,total_processed_loss_samples,elapsed_seconds
0,M0_FIXED_2RC__lambda_0p000e00,M0_FIXED_2RC,0.000000,538549886,COMPLETED,MAX_EPOCHS_REACHED,60,60,21240,0,5,5,0.003833,0.003833,0.003833,1.398346e-07,0.000000,0.000000e+00,0.000003,18.399066,169920,538182720,2011.502400
1,M1_GLOBAL_DENSE_LPV__lambda_0p000e00,M1_GLOBAL_DENSE_LPV,0.000000,148632042,COMPLETED,MAX_EPOCHS_REACHED,60,60,21240,0,30,30,0.003833,0.003833,0.003833,1.062253e-07,0.000000,1.064326e-07,0.000008,15.987197,169920,538182720,1954.605371
2,M2_GLOBAL_SPARSE_LPV__lambda_1p000em06,M2_GLOBAL_SPARSE_LPV,0.000001,2743738360,COMPLETED,MAX_EPOCHS_REACHED,60,60,21240,0,30,30,0.003833,0.003833,0.003833,1.483973e-07,0.013833,0.000000e+00,0.000007,18.992742,169920,538182720,2038.111009
3,M2_GLOBAL_SPARSE_LPV__lambda_3p000em06,M2_GLOBAL_SPARSE_LPV,0.000003,1585725957,COMPLETED,MAX_EPOCHS_REACHED,60,60,21240,0,30,30,0.003833,0.003833,0.003833,1.408176e-07,0.012630,0.000000e+00,0.000017,18.453383,169920,538182720,1836.342835
4,M2_GLOBAL_SPARSE_LPV__lambda_1p000em05,M2_GLOBAL_SPARSE_LPV,0.000010,1137364735,COMPLETED,MAX_EPOCHS_REACHED,60,60,21240,0,30,30,0.003833,0.003833,0.003833,1.426252e-07,0.008223,0.000000e+00,0.000051,18.571722,169920,538182720,1877.716778
5,M2_GLOBAL_SPARSE_LPV__lambda_3p000em05,M2_GLOBAL_SPARSE_LPV,0.000030,1800691771,COMPLETED,MAX_EPOCHS_REACHED,60,60,21240,0,30,30,0.003833,0.003833,0.003833,1.424873e-07,0.007211,0.000000e+00,0.000151,18.556071,169920,538182720,1928.593761
6,M2_GLOBAL_SPARSE_LPV__lambda_1p000em04,M2_GLOBAL_SPARSE_LPV,0.000100,3937709777,COMPLETED,MAX_EPOCHS_REACHED,59,60,21240,0,30,30,0.003834,0.003834,0.003833,1.424063e-07,0.007003,0.000000e+00,0.000500,18.421713,169920,538182720,1982.663267
7,M3_HIERARCHICAL_DENSE_LPV__lambda_0p000e00,M3_HIERARCHICAL_DENSE_LPV,0.000000,4173798562,COMPLETED,MAX_EPOCHS_REACHED,60,60,21240,0,240,240,0.003828,0.003828,0.003827,1.872275e-07,0.000000,2.530637e-07,0.000012,21.152533,169920,538182720,1982.914321
8,M4_HIERARCHICAL_SPARSE_LPV__lambda_1p000em06,M4_HIERARCHICAL_SPARSE_LPV,0.000001,4013992606,COMPLETED,MAX_EPOCHS_REACHED,60,60,21240,0,240,240,0.003833,0.003833,0.003833,1.499791e-07,0.103273,0.000000e+00,0.000017,19.045682,169920,538182720,1867.530413
9,M4_HIERARCHICAL_SPARSE_LPV__lambda_3p000em06,M4_HIERARCHICAL_SPARSE_LPV,0.000003,568666322,COMPLETED,MAX_EPOCHS_REACHED,60,60,21240,0,240,240,0.003833,0.003833,0.003833,1.426641e-07,0.080212,0.000000e+00,0.000048,18.576621,169920,538182720,1990.958452


In [26]:
# ======================================================================================
# CELL 22 — EVALUATE EVERY TRAINED JOB ON THE CALIBRATION PARTITION
# ======================================================================================
#
# Purpose
# -------
# Evaluate every fitted candidate and regularization job on the calibration
# trajectories, save crash-safe per-job results, select the simplest model
# within the approved calibration-error tolerance, and preserve the selected
# training coefficient vector for support closure and refitting in Cell 23.
#
# Ordinary-test, strict-test, and embargo trajectories remain unopened.
# ======================================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import re
import time

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping, Sequence

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

from IPython.display import display


# ======================================================================================
# 1. ENABLE JAX FLOAT64
# ======================================================================================

jax.config.update(
    "jax_enable_x64",
    True,
)


# ======================================================================================
# 2. VERIFY REQUIRED UPSTREAM OBJECTS
# ======================================================================================

REQUIRED_CELL_22_OBJECTS: tuple[str, ...] = (
    "NOTEBOOK_12_CANDIDATE_FIT_JOBS",
    "CANDIDATE_ACTIVE_MASKS",
    "MODEL_CANDIDATE_SPECIFICATIONS",
    "CANDIDATE_PENALTY_GROUP_IDS",
    "PENALTY_GROUP_MEMBERS",
    "load_design_trajectory",
    "get_design_parquet",
    "NOTEBOOK_12_TABLE_DIR",
    "NOTEBOOK_12_MODEL_DIR",
)


missing_cell_22_objects = [
    object_name
    for object_name in REQUIRED_CELL_22_OBJECTS
    if object_name not in globals()
]


if missing_cell_22_objects:
    raise RuntimeError(
        "Cell 22 requires the completed candidate-fitting and trajectory-"
        "loading cells.\n\n"
        "Missing objects:\n  - "
        + "\n  - ".join(
            missing_cell_22_objects
        )
    )


trajectory_simulator_available = bool(
    "simulate_lpv_trajectory_jax" in globals()
)

batch_simulator_available = bool(
    "simulate_lpv_batch_jax" in globals()
    or
    "simulate_lpv_batch_core" in globals()
    or
    "_simulate_lpv_batch_core" in globals()
)


if not (
    trajectory_simulator_available
    or
    batch_simulator_available
):
    raise RuntimeError(
        "No compatible LPV trajectory or batch simulator is available."
    )


# ======================================================================================
# 3. NORMALIZE CONFIGURATION
# ======================================================================================

NOTEBOOK_12_TABLE_DIR = Path(
    NOTEBOOK_12_TABLE_DIR
).expanduser().resolve()

NOTEBOOK_12_MODEL_DIR = Path(
    NOTEBOOK_12_MODEL_DIR
).expanduser().resolve()


NOTEBOOK_12_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NOTEBOOK_12_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


NOTEBOOK_12_CALIBRATION_CHECKPOINT_DIR = (
    NOTEBOOK_12_MODEL_DIR
    /
    "calibration_evaluation_checkpoints"
)

NOTEBOOK_12_CALIBRATION_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


LOSS_START_SAMPLE_INDEX = int(
    globals().get(
        "LOSS_START_SAMPLE_INDEX",
        1,
    )
)


GROUP_SUPPORT_THRESHOLD = float(
    globals().get(
        "GROUP_SUPPORT_THRESHOLD",
        1.0e-8,
    )
)


CALIBRATION_COMPLEXITY_TOLERANCE = float(
    globals().get(
        "CALIBRATION_COMPLEXITY_TOLERANCE",
        0.01,
    )
)


EVALUATION_TRAJECTORY_LIMIT = globals().get(
    "EVALUATION_TRAJECTORY_LIMIT",
    None,
)


RESUME_CALIBRATION_EVALUATION: bool = bool(
    globals().get(
        "RESUME_CALIBRATION_EVALUATION",
        True,
    )
)


FORCE_RESTART_CALIBRATION_EVALUATION: bool = bool(
    globals().get(
        "FORCE_RESTART_CALIBRATION_EVALUATION",
        False,
    )
)


CALIBRATION_PROGRESS_INTERVAL: int = int(
    globals().get(
        "CALIBRATION_PROGRESS_INTERVAL",
        100,
    )
)


if LOSS_START_SAMPLE_INDEX < 0:
    raise ValueError(
        "LOSS_START_SAMPLE_INDEX must be nonnegative."
    )


if (
    not np.isfinite(
        GROUP_SUPPORT_THRESHOLD
    )
    or
    GROUP_SUPPORT_THRESHOLD
    <
    0.0
):
    raise ValueError(
        "GROUP_SUPPORT_THRESHOLD must be finite and nonnegative."
    )


if (
    not np.isfinite(
        CALIBRATION_COMPLEXITY_TOLERANCE
    )
    or
    CALIBRATION_COMPLEXITY_TOLERANCE
    <
    0.0
):
    raise ValueError(
        "CALIBRATION_COMPLEXITY_TOLERANCE must be finite and nonnegative."
    )


if EVALUATION_TRAJECTORY_LIMIT is not None:
    EVALUATION_TRAJECTORY_LIMIT = int(
        EVALUATION_TRAJECTORY_LIMIT
    )

    if EVALUATION_TRAJECTORY_LIMIT <= 0:
        raise ValueError(
            "EVALUATION_TRAJECTORY_LIMIT must be positive or None."
        )


if CALIBRATION_PROGRESS_INTERVAL <= 0:
    raise ValueError(
        "CALIBRATION_PROGRESS_INTERVAL must be positive."
    )


# ======================================================================================
# 4. DEFINE OUTPUT PATHS
# ======================================================================================

NOTEBOOK_12_CALIBRATION_JOB_METRICS_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_calibration_candidate_metrics.parquet"
)


NOTEBOOK_12_CALIBRATION_TRAJECTORY_METRICS_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_calibration_trajectory_metrics.parquet"
)


NOTEBOOK_12_CALIBRATION_SELECTION_POOL_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_calibration_selection_pool.parquet"
)


NOTEBOOK_12_CALIBRATION_SELECTION_CONTRACT_PATH = (
    NOTEBOOK_12_MODEL_DIR
    /
    "lpv_calibration_selection.json"
)


NOTEBOOK_12_SELECTED_TRAINED_COEFFICIENT_PATH = (
    NOTEBOOK_12_MODEL_DIR
    /
    "lpv_selected_trained_coefficients.npz"
)


# ======================================================================================
# 5. DEFINE ATOMIC FILE HELPERS
# ======================================================================================

def json_safe_cell_22(
    value: Any,
) -> Any:
    """
    Convert nested NumPy values to JSON-compatible Python values.
    """

    if isinstance(
        value,
        Mapping,
    ):
        return {
            str(key): json_safe_cell_22(
                item
            )
            for key, item in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        return [
            json_safe_cell_22(
                item
            )
            for item in value
        ]

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    return value


def atomic_write_json_cell_22(
    output_path: Path,
    payload: Mapping[str, Any],
) -> None:
    """
    Atomically write one JSON file.
    """

    output_path = Path(
        output_path
    ).expanduser().resolve()

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = output_path.with_name(
        output_path.name
        +
        ".tmp"
    )


    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        json.dump(
            json_safe_cell_22(
                dict(
                    payload
                )
            ),
            file_handle,
            indent=2,
            sort_keys=True,
        )

        file_handle.write(
            "\n"
        )


    os.replace(
        temporary_path,
        output_path,
    )


def atomic_write_parquet_cell_22(
    dataframe: pd.DataFrame,
    output_path: Path,
) -> None:
    """
    Atomically write one Parquet dataframe.
    """

    output_path = Path(
        output_path
    ).expanduser().resolve()

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = output_path.with_name(
        output_path.stem
        +
        ".tmp.parquet"
    )


    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )


    os.replace(
        temporary_path,
        output_path,
    )


def atomic_write_npz_cell_22(
    output_path: Path,
    **arrays: np.ndarray,
) -> None:
    """
    Atomically write one compressed NumPy archive.
    """

    output_path = Path(
        output_path
    ).expanduser().resolve()

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = output_path.with_name(
        output_path.stem
        +
        ".tmp.npz"
    )


    if temporary_path.exists():
        temporary_path.unlink()


    np.savez_compressed(
        temporary_path,
        **arrays,
    )


    os.replace(
        temporary_path,
        output_path,
    )


# ======================================================================================
# 6. DEFINE GENERIC JOB ACCESSORS
# ======================================================================================

_MISSING_JOB_VALUE_CELL_22 = object()


def get_job_value_cell_22(
    job: Any,
    names: Sequence[str],
    *,
    default: Any = _MISSING_JOB_VALUE_CELL_22,
) -> Any:
    """
    Read one field from a mapping, dataclass, named tuple, or object.
    """

    for name in names:
        if isinstance(
            job,
            Mapping,
        ):
            if name in job:
                return job[
                    name
                ]

        if hasattr(
            job,
            name,
        ):
            return getattr(
                job,
                name,
            )


    if default is not _MISSING_JOB_VALUE_CELL_22:
        return default


    raise AttributeError(
        "The candidate job does not provide any of these fields:\n  - "
        + "\n  - ".join(
            names
        )
    )


def normalize_job_id_cell_22(
    job: Any,
) -> str:
    """
    Return the candidate fitting-job identifier.
    """

    return str(
        get_job_value_cell_22(
            job,
            (
                "job_id",
                "fit_job_id",
                "id",
            ),
        )
    ).strip()


def normalize_candidate_id_cell_22(
    job: Any,
) -> str:
    """
    Return the candidate model-family identifier.
    """

    candidate_id = get_job_value_cell_22(
        job,
        (
            "candidate_id",
            "model_candidate_id",
            "model_id",
        ),
        default=None,
    )


    if candidate_id is not None:
        return str(
            candidate_id
        ).strip()


    job_id = normalize_job_id_cell_22(
        job
    )


    if "__lambda_" in job_id:
        return job_id.split(
            "__lambda_",
            maxsplit=1,
        )[
            0
        ]


    raise RuntimeError(
        "The candidate ID could not be recovered from job ID:\n"
        f"  {job_id}"
    )


def normalize_lambda_cell_22(
    job: Any,
) -> float:
    """
    Return the regularization value associated with one fitting job.
    """

    value = get_job_value_cell_22(
        job,
        (
            "regularization_lambda",
            "lambda_value",
            "sparse_lambda",
            "penalty_lambda",
        ),
        default=None,
    )


    if value is None:
        job_id = normalize_job_id_cell_22(
            job
        )


        if "__lambda_" not in job_id:
            return 0.0


        encoded_value = job_id.split(
            "__lambda_",
            maxsplit=1,
        )[
            1
        ]


        encoded_value = encoded_value.replace(
            "em",
            "e-",
        )

        encoded_value = encoded_value.replace(
            "ep",
            "e+",
        )

        encoded_value = encoded_value.replace(
            "p",
            ".",
        )


        value = float(
            encoded_value
        )


    value = float(
        value
    )


    if not np.isfinite(
        value
    ):
        raise ValueError(
            "A regularization value is nonfinite."
        )


    if value < 0.0:
        raise ValueError(
            "A regularization value is negative."
        )


    return value


def candidate_is_sparse_cell_22(
    candidate_id: str,
) -> bool:
    """
    Return whether a candidate is registered as sparse.
    """

    specification = MODEL_CANDIDATE_SPECIFICATIONS[
        candidate_id
    ]


    if isinstance(
        specification,
        Mapping,
    ):
        return bool(
            specification.get(
                "sparse",
                False,
            )
        )


    return bool(
        getattr(
            specification,
            "sparse",
            False,
        )
    )


# ======================================================================================
# 7. NORMALIZE THE CANDIDATE-JOB REGISTRY
# ======================================================================================

candidate_fit_jobs_cell_22 = list(
    NOTEBOOK_12_CANDIDATE_FIT_JOBS
)


if len(
    candidate_fit_jobs_cell_22
) == 0:
    raise RuntimeError(
        "No candidate fitting jobs were registered."
    )


NORMALIZED_CANDIDATE_JOBS_CELL_22: list[
    dict[str, Any]
] = []


for job_position, job in enumerate(
    candidate_fit_jobs_cell_22
):
    job_id = normalize_job_id_cell_22(
        job
    )

    candidate_id = normalize_candidate_id_cell_22(
        job
    )

    regularization_lambda = normalize_lambda_cell_22(
        job
    )


    if candidate_id not in CANDIDATE_ACTIVE_MASKS:
        raise RuntimeError(
            "A calibration job references an unknown candidate.\n"
            f"Job ID: {job_id}\n"
            f"Candidate ID: {candidate_id}"
        )


    active_mask = np.asarray(
        CANDIDATE_ACTIVE_MASKS[
            candidate_id
        ],
        dtype=bool,
    ).reshape(
        -1
    )


    if not active_mask.any():
        raise RuntimeError(
            f"Candidate {candidate_id!r} has no active coefficients."
        )


    NORMALIZED_CANDIDATE_JOBS_CELL_22.append(
        {
            "job_position": int(
                job_position
            ),
            "job": job,
            "job_id": job_id,
            "candidate_id": candidate_id,
            "regularization_lambda": float(
                regularization_lambda
            ),
        }
    )


normalized_job_ids_cell_22 = [
    record[
        "job_id"
    ]
    for record in NORMALIZED_CANDIDATE_JOBS_CELL_22
]


if len(
    set(
        normalized_job_ids_cell_22
    )
) != len(
    normalized_job_ids_cell_22
):
    raise RuntimeError(
        "The candidate fitting-job registry contains duplicate job IDs."
    )


candidate_mask_lengths_cell_22 = {
    int(
        np.asarray(
            candidate_mask,
            dtype=bool,
        ).size
    )
    for candidate_mask in CANDIDATE_ACTIVE_MASKS.values()
}


if len(
    candidate_mask_lengths_cell_22
) != 1:
    raise RuntimeError(
        "Candidate active masks do not share one reduced dimension."
    )


NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT = int(
    next(
        iter(
            candidate_mask_lengths_cell_22
        )
    )
)


# ======================================================================================
# 8. RECOVER FITTED COEFFICIENTS FROM MEMORY OR HARD DRIVE
# ======================================================================================

def build_candidate_coefficient_lookup_cell_22() -> dict[
    str,
    np.ndarray,
]:
    """
    Recover the best fitted coefficient vector for every candidate job.
    """

    coefficient_lookup: dict[
        str,
        np.ndarray,
    ] = {}


    # ------------------------------------------------------------------
    # Preferred in-memory mapping produced by checkpointed Cell 21
    # ------------------------------------------------------------------

    if (
        "NOTEBOOK_12_CANDIDATE_COEFFICIENT_BY_JOB_ID"
        in globals()
    ):
        source_mapping = globals()[
            "NOTEBOOK_12_CANDIDATE_COEFFICIENT_BY_JOB_ID"
        ]


        for job_id, coefficients in source_mapping.items():
            coefficient_lookup[
                str(
                    job_id
                )
            ] = np.array(
                coefficients,
                dtype=np.float64,
                copy=True,
            ).reshape(
                -1
            )


    # ------------------------------------------------------------------
    # Result mapping produced by checkpointed Cell 21
    # ------------------------------------------------------------------

    if (
        "NOTEBOOK_12_CANDIDATE_FIT_RESULT_BY_JOB_ID"
        in globals()
    ):
        result_mapping = globals()[
            "NOTEBOOK_12_CANDIDATE_FIT_RESULT_BY_JOB_ID"
        ]


        for job_id, result in result_mapping.items():
            normalized_job_id = str(
                job_id
            )


            if normalized_job_id in coefficient_lookup:
                continue


            if not isinstance(
                result,
                Mapping,
            ):
                continue


            for coefficient_key in (
                "best_reduced_coefficients",
                "coefficients",
                "final_reduced_coefficients",
            ):
                if coefficient_key in result:
                    coefficient_lookup[
                        normalized_job_id
                    ] = np.array(
                        result[
                            coefficient_key
                        ],
                        dtype=np.float64,
                        copy=True,
                    ).reshape(
                        -1
                    )

                    break


    # ------------------------------------------------------------------
    # Compatibility with the original Cell 21 result mapping
    # ------------------------------------------------------------------

    if "CANDIDATE_FIT_RESULTS" in globals():
        legacy_mapping = globals()[
            "CANDIDATE_FIT_RESULTS"
        ]


        if isinstance(
            legacy_mapping,
            Mapping,
        ):
            for job_id, result in legacy_mapping.items():
                normalized_job_id = str(
                    job_id
                )


                if normalized_job_id in coefficient_lookup:
                    continue


                if not isinstance(
                    result,
                    Mapping,
                ):
                    continue


                for coefficient_key in (
                    "coefficients",
                    "best_reduced_coefficients",
                    "final_reduced_coefficients",
                ):
                    if coefficient_key in result:
                        coefficient_lookup[
                            normalized_job_id
                        ] = np.array(
                            result[
                                coefficient_key
                            ],
                            dtype=np.float64,
                            copy=True,
                        ).reshape(
                            -1
                        )

                        break


    # ------------------------------------------------------------------
    # Saved consolidated coefficient archive
    # ------------------------------------------------------------------

    candidate_coefficient_path = globals().get(
        "NOTEBOOK_12_CANDIDATE_FIT_COEFFICIENT_PATH",
        (
            NOTEBOOK_12_MODEL_DIR
            /
            "lpv_candidate_fit_coefficients.npz"
        ),
    )


    candidate_coefficient_path = Path(
        candidate_coefficient_path
    ).expanduser().resolve()


    if candidate_coefficient_path.is_file():
        with np.load(
            candidate_coefficient_path,
            allow_pickle=False,
        ) as coefficient_archive:
            archived_job_ids = coefficient_archive[
                "job_ids"
            ].astype(str)


            archived_coefficients = np.asarray(
                coefficient_archive[
                    "best_reduced_coefficients"
                ],
                dtype=np.float64,
            )


        if archived_coefficients.ndim != 2:
            raise RuntimeError(
                "The saved candidate coefficient matrix is not "
                "two-dimensional."
            )


        if archived_coefficients.shape[
            0
        ] != archived_job_ids.size:
            raise RuntimeError(
                "The saved candidate job-ID and coefficient counts differ."
            )


        for row_position, job_id in enumerate(
            archived_job_ids
        ):
            normalized_job_id = str(
                job_id
            )


            if normalized_job_id not in coefficient_lookup:
                coefficient_lookup[
                    normalized_job_id
                ] = np.array(
                    archived_coefficients[
                        row_position
                    ],
                    dtype=np.float64,
                    copy=True,
                )


    # ------------------------------------------------------------------
    # Final validation
    # ------------------------------------------------------------------

    missing_job_coefficients = [
        job_id
        for job_id in normalized_job_ids_cell_22
        if job_id not in coefficient_lookup
    ]


    if missing_job_coefficients:
        raise RuntimeError(
            "Fitted coefficients could not be recovered for these jobs:\n  - "
            + "\n  - ".join(
                missing_job_coefficients
            )
            +
            "\n\nExecute the checkpointed Cell 21 or verify the saved "
            "candidate coefficient archive."
        )


    for job_id in normalized_job_ids_cell_22:
        coefficient_array = np.array(
            coefficient_lookup[
                job_id
            ],
            dtype=np.float64,
            copy=True,
        ).reshape(
            -1
        )


        if coefficient_array.shape != (
            NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
        ):
            raise RuntimeError(
                f"Job {job_id!r} has an incorrect coefficient dimension.\n"
                f"Expected: "
                f"{NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT:,}\n"
                f"Observed: {coefficient_array.size:,}"
            )


        if not np.isfinite(
            coefficient_array
        ).all():
            raise RuntimeError(
                f"Job {job_id!r} has nonfinite fitted coefficients."
            )


        coefficient_lookup[
            job_id
        ] = coefficient_array


    return coefficient_lookup


CANDIDATE_COEFFICIENT_LOOKUP_CELL_22 = (
    build_candidate_coefficient_lookup_cell_22()
)


# Preserve compatibility with the original downstream notebook code.

CANDIDATE_FIT_RESULTS = {
    job_id: {
        "coefficients": np.array(
            coefficients,
            dtype=np.float64,
            copy=True,
        )
    }
    for job_id, coefficients in (
        CANDIDATE_COEFFICIENT_LOOKUP_CELL_22.items()
    )
}


# ======================================================================================
# 9. DEFINE THE TRAJECTORY SIMULATION WRAPPER
# ======================================================================================

def simulate_loaded_trajectory_numpy(
    coefficients: np.ndarray,
    trajectory_data: Mapping[str, Any],
) -> dict[str, Any]:
    """
    Simulate one complete trajectory using the fitted reduced coefficients.
    """

    coefficient_array = np.array(
        coefficients,
        dtype=np.float64,
        copy=True,
    ).reshape(
        -1
    )


    basis_matrix = np.asarray(
        trajectory_data[
            "basis_matrix"
        ],
        dtype=np.float64,
    )


    current_discharge_a = np.asarray(
        trajectory_data[
            "current_discharge_a"
        ],
        dtype=np.float64,
    )


    voltage_v = np.asarray(
        trajectory_data[
            "voltage_v"
        ],
        dtype=np.float64,
    )


    ocv_v = np.asarray(
        trajectory_data[
            "ocv_v"
        ],
        dtype=np.float64,
    )


    delta_t_s = np.asarray(
        trajectory_data[
            "delta_t_s"
        ],
        dtype=np.float64,
    )


    sample_count = int(
        voltage_v.size
    )


    if sample_count < 2:
        raise RuntimeError(
            "A calibration trajectory contains fewer than two samples."
        )


    expected_vector_shape = (
        sample_count,
    )


    for array_name, array in (
        (
            "current_discharge_a",
            current_discharge_a,
        ),
        (
            "voltage_v",
            voltage_v,
        ),
        (
            "ocv_v",
            ocv_v,
        ),
        (
            "delta_t_s",
            delta_t_s,
        ),
    ):
        if array.shape != expected_vector_shape:
            raise RuntimeError(
                f"Trajectory array {array_name!r} has an incorrect shape."
            )


        if not np.isfinite(
            array
        ).all():
            raise RuntimeError(
                f"Trajectory array {array_name!r} contains nonfinite values."
            )


    if basis_matrix.shape[
        0
    ] != sample_count:
        raise RuntimeError(
            "The basis matrix and trajectory sample count differ."
        )


    if not np.isfinite(
        basis_matrix
    ).all():
        raise RuntimeError(
            "The trajectory basis matrix contains nonfinite values."
        )


    used_batch_simulator = False


    if "simulate_lpv_trajectory_jax" in globals():
        outputs = simulate_lpv_trajectory_jax(
            jnp.asarray(
                coefficient_array,
                dtype=jnp.float64,
            ),

            jnp.asarray(
                basis_matrix,
                dtype=jnp.float64,
            ),

            jnp.asarray(
                current_discharge_a,
                dtype=jnp.float64,
            ),

            jnp.asarray(
                voltage_v,
                dtype=jnp.float64,
            ),

            jnp.asarray(
                ocv_v,
                dtype=jnp.float64,
            ),

            jnp.asarray(
                delta_t_s,
                dtype=jnp.float64,
            ),

            jnp.asarray(
                trajectory_data[
                    "global_full_indices"
                ],
                dtype=jnp.int32,
            ),

            jnp.asarray(
                trajectory_data[
                    "chemistry_full_indices"
                ],
                dtype=jnp.int32,
            ),

            jnp.asarray(
                trajectory_data[
                    "cell_full_indices"
                ],
                dtype=jnp.int32,
            ),

            jnp.asarray(
                trajectory_data[
                    "has_cell_effect"
                ],
                dtype=jnp.bool_,
            ),
        )

    else:
        used_batch_simulator = True


        if "simulate_lpv_batch_jax" in globals():
            batch_simulator = globals()[
                "simulate_lpv_batch_jax"
            ]

        elif "simulate_lpv_batch_core" in globals():
            batch_simulator = globals()[
                "simulate_lpv_batch_core"
            ]

        else:
            batch_simulator = globals()[
                "_simulate_lpv_batch_core"
            ]


        valid_mask = np.ones(
            (
                1,
                sample_count,
            ),
            dtype=bool,
        )


        loss_mask = valid_mask.copy()

        loss_mask[
            :,
            0,
        ] = False


        single_trajectory_batch = {
            "basis": jnp.asarray(
                basis_matrix[
                    None,
                    :,
                    :,
                ],
                dtype=jnp.float64,
            ),

            "current": jnp.asarray(
                current_discharge_a[
                    None,
                    :,
                ],
                dtype=jnp.float64,
            ),

            "voltage": jnp.asarray(
                voltage_v[
                    None,
                    :,
                ],
                dtype=jnp.float64,
            ),

            "ocv": jnp.asarray(
                ocv_v[
                    None,
                    :,
                ],
                dtype=jnp.float64,
            ),

            "delta_t": jnp.asarray(
                delta_t_s[
                    None,
                    :,
                ],
                dtype=jnp.float64,
            ),

            "valid_mask": jnp.asarray(
                valid_mask,
                dtype=jnp.bool_,
            ),

            "loss_mask": jnp.asarray(
                loss_mask,
                dtype=jnp.bool_,
            ),

            "global_indices": jnp.asarray(
                np.asarray(
                    trajectory_data[
                        "global_full_indices"
                    ],
                    dtype=np.int32,
                )[
                    None,
                    :,
                    :,
                ],
                dtype=jnp.int32,
            ),

            "chemistry_indices": jnp.asarray(
                np.asarray(
                    trajectory_data[
                        "chemistry_full_indices"
                    ],
                    dtype=np.int32,
                )[
                    None,
                    :,
                    :,
                ],
                dtype=jnp.int32,
            ),

            "cell_indices": jnp.asarray(
                np.asarray(
                    trajectory_data[
                        "cell_full_indices"
                    ],
                    dtype=np.int32,
                )[
                    None,
                    :,
                    :,
                ],
                dtype=jnp.int32,
            ),

            "has_cell_effect": jnp.asarray(
                [
                    bool(
                        trajectory_data[
                            "has_cell_effect"
                        ]
                    )
                ],
                dtype=jnp.bool_,
            ),
        }


        outputs = batch_simulator(
            jnp.asarray(
                coefficient_array,
                dtype=jnp.float64,
            ),
            single_trajectory_batch,
        )


    if len(
        outputs
    ) < 12:
        raise RuntimeError(
            "The LPV simulator returned fewer outputs than expected."
        )


    def convert_simulator_output(
        output_index: int,
    ) -> np.ndarray:
        """
        Convert one simulator output to an owned NumPy array.
        """

        output_array = np.array(
            outputs[
                output_index
            ],
            dtype=np.float64,
            copy=True,
        )


        if used_batch_simulator:
            if (
                output_array.ndim >= 1
                and
                output_array.shape[
                    0
                ]
                ==
                1
            ):
                output_array = np.array(
                    output_array[
                        0
                    ],
                    dtype=np.float64,
                    copy=True,
                )


        return output_array


    if len(
        outputs
    ) > 14:
        rest_start_array = np.asarray(
            outputs[
                14
            ]
        ).reshape(
            -1
        )


        rest_start_detected = bool(
            rest_start_array[
                0
            ]
        )

    else:
        rest_current_threshold = float(
            globals().get(
                "REST_CURRENT_A",
                0.05,
            )
        )

        rest_sample_count = min(
            int(
                globals().get(
                    "INITIAL_REST_SAMPLES",
                    5,
                )
            ),
            sample_count,
        )


        rest_start_detected = bool(
            np.all(
                np.abs(
                    current_discharge_a[
                        :
                        rest_sample_count
                    ]
                )
                <=
                rest_current_threshold
            )
        )


    result = {
        "predicted_voltage_v": convert_simulator_output(
            0
        ),

        "residual_v": convert_simulator_output(
            1
        ),

        "transformed_parameters": convert_simulator_output(
            2
        ),

        "R0_ohm": convert_simulator_output(
            3
        ),

        "R1_ohm": convert_simulator_output(
            4
        ),

        "R2_ohm": convert_simulator_output(
            5
        ),

        "tau1_s": convert_simulator_output(
            6
        ),

        "tau2_s": convert_simulator_output(
            7
        ),

        "alpha1": convert_simulator_output(
            8
        ),

        "alpha2": convert_simulator_output(
            9
        ),

        "v1_state_v": convert_simulator_output(
            10
        ),

        "v2_state_v": convert_simulator_output(
            11
        ),

        "rest_start_detected": bool(
            rest_start_detected
        ),
    }


    for array_name in (
        "predicted_voltage_v",
        "residual_v",
        "R0_ohm",
        "R1_ohm",
        "R2_ohm",
        "tau1_s",
        "tau2_s",
        "alpha1",
        "alpha2",
        "v1_state_v",
        "v2_state_v",
    ):
        array = np.asarray(
            result[
                array_name
            ],
            dtype=np.float64,
        )


        if array.shape != (
            sample_count,
        ):
            raise RuntimeError(
                f"Simulator output {array_name!r} has an incorrect shape.\n"
                f"Expected: ({sample_count},)\n"
                f"Observed: {array.shape}"
            )


        if not np.isfinite(
            array
        ).all():
            raise RuntimeError(
                f"Simulator output {array_name!r} contains nonfinite values."
            )


    return result


# ======================================================================================
# 10. DEFINE RESIDUAL METRICS
# ======================================================================================

def calculate_residual_metrics(
    residual_v: np.ndarray,
) -> dict[str, float]:
    """
    Calculate trajectory-level voltage residual metrics.
    """

    residual_array = np.asarray(
        residual_v,
        dtype=np.float64,
    ).reshape(
        -1
    )


    if residual_array.size == 0:
        raise RuntimeError(
            "A calibration trajectory contains no evaluation residuals."
        )


    if not np.isfinite(
        residual_array
    ).all():
        raise RuntimeError(
            "A calibration residual vector contains nonfinite values."
        )


    squared_residual = np.square(
        residual_array
    )

    absolute_residual = np.abs(
        residual_array
    )


    return {
        "mse_v2": float(
            np.mean(
                squared_residual
            )
        ),

        "rmse_v": float(
            np.sqrt(
                np.mean(
                    squared_residual
                )
            )
        ),

        "mae_v": float(
            np.mean(
                absolute_residual
            )
        ),

        "bias_v": float(
            np.mean(
                residual_array
            )
        ),

        "residual_standard_deviation_v": float(
            np.std(
                residual_array,
                ddof=0,
            )
        ),

        "maximum_absolute_residual_v": float(
            np.max(
                absolute_residual
            )
        ),
    }


# ======================================================================================
# 11. DEFINE CALIBRATION-PARTITION EVALUATION
# ======================================================================================

def evaluate_development_partition(
    coefficients: np.ndarray,
    *,
    partition: str,
    trajectory_limit: int | None = None,
) -> tuple[
    dict[str, Any],
    pd.DataFrame,
]:
    """
    Evaluate one fitted coefficient vector on one approved partition.
    """

    normalized_partition = str(
        partition
    ).strip().lower()


    if normalized_partition != "calibration":
        raise ValueError(
            "Cell 22 permits calibration-partition evaluation only."
        )


    parquet_file = get_design_parquet(
        normalized_partition
    )


    available_row_group_count = int(
        parquet_file.metadata.num_row_groups
    )


    if available_row_group_count <= 0:
        raise RuntimeError(
            "The calibration design file contains no row groups."
        )


    evaluation_row_group_count = (
        available_row_group_count
    )


    if trajectory_limit is not None:
        evaluation_row_group_count = min(
            available_row_group_count,
            int(
                trajectory_limit
            ),
        )


    trajectory_metric_records: list[
        dict[str, Any]
    ] = []


    pooled_squared_error = 0.0
    pooled_absolute_error = 0.0
    pooled_residual_sum = 0.0
    pooled_maximum_absolute_residual = 0.0
    pooled_sample_count = 0


    evaluation_start_time = time.perf_counter()


    progress_interval = min(
        CALIBRATION_PROGRESS_INTERVAL,
        max(
            1,
            evaluation_row_group_count,
        ),
    )


    for row_group_index in range(
        evaluation_row_group_count
    ):
        trajectory_data = load_design_trajectory(
            normalized_partition,
            row_group_index,
        )


        simulation = simulate_loaded_trajectory_numpy(
            coefficients,
            trajectory_data,
        )


        full_residual = np.asarray(
            simulation[
                "residual_v"
            ],
            dtype=np.float64,
        )


        if LOSS_START_SAMPLE_INDEX >= full_residual.size:
            raise RuntimeError(
                "LOSS_START_SAMPLE_INDEX removes every sample from a "
                "calibration trajectory.\n"
                f"Trajectory: {trajectory_data['trajectory_uid']}\n"
                f"Sample count: {full_residual.size:,}\n"
                f"Loss start index: {LOSS_START_SAMPLE_INDEX:,}"
            )


        residual = full_residual[
            LOSS_START_SAMPLE_INDEX:
        ]


        metrics = calculate_residual_metrics(
            residual
        )


        residual_squared_sum = float(
            np.sum(
                np.square(
                    residual
                )
            )
        )


        residual_absolute_sum = float(
            np.sum(
                np.abs(
                    residual
                )
            )
        )


        residual_sum = float(
            np.sum(
                residual
            )
        )


        residual_maximum_absolute = float(
            np.max(
                np.abs(
                    residual
                )
            )
        )


        pooled_squared_error += (
            residual_squared_sum
        )

        pooled_absolute_error += (
            residual_absolute_sum
        )

        pooled_residual_sum += residual_sum

        pooled_maximum_absolute_residual = max(
            pooled_maximum_absolute_residual,
            residual_maximum_absolute,
        )

        pooled_sample_count += int(
            residual.size
        )


        nonterminal_slice = slice(
            0,
            max(
                1,
                int(
                    trajectory_data[
                        "sample_count"
                    ]
                )
                -
                1,
            ),
        )


        tau_gap_s = (
            np.asarray(
                simulation[
                    "tau2_s"
                ],
                dtype=np.float64,
            )
            -
            np.asarray(
                simulation[
                    "tau1_s"
                ],
                dtype=np.float64,
            )
        )


        trajectory_metric_records.append(
            {
                "partition": normalized_partition,

                "row_group_index": int(
                    row_group_index
                ),

                "trajectory_uid": str(
                    trajectory_data[
                        "trajectory_uid"
                    ]
                ),

                "chemistry": str(
                    trajectory_data[
                        "chemistry"
                    ]
                ),

                "cell_entity_id": str(
                    trajectory_data[
                        "cell_entity_id"
                    ]
                ),

                "sample_count": int(
                    trajectory_data[
                        "sample_count"
                    ]
                ),

                "evaluation_sample_count": int(
                    residual.size
                ),

                "rest_start_detected": bool(
                    simulation[
                        "rest_start_detected"
                    ]
                ),

                **metrics,

                "minimum_R0_ohm": float(
                    np.min(
                        simulation[
                            "R0_ohm"
                        ]
                    )
                ),

                "maximum_R0_ohm": float(
                    np.max(
                        simulation[
                            "R0_ohm"
                        ]
                    )
                ),

                "minimum_R1_ohm": float(
                    np.min(
                        simulation[
                            "R1_ohm"
                        ]
                    )
                ),

                "maximum_R1_ohm": float(
                    np.max(
                        simulation[
                            "R1_ohm"
                        ]
                    )
                ),

                "minimum_R2_ohm": float(
                    np.min(
                        simulation[
                            "R2_ohm"
                        ]
                    )
                ),

                "maximum_R2_ohm": float(
                    np.max(
                        simulation[
                            "R2_ohm"
                        ]
                    )
                ),

                "minimum_tau1_s": float(
                    np.min(
                        simulation[
                            "tau1_s"
                        ]
                    )
                ),

                "maximum_tau1_s": float(
                    np.max(
                        simulation[
                            "tau1_s"
                        ]
                    )
                ),

                "minimum_tau_gap_s": float(
                    np.min(
                        tau_gap_s
                    )
                ),

                "maximum_tau_gap_s": float(
                    np.max(
                        tau_gap_s
                    )
                ),

                "minimum_tau2_s": float(
                    np.min(
                        simulation[
                            "tau2_s"
                        ]
                    )
                ),

                "maximum_tau2_s": float(
                    np.max(
                        simulation[
                            "tau2_s"
                        ]
                    )
                ),

                "maximum_alpha1": float(
                    np.max(
                        np.asarray(
                            simulation[
                                "alpha1"
                            ],
                            dtype=np.float64,
                        )[
                            nonterminal_slice
                        ]
                    )
                ),

                "maximum_alpha2": float(
                    np.max(
                        np.asarray(
                            simulation[
                                "alpha2"
                            ],
                            dtype=np.float64,
                        )[
                            nonterminal_slice
                        ]
                    )
                ),
            }
        )


        completed_trajectory_count = (
            row_group_index
            +
            1
        )


        if (
            completed_trajectory_count
            ==
            1
            or
            completed_trajectory_count
            %
            progress_interval
            ==
            0
            or
            completed_trajectory_count
            ==
            evaluation_row_group_count
        ):
            elapsed_seconds = (
                time.perf_counter()
                -
                evaluation_start_time
            )


            trajectories_per_second = (
                completed_trajectory_count
                /
                max(
                    elapsed_seconds,
                    np.finfo(
                        np.float64
                    ).tiny,
                )
            )


            remaining_trajectory_count = (
                evaluation_row_group_count
                -
                completed_trajectory_count
            )


            estimated_remaining_seconds = (
                remaining_trajectory_count
                /
                max(
                    trajectories_per_second,
                    np.finfo(
                        np.float64
                    ).tiny,
                )
            )


            print(
                f"        Calibration trajectories "
                f"{completed_trajectory_count:,}/"
                f"{evaluation_row_group_count:,} | "
                f"elapsed={elapsed_seconds:.1f} s | "
                f"ETA={estimated_remaining_seconds:.1f} s"
            )


    trajectory_metrics_df = pd.DataFrame(
        trajectory_metric_records
    )


    if trajectory_metrics_df.empty:
        raise RuntimeError(
            "The calibration trajectory-metrics table is empty."
        )


    if pooled_sample_count <= 0:
        raise RuntimeError(
            "No calibration evaluation samples were available."
        )


    trajectory_rmse_series = pd.to_numeric(
        trajectory_metrics_df[
            "rmse_v"
        ],
        errors="raise",
    )


    summary = {
        "partition": normalized_partition,

        "available_trajectory_count": int(
            available_row_group_count
        ),

        "trajectory_count": int(
            len(
                trajectory_metrics_df
            )
        ),

        "sample_count": int(
            pooled_sample_count
        ),

        "pooled_rmse_v": float(
            math.sqrt(
                pooled_squared_error
                /
                pooled_sample_count
            )
        ),

        "pooled_mae_v": float(
            pooled_absolute_error
            /
            pooled_sample_count
        ),

        "pooled_bias_v": float(
            pooled_residual_sum
            /
            pooled_sample_count
        ),

        "pooled_maximum_absolute_residual_v": float(
            pooled_maximum_absolute_residual
        ),

        "trajectory_weighted_rmse_v": float(
            trajectory_rmse_series.mean()
        ),

        "median_trajectory_rmse_v": float(
            trajectory_rmse_series.median()
        ),

        "p90_trajectory_rmse_v": float(
            trajectory_rmse_series.quantile(
                0.90
            )
        ),

        "p95_trajectory_rmse_v": float(
            trajectory_rmse_series.quantile(
                0.95
            )
        ),

        "maximum_trajectory_rmse_v": float(
            trajectory_rmse_series.max()
        ),

        "elapsed_seconds": float(
            time.perf_counter()
            -
            evaluation_start_time
        ),
    }


    for summary_name, summary_value in summary.items():
        if isinstance(
            summary_value,
            float,
        ):
            if not np.isfinite(
                summary_value
            ):
                raise RuntimeError(
                    f"Calibration summary {summary_name!r} is nonfinite."
                )


    return summary, trajectory_metrics_df


# ======================================================================================
# 12. DEFINE SUPPORT AND GROUP-PENALTY SUMMARIES
# ======================================================================================

def coefficient_support_summary(
    coefficients: np.ndarray,
    *,
    candidate_id: str,
) -> dict[str, int]:
    """
    Count active reduced coefficients and active hierarchy groups.
    """

    coefficient_array = np.asarray(
        coefficients,
        dtype=np.float64,
    ).reshape(
        -1
    )


    candidate_mask = np.asarray(
        CANDIDATE_ACTIVE_MASKS[
            candidate_id
        ],
        dtype=bool,
    ).reshape(
        -1
    )


    if coefficient_array.shape != candidate_mask.shape:
        raise RuntimeError(
            "The coefficient vector and candidate active mask differ in shape."
        )


    active_candidate_coefficients = coefficient_array[
        candidate_mask
    ]


    nonzero_coefficient_count = int(
        (
            np.abs(
                active_candidate_coefficients
            )
            >
            GROUP_SUPPORT_THRESHOLD
        ).sum()
    )


    relevant_group_ids = tuple(
        str(
            group_id
        )
        for group_id in CANDIDATE_PENALTY_GROUP_IDS.get(
            candidate_id,
            (),
        )
    )


    if candidate_is_sparse_cell_22(
        candidate_id
    ):
        active_group_count = 0


        for group_id in relevant_group_ids:
            members = np.asarray(
                PENALTY_GROUP_MEMBERS[
                    group_id
                ],
                dtype=np.int64,
            )


            if np.linalg.norm(
                coefficient_array[
                    members
                ]
            ) > GROUP_SUPPORT_THRESHOLD:
                active_group_count += 1

    else:
        active_group_count = 0


        for group_id, raw_members in (
            PENALTY_GROUP_MEMBERS.items()
        ):
            members = np.asarray(
                raw_members,
                dtype=np.int64,
            )


            if np.any(
                candidate_mask[
                    members
                ]
            ):
                active_group_count += 1


    return {
        "active_candidate_coefficient_count": int(
            candidate_mask.sum()
        ),

        "nonzero_coefficient_count": int(
            nonzero_coefficient_count
        ),

        "active_group_count": int(
            active_group_count
        ),
    }


def group_lasso_penalty_numpy_cell_22(
    coefficients: np.ndarray,
    *,
    candidate_id: str,
) -> float:
    """
    Calculate the weighted group-Lasso norm for reporting.
    """

    coefficient_array = np.asarray(
        coefficients,
        dtype=np.float64,
    ).reshape(
        -1
    )


    group_weights = globals().get(
        "PENALTY_GROUP_WEIGHTS",
        {},
    )


    penalty_value = 0.0


    for group_id in CANDIDATE_PENALTY_GROUP_IDS.get(
        candidate_id,
        (),
    ):
        members = np.asarray(
            PENALTY_GROUP_MEMBERS[
                group_id
            ],
            dtype=np.int64,
        )


        group_weight = float(
            group_weights.get(
                group_id,
                1.0,
            )
        )


        penalty_value += (
            group_weight
            *
            float(
                np.linalg.norm(
                    coefficient_array[
                        members
                    ]
                )
            )
        )


    return float(
        penalty_value
    )


# ======================================================================================
# 13. DEFINE CALIBRATION CHECKPOINT HELPERS
# ======================================================================================

def safe_job_name_cell_22(
    job_id: str,
) -> str:
    """
    Create a filesystem-safe calibration checkpoint name.
    """

    cleaned_name = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(
            job_id
        ),
    ).strip(
        "._"
    )


    digest = hashlib.sha256(
        str(
            job_id
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :
        10
    ]


    return (
        cleaned_name[
            :
            100
        ]
        +
        "__"
        +
        digest
    )


def coefficient_sha256_cell_22(
    coefficients: np.ndarray,
) -> str:
    """
    Return the SHA-256 hash of one float64 coefficient vector.
    """

    coefficient_array = np.ascontiguousarray(
        np.asarray(
            coefficients,
            dtype=np.float64,
        )
    )


    return hashlib.sha256(
        coefficient_array.tobytes()
    ).hexdigest()


def calibration_evaluation_signature_cell_22(
    *,
    job_id: str,
    candidate_id: str,
    regularization_lambda: float,
    coefficients: np.ndarray,
) -> str:
    """
    Build a signature for one calibration-evaluation checkpoint.
    """

    signature_payload = {
        "job_id": str(
            job_id
        ),

        "candidate_id": str(
            candidate_id
        ),

        "regularization_lambda": float(
            regularization_lambda
        ),

        "coefficient_sha256": coefficient_sha256_cell_22(
            coefficients
        ),

        "loss_start_sample_index": int(
            LOSS_START_SAMPLE_INDEX
        ),

        "evaluation_trajectory_limit": (
            None
            if EVALUATION_TRAJECTORY_LIMIT is None
            else int(
                EVALUATION_TRAJECTORY_LIMIT
            )
        ),

        "calibration_complexity_tolerance": float(
            CALIBRATION_COMPLEXITY_TOLERANCE
        ),
    }


    canonical_signature_text = json.dumps(
        signature_payload,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


    return hashlib.sha256(
        canonical_signature_text.encode(
            "utf-8"
        )
    ).hexdigest()


def calibration_checkpoint_paths_cell_22(
    job_id: str,
) -> dict[str, Path]:
    """
    Return calibration checkpoint paths for one fitting job.
    """

    job_directory = (
        NOTEBOOK_12_CALIBRATION_CHECKPOINT_DIR
        /
        safe_job_name_cell_22(
            job_id
        )
    )


    job_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    return {
        "directory": job_directory,

        "summary_json": (
            job_directory
            /
            "calibration_summary.json"
        ),

        "trajectory_metrics": (
            job_directory
            /
            "calibration_trajectory_metrics.parquet"
        ),
    }


def load_calibration_checkpoint_cell_22(
    *,
    checkpoint_paths: Mapping[str, Path],
    expected_signature: str,
) -> tuple[
    dict[str, Any],
    pd.DataFrame,
] | None:
    """
    Load one compatible completed calibration checkpoint.
    """

    summary_path = checkpoint_paths[
        "summary_json"
    ]

    trajectory_metric_path = checkpoint_paths[
        "trajectory_metrics"
    ]


    if not (
        summary_path.is_file()
        and
        trajectory_metric_path.is_file()
    ):
        return None


    with summary_path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        saved_summary = json.load(
            file_handle
        )


    if str(
        saved_summary.get(
            "evaluation_signature",
            "",
        )
    ) != str(
        expected_signature
    ):
        return None


    trajectory_metrics_df = pd.read_parquet(
        trajectory_metric_path
    )


    expected_trajectory_count = int(
        saved_summary[
            "trajectory_count"
        ]
    )


    if len(
        trajectory_metrics_df
    ) != expected_trajectory_count:
        raise RuntimeError(
            "A calibration checkpoint has an incorrect trajectory count."
        )


    return saved_summary, trajectory_metrics_df


# ======================================================================================
# 14. EVALUATE OR RESUME ALL CANDIDATE JOBS
# ======================================================================================

calibration_job_records: list[
    dict[str, Any]
] = []


calibration_trajectory_metric_frames: list[
    pd.DataFrame
] = []


cell_22_start_time = time.perf_counter()


for job_number, job_record in enumerate(
    NORMALIZED_CANDIDATE_JOBS_CELL_22,
    start=1,
):
    job_id = str(
        job_record[
            "job_id"
        ]
    )

    candidate_id = str(
        job_record[
            "candidate_id"
        ]
    )

    regularization_lambda = float(
        job_record[
            "regularization_lambda"
        ]
    )


    coefficients = np.array(
        CANDIDATE_COEFFICIENT_LOOKUP_CELL_22[
            job_id
        ],
        dtype=np.float64,
        copy=True,
    )


    print(
        f"[{job_number}/{len(NORMALIZED_CANDIDATE_JOBS_CELL_22)}] "
        f"Calibration evaluation: {job_id}"
    )


    checkpoint_paths = calibration_checkpoint_paths_cell_22(
        job_id
    )


    evaluation_signature = (
        calibration_evaluation_signature_cell_22(
            job_id=job_id,
            candidate_id=candidate_id,
            regularization_lambda=(
                regularization_lambda
            ),
            coefficients=coefficients,
        )
    )


    loaded_checkpoint = None


    if (
        RESUME_CALIBRATION_EVALUATION
        and
        not FORCE_RESTART_CALIBRATION_EVALUATION
    ):
        loaded_checkpoint = (
            load_calibration_checkpoint_cell_22(
                checkpoint_paths=(
                    checkpoint_paths
                ),
                expected_signature=(
                    evaluation_signature
                ),
            )
        )


    if loaded_checkpoint is not None:
        saved_summary, trajectory_metrics_df = (
            loaded_checkpoint
        )


        summary = {
            key: value
            for key, value in saved_summary.items()
            if key not in (
                "created_utc",
                "evaluation_signature",
                "job_id",
                "candidate_id",
                "regularization_lambda",
                "coefficient_sha256",
                "active_candidate_coefficient_count",
                "nonzero_coefficient_count",
                "active_group_count",
                "coefficient_norm",
                "group_penalty",
            )
        }


        support_summary = {
            "active_candidate_coefficient_count": int(
                saved_summary[
                    "active_candidate_coefficient_count"
                ]
            ),

            "nonzero_coefficient_count": int(
                saved_summary[
                    "nonzero_coefficient_count"
                ]
            ),

            "active_group_count": int(
                saved_summary[
                    "active_group_count"
                ]
            ),
        }


        coefficient_norm = float(
            saved_summary[
                "coefficient_norm"
            ]
        )


        reported_group_penalty = float(
            saved_summary[
                "group_penalty"
            ]
        )


        print(
            "    Compatible saved calibration result found; "
            "evaluation skipped."
        )

    else:
        summary, trajectory_metrics_df = (
            evaluate_development_partition(
                coefficients,
                partition="calibration",
                trajectory_limit=(
                    EVALUATION_TRAJECTORY_LIMIT
                ),
            )
        )


        support_summary = coefficient_support_summary(
            coefficients,
            candidate_id=candidate_id,
        )


        coefficient_norm = float(
            np.linalg.norm(
                coefficients
            )
        )


        reported_group_penalty = (
            group_lasso_penalty_numpy_cell_22(
                coefficients,
                candidate_id=candidate_id,
            )
        )


        trajectory_metrics_df[
            "job_id"
        ] = job_id

        trajectory_metrics_df[
            "candidate_id"
        ] = candidate_id

        trajectory_metrics_df[
            "regularization_lambda"
        ] = float(
            regularization_lambda
        )


        atomic_write_parquet_cell_22(
            trajectory_metrics_df,
            checkpoint_paths[
                "trajectory_metrics"
            ],
        )


        checkpoint_summary = {
            "created_utc": datetime.now(
                timezone.utc
            ).isoformat(),

            "evaluation_signature": (
                evaluation_signature
            ),

            "job_id": job_id,

            "candidate_id": candidate_id,

            "regularization_lambda": float(
                regularization_lambda
            ),

            "coefficient_sha256": (
                coefficient_sha256_cell_22(
                    coefficients
                )
            ),

            **summary,

            **support_summary,

            "coefficient_norm": float(
                coefficient_norm
            ),

            "group_penalty": float(
                reported_group_penalty
            ),
        }


        atomic_write_json_cell_22(
            checkpoint_paths[
                "summary_json"
            ],
            checkpoint_summary,
        )


        print(
            f"    Saved calibration checkpoint: "
            f"{checkpoint_paths['directory']}"
        )


    # Ensure metadata columns exist when loading a checkpoint.

    trajectory_metrics_df = trajectory_metrics_df.copy()


    trajectory_metrics_df[
        "job_id"
    ] = job_id

    trajectory_metrics_df[
        "candidate_id"
    ] = candidate_id

    trajectory_metrics_df[
        "regularization_lambda"
    ] = float(
        regularization_lambda
    )


    calibration_job_record = {
        "job_id": job_id,

        "candidate_id": candidate_id,

        "regularization_lambda": float(
            regularization_lambda
        ),

        **summary,

        **support_summary,

        "coefficient_norm": float(
            coefficient_norm
        ),

        "group_penalty": float(
            reported_group_penalty
        ),

        "coefficient_sha256": (
            coefficient_sha256_cell_22(
                coefficients
            )
        ),

        "calibration_checkpoint_directory": str(
            checkpoint_paths[
                "directory"
            ]
        ),
    }


    calibration_job_records.append(
        calibration_job_record
    )


    calibration_trajectory_metric_frames.append(
        trajectory_metrics_df
    )


    # ------------------------------------------------------------------
    # Save consolidated partial outputs after each completed candidate.
    # ------------------------------------------------------------------

    partial_job_metrics_df = pd.DataFrame(
        calibration_job_records
    )


    partial_trajectory_metrics_df = pd.concat(
        calibration_trajectory_metric_frames,
        ignore_index=True,
    )


    atomic_write_parquet_cell_22(
        partial_job_metrics_df,
        NOTEBOOK_12_CALIBRATION_JOB_METRICS_PATH,
    )


    atomic_write_parquet_cell_22(
        partial_trajectory_metrics_df,
        NOTEBOOK_12_CALIBRATION_TRAJECTORY_METRICS_PATH,
    )


    print(
        f"    Trajectory-weighted calibration RMSE: "
        f"{float(calibration_job_record['trajectory_weighted_rmse_v']):.8f} V"
    )

    print(
        f"    Evaluation time: "
        f"{float(calibration_job_record['elapsed_seconds']):.2f} s"
    )


# ======================================================================================
# 15. BUILD FINAL CALIBRATION TABLES
# ======================================================================================

NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF = (
    pd.DataFrame(
        calibration_job_records
    )
    .sort_values(
        by=[
            "trajectory_weighted_rmse_v",
            "active_group_count",
            "nonzero_coefficient_count",
            "candidate_id",
            "regularization_lambda",
            "job_id",
        ],
        kind="mergesort",
        ignore_index=True,
    )
)


NOTEBOOK_12_CALIBRATION_TRAJECTORY_METRICS_DF = (
    pd.concat(
        calibration_trajectory_metric_frames,
        ignore_index=True,
    )
    .sort_values(
        by=[
            "job_id",
            "row_group_index",
        ],
        kind="mergesort",
        ignore_index=True,
    )
)


if len(
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF
) != len(
    NORMALIZED_CANDIDATE_JOBS_CELL_22
):
    raise RuntimeError(
        "The final calibration job-metrics table has an incorrect row count."
    )


if NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF[
    "job_id"
].duplicated().any():
    raise RuntimeError(
        "The final calibration job-metrics table contains duplicate job IDs."
    )


required_finite_job_metric_columns = (
    "pooled_rmse_v",
    "pooled_mae_v",
    "pooled_bias_v",
    "trajectory_weighted_rmse_v",
    "median_trajectory_rmse_v",
    "maximum_trajectory_rmse_v",
    "coefficient_norm",
    "group_penalty",
)


for column_name in required_finite_job_metric_columns:
    column_values = pd.to_numeric(
        NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF[
            column_name
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.float64
    )


    if not np.isfinite(
        column_values
    ).all():
        raise RuntimeError(
            f"Calibration metric column {column_name!r} contains "
            "nonfinite values."
        )


# ======================================================================================
# 16. APPLY THE CALIBRATION-ONLY MODEL-SELECTION RULE
# ======================================================================================

best_calibration_rmse = float(
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF[
        "trajectory_weighted_rmse_v"
    ].min()
)


calibration_acceptance_limit = float(
    best_calibration_rmse
    *
    (
        1.0
        +
        CALIBRATION_COMPLEXITY_TOLERANCE
    )
)


NOTEBOOK_12_CALIBRATION_SELECTION_POOL_DF = (
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF.loc[
        NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF[
            "trajectory_weighted_rmse_v"
        ]
        <=
        calibration_acceptance_limit
    ]
    .sort_values(
        by=[
            "active_group_count",
            "nonzero_coefficient_count",
            "active_candidate_coefficient_count",
            "trajectory_weighted_rmse_v",
            "candidate_id",
            "regularization_lambda",
            "job_id",
        ],
        kind="mergesort",
        ignore_index=True,
    )
)


if NOTEBOOK_12_CALIBRATION_SELECTION_POOL_DF.empty:
    raise RuntimeError(
        "The calibration selection pool is empty."
    )


selected_row = (
    NOTEBOOK_12_CALIBRATION_SELECTION_POOL_DF.iloc[
        0
    ]
)


SELECTED_JOB_ID = str(
    selected_row[
        "job_id"
    ]
)


SELECTED_CANDIDATE_ID = str(
    selected_row[
        "candidate_id"
    ]
)


SELECTED_REGULARIZATION_LAMBDA = float(
    selected_row[
        "regularization_lambda"
    ]
)


SELECTED_TRAINED_COEFFICIENTS = np.array(
    CANDIDATE_COEFFICIENT_LOOKUP_CELL_22[
        SELECTED_JOB_ID
    ],
    dtype=np.float64,
    copy=True,
)


if SELECTED_TRAINED_COEFFICIENTS.shape != (
    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
):
    raise RuntimeError(
        "The selected trained coefficient vector has an incorrect shape."
    )


if not np.isfinite(
    SELECTED_TRAINED_COEFFICIENTS
).all():
    raise RuntimeError(
        "The selected trained coefficient vector contains nonfinite values."
    )


# ======================================================================================
# 17. SAVE FINAL CALIBRATION AND SELECTION ARTIFACTS
# ======================================================================================

atomic_write_parquet_cell_22(
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF,
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_PATH,
)


atomic_write_parquet_cell_22(
    NOTEBOOK_12_CALIBRATION_TRAJECTORY_METRICS_DF,
    NOTEBOOK_12_CALIBRATION_TRAJECTORY_METRICS_PATH,
)


atomic_write_parquet_cell_22(
    NOTEBOOK_12_CALIBRATION_SELECTION_POOL_DF,
    NOTEBOOK_12_CALIBRATION_SELECTION_POOL_PATH,
)


atomic_write_npz_cell_22(
    NOTEBOOK_12_SELECTED_TRAINED_COEFFICIENT_PATH,

    selected_trained_coefficients=(
        SELECTED_TRAINED_COEFFICIENTS
    ),

    selected_job_id=np.asarray(
        SELECTED_JOB_ID,
        dtype=np.str_,
    ),

    selected_candidate_id=np.asarray(
        SELECTED_CANDIDATE_ID,
        dtype=np.str_,
    ),

    selected_regularization_lambda=np.asarray(
        SELECTED_REGULARIZATION_LAMBDA,
        dtype=np.float64,
    ),

    calibration_best_rmse_v=np.asarray(
        best_calibration_rmse,
        dtype=np.float64,
    ),

    calibration_acceptance_limit_v=np.asarray(
        calibration_acceptance_limit,
        dtype=np.float64,
    ),
)


selected_job_metric_record = (
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF.loc[
        NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF[
            "job_id"
        ].eq(
            SELECTED_JOB_ID
        )
    ]
    .iloc[
        0
    ]
    .to_dict()
)


NOTEBOOK_12_CALIBRATION_SELECTION_CONTRACT = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "status": "CALIBRATION_SELECTION_COMPLETE",

    "selection_partition": "calibration",

    "selection_metric": (
        "trajectory_weighted_rmse_v"
    ),

    "complexity_tolerance": float(
        CALIBRATION_COMPLEXITY_TOLERANCE
    ),

    "best_calibration_rmse_v": float(
        best_calibration_rmse
    ),

    "calibration_acceptance_limit_v": float(
        calibration_acceptance_limit
    ),

    "selection_pool_job_count": int(
        len(
            NOTEBOOK_12_CALIBRATION_SELECTION_POOL_DF
        )
    ),

    "selected_job_id": SELECTED_JOB_ID,

    "selected_candidate_id": (
        SELECTED_CANDIDATE_ID
    ),

    "selected_regularization_lambda": float(
        SELECTED_REGULARIZATION_LAMBDA
    ),

    "selected_job_metrics": (
        selected_job_metric_record
    ),

    "training_data_used_for_parameter_fitting": True,

    "calibration_data_used_for_model_selection": True,

    "ordinary_test_data_opened": False,

    "strict_test_data_opened": False,

    "ordinary_test_data_used_for_selection": False,

    "strict_test_data_used_for_selection": False,

    "candidate_job_metric_path": str(
        NOTEBOOK_12_CALIBRATION_JOB_METRICS_PATH
    ),

    "trajectory_metric_path": str(
        NOTEBOOK_12_CALIBRATION_TRAJECTORY_METRICS_PATH
    ),

    "selection_pool_path": str(
        NOTEBOOK_12_CALIBRATION_SELECTION_POOL_PATH
    ),

    "selected_trained_coefficient_path": str(
        NOTEBOOK_12_SELECTED_TRAINED_COEFFICIENT_PATH
    ),

    "checkpoint_directory": str(
        NOTEBOOK_12_CALIBRATION_CHECKPOINT_DIR
    ),
}


atomic_write_json_cell_22(
    NOTEBOOK_12_CALIBRATION_SELECTION_CONTRACT_PATH,
    NOTEBOOK_12_CALIBRATION_SELECTION_CONTRACT,
)


# ======================================================================================
# 18. VERIFY SAVED OUTPUTS
# ======================================================================================

required_cell_22_output_paths = (
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_PATH,
    NOTEBOOK_12_CALIBRATION_TRAJECTORY_METRICS_PATH,
    NOTEBOOK_12_CALIBRATION_SELECTION_POOL_PATH,
    NOTEBOOK_12_CALIBRATION_SELECTION_CONTRACT_PATH,
    NOTEBOOK_12_SELECTED_TRAINED_COEFFICIENT_PATH,
)


missing_cell_22_output_paths = [
    output_path
    for output_path in required_cell_22_output_paths
    if not Path(
        output_path
    ).is_file()
]


if missing_cell_22_output_paths:
    raise FileNotFoundError(
        "Cell 22 failed to create required outputs:\n  - "
        + "\n  - ".join(
            str(
                output_path
            )
            for output_path in missing_cell_22_output_paths
        )
    )


with np.load(
    NOTEBOOK_12_SELECTED_TRAINED_COEFFICIENT_PATH,
    allow_pickle=False,
) as selected_coefficient_archive:
    reloaded_selected_coefficients = np.asarray(
        selected_coefficient_archive[
            "selected_trained_coefficients"
        ],
        dtype=np.float64,
    )


if not np.array_equal(
    reloaded_selected_coefficients,
    SELECTED_TRAINED_COEFFICIENTS,
):
    raise RuntimeError(
        "The reloaded selected coefficient vector differs from memory."
    )


# ======================================================================================
# 19. FINAL EXECUTION SUMMARY
# ======================================================================================

cell_22_elapsed_seconds = float(
    time.perf_counter()
    -
    cell_22_start_time
)


print("=" * 118)
print("CALIBRATION-ONLY MODEL SELECTION")
print("=" * 118)

print(
    f"Evaluated candidate jobs          : "
    f"{len(NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF):,}"
)

print(
    f"Calibration trajectories per job  : "
    f"{int(NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF['trajectory_count'].max()):,}"
)

print(
    f"Best calibration RMSE             : "
    f"{best_calibration_rmse:.8f} V"
)

print(
    f"Complexity tolerance              : "
    f"{CALIBRATION_COMPLEXITY_TOLERANCE:.4%}"
)

print(
    f"Complexity-equivalent RMSE limit  : "
    f"{calibration_acceptance_limit:.8f} V"
)

print(
    f"Jobs inside selection pool        : "
    f"{len(NOTEBOOK_12_CALIBRATION_SELECTION_POOL_DF):,}"
)

print(
    f"Selected job                      : "
    f"{SELECTED_JOB_ID}"
)

print(
    f"Selected candidate                : "
    f"{SELECTED_CANDIDATE_ID}"
)

print(
    f"Selected regularization lambda    : "
    f"{SELECTED_REGULARIZATION_LAMBDA:.8e}"
)

print(
    f"Selected active hierarchy groups  : "
    f"{int(selected_row['active_group_count']):,}"
)

print(
    f"Selected nonzero coefficients     : "
    f"{int(selected_row['nonzero_coefficient_count']):,}"
)

print(
    f"Total Cell 22 execution time      : "
    f"{cell_22_elapsed_seconds:.3f} s"
)

print(
    f"Calibration checkpoint directory  : "
    f"{NOTEBOOK_12_CALIBRATION_CHECKPOINT_DIR}"
)

print(
    f"Candidate metric table            : "
    f"{NOTEBOOK_12_CALIBRATION_JOB_METRICS_PATH}"
)

print(
    f"Trajectory metric table           : "
    f"{NOTEBOOK_12_CALIBRATION_TRAJECTORY_METRICS_PATH}"
)

print(
    f"Selected coefficient archive      : "
    f"{NOTEBOOK_12_SELECTED_TRAINED_COEFFICIENT_PATH}"
)

print(
    f"Ordinary-test data opened         : "
    f"False"
)

print(
    f"Strict-test data opened           : "
    f"False"
)

print("-" * 118)

print(
    "All trained jobs were evaluated using calibration trajectories only. "
    "The selected training model is ready for heredity closure and "
    "support refitting in Cell 23."
)

print("=" * 118)


display(
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF
)


display(
    NOTEBOOK_12_CALIBRATION_SELECTION_POOL_DF
)

[1/13] Calibration evaluation: M0_FIXED_2RC__lambda_0p000e00
        Calibration trajectories 1/1,017 | elapsed=0.4 s | ETA=406.5 s
        Calibration trajectories 100/1,017 | elapsed=7.4 s | ETA=67.6 s
        Calibration trajectories 200/1,017 | elapsed=11.6 s | ETA=47.6 s
        Calibration trajectories 300/1,017 | elapsed=18.8 s | ETA=45.0 s
        Calibration trajectories 400/1,017 | elapsed=28.2 s | ETA=43.6 s
        Calibration trajectories 500/1,017 | elapsed=38.1 s | ETA=39.4 s
        Calibration trajectories 600/1,017 | elapsed=45.1 s | ETA=31.4 s
        Calibration trajectories 700/1,017 | elapsed=55.4 s | ETA=25.1 s
        Calibration trajectories 800/1,017 | elapsed=63.6 s | ETA=17.3 s
        Calibration trajectories 900/1,017 | elapsed=69.1 s | ETA=9.0 s
        Calibration trajectories 1,000/1,017 | elapsed=73.7 s | ETA=1.3 s
        Calibration trajectories 1,017/1,017 | elapsed=74.3 s | ETA=0.0 s
    Saved calibration checkpoint: /home/xzkhu/Downloads/batter_lp

,job_id,candidate_id,regularization_lambda,partition,available_trajectory_count,trajectory_count,sample_count,pooled_rmse_v,pooled_mae_v,pooled_bias_v,pooled_maximum_absolute_residual_v,trajectory_weighted_rmse_v,median_trajectory_rmse_v,p90_trajectory_rmse_v,p95_trajectory_rmse_v,maximum_trajectory_rmse_v,elapsed_seconds,active_candidate_coefficient_count,nonzero_coefficient_count,active_group_count,coefficient_norm,group_penalty,coefficient_sha256,calibration_checkpoint_directory
0,M2_GLOBAL_SPARSE_LPV__lambda_1p000em06,M2_GLOBAL_SPARSE_LPV,0.000001,calibration,1017,1017,4391688,0.392779,0.290811,0.097033,1.417400,0.291057,0.205078,0.565283,0.635497,0.669025,2.887162,30,24,19,18.992742,0.028709,026a083839110ed1f32dffbb19d1e83f908a9378f948e4...,/home/xzkhu/Downloads/batter_lpv_project/data/...
1,M4_HIERARCHICAL_SPARSE_LPV__lambda_1p000em06,M4_HIERARCHICAL_SPARSE_LPV,0.000001,calibration,1017,1017,4391688,0.392786,0.290809,0.097040,1.417466,0.291063,0.205092,0.565283,0.635512,0.669044,2.657301,240,195,127,19.045682,0.132355,acec82f63eee60594977a56935431b97282dae4cb2db4c...,/home/xzkhu/Downloads/batter_lpv_project/data/...
2,M2_GLOBAL_SPARSE_LPV__lambda_1p000em05,M2_GLOBAL_SPARSE_LPV,0.000010,calibration,1017,1017,4391688,0.392800,0.290805,0.097060,1.417591,0.291076,0.205092,0.565283,0.635540,0.669082,2.794594,30,25,20,18.571722,0.009591,ec299ab0d013c41a138f3e8e725390afcfabd41c52920d...,/home/xzkhu/Downloads/batter_lpv_project/data/...
3,M2_GLOBAL_SPARSE_LPV__lambda_3p000em05,M2_GLOBAL_SPARSE_LPV,0.000030,calibration,1017,1017,4391688,0.392800,0.290805,0.097059,1.417593,0.291076,0.205092,0.565283,0.635540,0.669082,3.184534,30,27,22,18.556071,0.009058,217a791fe6c7eaa0da8c9edc21c7cede98c1d621a7f44f...,/home/xzkhu/Downloads/batter_lpv_project/data/...
4,M4_HIERARCHICAL_SPARSE_LPV__lambda_3p000em05,M4_HIERARCHICAL_SPARSE_LPV,0.000030,calibration,1017,1017,4391688,0.392800,0.290805,0.097059,1.417594,0.291076,0.205092,0.565283,0.635540,0.669083,2.834061,240,145,123,18.558265,0.052705,77cb203d4748af91d3233ece23743ea29afd1f6ade86c8...,/home/xzkhu/Downloads/batter_lpv_project/data/...
5,M4_HIERARCHICAL_SPARSE_LPV__lambda_3p000em06,M4_HIERARCHICAL_SPARSE_LPV,0.000003,calibration,1017,1017,4391688,0.392801,0.290807,0.097062,1.417595,0.291076,0.205092,0.565283,0.635541,0.669083,2.529534,240,157,120,18.576621,0.061801,250e7469c512518d38e7e242e5688558ad0ab349baa226...,/home/xzkhu/Downloads/batter_lpv_project/data/...
6,M4_HIERARCHICAL_SPARSE_LPV__lambda_1p000em04,M4_HIERARCHICAL_SPARSE_LPV,0.000100,calibration,1017,1017,4391688,0.392802,0.290809,0.097064,1.417603,0.291077,0.205092,0.565283,0.635542,0.669085,2.607463,240,133,115,18.540689,0.044549,84ae60adec22ee95cc4fc27e87e7f8e90f92e17483d89a...,/home/xzkhu/Downloads/batter_lpv_project/data/...
7,M2_GLOBAL_SPARSE_LPV__lambda_3p000em06,M2_GLOBAL_SPARSE_LPV,0.000003,calibration,1017,1017,4391688,0.392803,0.290805,0.097055,1.417620,0.291078,0.205099,0.565283,0.635546,0.669091,2.842867,30,26,21,18.453383,0.010134,14fe216267f58bdf443e3f8880de083b39aeba2acccfa0...,/home/xzkhu/Downloads/batter_lpv_project/data/...
8,M0_FIXED_2RC__lambda_0p000e00,M0_FIXED_2RC,0.000000,calibration,1017,1017,4391688,0.392803,0.290808,0.097060,1.417623,0.291078,0.205101,0.565283,0.635547,0.669092,74.280054,5,5,5,18.399066,0.000000,c735ffcd21739f6ee2fd33c8b5b1b180d71bb549b90fc9...,/home/xzkhu/Downloads/batter_lpv_project/data/...
9,M2_GLOBAL_SPARSE_LPV__lambda_1p000em04,M2_GLOBAL_SPARSE_LPV,0.000100,calibration,1017,1017,4391688,0.392803,0.290809,0.097063,1.417622,0.291078,0.205100,0.565283,0.635547,0.669091,2.739704,30,25,20,18.421713,0.006077,6f692fdc2ed839aa97ea37a08cf5d101b74da3820c97f6...,/home/xzkhu/Downloads/batter_lpv_project/data/...


,job_id,candidate_id,regularization_lambda,partition,available_trajectory_count,trajectory_count,sample_count,pooled_rmse_v,pooled_mae_v,pooled_bias_v,pooled_maximum_absolute_residual_v,trajectory_weighted_rmse_v,median_trajectory_rmse_v,p90_trajectory_rmse_v,p95_trajectory_rmse_v,maximum_trajectory_rmse_v,elapsed_seconds,active_candidate_coefficient_count,nonzero_coefficient_count,active_group_count,coefficient_norm,group_penalty,coefficient_sha256,calibration_checkpoint_directory
0,M0_FIXED_2RC__lambda_0p000e00,M0_FIXED_2RC,0.000000,calibration,1017,1017,4391688,0.392803,0.290808,0.097060,1.417623,0.291078,0.205101,0.565283,0.635547,0.669092,74.280054,5,5,5,18.399066,0.000000,c735ffcd21739f6ee2fd33c8b5b1b180d71bb549b90fc9...,/home/xzkhu/Downloads/batter_lpv_project/data/...
1,M2_GLOBAL_SPARSE_LPV__lambda_1p000em06,M2_GLOBAL_SPARSE_LPV,0.000001,calibration,1017,1017,4391688,0.392779,0.290811,0.097033,1.417400,0.291057,0.205078,0.565283,0.635497,0.669025,2.887162,30,24,19,18.992742,0.028709,026a083839110ed1f32dffbb19d1e83f908a9378f948e4...,/home/xzkhu/Downloads/batter_lpv_project/data/...
2,M2_GLOBAL_SPARSE_LPV__lambda_1p000em05,M2_GLOBAL_SPARSE_LPV,0.000010,calibration,1017,1017,4391688,0.392800,0.290805,0.097060,1.417591,0.291076,0.205092,0.565283,0.635540,0.669082,2.794594,30,25,20,18.571722,0.009591,ec299ab0d013c41a138f3e8e725390afcfabd41c52920d...,/home/xzkhu/Downloads/batter_lpv_project/data/...
3,M2_GLOBAL_SPARSE_LPV__lambda_1p000em04,M2_GLOBAL_SPARSE_LPV,0.000100,calibration,1017,1017,4391688,0.392803,0.290809,0.097063,1.417622,0.291078,0.205100,0.565283,0.635547,0.669091,2.739704,30,25,20,18.421713,0.006077,6f692fdc2ed839aa97ea37a08cf5d101b74da3820c97f6...,/home/xzkhu/Downloads/batter_lpv_project/data/...
4,M2_GLOBAL_SPARSE_LPV__lambda_3p000em06,M2_GLOBAL_SPARSE_LPV,0.000003,calibration,1017,1017,4391688,0.392803,0.290805,0.097055,1.417620,0.291078,0.205099,0.565283,0.635546,0.669091,2.842867,30,26,21,18.453383,0.010134,14fe216267f58bdf443e3f8880de083b39aeba2acccfa0...,/home/xzkhu/Downloads/batter_lpv_project/data/...
5,M2_GLOBAL_SPARSE_LPV__lambda_3p000em05,M2_GLOBAL_SPARSE_LPV,0.000030,calibration,1017,1017,4391688,0.392800,0.290805,0.097059,1.417593,0.291076,0.205092,0.565283,0.635540,0.669082,3.184534,30,27,22,18.556071,0.009058,217a791fe6c7eaa0da8c9edc21c7cede98c1d621a7f44f...,/home/xzkhu/Downloads/batter_lpv_project/data/...
6,M1_GLOBAL_DENSE_LPV__lambda_0p000e00,M1_GLOBAL_DENSE_LPV,0.000000,calibration,1017,1017,4391688,0.392914,0.290956,0.097203,1.418401,0.291152,0.205390,0.565283,0.635740,0.669370,2.685478,30,30,30,15.987197,0.000000,8c05e02aa1ee64de654e7f1ae1f1705b1fbf5861c8278e...,/home/xzkhu/Downloads/batter_lpv_project/data/...
7,M4_HIERARCHICAL_SPARSE_LPV__lambda_1p000em04,M4_HIERARCHICAL_SPARSE_LPV,0.000100,calibration,1017,1017,4391688,0.392802,0.290809,0.097064,1.417603,0.291077,0.205092,0.565283,0.635542,0.669085,2.607463,240,133,115,18.540689,0.044549,84ae60adec22ee95cc4fc27e87e7f8e90f92e17483d89a...,/home/xzkhu/Downloads/batter_lpv_project/data/...
8,M4_HIERARCHICAL_SPARSE_LPV__lambda_3p000em06,M4_HIERARCHICAL_SPARSE_LPV,0.000003,calibration,1017,1017,4391688,0.392801,0.290807,0.097062,1.417595,0.291076,0.205092,0.565283,0.635541,0.669083,2.529534,240,157,120,18.576621,0.061801,250e7469c512518d38e7e242e5688558ad0ab349baa226...,/home/xzkhu/Downloads/batter_lpv_project/data/...
9,M4_HIERARCHICAL_SPARSE_LPV__lambda_3p000em05,M4_HIERARCHICAL_SPARSE_LPV,0.000030,calibration,1017,1017,4391688,0.392800,0.290805,0.097059,1.417594,0.291076,0.205092,0.565283,0.635540,0.669083,2.834061,240,145,123,18.558265,0.052705,77cb203d4748af91d3233ece23743ea29afd1f6ade86c8...,/home/xzkhu/Downloads/batter_lpv_project/data/...


In [28]:
# ======================================================================================
# CELL 23 — CLOSE HEREDITY, REFIT THE SELECTED SUPPORT, AND FREEZE THE MODEL
# ======================================================================================
#
# Purpose
# -------
# Close the calibration-selected coefficient support under basis and scope heredity,
# refit only that fixed support on training trajectories, save resumable checkpoints,
# freeze the final reduced/full coefficient vectors, and permit final test evaluation.
# ======================================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import time

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

from IPython.display import display


jax.config.update(
    "jax_enable_x64",
    True,
)


# ======================================================================================
# 1. VERIFY REQUIRED UPSTREAM OBJECTS
# ======================================================================================

REQUIRED_CELL_23_OBJECTS: tuple[str, ...] = (
    "SELECTED_JOB_ID",
    "SELECTED_CANDIDATE_ID",
    "SELECTED_REGULARIZATION_LAMBDA",
    "SELECTED_TRAINED_COEFFICIENTS",
    "NOTEBOOK_12_PENALTY_GROUP_REGISTRY_DF",
    "PENALTY_GROUP_MEMBERS",
    "PENALTY_GROUP_METADATA",
    "CANDIDATE_ACTIVE_MASKS",
    "MODEL_CANDIDATE_SPECIFICATIONS",
    "training_row_group_indices",
    "TRAINING_BATCH_TRAJECTORIES",
    "load_padded_trajectory_batch",
    "batch_to_jax",
    "NOTEBOOK_12_TABLE_DIR",
    "NOTEBOOK_12_MODEL_DIR",
    "NOTEBOOK_12_CONTRACT_DIR",
)


missing_cell_23_objects = [
    object_name
    for object_name in REQUIRED_CELL_23_OBJECTS
    if object_name not in globals()
]


if missing_cell_23_objects:
    raise RuntimeError(
        "Cell 23 requires the completed training and calibration-selection "
        "cells.\n\n"
        "Missing objects:\n  - "
        + "\n  - ".join(
            missing_cell_23_objects
        )
    )


# Resolve simulator, data-loss and physical-barrier functions across
# the earlier corrected Cell 19 variants.

if "simulate_lpv_batch_core" in globals():
    CELL_23_SIMULATOR_CORE = globals()[
        "simulate_lpv_batch_core"
    ]

elif "_simulate_lpv_batch_core" in globals():
    CELL_23_SIMULATOR_CORE = globals()[
        "_simulate_lpv_batch_core"
    ]

else:
    raise RuntimeError(
        "No compatible batched LPV simulator core is available. "
        "Execute the corrected Cell 19 first."
    )


if "masked_pseudo_huber_mean" in globals():
    CELL_23_DATA_LOSS = globals()[
        "masked_pseudo_huber_mean"
    ]

elif "_pseudo_huber_mean" in globals():
    CELL_23_DATA_LOSS = globals()[
        "_pseudo_huber_mean"
    ]

else:
    raise RuntimeError(
        "No compatible masked pseudo-Huber loss is available. "
        "Execute the corrected Cell 19 first."
    )


if "physical_parameter_barrier" in globals():
    CELL_23_PHYSICAL_BARRIER = globals()[
        "physical_parameter_barrier"
    ]

elif "_physical_barrier" in globals():
    CELL_23_PHYSICAL_BARRIER = globals()[
        "_physical_barrier"
    ]

else:
    raise RuntimeError(
        "No compatible physical-parameter barrier is available. "
        "Execute the corrected Cell 19 first."
    )


# ======================================================================================
# 2. NORMALIZE PATHS AND CONFIGURATION
# ======================================================================================

NOTEBOOK_12_TABLE_DIR = Path(
    NOTEBOOK_12_TABLE_DIR
).expanduser().resolve()

NOTEBOOK_12_MODEL_DIR = Path(
    NOTEBOOK_12_MODEL_DIR
).expanduser().resolve()

NOTEBOOK_12_CONTRACT_DIR = Path(
    NOTEBOOK_12_CONTRACT_DIR
).expanduser().resolve()


for directory in (
    NOTEBOOK_12_TABLE_DIR,
    NOTEBOOK_12_MODEL_DIR,
    NOTEBOOK_12_CONTRACT_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


GROUP_SUPPORT_THRESHOLD = float(
    globals().get(
        "GROUP_SUPPORT_THRESHOLD",
        1.0e-8,
    )
)


REFIT_STEPS = int(
    globals().get(
        "REFIT_STEPS",
        1000,
    )
)


CHECKPOINT_INTERVAL_STEPS = int(
    globals().get(
        "CHECKPOINT_INTERVAL_STEPS",
        50,
    )
)


RANDOM_SEED = int(
    globals().get(
        "RANDOM_SEED",
        20260701,
    )
)


TRAINING_BATCH_TRAJECTORIES = int(
    TRAINING_BATCH_TRAJECTORIES
)


REFIT_LEARNING_RATE = float(
    globals().get(
        "REFIT_LEARNING_RATE",
        globals().get(
            "LEARNING_RATE",
            globals().get(
                "ADAM_LEARNING_RATE",
                2.5e-3,
            ),
        ),
    )
)


REFIT_ADAM_BETA1 = float(
    globals().get(
        "ADAM_BETA1",
        0.9,
    )
)


REFIT_ADAM_BETA2 = float(
    globals().get(
        "ADAM_BETA2",
        0.999,
    )
)


REFIT_ADAM_EPSILON = float(
    globals().get(
        "ADAM_EPSILON",
        1.0e-8,
    )
)


REFIT_GRADIENT_NORM_CLIP = float(
    globals().get(
        "GRADIENT_NORM_CLIP",
        globals().get(
            "GRADIENT_CLIP_NORM",
            10.0,
        ),
    )
)


REFIT_MAX_PARAMETER_ABS = float(
    globals().get(
        "MAX_REDUCED_COEFFICIENT_ABS",
        20.0,
    )
)


PSEUDO_HUBER_DELTA_V = float(
    globals().get(
        "PSEUDO_HUBER_DELTA_V",
        0.02,
    )
)


RIDGE_WEIGHT = float(
    globals().get(
        "RIDGE_WEIGHT",
        0.0,
    )
)


PHYSICAL_BARRIER_WEIGHT = float(
    globals().get(
        "PHYSICAL_BARRIER_WEIGHT",
        1.0e-3,
    )
)


RESUME_SUPPORT_REFIT: bool = bool(
    globals().get(
        "RESUME_SUPPORT_REFIT",
        True,
    )
)


FORCE_RESTART_SUPPORT_REFIT: bool = bool(
    globals().get(
        "FORCE_RESTART_SUPPORT_REFIT",
        False,
    )
)


if (
    not np.isfinite(
        GROUP_SUPPORT_THRESHOLD
    )
    or
    GROUP_SUPPORT_THRESHOLD < 0.0
):
    raise ValueError(
        "GROUP_SUPPORT_THRESHOLD must be finite and nonnegative."
    )


if REFIT_STEPS <= 0:
    raise ValueError(
        "REFIT_STEPS must be positive."
    )


if CHECKPOINT_INTERVAL_STEPS <= 0:
    raise ValueError(
        "CHECKPOINT_INTERVAL_STEPS must be positive."
    )


if TRAINING_BATCH_TRAJECTORIES <= 0:
    raise ValueError(
        "TRAINING_BATCH_TRAJECTORIES must be positive."
    )


if (
    not np.isfinite(
        REFIT_LEARNING_RATE
    )
    or
    REFIT_LEARNING_RATE <= 0.0
):
    raise ValueError(
        "REFIT_LEARNING_RATE must be finite and positive."
    )


if not (
    0.0
    <
    REFIT_ADAM_BETA1
    <
    1.0
):
    raise ValueError(
        "REFIT_ADAM_BETA1 must lie strictly between zero and one."
    )


if not (
    0.0
    <
    REFIT_ADAM_BETA2
    <
    1.0
):
    raise ValueError(
        "REFIT_ADAM_BETA2 must lie strictly between zero and one."
    )


if (
    not np.isfinite(
        REFIT_ADAM_EPSILON
    )
    or
    REFIT_ADAM_EPSILON <= 0.0
):
    raise ValueError(
        "REFIT_ADAM_EPSILON must be finite and positive."
    )


if (
    not np.isfinite(
        REFIT_GRADIENT_NORM_CLIP
    )
    or
    REFIT_GRADIENT_NORM_CLIP <= 0.0
):
    raise ValueError(
        "REFIT_GRADIENT_NORM_CLIP must be finite and positive."
    )


if (
    not np.isfinite(
        REFIT_MAX_PARAMETER_ABS
    )
    or
    REFIT_MAX_PARAMETER_ABS <= 0.0
):
    raise ValueError(
        "REFIT_MAX_PARAMETER_ABS must be finite and positive."
    )


training_row_group_indices = np.asarray(
    training_row_group_indices,
    dtype=np.int32,
).reshape(
    -1
)


if training_row_group_indices.size == 0:
    raise RuntimeError(
        "No training row groups are available for support refitting."
    )


if (
    training_row_group_indices
    <
    0
).any():
    raise RuntimeError(
        "The training row-group registry contains a negative index."
    )


TRAIN_ROW_GROUP_COUNT = int(
    training_row_group_indices.size
)


# ======================================================================================
# 3. DETERMINE REDUCED DIMENSION AND VALIDATE SELECTED COEFFICIENTS
# ======================================================================================

selected_candidate_mask = np.asarray(
    CANDIDATE_ACTIVE_MASKS[
        SELECTED_CANDIDATE_ID
    ],
    dtype=bool,
).reshape(
    -1
)


NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT = int(
    selected_candidate_mask.size
)


SELECTED_TRAINED_COEFFICIENTS = np.array(
    SELECTED_TRAINED_COEFFICIENTS,
    dtype=np.float64,
    copy=True,
).reshape(
    -1
)


if SELECTED_TRAINED_COEFFICIENTS.shape != (
    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
):
    raise RuntimeError(
        "The selected trained coefficient vector has an incorrect "
        "dimension.\n"
        f"Expected: {NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT:,}\n"
        f"Observed: {SELECTED_TRAINED_COEFFICIENTS.size:,}"
    )


if not np.isfinite(
    SELECTED_TRAINED_COEFFICIENTS
).all():
    raise RuntimeError(
        "The selected trained coefficient vector contains nonfinite values."
    )


# ======================================================================================
# 4. DEFINE ATOMIC OUTPUT HELPERS
# ======================================================================================

def _json_safe_cell_23(
    value: Any,
) -> Any:
    """
    Convert nested NumPy and Path values into JSON-safe Python values.
    """

    if isinstance(
        value,
        Mapping,
    ):
        return {
            str(
                key
            ): _json_safe_cell_23(
                item
            )
            for key, item in value.items()
        }


    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        return [
            _json_safe_cell_23(
                item
            )
            for item in value
        ]


    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()


    if isinstance(
        value,
        np.generic,
    ):
        return value.item()


    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )


    return value


def _atomic_write_json_cell_23(
    output_path: Path,
    payload: Mapping[str, Any],
) -> None:
    """
    Atomically write one JSON object.
    """

    output_path = Path(
        output_path
    ).expanduser().resolve()


    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = output_path.with_name(
        output_path.name
        +
        ".tmp"
    )


    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        json.dump(
            _json_safe_cell_23(
                dict(
                    payload
                )
            ),
            file_handle,
            indent=2,
            sort_keys=True,
        )

        file_handle.write(
            "\n"
        )


    os.replace(
        temporary_path,
        output_path,
    )


def _atomic_write_parquet_cell_23(
    dataframe: pd.DataFrame,
    output_path: Path,
) -> None:
    """
    Atomically write one Parquet dataframe.
    """

    output_path = Path(
        output_path
    ).expanduser().resolve()


    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = output_path.with_name(
        output_path.stem
        +
        ".tmp.parquet"
    )


    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )


    os.replace(
        temporary_path,
        output_path,
    )


def _atomic_write_npz_cell_23(
    output_path: Path,
    **arrays: np.ndarray,
) -> None:
    """
    Atomically write one compressed NumPy archive.
    """

    output_path = Path(
        output_path
    ).expanduser().resolve()


    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = output_path.with_name(
        output_path.stem
        +
        ".tmp.npz"
    )


    if temporary_path.exists():
        temporary_path.unlink()


    np.savez_compressed(
        temporary_path,
        **arrays,
    )


    os.replace(
        temporary_path,
        output_path,
    )


def _utc_now_iso_cell_23() -> str:
    """
    Return the current UTC timestamp.
    """

    return datetime.now(
        timezone.utc
    ).isoformat()


# ======================================================================================
# 5. NORMALIZE THE PENALTY-GROUP REGISTRY
# ======================================================================================

group_registry_df = (
    NOTEBOOK_12_PENALTY_GROUP_REGISTRY_DF
    .copy()
)


required_group_columns = (
    "penalty_group_id",
    "transformed_parameter",
    "scope",
    "entity_block_id",
    "basis_id",
)


missing_group_columns = [
    column_name
    for column_name in required_group_columns
    if column_name not in group_registry_df.columns
]


if missing_group_columns:
    raise RuntimeError(
        "The penalty-group registry is missing required columns:\n  - "
        +
        "\n  - ".join(
            missing_group_columns
        )
    )


for column_name in required_group_columns:
    group_registry_df[
        column_name
    ] = (
        group_registry_df[
            column_name
        ]
        .astype("string")
        .str.strip()
    )


group_registry_df[
    "scope"
] = (
    group_registry_df[
        "scope"
    ]
    .str.upper()
)


if group_registry_df[
    "penalty_group_id"
].duplicated().any():
    raise RuntimeError(
        "The penalty-group registry contains duplicate group IDs."
    )


group_key_to_id: dict[
    tuple[str, str, str, str],
    str,
] = {
    (
        str(
            row.transformed_parameter
        ),
        str(
            row.scope
        ).upper(),
        str(
            row.entity_block_id
        ),
        str(
            row.basis_id
        ),
    ): str(
        row.penalty_group_id
    )
    for row in group_registry_df.itertuples(
        index=False
    )
}


if len(
    group_key_to_id
) != len(
    group_registry_df
):
    raise RuntimeError(
        "The penalty-group key registry is not one-to-one."
    )


# Resolve the intercept basis ID.

if "intercept_basis_id" in globals():
    INTERCEPT_BASIS_ID_CELL_23 = str(
        globals()[
            "intercept_basis_id"
        ]
    )


elif "NOTEBOOK_12_BASIS_TERM_DF" in globals():
    basis_registry = globals()[
        "NOTEBOOK_12_BASIS_TERM_DF"
    ].copy()


    if (
        "basis_position"
        not in basis_registry.columns
        or
        "basis_id"
        not in basis_registry.columns
    ):
        raise RuntimeError(
            "The basis registry cannot identify the intercept basis."
        )


    INTERCEPT_BASIS_ID_CELL_23 = str(
        basis_registry
        .sort_values(
            "basis_position",
            kind="mergesort",
        )
        .iloc[
            0
        ][
            "basis_id"
        ]
    )


else:
    raise RuntimeError(
        "The intercept basis ID is unavailable. "
        "Execute the basis-construction cell first."
    )


# Build a cell-to-chemistry map for CELL-to-CHEMISTRY scope heredity.

cell_to_chemistry_cell_23: dict[
    str,
    str,
] = {}


for manifest_name in (
    "DEVELOPMENT_TRAJECTORY_MANIFEST_DF",
    "NOTEBOOK_12_DESIGN_TRAJECTORY_MANIFEST_DF",
):
    if manifest_name not in globals():
        continue


    manifest_df = globals()[
        manifest_name
    ]


    if not {
        "cell_entity_id",
        "chemistry",
    }.issubset(
        manifest_df.columns
    ):
        continue


    cell_chemistry_df = (
        manifest_df[
            [
                "cell_entity_id",
                "chemistry",
            ]
        ]
        .dropna()
        .astype("string")
        .drop_duplicates()
    )


    cell_chemistry_df[
        "chemistry"
    ] = (
        cell_chemistry_df[
            "chemistry"
        ]
        .str.strip()
        .str.upper()
    )


    ambiguous_cells = (
        cell_chemistry_df
        .groupby(
            "cell_entity_id",
            observed=True,
        )[
            "chemistry"
        ]
        .nunique()
        .loc[
            lambda series: series
            >
            1
        ]
    )


    if not ambiguous_cells.empty:
        raise RuntimeError(
            "At least one cell entity maps to multiple chemistries "
            "in the trajectory manifest."
        )


    cell_to_chemistry_cell_23.update(
        dict(
            zip(
                cell_chemistry_df[
                    "cell_entity_id"
                ].astype(str),

                cell_chemistry_df[
                    "chemistry"
                ].astype(str),
            )
        )
    )


# ======================================================================================
# 6. DEFINE CANDIDATE AND HEREDITY HELPERS
# ======================================================================================

def _candidate_is_sparse_cell_23(
    candidate_id: str,
) -> bool:
    """
    Return whether the selected candidate is sparse.
    """

    specification = (
        MODEL_CANDIDATE_SPECIFICATIONS[
            candidate_id
        ]
    )


    if isinstance(
        specification,
        Mapping,
    ):
        return bool(
            specification.get(
                "sparse",
                False,
            )
        )


    return bool(
        getattr(
            specification,
            "sparse",
            False,
        )
    )


def _group_members_cell_23(
    group_id: str,
) -> np.ndarray:
    """
    Return and validate the reduced coefficient indices of one group.
    """

    if group_id not in PENALTY_GROUP_MEMBERS:
        raise KeyError(
            f"Penalty group {group_id!r} has no member registry."
        )


    members = np.asarray(
        PENALTY_GROUP_MEMBERS[
            group_id
        ],
        dtype=np.int64,
    ).reshape(
        -1
    )


    if members.size == 0:
        raise RuntimeError(
            f"Penalty group {group_id!r} is empty."
        )


    if (
        members
        <
        0
    ).any() or (
        members
        >=
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT
    ).any():
        raise RuntimeError(
            f"Penalty group {group_id!r} contains an invalid "
            "coefficient index."
        )


    return members


def _candidate_relevant_group_ids_cell_23(
    candidate_id: str,
) -> set[str]:
    """
    Return all penalty groups permitted by one candidate family.
    """

    candidate_mask = np.asarray(
        CANDIDATE_ACTIVE_MASKS[
            candidate_id
        ],
        dtype=bool,
    )


    relevant: set[str] = set()


    for group_id in group_registry_df[
        "penalty_group_id"
    ].astype(str):
        members = _group_members_cell_23(
            group_id
        )


        if np.any(
            candidate_mask[
                members
            ]
        ):
            relevant.add(
                group_id
            )


    return relevant


def _metadata_value_cell_23(
    metadata: Mapping[str, Any],
    key: str,
    default: Any = None,
) -> Any:
    """
    Read a field from mapping-like or attribute-like metadata.
    """

    if isinstance(
        metadata,
        Mapping,
    ):
        return metadata.get(
            key,
            default,
        )


    return getattr(
        metadata,
        key,
        default,
    )


def _parent_groups_cell_23(
    group_id: str,
) -> list[str]:
    """
    Return the strong-basis and scope-heredity parents of one group.
    """

    metadata = PENALTY_GROUP_METADATA[
        group_id
    ]


    transformed_parameter = str(
        _metadata_value_cell_23(
            metadata,
            "transformed_parameter",
        )
    )


    scope = str(
        _metadata_value_cell_23(
            metadata,
            "scope",
        )
    ).upper()


    entity_block_id = str(
        _metadata_value_cell_23(
            metadata,
            "entity_block_id",
        )
    )


    basis_id = str(
        _metadata_value_cell_23(
            metadata,
            "basis_id",
        )
    )


    parent_group_ids: list[str] = []


    parent_basis_ids = _metadata_value_cell_23(
        metadata,
        "parent_basis_ids",
        (),
    )


    if parent_basis_ids is None:
        parent_basis_ids = ()


    # Strong basis heredity within the same scope and entity block.

    for parent_basis_id in parent_basis_ids:
        parent_key = (
            transformed_parameter,
            scope,
            entity_block_id,
            str(
                parent_basis_id
            ),
        )


        parent_group_id = group_key_to_id.get(
            parent_key
        )


        if parent_group_id is not None:
            parent_group_ids.append(
                parent_group_id
            )


    # Scope heredity.

    if scope == "CHEMISTRY":
        global_parent_key = (
            transformed_parameter,
            "GLOBAL",
            "GLOBAL",
            basis_id,
        )


        global_parent_id = group_key_to_id.get(
            global_parent_key
        )


        if global_parent_id is not None:
            parent_group_ids.append(
                global_parent_id
            )


    elif scope == "CELL":
        explicit_parent_entity = (
            _metadata_value_cell_23(
                metadata,
                "parent_entity_block_id",
                None,
            )
        )


        if explicit_parent_entity is None:
            explicit_parent_entity = (
                _metadata_value_cell_23(
                    metadata,
                    "chemistry",
                    None,
                )
            )


        if explicit_parent_entity is None:
            explicit_parent_entity = (
                cell_to_chemistry_cell_23.get(
                    entity_block_id
                )
            )


        if explicit_parent_entity is not None:
            explicit_parent_entity = str(
                explicit_parent_entity
            ).upper()


        chemistry_parent_id: str | None = None


        if explicit_parent_entity is not None:
            chemistry_parent_key = (
                transformed_parameter,
                "CHEMISTRY",
                explicit_parent_entity,
                basis_id,
            )


            chemistry_parent_id = (
                group_key_to_id.get(
                    chemistry_parent_key
                )
            )


        # Some registries use one shared chemistry group with
        # entity_block_id="GLOBAL".

        if chemistry_parent_id is None:
            shared_chemistry_parent_key = (
                transformed_parameter,
                "CHEMISTRY",
                "GLOBAL",
                basis_id,
            )


            chemistry_parent_id = (
                group_key_to_id.get(
                    shared_chemistry_parent_key
                )
            )


        # Final fallback: accept a unique chemistry group.

        if chemistry_parent_id is None:
            matching_chemistry_groups = [
                candidate_group_id

                for (
                    parameter_key,
                    scope_key,
                    _entity_key,
                    basis_key,
                ), candidate_group_id in (
                    group_key_to_id.items()
                )

                if (
                    parameter_key
                    ==
                    transformed_parameter
                    and
                    scope_key
                    ==
                    "CHEMISTRY"
                    and
                    basis_key
                    ==
                    basis_id
                )
            ]


            if len(
                matching_chemistry_groups
            ) == 1:
                chemistry_parent_id = (
                    matching_chemistry_groups[
                        0
                    ]
                )


            elif len(
                matching_chemistry_groups
            ) > 1:
                raise RuntimeError(
                    "The chemistry parent of a cell-level penalty group "
                    "could not be identified uniquely.\n"
                    f"Cell group: {group_id}\n"
                    f"Cell entity: {entity_block_id}\n"
                    f"Candidate chemistry parents: "
                    f"{matching_chemistry_groups[:10]}"
                )


        if chemistry_parent_id is not None:
            parent_group_ids.append(
                chemistry_parent_id
            )


    return list(
        dict.fromkeys(
            parent_group_ids
        )
    )


def close_group_support_under_heredity(
    coefficients: np.ndarray,
    *,
    candidate_id: str,
    threshold: float,
) -> tuple[
    set[str],
    pd.DataFrame,
]:
    """
    Close the selected hierarchy under basis and scope heredity.
    """

    coefficient_array = np.asarray(
        coefficients,
        dtype=np.float64,
    ).reshape(
        -1
    )


    candidate_mask = np.asarray(
        CANDIDATE_ACTIVE_MASKS[
            candidate_id
        ],
        dtype=bool,
    )


    relevant_group_ids = (
        _candidate_relevant_group_ids_cell_23(
            candidate_id
        )
    )


    sparse_candidate = (
        _candidate_is_sparse_cell_23(
            candidate_id
        )
    )


    active_groups: set[str] = set()


    support_records: list[
        dict[str, Any]
    ] = []


    for row in group_registry_df.itertuples(
        index=False
    ):
        group_id = str(
            row.penalty_group_id
        )


        if group_id not in relevant_group_ids:
            continue


        members = _group_members_cell_23(
            group_id
        )


        group_norm = float(
            np.linalg.norm(
                coefficient_array[
                    members
                ]
            )
        )


        initially_active = (
            bool(
                group_norm
                >
                threshold
            )
            if sparse_candidate
            else True
        )


        if initially_active:
            active_groups.add(
                group_id
            )


        metadata = PENALTY_GROUP_METADATA[
            group_id
        ]


        support_records.append(
            {
                "penalty_group_id": (
                    group_id
                ),

                "transformed_parameter": str(
                    _metadata_value_cell_23(
                        metadata,
                        "transformed_parameter",
                    )
                ),

                "scope": str(
                    _metadata_value_cell_23(
                        metadata,
                        "scope",
                    )
                ).upper(),

                "scope_level": (
                    _metadata_value_cell_23(
                        metadata,
                        "scope_level",
                        np.nan,
                    )
                ),

                "entity_block_id": str(
                    _metadata_value_cell_23(
                        metadata,
                        "entity_block_id",
                    )
                ),

                "basis_id": str(
                    _metadata_value_cell_23(
                        metadata,
                        "basis_id",
                    )
                ),

                "basis_position": (
                    _metadata_value_cell_23(
                        metadata,
                        "basis_position",
                        np.nan,
                    )
                ),

                "group_norm_before_closure": (
                    group_norm
                ),

                "initially_active": (
                    initially_active
                ),
            }
        )


    # Global intercept groups are mandatory whenever the candidate
    # permits those groups.

    for row in group_registry_df.itertuples(
        index=False
    ):
        group_id = str(
            row.penalty_group_id
        )


        if group_id not in relevant_group_ids:
            continue


        if (
            str(
                row.scope
            ).upper()
            ==
            "GLOBAL"
            and
            str(
                row.basis_id
            )
            ==
            INTERCEPT_BASIS_ID_CELL_23
        ):
            active_groups.add(
                group_id
            )


    changed = True


    while changed:
        changed = False


        for group_id in tuple(
            active_groups
        ):
            for parent_group_id in (
                _parent_groups_cell_23(
                    group_id
                )
            ):
                if parent_group_id not in relevant_group_ids:
                    raise RuntimeError(
                        "Heredity requires a parent group excluded by the "
                        "selected candidate family.\n"
                        f"Active child: {group_id}\n"
                        f"Required parent: {parent_group_id}\n"
                        f"Candidate: {candidate_id}"
                    )


                if parent_group_id not in active_groups:
                    active_groups.add(
                        parent_group_id
                    )

                    changed = True


    support_df = pd.DataFrame(
        support_records
    )


    if support_df.empty:
        raise RuntimeError(
            "No candidate-relevant penalty groups were found."
        )


    support_df[
        "active_after_heredity_closure"
    ] = support_df[
        "penalty_group_id"
    ].isin(
        active_groups
    )


    support_df[
        "activated_by_closure"
    ] = (
        support_df[
            "active_after_heredity_closure"
        ]
        &
        ~support_df[
            "initially_active"
        ]
    )


    return active_groups, support_df


# ======================================================================================
# 7. CLOSE THE SELECTED SUPPORT UNDER HEREDITY
# ======================================================================================

(
    SELECTED_ACTIVE_GROUP_IDS,
    NOTEBOOK_12_SELECTED_SUPPORT_DF,
) = close_group_support_under_heredity(
    SELECTED_TRAINED_COEFFICIENTS,

    candidate_id=(
        SELECTED_CANDIDATE_ID
    ),

    threshold=(
        GROUP_SUPPORT_THRESHOLD
    ),
)


SELECTED_SUPPORT_MASK = np.zeros(
    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
    dtype=bool,
)


for group_id in SELECTED_ACTIVE_GROUP_IDS:
    SELECTED_SUPPORT_MASK[
        _group_members_cell_23(
            group_id
        )
    ] = True


# Respect the selected model family after heredity closure.

SELECTED_SUPPORT_MASK = (
    SELECTED_SUPPORT_MASK
    &
    selected_candidate_mask
)


if not SELECTED_SUPPORT_MASK.any():
    raise RuntimeError(
        "The selected support is empty after heredity closure."
    )


selected_support_float = (
    SELECTED_SUPPORT_MASK.astype(
        np.float64
    )
)


# Verify that every active group retains at least one active coefficient.

for group_id in SELECTED_ACTIVE_GROUP_IDS:
    members = _group_members_cell_23(
        group_id
    )


    if not np.any(
        SELECTED_SUPPORT_MASK[
            members
        ]
    ):
        raise RuntimeError(
            f"Active heredity group {group_id!r} has no retained "
            "coefficient."
        )


# ======================================================================================
# 8. DEFINE THE SUPPORT-REFIT OBJECTIVE
# ======================================================================================

def make_support_refit_objective(
    support_mask: np.ndarray,
):
    """
    Create the differentiable objective for fixed-support debiasing.
    """

    mask = jnp.asarray(
        support_mask,
        dtype=jnp.float64,
    )


    def objective(
        unconstrained_coefficients: jax.Array,
        batch: Mapping[
            str,
            jax.Array,
        ],
    ) -> tuple[
        jax.Array,
        tuple[
            jax.Array,
            jax.Array,
            jax.Array,
        ],
    ]:
        coefficients = (
            unconstrained_coefficients
            *
            mask
        )


        simulation_outputs = (
            CELL_23_SIMULATOR_CORE(
                coefficients,
                batch,
            )
        )


        residual = simulation_outputs[
            1
        ]


        transformed = simulation_outputs[
            2
        ]


        data_loss = CELL_23_DATA_LOSS(
            residual,
            batch[
                "loss_mask"
            ],
            PSEUDO_HUBER_DELTA_V,
        )


        ridge_penalty = (
            RIDGE_WEIGHT
            *
            jnp.mean(
                jnp.square(
                    coefficients
                )
            )
        )


        physical_penalty = (
            PHYSICAL_BARRIER_WEIGHT
            *
            CELL_23_PHYSICAL_BARRIER(
                transformed,
                batch[
                    "loss_mask"
                ],
            )
        )


        total_loss = (
            data_loss
            +
            ridge_penalty
            +
            physical_penalty
        )


        return (
            total_loss,
            (
                data_loss,
                ridge_penalty,
                physical_penalty,
            ),
        )


    return jax.jit(
        jax.value_and_grad(
            objective,
            argnums=0,
            has_aux=True,
        )
    )


refit_objective = (
    make_support_refit_objective(
        SELECTED_SUPPORT_MASK
    )
)


# ======================================================================================
# 9. DEFINE NUMPY OPTIMIZER HELPERS
# ======================================================================================

def _writable_float64_cell_23(
    value: Any,
    *,
    name: str,
    expected_shape: tuple[int, ...],
) -> np.ndarray:
    """
    Convert a JAX or NumPy value into a writable owned float64 array.
    """

    output = np.array(
        value,
        dtype=np.float64,
        copy=True,
        order="C",
    )


    if output.shape != expected_shape:
        raise RuntimeError(
            f"{name} has an incorrect shape.\n"
            f"Expected: {expected_shape}\n"
            f"Observed: {output.shape}"
        )


    if not output.flags[
        "WRITEABLE"
    ]:
        output = output.copy(
            order="C"
        )


    if not np.isfinite(
        output
    ).all():
        raise RuntimeError(
            f"{name} contains nonfinite values."
        )


    return output


def _clip_gradient_cell_23(
    gradient: np.ndarray,
    maximum_norm: float,
) -> tuple[
    np.ndarray,
    float,
    bool,
]:
    """
    Clip one gradient by its global Euclidean norm.
    """

    gradient_array = np.array(
        gradient,
        dtype=np.float64,
        copy=True,
    )


    gradient_norm = float(
        np.linalg.norm(
            gradient_array
        )
    )


    if not np.isfinite(
        gradient_norm
    ):
        raise RuntimeError(
            "The support-refit gradient norm is nonfinite."
        )


    clipped = bool(
        gradient_norm
        >
        maximum_norm
    )


    if clipped:
        scale = (
            maximum_norm
            /
            max(
                gradient_norm,
                np.finfo(
                    np.float64
                ).tiny,
            )
        )


        clipped_gradient = (
            gradient_array
            *
            scale
        )


    else:
        clipped_gradient = (
            gradient_array.copy()
        )


    return (
        clipped_gradient,
        gradient_norm,
        clipped,
    )


# ======================================================================================
# 10. CONFIGURE RESUMABLE SUPPORT-REFIT CHECKPOINTS
# ======================================================================================

NOTEBOOK_12_SUPPORT_REFIT_CHECKPOINT_DIR = (
    NOTEBOOK_12_MODEL_DIR
    /
    "selected_support_refit_checkpoint"
)


NOTEBOOK_12_SUPPORT_REFIT_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


REFIT_CHECKPOINT_STATE_PATH = (
    NOTEBOOK_12_SUPPORT_REFIT_CHECKPOINT_DIR
    /
    "checkpoint_state.json"
)


REFIT_CHECKPOINT_ARRAY_PATH = (
    NOTEBOOK_12_SUPPORT_REFIT_CHECKPOINT_DIR
    /
    "checkpoint_arrays.npz"
)


REFIT_CHECKPOINT_HISTORY_PATH = (
    NOTEBOOK_12_SUPPORT_REFIT_CHECKPOINT_DIR
    /
    "checkpoint_history.parquet"
)


if (
    FORCE_RESTART_SUPPORT_REFIT
    and
    NOTEBOOK_12_SUPPORT_REFIT_CHECKPOINT_DIR.exists()
):
    shutil.rmtree(
        NOTEBOOK_12_SUPPORT_REFIT_CHECKPOINT_DIR
    )


    NOTEBOOK_12_SUPPORT_REFIT_CHECKPOINT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


training_registry_hash_cell_23 = hashlib.sha256(
    np.ascontiguousarray(
        training_row_group_indices
    ).tobytes()
).hexdigest()


support_hash_cell_23 = hashlib.sha256(
    np.ascontiguousarray(
        SELECTED_SUPPORT_MASK.astype(
            np.uint8
        )
    ).tobytes()
).hexdigest()


refit_signature_payload_cell_23 = {
    "selected_job_id": str(
        SELECTED_JOB_ID
    ),

    "selected_candidate_id": str(
        SELECTED_CANDIDATE_ID
    ),

    "selected_regularization_lambda": float(
        SELECTED_REGULARIZATION_LAMBDA
    ),

    "selected_support_sha256": (
        support_hash_cell_23
    ),

    "training_registry_sha256": (
        training_registry_hash_cell_23
    ),

    "refit_steps": int(
        REFIT_STEPS
    ),

    "training_batch_trajectories": int(
        TRAINING_BATCH_TRAJECTORIES
    ),

    "learning_rate": float(
        REFIT_LEARNING_RATE
    ),

    "adam_beta1": float(
        REFIT_ADAM_BETA1
    ),

    "adam_beta2": float(
        REFIT_ADAM_BETA2
    ),

    "adam_epsilon": float(
        REFIT_ADAM_EPSILON
    ),

    "gradient_norm_clip": float(
        REFIT_GRADIENT_NORM_CLIP
    ),

    "ridge_weight": float(
        RIDGE_WEIGHT
    ),

    "physical_barrier_weight": float(
        PHYSICAL_BARRIER_WEIGHT
    ),

    "pseudo_huber_delta_v": float(
        PSEUDO_HUBER_DELTA_V
    ),
}


REFIT_SIGNATURE_CELL_23 = hashlib.sha256(
    json.dumps(
        refit_signature_payload_cell_23,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


def _save_refit_checkpoint_cell_23(
    *,
    completed_step: int,
    coefficients: np.ndarray,
    first_moment: np.ndarray,
    second_moment: np.ndarray,
    rng: np.random.Generator,
    history_records: list[
        dict[str, Any]
    ],
    clipped_step_count: int,
    elapsed_seconds: float,
    status: str,
) -> None:
    """
    Save all state required to resume the support refit.
    """

    _atomic_write_npz_cell_23(
        REFIT_CHECKPOINT_ARRAY_PATH,

        coefficients=np.asarray(
            coefficients,
            dtype=np.float64,
        ),

        first_moment=np.asarray(
            first_moment,
            dtype=np.float64,
        ),

        second_moment=np.asarray(
            second_moment,
            dtype=np.float64,
        ),
    )


    _atomic_write_parquet_cell_23(
        pd.DataFrame(
            history_records
        ),
        REFIT_CHECKPOINT_HISTORY_PATH,
    )


    _atomic_write_json_cell_23(
        REFIT_CHECKPOINT_STATE_PATH,
        {
            "saved_utc": (
                _utc_now_iso_cell_23()
            ),

            "status": str(
                status
            ),

            "refit_signature": (
                REFIT_SIGNATURE_CELL_23
            ),

            "completed_step": int(
                completed_step
            ),

            "next_step": int(
                completed_step
                +
                1
            ),

            "clipped_gradient_step_count": int(
                clipped_step_count
            ),

            "elapsed_seconds": float(
                elapsed_seconds
            ),

            "history_row_count": int(
                len(
                    history_records
                )
            ),

            "rng_state": (
                rng.bit_generator.state
            ),
        },
    )


def _load_refit_checkpoint_cell_23() -> dict[
    str,
    Any,
] | None:
    """
    Load a compatible support-refit checkpoint.
    """

    if (
        not RESUME_SUPPORT_REFIT
        or
        FORCE_RESTART_SUPPORT_REFIT
    ):
        return None


    required_paths = (
        REFIT_CHECKPOINT_STATE_PATH,
        REFIT_CHECKPOINT_ARRAY_PATH,
        REFIT_CHECKPOINT_HISTORY_PATH,
    )


    if not all(
        path.is_file()
        for path in required_paths
    ):
        return None


    with REFIT_CHECKPOINT_STATE_PATH.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        state = json.load(
            file_handle
        )


    if str(
        state.get(
            "refit_signature",
            "",
        )
    ) != REFIT_SIGNATURE_CELL_23:
        raise RuntimeError(
            "The saved support-refit checkpoint is incompatible with "
            "the current selection or optimization configuration. "
            "Set FORCE_RESTART_SUPPORT_REFIT=True to discard it."
        )


    with np.load(
        REFIT_CHECKPOINT_ARRAY_PATH,
        allow_pickle=False,
    ) as archive:
        coefficients = (
            _writable_float64_cell_23(
                archive[
                    "coefficients"
                ],
                name=(
                    "checkpoint coefficients"
                ),
                expected_shape=(
                    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
                ),
            )
        )


        first_moment = (
            _writable_float64_cell_23(
                archive[
                    "first_moment"
                ],
                name=(
                    "checkpoint first moment"
                ),
                expected_shape=(
                    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
                ),
            )
        )


        second_moment = (
            _writable_float64_cell_23(
                archive[
                    "second_moment"
                ],
                name=(
                    "checkpoint second moment"
                ),
                expected_shape=(
                    NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
                ),
            )
        )


    history_df = pd.read_parquet(
        REFIT_CHECKPOINT_HISTORY_PATH
    )


    history_records = history_df.to_dict(
        orient="records"
    )


    completed_step = int(
        state[
            "completed_step"
        ]
    )


    if (
        len(
            history_records
        )
        ==
        0
        and
        completed_step
        >
        0
    ):
        raise RuntimeError(
            "The support-refit checkpoint history is empty."
        )


    rng = np.random.default_rng()


    rng.bit_generator.state = state[
        "rng_state"
    ]


    return {
        "state": state,
        "coefficients": coefficients,
        "first_moment": first_moment,
        "second_moment": second_moment,
        "history_records": (
            history_records
        ),
        "rng": rng,
    }


# ======================================================================================
# 11. INITIALIZE OR RESUME THE SUPPORT REFIT
# ======================================================================================

checkpoint_cell_23 = (
    _load_refit_checkpoint_cell_23()
)


if checkpoint_cell_23 is None:
    refit_coefficients = (
        _writable_float64_cell_23(
            SELECTED_TRAINED_COEFFICIENTS
            *
            selected_support_float,

            name=(
                "initial support-refit coefficients"
            ),

            expected_shape=(
                NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
            ),
        )
    )


    refit_first_moment = np.zeros_like(
        refit_coefficients
    )


    refit_second_moment = np.zeros_like(
        refit_coefficients
    )


    refit_rng = np.random.default_rng(
        RANDOM_SEED
        +
        120_000
    )


    refit_history_records: list[
        dict[str, Any]
    ] = []


    next_refit_step = 1

    clipped_gradient_step_count = 0

    prior_elapsed_seconds = 0.0


else:
    refit_coefficients = checkpoint_cell_23[
        "coefficients"
    ]


    refit_first_moment = checkpoint_cell_23[
        "first_moment"
    ]


    refit_second_moment = checkpoint_cell_23[
        "second_moment"
    ]


    refit_history_records = checkpoint_cell_23[
        "history_records"
    ]


    refit_rng = checkpoint_cell_23[
        "rng"
    ]


    next_refit_step = int(
        checkpoint_cell_23[
            "state"
        ][
            "next_step"
        ]
    )


    clipped_gradient_step_count = int(
        checkpoint_cell_23[
            "state"
        ][
            "clipped_gradient_step_count"
        ]
    )


    prior_elapsed_seconds = float(
        checkpoint_cell_23[
            "state"
        ][
            "elapsed_seconds"
        ]
    )


    print(
        f"Resuming support refit from step "
        f"{next_refit_step:,}; "
        f"last saved step was "
        f"{next_refit_step - 1:,}."
    )


# ======================================================================================
# 12. REFIT THE CLOSED SUPPORT ON TRAINING TRAJECTORIES ONLY
# ======================================================================================

refit_session_start_time = (
    time.perf_counter()
)


for update_step in range(
    next_refit_step,
    REFIT_STEPS
    +
    1,
):
    sampled_row_groups = (
        refit_rng.choice(
            training_row_group_indices,

            size=min(
                TRAINING_BATCH_TRAJECTORIES,
                TRAIN_ROW_GROUP_COUNT,
            ),

            replace=(
                TRAIN_ROW_GROUP_COUNT
                <
                TRAINING_BATCH_TRAJECTORIES
            ),
        )
    )


    batch = batch_to_jax(
        load_padded_trajectory_batch(
            partition="train",

            row_group_indices=(
                sampled_row_groups
            ),
        )
    )


    (
        (
            total_loss,
            auxiliary_losses,
        ),
        gradient,
    ) = refit_objective(
        jnp.asarray(
            refit_coefficients,
            dtype=jnp.float64,
        ),
        batch,
    )


    (
        data_loss,
        ridge_penalty,
        physical_penalty,
    ) = auxiliary_losses


    # ------------------------------------------------------------------
    # Critical correction:
    #
    # np.asarray() can expose a read-only JAX buffer. np.array(...,
    # copy=True) creates an owned writable NumPy array.
    # ------------------------------------------------------------------

    gradient_array = np.array(
        gradient,
        dtype=np.float64,
        copy=True,
        order="C",
    )


    if gradient_array.shape != (
        NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
    ):
        raise RuntimeError(
            "The support-refit gradient has an incorrect shape."
        )


    if not np.isfinite(
        gradient_array
    ).all():
        nonfinite_positions = np.flatnonzero(
            ~np.isfinite(
                gradient_array
            )
        )


        raise RuntimeError(
            "The support-refit gradient contains nonfinite values.\n"
            f"Count: {nonfinite_positions.size:,}\n"
            f"First positions: "
            f"{nonfinite_positions[:20].tolist()}"
        )


    # Non-in-place masking prevents the read-only-array error.

    gradient_array = (
        gradient_array
        *
        selected_support_float
    )


    (
        clipped_gradient,
        gradient_norm,
        gradient_was_clipped,
    ) = _clip_gradient_cell_23(
        gradient_array,
        REFIT_GRADIENT_NORM_CLIP,
    )


    if gradient_was_clipped:
        clipped_gradient_step_count += 1


    refit_first_moment = (
        REFIT_ADAM_BETA1
        *
        refit_first_moment
        +
        (
            1.0
            -
            REFIT_ADAM_BETA1
        )
        *
        clipped_gradient
    )


    refit_second_moment = (
        REFIT_ADAM_BETA2
        *
        refit_second_moment
        +
        (
            1.0
            -
            REFIT_ADAM_BETA2
        )
        *
        np.square(
            clipped_gradient
        )
    )


    first_hat = (
        refit_first_moment
        /
        (
            1.0
            -
            REFIT_ADAM_BETA1
            **
            update_step
        )
    )


    second_hat = (
        refit_second_moment
        /
        (
            1.0
            -
            REFIT_ADAM_BETA2
            **
            update_step
        )
    )


    updated_coefficients = (
        refit_coefficients
        -
        REFIT_LEARNING_RATE
        *
        first_hat
        /
        (
            np.sqrt(
                second_hat
            )
            +
            REFIT_ADAM_EPSILON
        )
    )


    updated_coefficients = np.clip(
        updated_coefficients,
        -
        REFIT_MAX_PARAMETER_ABS,
        REFIT_MAX_PARAMETER_ABS,
    )


    # Non-in-place support masking prevents aliasing and writeability errors.

    refit_coefficients = (
        _writable_float64_cell_23(
            updated_coefficients
            *
            selected_support_float,

            name=(
                "updated support-refit coefficients"
            ),

            expected_shape=(
                NOTEBOOK_12_REDUCED_COEFFICIENT_COUNT,
            ),
        )
    )


    should_checkpoint = (
        update_step
        ==
        1
        or
        update_step
        %
        CHECKPOINT_INTERVAL_STEPS
        ==
        0
        or
        update_step
        ==
        REFIT_STEPS
    )


    if should_checkpoint:
        total_loss_value = float(
            np.asarray(
                total_loss
            )
        )


        data_loss_value = float(
            np.asarray(
                data_loss
            )
        )


        ridge_penalty_value = float(
            np.asarray(
                ridge_penalty
            )
        )


        physical_penalty_value = float(
            np.asarray(
                physical_penalty
            )
        )


        for value_name, value in (
            (
                "total loss",
                total_loss_value,
            ),
            (
                "data loss",
                data_loss_value,
            ),
            (
                "ridge penalty",
                ridge_penalty_value,
            ),
            (
                "physical penalty",
                physical_penalty_value,
            ),
            (
                "gradient norm",
                gradient_norm,
            ),
        ):
            if not np.isfinite(
                value
            ):
                raise RuntimeError(
                    f"The support-refit {value_name} is nonfinite."
                )


        cumulative_elapsed_seconds = float(
            prior_elapsed_seconds
            +
            time.perf_counter()
            -
            refit_session_start_time
        )


        refit_history_records.append(
            {
                "step": int(
                    update_step
                ),

                "total_loss": (
                    total_loss_value
                ),

                "data_loss": (
                    data_loss_value
                ),

                "ridge_penalty": (
                    ridge_penalty_value
                ),

                "physical_penalty": (
                    physical_penalty_value
                ),

                "gradient_norm": float(
                    gradient_norm
                ),

                "gradient_was_clipped": bool(
                    gradient_was_clipped
                ),

                "coefficient_norm": float(
                    np.linalg.norm(
                        refit_coefficients
                    )
                ),

                "active_coefficient_count": int(
                    SELECTED_SUPPORT_MASK.sum()
                ),

                "elapsed_seconds": float(
                    cumulative_elapsed_seconds
                ),
            }
        )


        _save_refit_checkpoint_cell_23(
            completed_step=(
                update_step
            ),

            coefficients=(
                refit_coefficients
            ),

            first_moment=(
                refit_first_moment
            ),

            second_moment=(
                refit_second_moment
            ),

            rng=(
                refit_rng
            ),

            history_records=(
                refit_history_records
            ),

            clipped_step_count=(
                clipped_gradient_step_count
            ),

            elapsed_seconds=(
                cumulative_elapsed_seconds
            ),

            status=(
                "COMPLETED"
                if update_step
                ==
                REFIT_STEPS
                else
                "IN_PROGRESS"
            ),
        )


        print(
            f"Support refit step "
            f"{update_step:05d}/"
            f"{REFIT_STEPS:05d} | "
            f"loss={total_loss_value:.8e} | "
            f"data={data_loss_value:.8e} | "
            f"grad={gradient_norm:.4e} | "
            f"checkpoint=saved"
        )


# ======================================================================================
# 13. FREEZE THE REDUCED AND FULL COEFFICIENT VECTORS
# ======================================================================================

FROZEN_REDUCED_COEFFICIENTS = np.array(
    refit_coefficients,
    dtype=np.float64,
    copy=True,
)


if not np.isfinite(
    FROZEN_REDUCED_COEFFICIENTS
).all():
    raise RuntimeError(
        "The frozen reduced coefficient vector contains nonfinite values."
    )


if np.any(
    np.abs(
        FROZEN_REDUCED_COEFFICIENTS[
            ~SELECTED_SUPPORT_MASK
        ]
    )
    >
    0.0
):
    raise RuntimeError(
        "A frozen coefficient is nonzero outside the selected support."
    )


# Resolve the identifiable expansion matrix across earlier cell variants.

if (
    "NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX"
    in globals()
):
    expansion_matrix_cell_23 = globals()[
        "NOTEBOOK_12_IDENTIFIABLE_EXPANSION_MATRIX"
    ]


elif (
    "JAX_IDENTIFIABLE_EXPANSION_MATRIX"
    in globals()
):
    expansion_matrix_cell_23 = np.asarray(
        globals()[
            "JAX_IDENTIFIABLE_EXPANSION_MATRIX"
        ],
        dtype=np.float64,
    )


else:
    raise RuntimeError(
        "The identifiable expansion matrix is unavailable. "
        "Execute the coefficient-reduction cell before Cell 23."
    )


FROZEN_FULL_COEFFICIENTS = np.asarray(
    expansion_matrix_cell_23
    @
    FROZEN_REDUCED_COEFFICIENTS,
    dtype=np.float64,
).reshape(
    -1
)


if not np.isfinite(
    FROZEN_FULL_COEFFICIENTS
).all():
    raise RuntimeError(
        "The frozen full coefficient vector contains nonfinite values."
    )


MODEL_SELECTION_FROZEN = True

ORDINARY_TEST_ACCESS_PERMITTED = True

STRICT_TEST_ACCESS_PERMITTED = True


# ======================================================================================
# 14. DEFINE AND SAVE FINAL MODEL ARTIFACTS
# ======================================================================================

NOTEBOOK_12_SELECTED_SUPPORT_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_selected_support.parquet"
)


NOTEBOOK_12_REFIT_HISTORY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_selected_support_refit_history.parquet"
)


NOTEBOOK_12_FROZEN_COEFFICIENT_PATH = (
    NOTEBOOK_12_MODEL_DIR
    /
    "lpv_frozen_coefficients.npz"
)


NOTEBOOK_12_MODEL_SELECTION_CONTRACT_PATH = (
    NOTEBOOK_12_CONTRACT_DIR
    /
    "notebook12_model_selection_contract.json"
)


NOTEBOOK_12_STRICT_TEST_QUARANTINE_PATH = Path(
    globals().get(
        "NOTEBOOK_12_STRICT_TEST_QUARANTINE_PATH",

        NOTEBOOK_12_CONTRACT_DIR
        /
        "notebook12_strict_test_quarantine.json",
    )
).expanduser().resolve()


_atomic_write_parquet_cell_23(
    NOTEBOOK_12_SELECTED_SUPPORT_DF,
    NOTEBOOK_12_SELECTED_SUPPORT_PATH,
)


_atomic_write_parquet_cell_23(
    pd.DataFrame(
        refit_history_records
    ),
    NOTEBOOK_12_REFIT_HISTORY_PATH,
)


_atomic_write_npz_cell_23(
    NOTEBOOK_12_FROZEN_COEFFICIENT_PATH,

    reduced_coefficients=(
        FROZEN_REDUCED_COEFFICIENTS
    ),

    full_coefficients=(
        FROZEN_FULL_COEFFICIENTS
    ),

    support_mask=(
        SELECTED_SUPPORT_MASK.astype(
            bool
        )
    ),

    selected_candidate_id=np.asarray(
        SELECTED_CANDIDATE_ID,
        dtype=np.str_,
    ),

    selected_job_id=np.asarray(
        SELECTED_JOB_ID,
        dtype=np.str_,
    ),

    selected_regularization_lambda=np.asarray(
        SELECTED_REGULARIZATION_LAMBDA,
        dtype=np.float64,
    ),
)


final_refit_elapsed_seconds = float(
    prior_elapsed_seconds
    +
    time.perf_counter()
    -
    refit_session_start_time
)


_atomic_write_json_cell_23(
    NOTEBOOK_12_MODEL_SELECTION_CONTRACT_PATH,
    {
        "created_utc": (
            _utc_now_iso_cell_23()
        ),

        "status": (
            "CALIBRATION_COMPLETE_MODEL_FROZEN"
        ),

        "selected_job_id": (
            SELECTED_JOB_ID
        ),

        "selected_candidate_id": (
            SELECTED_CANDIDATE_ID
        ),

        "selected_regularization_lambda": float(
            SELECTED_REGULARIZATION_LAMBDA
        ),

        "support_threshold": float(
            GROUP_SUPPORT_THRESHOLD
        ),

        "active_group_count": int(
            len(
                SELECTED_ACTIVE_GROUP_IDS
            )
        ),

        "active_reduced_coefficient_count": int(
            SELECTED_SUPPORT_MASK.sum()
        ),

        "basis_strong_heredity_closed": True,

        "scope_heredity_closed": True,

        "support_refit_steps": int(
            REFIT_STEPS
        ),

        "support_refit_elapsed_seconds": float(
            final_refit_elapsed_seconds
        ),

        "support_refit_checkpoint_directory": str(
            NOTEBOOK_12_SUPPORT_REFIT_CHECKPOINT_DIR
        ),

        "training_data_used_for_fitting": True,

        "calibration_data_used_for_selection": True,

        "ordinary_test_data_used_for_selection": False,

        "strict_test_data_used_for_selection": False,

        "ordinary_test_access_permitted_after_freeze": True,

        "strict_test_access_permitted_after_freeze": True,

        "selected_support_path": str(
            NOTEBOOK_12_SELECTED_SUPPORT_PATH
        ),

        "refit_history_path": str(
            NOTEBOOK_12_REFIT_HISTORY_PATH
        ),

        "frozen_coefficient_path": str(
            NOTEBOOK_12_FROZEN_COEFFICIENT_PATH
        ),
    },
)


_atomic_write_json_cell_23(
    NOTEBOOK_12_STRICT_TEST_QUARANTINE_PATH,
    {
        "updated_utc": (
            _utc_now_iso_cell_23()
        ),

        "status": (
            "MODEL_FROZEN_FINAL_EVALUATION_PERMITTED"
        ),

        "selected_candidate_id": (
            SELECTED_CANDIDATE_ID
        ),

        "selected_job_id": (
            SELECTED_JOB_ID
        ),

        "ordinary_test_arrays_opened": False,

        "strict_test_arrays_opened": False,

        "ordinary_test_access_permitted": True,

        "strict_test_access_permitted": True,

        "test_data_used_for_selection": False,

        "strict_test_data_used_for_selection": False,
    },
)


# ======================================================================================
# 15. VERIFY SAVED ARTIFACTS
# ======================================================================================

required_cell_23_outputs = (
    NOTEBOOK_12_SELECTED_SUPPORT_PATH,
    NOTEBOOK_12_REFIT_HISTORY_PATH,
    NOTEBOOK_12_FROZEN_COEFFICIENT_PATH,
    NOTEBOOK_12_MODEL_SELECTION_CONTRACT_PATH,
    NOTEBOOK_12_STRICT_TEST_QUARANTINE_PATH,
)


missing_cell_23_outputs = [
    path
    for path in required_cell_23_outputs
    if not Path(
        path
    ).is_file()
]


if missing_cell_23_outputs:
    raise FileNotFoundError(
        "Cell 23 failed to create required outputs:\n  - "
        +
        "\n  - ".join(
            str(
                path
            )
            for path in missing_cell_23_outputs
        )
    )


with np.load(
    NOTEBOOK_12_FROZEN_COEFFICIENT_PATH,
    allow_pickle=False,
) as archive:
    reloaded_reduced = np.asarray(
        archive[
            "reduced_coefficients"
        ],
        dtype=np.float64,
    )


    reloaded_full = np.asarray(
        archive[
            "full_coefficients"
        ],
        dtype=np.float64,
    )


    reloaded_support = np.asarray(
        archive[
            "support_mask"
        ],
        dtype=bool,
    )


if not np.array_equal(
    reloaded_reduced,
    FROZEN_REDUCED_COEFFICIENTS,
):
    raise RuntimeError(
        "The reloaded frozen reduced coefficients differ from memory."
    )


if not np.array_equal(
    reloaded_full,
    FROZEN_FULL_COEFFICIENTS,
):
    raise RuntimeError(
        "The reloaded frozen full coefficients differ from memory."
    )


if not np.array_equal(
    reloaded_support,
    SELECTED_SUPPORT_MASK,
):
    raise RuntimeError(
        "The reloaded selected-support mask differs from memory."
    )


# ======================================================================================
# 16. FINAL EXECUTION SUMMARY
# ======================================================================================

print("=" * 118)
print("HEREDITY-CLOSED SUPPORT REFIT AND FROZEN MODEL")
print("=" * 118)


print(
    f"Selected candidate             : "
    f"{SELECTED_CANDIDATE_ID}"
)


print(
    f"Selected training job          : "
    f"{SELECTED_JOB_ID}"
)


print(
    f"Selected lambda                : "
    f"{SELECTED_REGULARIZATION_LAMBDA:.8e}"
)


print(
    f"Initially active groups        : "
    f"{int(NOTEBOOK_12_SELECTED_SUPPORT_DF['initially_active'].sum()):,}"
)


print(
    f"Groups activated by closure    : "
    f"{int(NOTEBOOK_12_SELECTED_SUPPORT_DF['activated_by_closure'].sum()):,}"
)


print(
    f"Active hierarchy groups        : "
    f"{len(SELECTED_ACTIVE_GROUP_IDS):,}"
)


print(
    f"Active reduced coefficients    : "
    f"{int(SELECTED_SUPPORT_MASK.sum()):,}"
)


print(
    f"Support-refit steps            : "
    f"{REFIT_STEPS:,}"
)


print(
    f"Clipped-gradient steps         : "
    f"{clipped_gradient_step_count:,}"
)


print(
    f"Support-refit elapsed time     : "
    f"{final_refit_elapsed_seconds:.3f} s"
)


print(
    f"Frozen reduced dimension       : "
    f"{FROZEN_REDUCED_COEFFICIENTS.size:,}"
)


print(
    f"Frozen full dimension          : "
    f"{FROZEN_FULL_COEFFICIENTS.size:,}"
)


print(
    f"Selected support table         : "
    f"{NOTEBOOK_12_SELECTED_SUPPORT_PATH}"
)


print(
    f"Support-refit history          : "
    f"{NOTEBOOK_12_REFIT_HISTORY_PATH}"
)


print(
    f"Frozen coefficient archive     : "
    f"{NOTEBOOK_12_FROZEN_COEFFICIENT_PATH}"
)


print(
    "Ordinary-test access           : permitted"
)


print(
    "Strict-test access             : permitted"
)


print("-" * 118)


print(
    "The selected hierarchy was closed under heredity, refitted on "
    "training trajectories only, checkpointed, and frozen before final "
    "test evaluation."
)


print("=" * 118)


display(
    NOTEBOOK_12_SELECTED_SUPPORT_DF
)


display(
    pd.DataFrame(
        refit_history_records
    )
)

Support refit step 00001/01200 | loss=5.97286759e-03 | data=5.97272654e-03 | grad=1.6139e-05 | checkpoint=saved
Support refit step 00100/01200 | loss=2.45020581e-03 | data=2.45006339e-03 | grad=1.5407e-08 | checkpoint=saved
Support refit step 00200/01200 | loss=2.98217848e-03 | data=2.98203527e-03 | grad=1.5449e-08 | checkpoint=saved
Support refit step 00300/01200 | loss=2.05094103e-03 | data=2.05079702e-03 | grad=2.7854e-07 | checkpoint=saved
Support refit step 00400/01200 | loss=3.26235435e-03 | data=3.26220912e-03 | grad=9.5959e-08 | checkpoint=saved
Support refit step 00500/01200 | loss=3.90485010e-03 | data=3.90470413e-03 | grad=1.5597e-08 | checkpoint=saved
Support refit step 00600/01200 | loss=1.04036789e-03 | data=1.04022106e-03 | grad=2.7277e-06 | checkpoint=saved
Support refit step 00700/01200 | loss=3.09695355e-03 | data=3.09680593e-03 | grad=1.5685e-08 | checkpoint=saved
Support refit step 00800/01200 | loss=1.13759916e-03 | data=1.13745084e-03 | grad=1.5723e-08 | checkpoin

,penalty_group_id,transformed_parameter,scope,scope_level,entity_block_id,basis_id,basis_position,group_norm_before_closure,initially_active,active_after_heredity_closure,activated_by_closure
0,group__a93b368b2c0c9140,eta_R0,GLOBAL,0,GLOBAL,basis__intercept,0,10.584670,True,True,False
1,group__8eec8179a081cd32,eta_R1,GLOBAL,0,GLOBAL,basis__intercept,0,9.988978,True,True,False
2,group__92b4e249facd0a05,eta_R2,GLOBAL,0,GLOBAL,basis__intercept,0,9.298206,True,True,False
3,group__00eb4211770ab61c,eta_tau1,GLOBAL,0,GLOBAL,basis__intercept,0,3.667911,True,True,False
4,group__8e7c8512e7546f66,eta_tau_gap,GLOBAL,0,GLOBAL,basis__intercept,0,5.176919,True,True,False


,step,total_loss,data_loss,ridge_penalty,physical_penalty,gradient_norm,gradient_was_clipped,coefficient_norm,active_coefficient_count,elapsed_seconds
0,1,0.005973,0.005973,1.410523e-07,0.0,1.613946e-05,False,18.401208,5,0.418348
1,100,0.002450,0.002450,1.424262e-07,0.0,1.540704e-08,False,18.488797,5,28.439287
2,200,0.002982,0.002982,1.432098e-07,0.0,1.544937e-08,False,18.539719,5,51.929811
3,300,0.002051,0.002051,1.440184e-07,0.0,2.785387e-07,False,18.592086,5,75.437565
4,400,0.003262,0.003262,1.452255e-07,0.0,9.595905e-08,False,18.669855,5,95.115151
5,500,0.003905,0.003905,1.459627e-07,0.0,1.559715e-08,False,18.716814,5,111.304963
6,600,0.001040,0.001040,1.468306e-07,0.0,2.727670e-06,False,18.772687,5,132.647511
7,700,0.003097,0.003097,1.476207e-07,0.0,1.568548e-08,False,18.823559,5,152.989823
8,800,0.001138,0.001137,1.483237e-07,0.0,1.572279e-08,False,18.867464,5,173.011860
9,900,0.003173,0.003173,1.491737e-07,0.0,1.642128e-07,False,18.921546,5,188.948131


In [36]:
# ======================================================================================
# CELL 24 — MATERIALIZE ORDINARY-TEST AND STRICT-TEST DESIGN DATA AFTER FREEZING
# ======================================================================================

if not bool(
    globals().get(
        "MODEL_SELECTION_FROZEN",
        False,
    )
):
    raise RuntimeError(
        "Final evaluation data cannot be opened before the model is frozen."
    )

if not bool(
    globals().get(
        "ORDINARY_TEST_ACCESS_PERMITTED",
        False,
    )
):
    raise RuntimeError(
        "Ordinary-test access has not been permitted."
    )

if not bool(
    globals().get(
        "STRICT_TEST_ACCESS_PERMITTED",
        False,
    )
):
    raise RuntimeError(
        "Strict-test access has not been permitted."
    )

NOTEBOOK_12_TEST_DESIGN_SAMPLE_PATH = (
    NOTEBOOK_12_DATA_DIR
    /
    "lpv_ordinary_test_design_samples.parquet"
)

NOTEBOOK_12_STRICT_TEST_DESIGN_SAMPLE_PATH = (
    NOTEBOOK_12_DATA_DIR
    /
    "lpv_strict_test_design_samples.parquet"
)

NOTEBOOK_12_FINAL_EVALUATION_MANIFEST_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_final_evaluation_trajectory_manifest.parquet"
)


def replace_arrow_column(
    table: pa.Table,
    column_name: str,
    replacement: pa.Array | pa.ChunkedArray,
) -> pa.Table:
    column_index = table.schema.get_field_index(
        column_name
    )

    if column_index < 0:
        raise KeyError(
            f"Column {column_name!r} is absent from the Arrow table."
        )

    return table.set_column(
        column_index,
        column_name,
        replacement,
    )


def construct_final_design_table(
    source_table: pa.Table,
    *,
    evaluation_partition: str,
    source_row_group_index: int,
    strict_test: bool,
) -> tuple[
    pa.Table,
    dict[str, Any],
]:
    working_table = source_table

    strict_source_column = (
        STATE_SAMPLE_COLUMN_BINDINGS[
            "is_strict_test"
        ]
    )

    if strict_source_column is not None:
        working_table = replace_arrow_column(
            working_table,
            str(
                strict_source_column
            ),
            pa.array(
                np.zeros(
                    working_table.num_rows,
                    dtype=bool,
                ),
                type=pa.bool_(),
            ),
        )

    output_table, manifest_record = (
        construct_design_table(
            working_table,
            partition="test",
            source_row_group_index=(
                source_row_group_index
            ),
        )
    )

    if "is_strict_test" in output_table.column_names:
        output_table = replace_arrow_column(
            output_table,
            "is_strict_test",
            pa.array(
                np.full(
                    output_table.num_rows,
                    strict_test,
                    dtype=bool,
                ),
                type=pa.bool_(),
            ),
        )

    manifest_record[
        "partition"
    ] = evaluation_partition

    manifest_record[
        "is_strict_test"
    ] = bool(
        strict_test
    )

    return output_table, manifest_record


def materialize_final_evaluation_partition(
    *,
    output_path: Path,
    evaluation_partition: str,
    strict_test: bool,
) -> dict[str, Any]:
    selected_access_role = (
        "STRICT_TEST_QUARANTINE"
        if strict_test
        else "ORDINARY_TEST_HOLDOUT"
    )

    selected_df = (
        NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF.loc[
            NOTEBOOK_12_SAMPLE_ACCESS_PLAN_DF[
                "access_role"
            ].eq(
                selected_access_role
            )
        ]
        .sort_values(
            "source_row_group_index",
            kind="mergesort",
        )
    )

    if selected_df.empty:
        raise RuntimeError(
            f"No trajectories are available for {evaluation_partition!r}."
        )

    temporary_path = output_path.with_suffix(
        output_path.suffix
        +
        ".tmp"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    writer: pq.ParquetWriter | None = None
    output_schema: pa.Schema | None = None
    manifest_records: list[
        dict[str, Any]
    ] = []

    try:
        for output_row_group_index, source_row_group_index in enumerate(
            selected_df[
                "source_row_group_index"
            ].astype(int)
        ):
            source_table = STATE_SAMPLE_PARQUET.read_row_group(
                source_row_group_index,
                use_threads=True,
            )

            output_table, manifest_record = (
                construct_final_design_table(
                    source_table,
                    evaluation_partition=(
                        evaluation_partition
                    ),
                    source_row_group_index=(
                        source_row_group_index
                    ),
                    strict_test=(
                        strict_test
                    ),
                )
            )

            manifest_record[
                "design_row_group_index"
            ] = int(
                output_row_group_index
            )

            if writer is None:
                output_schema = output_table.schema
                writer = pq.ParquetWriter(
                    temporary_path,
                    output_schema,
                    compression="zstd",
                    use_dictionary=True,
                    write_statistics=True,
                )
            elif not output_table.schema.equals(
                output_schema,
                check_metadata=False,
            ):
                raise RuntimeError(
                    "The final evaluation design schema changed between trajectories."
                )

            writer.write_table(
                output_table,
                row_group_size=(
                    output_table.num_rows
                ),
            )

            manifest_records.append(
                manifest_record
            )

    except Exception:
        if writer is not None:
            writer.close()
        if temporary_path.exists():
            temporary_path.unlink()
        raise

    if writer is None:
        raise RuntimeError(
            f"No {evaluation_partition} design rows were written."
        )

    writer.close()
    temporary_path.replace(
        output_path
    )

    saved = pq.ParquetFile(
        output_path
    )

    return {
        "partition": evaluation_partition,
        "path": str(
            output_path
        ),
        "sample_count": int(
            saved.metadata.num_rows
        ),
        "trajectory_count": int(
            saved.metadata.num_row_groups
        ),
        "schema": (
            saved.schema_arrow
        ),
        "sha256": calculate_streaming_sha256(
            output_path,
            chunk_bytes=(
                NOTEBOOK_12_HASH_CHUNK_BYTES
            ),
        ),
        "manifest_records": (
            manifest_records
        ),
    }


ORDINARY_TEST_DESIGN_RESULT = (
    materialize_final_evaluation_partition(
        output_path=(
            NOTEBOOK_12_TEST_DESIGN_SAMPLE_PATH
        ),
        evaluation_partition="test",
        strict_test=False,
    )
)

STRICT_TEST_DESIGN_RESULT = (
    materialize_final_evaluation_partition(
        output_path=(
            NOTEBOOK_12_STRICT_TEST_DESIGN_SAMPLE_PATH
        ),
        evaluation_partition="strict_test",
        strict_test=True,
    )
)

if not ORDINARY_TEST_DESIGN_RESULT[
    "schema"
].equals(
    STRICT_TEST_DESIGN_RESULT[
        "schema"
    ],
    check_metadata=False,
):
    raise RuntimeError(
        "Ordinary-test and strict-test design schemas do not match."
    )

NOTEBOOK_12_FINAL_EVALUATION_MANIFEST_DF = pd.DataFrame(
    ORDINARY_TEST_DESIGN_RESULT[
        "manifest_records"
    ]
    +
    STRICT_TEST_DESIGN_RESULT[
        "manifest_records"
    ]
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_FINAL_EVALUATION_MANIFEST_DF,
    NOTEBOOK_12_FINAL_EVALUATION_MANIFEST_PATH,
)

write_json_atomic(
    NOTEBOOK_12_STRICT_TEST_QUARANTINE_PATH,
    {
        "updated_utc": utc_now_iso(),
        "status": (
            "FINAL_EVALUATION_DATA_MATERIALIZED"
        ),
        "model_selection_frozen": True,
        "selected_candidate_id": (
            SELECTED_CANDIDATE_ID
        ),
        "ordinary_test_arrays_opened": True,
        "strict_test_arrays_opened": True,
        "ordinary_test_data_used_for_selection": False,
        "strict_test_data_used_for_selection": False,
        "ordinary_test_design_path": str(
            NOTEBOOK_12_TEST_DESIGN_SAMPLE_PATH
        ),
        "strict_test_design_path": str(
            NOTEBOOK_12_STRICT_TEST_DESIGN_SAMPLE_PATH
        ),
    },
)

print("=" * 110)
print("FINAL EVALUATION MATERIALIZATION")
print("=" * 110)
print(
    f"Ordinary-test trajectories : "
    f"{ORDINARY_TEST_DESIGN_RESULT['trajectory_count']:,}"
)
print(
    f"Strict-test trajectories   : "
    f"{STRICT_TEST_DESIGN_RESULT['trajectory_count']:,}"
)
print(
    f"Ordinary-test samples      : "
    f"{ORDINARY_TEST_DESIGN_RESULT['sample_count']:,}"
)
print(
    f"Strict-test samples        : "
    f"{STRICT_TEST_DESIGN_RESULT['sample_count']:,}"
)
print("=" * 110)


FINAL EVALUATION MATERIALIZATION
Ordinary-test trajectories : 1,080
Strict-test trajectories   : 9
Ordinary-test samples      : 4,918,294
Strict-test samples        : 35,330


In [42]:
# ======================================================================================
# CELL 25 — EVALUATE THE FROZEN MODEL ON TRAIN, CALIBRATION, TEST, AND STRICT TEST
# ======================================================================================

# Cell 22 may intentionally restrict its evaluator to calibration only. Cell 25 runs
# after model freezing, so it defines a separate post-freeze evaluator for train and
# calibration and a held-out evaluator for ordinary test and strict test.

if not bool(globals().get("MODEL_SELECTION_FROZEN", False)):
    raise RuntimeError(
        "Cell 25 cannot run before Cell 23 freezes the selected model."
    )

if not bool(globals().get("ORDINARY_TEST_ACCESS_PERMITTED", False)):
    raise RuntimeError(
        "Ordinary-test access has not been permitted."
    )

if not bool(globals().get("STRICT_TEST_ACCESS_PERMITTED", False)):
    raise RuntimeError(
        "Strict-test access has not been permitted."
    )

for required_path in (
    NOTEBOOK_12_TRAIN_DESIGN_SAMPLE_PATH,
    NOTEBOOK_12_CALIBRATION_DESIGN_SAMPLE_PATH,
    NOTEBOOK_12_TEST_DESIGN_SAMPLE_PATH,
    NOTEBOOK_12_STRICT_TEST_DESIGN_SAMPLE_PATH,
):
    required_path = Path(required_path)

    if not required_path.is_file():
        raise FileNotFoundError(
            f"A design parquet required by Cell 25 is missing: {required_path}"
        )


FINAL_DESIGN_PARQUETS: dict[str, pq.ParquetFile] = {
    "test": pq.ParquetFile(
        NOTEBOOK_12_TEST_DESIGN_SAMPLE_PATH
    ),
    "strict_test": pq.ParquetFile(
        NOTEBOOK_12_STRICT_TEST_DESIGN_SAMPLE_PATH
    ),
}


def build_selector_arrays_for_identity(
    *,
    chemistry: str,
    cell_entity_id: str,
) -> dict[str, Any]:
    """
    Build frozen global, chemistry, and optional cell selector arrays.
    """

    chemistry = str(
        chemistry
    ).strip().upper()

    cell_entity_id = str(
        cell_entity_id
    ).strip()

    known_chemistries = {
        str(value).strip().upper()
        for value in NOTEBOOK_12_CHEMISTRIES
    }

    if chemistry not in known_chemistries:
        raise RuntimeError(
            "Final evaluation encountered a chemistry absent from the frozen "
            f"hierarchy: {chemistry!r}."
        )

    expected_prefix = (
        chemistry
        +
        TRAJECTORY_ID_SEPARATOR
    )

    if not cell_entity_id.startswith(
        expected_prefix
    ):
        raise RuntimeError(
            "Inconsistent chemistry/cell identity in final evaluation: "
            f"chemistry={chemistry!r}, "
            f"cell_entity_id={cell_entity_id!r}."
        )

    has_cell_effect = bool(
        cell_entity_id
        in
        CELL_ENTITY_TO_INDEX
    )

    selector_shape = (
        NOTEBOOK_12_PARAMETER_COUNT,
        NOTEBOOK_12_BASIS_TERM_COUNT,
    )

    global_indices = np.empty(
        selector_shape,
        dtype=np.int32,
    )

    chemistry_indices = np.empty(
        selector_shape,
        dtype=np.int32,
    )

    cell_indices = np.full(
        selector_shape,
        -1,
        dtype=np.int32,
    )

    def required_index(
        key: tuple[
            str,
            str,
            str,
            str,
        ],
        selector_level: str,
    ) -> int:
        value = (
            FULL_COEFFICIENT_INDEX_LOOKUP.get(
                key
            )
        )

        if value is None:
            raise RuntimeError(
                f"Missing frozen {selector_level} coefficient selector: "
                f"{key!r}."
            )

        value = int(
            value
        )

        if not (
            0
            <=
            value
            <
            NOTEBOOK_12_FULL_COEFFICIENT_COUNT
        ):
            raise RuntimeError(
                f"Invalid frozen {selector_level} selector index "
                f"{value} for {key!r}."
            )

        return value

    for parameter_position, transformed_parameter in enumerate(
        LPV_TRANSFORMED_PARAMETERS
    ):
        for basis_position, basis_id in enumerate(
            BASIS_ID_BY_POSITION
        ):
            global_indices[
                parameter_position,
                basis_position,
            ] = required_index(
                (
                    transformed_parameter,
                    "GLOBAL",
                    "GLOBAL",
                    basis_id,
                ),
                "global",
            )

            chemistry_indices[
                parameter_position,
                basis_position,
            ] = required_index(
                (
                    transformed_parameter,
                    "CHEMISTRY",
                    chemistry,
                    basis_id,
                ),
                "chemistry",
            )

            if has_cell_effect:
                cell_indices[
                    parameter_position,
                    basis_position,
                ] = required_index(
                    (
                        transformed_parameter,
                        "CELL",
                        cell_entity_id,
                        basis_id,
                    ),
                    "cell",
                )

    if (
        has_cell_effect
        and
        np.any(
            cell_indices
            <
            0
        )
    ):
        raise RuntimeError(
            "A training-seen cell is missing one or more frozen cell selectors."
        )

    return {
        "global_full_indices": (
            global_indices
        ),
        "chemistry_full_indices": (
            chemistry_indices
        ),
        "cell_full_indices": (
            cell_indices
        ),
        "has_cell_effect": (
            has_cell_effect
        ),
    }


def load_final_design_trajectory(
    partition: str,
    row_group_index: int,
) -> dict[str, Any]:
    """
    Load one ordinary-test or strict-test trajectory.

    The source sample index may start at a nonzero value. Row order and
    delta_t_to_next_s define the recursive simulation, so a separate zero-based
    trajectory index is created while the original source index is retained.
    """

    partition = str(
        partition
    ).strip().lower()

    row_group_index = int(
        row_group_index
    )

    if partition not in FINAL_DESIGN_PARQUETS:
        raise ValueError(
            "Final loading permits only 'test' and 'strict_test'."
        )

    parquet_file = (
        FINAL_DESIGN_PARQUETS[
            partition
        ]
    )

    row_group_count = int(
        parquet_file.metadata.num_row_groups
    )

    if not (
        0
        <=
        row_group_index
        <
        row_group_count
    ):
        raise IndexError(
            f"Invalid row group {row_group_index} for "
            f"{partition!r}; available row groups: "
            f"{row_group_count}."
        )

    table = parquet_file.read_row_group(
        row_group_index,
        columns=[
            "trajectory_uid",
            "chemistry",
            "cell_entity_id",
            "sample_index",
            "elapsed_time_s",
            "delta_t_to_next_s",
            "voltage_v",
            "current_discharge_a",
            "temperature_c",
            "soc_for_ocv",
            "ocv_v",
            *NOTEBOOK_12_BASIS_DESIGN_COLUMNS,
        ],
        use_threads=True,
    )

    sample_count = int(
        table.num_rows
    )

    if sample_count < 2:
        raise RuntimeError(
            "A final-evaluation trajectory must contain at least two samples."
        )

    def one_value(
        column_name: str,
    ) -> str:
        values = (
            unique_nonmissing_arrow_values(
                table[
                    column_name
                ]
            )
        )

        if len(
            values
        ) != 1:
            raise RuntimeError(
                f"Row group {row_group_index} in partition "
                f"{partition!r} has {len(values)} distinct "
                f"{column_name!r} values; expected one."
            )

        return str(
            values[
                0
            ]
        ).strip()

    trajectory_uid = one_value(
        "trajectory_uid"
    )

    chemistry = one_value(
        "chemistry"
    ).upper()

    cell_entity_id = one_value(
        "cell_entity_id"
    )

    sample_index_column = table[
        "sample_index"
    ]

    if sample_index_column.null_count != 0:
        raise RuntimeError(
            "The held-out source sample index contains null values: "
            f"partition={partition!r}, "
            f"trajectory_uid={trajectory_uid!r}."
        )

    source_sample_index = np.asarray(
        pc.cast(
            sample_index_column,
            pa.int64(),
            safe=False,
        ).to_numpy(
            zero_copy_only=False
        ),
        dtype=np.int64,
    )

    if source_sample_index.shape != (
        sample_count,
    ):
        raise RuntimeError(
            f"Invalid source sample-index shape "
            f"{source_sample_index.shape}; "
            f"expected {(sample_count,)}."
        )

    # The source index is retained only for traceability. It does not need to
    # begin at zero and does not need unit increments after source filtering.
    if (
        source_sample_index.size
        >
        1
        and
        np.any(
            np.diff(
                source_sample_index
            )
            <=
            0
        )
    ):
        raise RuntimeError(
            "The held-out source sample index is not strictly increasing: "
            f"partition={partition!r}, "
            f"trajectory_uid={trajectory_uid!r}."
        )

    trajectory_sample_index = np.arange(
        sample_count,
        dtype=np.int64,
    )

    def numeric(
        column_name: str,
        *,
        allow_terminal_nonfinite: bool = False,
    ) -> np.ndarray:
        column = table[
            column_name
        ]

        if allow_terminal_nonfinite:
            nonterminal_column = column.slice(
                0,
                max(
                    sample_count
                    -
                    1,
                    0,
                ),
            )

            if nonterminal_column.null_count != 0:
                raise RuntimeError(
                    f"Column {column_name!r} contains a null "
                    "nonterminal value."
                )

        elif column.null_count != 0:
            raise RuntimeError(
                f"Column {column_name!r} contains null values."
            )

        values = np.asarray(
            pc.cast(
                column,
                pa.float64(),
                safe=False,
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.float64,
        )

        if values.shape != (
            sample_count,
        ):
            raise RuntimeError(
                f"Column {column_name!r} has shape "
                f"{values.shape}; expected {(sample_count,)}."
            )

        checked_values = (
            values[
                :-1
            ]
            if allow_terminal_nonfinite
            else values
        )

        if not np.isfinite(
            checked_values
        ).all():
            qualifier = (
                "nonterminal "
                if allow_terminal_nonfinite
                else ""
            )

            raise RuntimeError(
                f"Column {column_name!r} contains a nonfinite "
                f"{qualifier}value."
            )

        return values

    elapsed_time_s = numeric(
        "elapsed_time_s"
    )

    delta_t_s = numeric(
        "delta_t_to_next_s",
        allow_terminal_nonfinite=True,
    )

    if not np.isfinite(
        delta_t_s[
            -1
        ]
    ):
        delta_t_s[
            -1
        ] = 0.0

    if np.any(
        delta_t_s[
            :-1
        ]
        <=
        0.0
    ):
        raise RuntimeError(
            "A held-out trajectory contains a nonpositive "
            "nonterminal sampling interval."
        )

    basis_matrix = np.column_stack(
        [
            numeric(
                column_name
            )
            for column_name in (
                NOTEBOOK_12_BASIS_DESIGN_COLUMNS
            )
        ]
    )

    expected_basis_shape = (
        sample_count,
        NOTEBOOK_12_BASIS_TERM_COUNT,
    )

    if basis_matrix.shape != expected_basis_shape:
        raise RuntimeError(
            f"Basis matrix has shape {basis_matrix.shape}; "
            f"expected {expected_basis_shape}."
        )

    selector_arrays = (
        build_selector_arrays_for_identity(
            chemistry=chemistry,
            cell_entity_id=(
                cell_entity_id
            ),
        )
    )

    return {
        "partition": (
            partition
        ),
        "row_group_index": (
            row_group_index
        ),
        "sample_count": (
            sample_count
        ),
        "trajectory_uid": (
            trajectory_uid
        ),
        "chemistry": (
            chemistry
        ),
        "cell_entity_id": (
            cell_entity_id
        ),

        # Zero-based index used inside this trajectory.
        "sample_index": (
            trajectory_sample_index
        ),
        "trajectory_sample_index": (
            trajectory_sample_index
        ),

        # Original source numbering retained for traceability.
        "source_sample_index": (
            source_sample_index
        ),
        "source_sample_index_origin": int(
            source_sample_index[
                0
            ]
        ),

        "elapsed_time_s": (
            elapsed_time_s
        ),
        "voltage_v": numeric(
            "voltage_v"
        ),
        "current_discharge_a": numeric(
            "current_discharge_a"
        ),
        "temperature_c": numeric(
            "temperature_c"
        ),
        "soc_for_ocv": numeric(
            "soc_for_ocv"
        ),
        "ocv_v": numeric(
            "ocv_v"
        ),
        "delta_t_s": (
            delta_t_s
        ),
        "basis_matrix": (
            basis_matrix
        ),
        **selector_arrays,
    }


def trajectory_metric_record(
    *,
    partition: str,
    row_group_index: int,
    trajectory_data: Mapping[str, Any],
    simulation: Mapping[str, Any],
) -> tuple[
    dict[str, Any],
    np.ndarray,
]:
    residual = np.asarray(
        simulation[
            "residual_v"
        ],
        dtype=np.float64,
    )[
        LOSS_START_SAMPLE_INDEX:
    ]

    if residual.size <= 0:
        raise RuntimeError(
            "No residual samples remain after "
            "LOSS_START_SAMPLE_INDEX for "
            f"{trajectory_data['trajectory_uid']!r}."
        )

    record = {
        "partition": (
            partition
        ),
        "row_group_index": int(
            row_group_index
        ),
        "trajectory_uid": str(
            trajectory_data[
                "trajectory_uid"
            ]
        ),
        "chemistry": str(
            trajectory_data[
                "chemistry"
            ]
        ),
        "cell_entity_id": str(
            trajectory_data[
                "cell_entity_id"
            ]
        ),
        "cell_effect_included": bool(
            trajectory_data.get(
                "has_cell_effect",
                True,
            )
        ),
        "sample_count": int(
            trajectory_data[
                "sample_count"
            ]
        ),
        "evaluation_sample_count": int(
            residual.size
        ),
        "rest_start_detected": bool(
            simulation[
                "rest_start_detected"
            ]
        ),
        **calculate_residual_metrics(
            residual
        ),
        "minimum_R0_ohm": float(
            np.min(
                simulation[
                    "R0_ohm"
                ]
            )
        ),
        "maximum_R0_ohm": float(
            np.max(
                simulation[
                    "R0_ohm"
                ]
            )
        ),
        "minimum_R1_ohm": float(
            np.min(
                simulation[
                    "R1_ohm"
                ]
            )
        ),
        "maximum_R1_ohm": float(
            np.max(
                simulation[
                    "R1_ohm"
                ]
            )
        ),
        "minimum_R2_ohm": float(
            np.min(
                simulation[
                    "R2_ohm"
                ]
            )
        ),
        "maximum_R2_ohm": float(
            np.max(
                simulation[
                    "R2_ohm"
                ]
            )
        ),
        "minimum_tau1_s": float(
            np.min(
                simulation[
                    "tau1_s"
                ]
            )
        ),
        "maximum_tau1_s": float(
            np.max(
                simulation[
                    "tau1_s"
                ]
            )
        ),
        "minimum_tau_gap_s": float(
            np.min(
                simulation[
                    "tau2_s"
                ]
                -
                simulation[
                    "tau1_s"
                ]
            )
        ),
        "maximum_tau2_s": float(
            np.max(
                simulation[
                    "tau2_s"
                ]
            )
        ),
        "maximum_alpha1": float(
            np.max(
                simulation[
                    "alpha1"
                ]
            )
        ),
        "maximum_alpha2": float(
            np.max(
                simulation[
                    "alpha2"
                ]
            )
        ),
    }

    return record, residual


def partition_summary(
    *,
    partition: str,
    trajectory_metrics_df: pd.DataFrame,
    pooled_squared_error: float,
    pooled_absolute_error: float,
    pooled_residual_sum: float,
    pooled_sample_count: int,
    elapsed_seconds: float,
    prediction_path: Path | None = None,
) -> dict[str, Any]:
    if (
        trajectory_metrics_df.empty
        or
        pooled_sample_count
        <=
        0
    ):
        raise RuntimeError(
            f"No evaluation results were produced for "
            f"{partition!r}."
        )

    summary: dict[str, Any] = {
        "partition": (
            partition
        ),
        "trajectory_count": int(
            len(
                trajectory_metrics_df
            )
        ),
        "sample_count": int(
            pooled_sample_count
        ),
        "pooled_rmse_v": float(
            math.sqrt(
                pooled_squared_error
                /
                pooled_sample_count
            )
        ),
        "pooled_mae_v": float(
            pooled_absolute_error
            /
            pooled_sample_count
        ),
        "pooled_bias_v": float(
            pooled_residual_sum
            /
            pooled_sample_count
        ),
        "trajectory_weighted_rmse_v": float(
            trajectory_metrics_df[
                "rmse_v"
            ].mean()
        ),
        "median_trajectory_rmse_v": float(
            trajectory_metrics_df[
                "rmse_v"
            ].median()
        ),
        "maximum_trajectory_rmse_v": float(
            trajectory_metrics_df[
                "rmse_v"
            ].max()
        ),
        "elapsed_seconds": float(
            elapsed_seconds
        ),
        "prediction_path": None,
        "prediction_sha256": None,
    }

    if prediction_path is not None:
        prediction_path = Path(
            prediction_path
        )

        summary[
            "prediction_path"
        ] = str(
            prediction_path
        )

        summary[
            "prediction_sha256"
        ] = calculate_streaming_sha256(
            prediction_path,
            chunk_bytes=(
                NOTEBOOK_12_HASH_CHUNK_BYTES
            ),
        )

    return summary


def evaluate_frozen_development_partition(
    coefficients: np.ndarray,
    *,
    partition: str,
    trajectory_limit: int | None = None,
) -> tuple[
    dict[str, Any],
    pd.DataFrame,
]:
    """
    Evaluate the frozen model on train or calibration without using
    Cell 22's calibration-only evaluator.
    """

    partition = str(
        partition
    ).strip().lower()

    if partition not in {
        "train",
        "calibration",
    }:
        raise ValueError(
            "Frozen development evaluation permits only "
            "'train' and 'calibration'."
        )

    parquet_file = get_design_parquet(
        partition
    )

    row_group_count = int(
        parquet_file.metadata.num_row_groups
    )

    if trajectory_limit is not None:
        trajectory_limit = int(
            trajectory_limit
        )

        if trajectory_limit <= 0:
            raise ValueError(
                "trajectory_limit must be positive when provided."
            )

        row_group_count = min(
            row_group_count,
            trajectory_limit,
        )

    if row_group_count <= 0:
        raise RuntimeError(
            f"No row groups are available for "
            f"{partition!r}."
        )

    records: list[
        dict[str, Any]
    ] = []

    pooled_squared_error = 0.0
    pooled_absolute_error = 0.0
    pooled_residual_sum = 0.0
    pooled_sample_count = 0

    start_time = time.perf_counter()

    for row_group_index in range(
        row_group_count
    ):
        trajectory_data = (
            load_design_trajectory(
                partition,
                row_group_index,
            )
        )

        simulation = (
            simulate_loaded_trajectory_numpy(
                coefficients,
                trajectory_data,
            )
        )

        record, residual = (
            trajectory_metric_record(
                partition=partition,
                row_group_index=(
                    row_group_index
                ),
                trajectory_data=(
                    trajectory_data
                ),
                simulation=(
                    simulation
                ),
            )
        )

        records.append(
            record
        )

        pooled_squared_error += float(
            np.sum(
                np.square(
                    residual
                )
            )
        )

        pooled_absolute_error += float(
            np.sum(
                np.abs(
                    residual
                )
            )
        )

        pooled_residual_sum += float(
            np.sum(
                residual
            )
        )

        pooled_sample_count += int(
            residual.size
        )

    trajectory_metrics_df = pd.DataFrame(
        records
    )

    summary = partition_summary(
        partition=partition,
        trajectory_metrics_df=(
            trajectory_metrics_df
        ),
        pooled_squared_error=(
            pooled_squared_error
        ),
        pooled_absolute_error=(
            pooled_absolute_error
        ),
        pooled_residual_sum=(
            pooled_residual_sum
        ),
        pooled_sample_count=(
            pooled_sample_count
        ),
        elapsed_seconds=(
            time.perf_counter()
            -
            start_time
        ),
    )

    return summary, trajectory_metrics_df


def evaluate_final_partition(
    coefficients: np.ndarray,
    *,
    partition: str,
    prediction_path: Path,
    trajectory_limit: int | None = None,
) -> tuple[
    dict[str, Any],
    pd.DataFrame,
]:
    """
    Evaluate ordinary test or strict test and save sample-level predictions.
    """

    partition = str(
        partition
    ).strip().lower()

    if partition not in {
        "test",
        "strict_test",
    }:
        raise ValueError(
            "Held-out evaluation permits only "
            "'test' and 'strict_test'."
        )

    prediction_path = Path(
        prediction_path
    )

    prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    parquet_file = (
        FINAL_DESIGN_PARQUETS[
            partition
        ]
    )

    row_group_count = int(
        parquet_file.metadata.num_row_groups
    )

    if trajectory_limit is not None:
        trajectory_limit = int(
            trajectory_limit
        )

        if trajectory_limit <= 0:
            raise ValueError(
                "trajectory_limit must be positive when provided."
            )

        row_group_count = min(
            row_group_count,
            trajectory_limit,
        )

    if row_group_count <= 0:
        raise RuntimeError(
            f"No row groups are available for "
            f"{partition!r}."
        )

    temporary_path = prediction_path.with_suffix(
        prediction_path.suffix
        +
        ".tmp"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    writer: pq.ParquetWriter | None = None
    prediction_schema: pa.Schema | None = None

    records: list[
        dict[str, Any]
    ] = []

    pooled_squared_error = 0.0
    pooled_absolute_error = 0.0
    pooled_residual_sum = 0.0
    pooled_sample_count = 0

    start_time = time.perf_counter()

    try:
        for row_group_index in range(
            row_group_count
        ):
            trajectory_data = (
                load_final_design_trajectory(
                    partition,
                    row_group_index,
                )
            )

            simulation = (
                simulate_loaded_trajectory_numpy(
                    coefficients,
                    trajectory_data,
                )
            )

            record, residual = (
                trajectory_metric_record(
                    partition=partition,
                    row_group_index=(
                        row_group_index
                    ),
                    trajectory_data=(
                        trajectory_data
                    ),
                    simulation=(
                        simulation
                    ),
                )
            )

            records.append(
                record
            )

            pooled_squared_error += float(
                np.sum(
                    np.square(
                        residual
                    )
                )
            )

            pooled_absolute_error += float(
                np.sum(
                    np.abs(
                        residual
                    )
                )
            )

            pooled_residual_sum += float(
                np.sum(
                    residual
                )
            )

            pooled_sample_count += int(
                residual.size
            )

            sample_count = int(
                trajectory_data[
                    "sample_count"
                ]
            )

            prediction_table = pa.table(
                {
                    "trajectory_uid": pa.array(
                        [
                            trajectory_data[
                                "trajectory_uid"
                            ]
                        ]
                        *
                        sample_count,
                        type=pa.string(),
                    ),
                    "chemistry": pa.array(
                        [
                            trajectory_data[
                                "chemistry"
                            ]
                        ]
                        *
                        sample_count,
                        type=pa.string(),
                    ),
                    "cell_entity_id": pa.array(
                        [
                            trajectory_data[
                                "cell_entity_id"
                            ]
                        ]
                        *
                        sample_count,
                        type=pa.string(),
                    ),
                    "partition": pa.array(
                        [
                            partition
                        ]
                        *
                        sample_count,
                        type=pa.string(),
                    ),

                    # Zero-based location within the trajectory.
                    "sample_index": pa.array(
                        trajectory_data[
                            "trajectory_sample_index"
                        ],
                        type=pa.int64(),
                    ),

                    # Original source numbering.
                    "source_sample_index": pa.array(
                        trajectory_data[
                            "source_sample_index"
                        ],
                        type=pa.int64(),
                    ),

                    "elapsed_time_s": pa.array(
                        trajectory_data[
                            "elapsed_time_s"
                        ],
                        type=pa.float64(),
                    ),
                    "soc_for_ocv": pa.array(
                        trajectory_data[
                            "soc_for_ocv"
                        ],
                        type=pa.float64(),
                    ),
                    "temperature_c": pa.array(
                        trajectory_data[
                            "temperature_c"
                        ],
                        type=pa.float64(),
                    ),
                    "current_discharge_a": pa.array(
                        trajectory_data[
                            "current_discharge_a"
                        ],
                        type=pa.float64(),
                    ),
                    "measured_voltage_v": pa.array(
                        trajectory_data[
                            "voltage_v"
                        ],
                        type=pa.float64(),
                    ),
                    "predicted_voltage_v": pa.array(
                        simulation[
                            "predicted_voltage_v"
                        ],
                        type=pa.float64(),
                    ),
                    "residual_v": pa.array(
                        simulation[
                            "residual_v"
                        ],
                        type=pa.float64(),
                    ),
                    "R0_ohm": pa.array(
                        simulation[
                            "R0_ohm"
                        ],
                        type=pa.float64(),
                    ),
                    "R1_ohm": pa.array(
                        simulation[
                            "R1_ohm"
                        ],
                        type=pa.float64(),
                    ),
                    "R2_ohm": pa.array(
                        simulation[
                            "R2_ohm"
                        ],
                        type=pa.float64(),
                    ),
                    "tau1_s": pa.array(
                        simulation[
                            "tau1_s"
                        ],
                        type=pa.float64(),
                    ),
                    "tau2_s": pa.array(
                        simulation[
                            "tau2_s"
                        ],
                        type=pa.float64(),
                    ),
                    "evaluation_sample": pa.array(
                        np.arange(
                            sample_count,
                            dtype=np.int64,
                        )
                        >=
                        LOSS_START_SAMPLE_INDEX,
                        type=pa.bool_(),
                    ),
                }
            )

            if writer is None:
                prediction_schema = (
                    prediction_table.schema
                )

                writer = pq.ParquetWriter(
                    temporary_path,
                    prediction_schema,
                    compression="zstd",
                    use_dictionary=True,
                    write_statistics=True,
                )

            elif not prediction_table.schema.equals(
                prediction_schema,
                check_metadata=False,
            ):
                raise RuntimeError(
                    "The final prediction schema changed between trajectories."
                )

            writer.write_table(
                prediction_table,
                row_group_size=(
                    prediction_table.num_rows
                ),
            )

    except Exception:
        if writer is not None:
            writer.close()

        if temporary_path.exists():
            temporary_path.unlink()

        raise

    if writer is None:
        if temporary_path.exists():
            temporary_path.unlink()

        raise RuntimeError(
            f"No prediction rows were written for "
            f"{partition!r}."
        )

    writer.close()

    if prediction_path.exists():
        prediction_path.unlink()

    temporary_path.replace(
        prediction_path
    )

    trajectory_metrics_df = pd.DataFrame(
        records
    )

    summary = partition_summary(
        partition=partition,
        trajectory_metrics_df=(
            trajectory_metrics_df
        ),
        pooled_squared_error=(
            pooled_squared_error
        ),
        pooled_absolute_error=(
            pooled_absolute_error
        ),
        pooled_residual_sum=(
            pooled_residual_sum
        ),
        pooled_sample_count=(
            pooled_sample_count
        ),
        elapsed_seconds=(
            time.perf_counter()
            -
            start_time
        ),
        prediction_path=(
            prediction_path
        ),
    )

    return summary, trajectory_metrics_df


# ======================================================================================
# RUN THE FROZEN EVALUATION
# ======================================================================================

TRAIN_FINAL_SUMMARY, TRAIN_FINAL_TRAJECTORY_METRICS_DF = (
    evaluate_frozen_development_partition(
        FROZEN_REDUCED_COEFFICIENTS,
        partition="train",
        trajectory_limit=(
            EVALUATION_TRAJECTORY_LIMIT
        ),
    )
)

CALIBRATION_FINAL_SUMMARY, CALIBRATION_FINAL_TRAJECTORY_METRICS_DF = (
    evaluate_frozen_development_partition(
        FROZEN_REDUCED_COEFFICIENTS,
        partition="calibration",
        trajectory_limit=(
            EVALUATION_TRAJECTORY_LIMIT
        ),
    )
)

NOTEBOOK_12_TEST_PREDICTION_PATH = (
    NOTEBOOK_12_DATA_DIR
    /
    "lpv_ordinary_test_predictions.parquet"
)

NOTEBOOK_12_STRICT_TEST_PREDICTION_PATH = (
    NOTEBOOK_12_DATA_DIR
    /
    "lpv_strict_test_predictions.parquet"
)

TEST_FINAL_SUMMARY, TEST_FINAL_TRAJECTORY_METRICS_DF = (
    evaluate_final_partition(
        FROZEN_REDUCED_COEFFICIENTS,
        partition="test",
        prediction_path=(
            NOTEBOOK_12_TEST_PREDICTION_PATH
        ),
        trajectory_limit=(
            EVALUATION_TRAJECTORY_LIMIT
        ),
    )
)

STRICT_TEST_FINAL_SUMMARY, STRICT_TEST_FINAL_TRAJECTORY_METRICS_DF = (
    evaluate_final_partition(
        FROZEN_REDUCED_COEFFICIENTS,
        partition="strict_test",
        prediction_path=(
            NOTEBOOK_12_STRICT_TEST_PREDICTION_PATH
        ),
        trajectory_limit=(
            EVALUATION_TRAJECTORY_LIMIT
        ),
    )
)


# ======================================================================================
# COMBINE AND SAVE FINAL METRICS
# ======================================================================================

NOTEBOOK_12_FINAL_SPLIT_METRICS_DF = pd.DataFrame(
    [
        TRAIN_FINAL_SUMMARY,
        CALIBRATION_FINAL_SUMMARY,
        TEST_FINAL_SUMMARY,
        STRICT_TEST_FINAL_SUMMARY,
    ]
)

NOTEBOOK_12_FINAL_TRAJECTORY_METRICS_DF = pd.concat(
    [
        TRAIN_FINAL_TRAJECTORY_METRICS_DF,
        CALIBRATION_FINAL_TRAJECTORY_METRICS_DF,
        TEST_FINAL_TRAJECTORY_METRICS_DF,
        STRICT_TEST_FINAL_TRAJECTORY_METRICS_DF,
    ],
    ignore_index=True,
)

expected_partition_order = [
    "train",
    "calibration",
    "test",
    "strict_test",
]

observed_partition_order = (
    NOTEBOOK_12_FINAL_SPLIT_METRICS_DF[
        "partition"
    ]
    .astype(
        str
    )
    .tolist()
)

if (
    observed_partition_order
    !=
    expected_partition_order
):
    raise RuntimeError(
        "Unexpected final partition order: "
        f"{observed_partition_order!r}."
    )

metric_columns = [
    "pooled_rmse_v",
    "pooled_mae_v",
    "pooled_bias_v",
    "trajectory_weighted_rmse_v",
    "median_trajectory_rmse_v",
    "maximum_trajectory_rmse_v",
]

if not np.isfinite(
    NOTEBOOK_12_FINAL_SPLIT_METRICS_DF[
        metric_columns
    ].to_numpy(
        dtype=np.float64
    )
).all():
    raise RuntimeError(
        "Final split metrics contain a nonfinite value."
    )

NOTEBOOK_12_FINAL_SPLIT_METRICS_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_final_split_metrics.parquet"
)

NOTEBOOK_12_FINAL_TRAJECTORY_METRICS_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_final_trajectory_metrics.parquet"
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_FINAL_SPLIT_METRICS_DF,
    NOTEBOOK_12_FINAL_SPLIT_METRICS_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_FINAL_TRAJECTORY_METRICS_DF,
    NOTEBOOK_12_FINAL_TRAJECTORY_METRICS_PATH,
)


# ======================================================================================
# REPORT FINAL RESULTS
# ======================================================================================

print("=" * 110)
print("FROZEN LPV FINAL EVALUATION")
print("=" * 110)

for row in (
    NOTEBOOK_12_FINAL_SPLIT_METRICS_DF
    .itertuples(
        index=False
    )
):
    print(
        f"{str(row.partition):12s} "
        f"pooled RMSE="
        f"{float(row.pooled_rmse_v):.6f} V, "
        f"trajectory-weighted RMSE="
        f"{float(row.trajectory_weighted_rmse_v):.6f} V"
    )

print("=" * 110)

print(
    "Final split metrics       : "
    f"{NOTEBOOK_12_FINAL_SPLIT_METRICS_PATH}"
)

print(
    "Final trajectory metrics  : "
    f"{NOTEBOOK_12_FINAL_TRAJECTORY_METRICS_PATH}"
)

print(
    "Ordinary-test predictions : "
    f"{NOTEBOOK_12_TEST_PREDICTION_PATH}"
)

print(
    "Strict-test predictions   : "
    f"{NOTEBOOK_12_STRICT_TEST_PREDICTION_PATH}"
)

print("=" * 110)

display(
    NOTEBOOK_12_FINAL_SPLIT_METRICS_DF
)

FROZEN LPV FINAL EVALUATION
train        pooled RMSE=0.289151 V, trajectory-weighted RMSE=0.139670 V
calibration  pooled RMSE=0.392795 V, trajectory-weighted RMSE=0.291071 V
test         pooled RMSE=0.432149 V, trajectory-weighted RMSE=0.355881 V
strict_test  pooled RMSE=0.607619 V, trajectory-weighted RMSE=0.545173 V
Final split metrics       : /home/xzkhu/Downloads/batter_lpv_project/data/processed/notebook_12/tables/lpv_final_split_metrics.parquet
Final trajectory metrics  : /home/xzkhu/Downloads/batter_lpv_project/data/processed/notebook_12/tables/lpv_final_trajectory_metrics.parquet
Ordinary-test predictions : /home/xzkhu/Downloads/batter_lpv_project/data/processed/notebook_12/data/lpv_ordinary_test_predictions.parquet
Strict-test predictions   : /home/xzkhu/Downloads/batter_lpv_project/data/processed/notebook_12/data/lpv_strict_test_predictions.parquet


,partition,trajectory_count,sample_count,pooled_rmse_v,pooled_mae_v,pooled_bias_v,trajectory_weighted_rmse_v,median_trajectory_rmse_v,maximum_trajectory_rmse_v,elapsed_seconds,prediction_path,prediction_sha256
0,train,2832,8969712,0.289151,0.208455,-0.060494,0.139670,0.104246,0.822234,6.909118,None,None
1,calibration,1017,4391688,0.392795,0.290801,0.097059,0.291071,0.205092,0.669065,2.864621,None,None
2,test,1080,4917214,0.432149,0.332122,0.167656,0.355881,0.391042,0.668338,6.509468,/home/xzkhu/Downloads/batter_lpv_project/data/...,cfc600a13746589b6ff114ee05e01775480a48d10feb1c...
3,strict_test,9,35321,0.607619,0.449563,-0.448338,0.545173,0.520922,0.709588,0.056382,/home/xzkhu/Downloads/batter_lpv_project/data/...,f77d0aec7a44d311b25402bc63d6349cc3bfa33b4ee58c...


In [43]:
# ======================================================================================
# CELL 26 — RESIDUAL, PHYSICAL-CONSISTENCY, STABILITY, AND SENSITIVITY DIAGNOSTICS
# ======================================================================================

def streaming_residual_correlations(
    prediction_path: Path,
    *,
    batch_size: int = 250_000,
) -> pd.DataFrame:
    parquet_file = pq.ParquetFile(
        prediction_path
    )

    predictor_columns = (
        "current_discharge_a",
        "soc_for_ocv",
        "temperature_c",
        "elapsed_time_s",
    )

    accumulators = {
        predictor: {
            "count": 0,
            "sum_x": 0.0,
            "sum_y": 0.0,
            "sum_x2": 0.0,
            "sum_y2": 0.0,
            "sum_xy": 0.0,
        }
        for predictor in predictor_columns
    }

    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=[
            "evaluation_sample",
            "residual_v",
            *predictor_columns,
        ],
        use_threads=True,
    ):
        table = pa.Table.from_batches(
            [
                batch
            ]
        )

        evaluation_mask = np.asarray(
            pc.cast(
                table[
                    "evaluation_sample"
                ],
                pa.bool_(),
                safe=False,
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=bool,
        )

        residual = np.asarray(
            pc.cast(
                table[
                    "residual_v"
                ],
                pa.float64(),
                safe=False,
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.float64,
        )

        for predictor in predictor_columns:
            predictor_values = np.asarray(
                pc.cast(
                    table[
                        predictor
                    ],
                    pa.float64(),
                    safe=False,
                ).to_numpy(
                    zero_copy_only=False
                ),
                dtype=np.float64,
            )

            valid = (
                evaluation_mask
                &
                np.isfinite(
                    residual
                )
                &
                np.isfinite(
                    predictor_values
                )
            )

            if not valid.any():
                continue

            x = predictor_values[
                valid
            ]

            y = residual[
                valid
            ]

            accumulator = accumulators[
                predictor
            ]

            accumulator[
                "count"
            ] += int(
                x.size
            )

            accumulator[
                "sum_x"
            ] += float(
                np.sum(
                    x
                )
            )

            accumulator[
                "sum_y"
            ] += float(
                np.sum(
                    y
                )
            )

            accumulator[
                "sum_x2"
            ] += float(
                np.sum(
                    np.square(
                        x
                    )
                )
            )

            accumulator[
                "sum_y2"
            ] += float(
                np.sum(
                    np.square(
                        y
                    )
                )
            )

            accumulator[
                "sum_xy"
            ] += float(
                np.sum(
                    x
                    *
                    y
                )
            )

    records: list[
        dict[str, Any]
    ] = []

    for predictor, accumulator in (
        accumulators.items()
    ):
        count = int(
            accumulator[
                "count"
            ]
        )

        if count < 2:
            correlation = np.nan

        else:
            numerator = (
                count
                *
                accumulator[
                    "sum_xy"
                ]
                -
                accumulator[
                    "sum_x"
                ]
                *
                accumulator[
                    "sum_y"
                ]
            )

            denominator_x = (
                count
                *
                accumulator[
                    "sum_x2"
                ]
                -
                accumulator[
                    "sum_x"
                ]
                **
                2
            )

            denominator_y = (
                count
                *
                accumulator[
                    "sum_y2"
                ]
                -
                accumulator[
                    "sum_y"
                ]
                **
                2
            )

            denominator = math.sqrt(
                max(
                    denominator_x,
                    0.0,
                )
                *
                max(
                    denominator_y,
                    0.0,
                )
            )

            correlation = (
                numerator
                /
                denominator
                if denominator
                >
                0.0
                else np.nan
            )

        records.append(
            {
                "predictor": predictor,
                "sample_count": count,
                "residual_correlation": (
                    float(
                        correlation
                    )
                    if np.isfinite(
                        correlation
                    )
                    else np.nan
                ),
                "absolute_residual_correlation": (
                    float(
                        abs(
                            correlation
                        )
                    )
                    if np.isfinite(
                        correlation
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        records
    )


NOTEBOOK_12_TEST_RESIDUAL_CORRELATION_DF = (
    streaming_residual_correlations(
        NOTEBOOK_12_TEST_PREDICTION_PATH
    )
)

NOTEBOOK_12_STRICT_TEST_RESIDUAL_CORRELATION_DF = (
    streaming_residual_correlations(
        NOTEBOOK_12_STRICT_TEST_PREDICTION_PATH
    )
)

NOTEBOOK_12_TEST_RESIDUAL_CORRELATION_DF[
    "partition"
] = "test"

NOTEBOOK_12_STRICT_TEST_RESIDUAL_CORRELATION_DF[
    "partition"
] = "strict_test"

NOTEBOOK_12_RESIDUAL_CORRELATION_DF = pd.concat(
    [
        NOTEBOOK_12_TEST_RESIDUAL_CORRELATION_DF,
        NOTEBOOK_12_STRICT_TEST_RESIDUAL_CORRELATION_DF,
    ],
    ignore_index=True,
)

physical_diagnostic_records: list[
    dict[str, Any]
] = []

for partition, partition_df in (
    NOTEBOOK_12_FINAL_TRAJECTORY_METRICS_DF
    .groupby(
        "partition",
        sort=False,
    )
):
    physical_diagnostic_records.append(
        {
            "partition": partition,
            "minimum_R0_ohm": float(
                partition_df[
                    "minimum_R0_ohm"
                ].min()
            ),
            "maximum_R0_ohm": float(
                partition_df[
                    "maximum_R0_ohm"
                ].max()
            ),
            "minimum_R1_ohm": float(
                partition_df[
                    "minimum_R1_ohm"
                ].min()
            ),
            "maximum_R1_ohm": float(
                partition_df[
                    "maximum_R1_ohm"
                ].max()
            ),
            "minimum_R2_ohm": float(
                partition_df[
                    "minimum_R2_ohm"
                ].min()
            ),
            "maximum_R2_ohm": float(
                partition_df[
                    "maximum_R2_ohm"
                ].max()
            ),
            "minimum_tau1_s": float(
                partition_df[
                    "minimum_tau1_s"
                ].min()
            ),
            "minimum_tau_gap_s": float(
                partition_df[
                    "minimum_tau_gap_s"
                ].min()
            ),
            "maximum_tau2_s": float(
                partition_df[
                    "maximum_tau2_s"
                ].max()
            ),
            "maximum_alpha1": float(
                partition_df[
                    "maximum_alpha1"
                ].max()
            ),
            "maximum_alpha2": float(
                partition_df[
                    "maximum_alpha2"
                ].max()
            ),
        }
    )

NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF = pd.DataFrame(
    physical_diagnostic_records
)

NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
    "positive_resistances"
] = (
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
        [
            "minimum_R0_ohm",
            "minimum_R1_ohm",
            "minimum_R2_ohm",
        ]
    ]
    .gt(
        0.0
    )
    .all(
        axis=1
    )
)

NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
    "ordered_positive_time_constants"
] = (
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
        "minimum_tau1_s"
    ].gt(
        0.0
    )
    &
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
        "minimum_tau_gap_s"
    ].gt(
        0.0
    )
)

NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
    "recursive_stability"
] = (
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
        "maximum_alpha1"
    ].le(
        1.0
        +
        1.0e-12
    )
    &
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
        "maximum_alpha2"
    ].le(
        1.0
        +
        1.0e-12
    )
)

# Local sensitivity diagnostic on training-only mini-batches.
SENSITIVITY_BATCH_COUNT = (
    20
    if not FAST_DEMO
    else 4
)

support_indices = np.flatnonzero(
    SELECTED_SUPPORT_MASK
)

if support_indices.size == 0:
    raise RuntimeError(
        "No active coefficient is available for sensitivity analysis."
    )


def data_loss_only(
    coefficients: jax.Array,
    batch: Mapping[str, jax.Array],
) -> jax.Array:
    residual = _simulate_lpv_batch_core(
        coefficients,
        batch,
    )[
        1
    ]

    return _pseudo_huber_mean(
        residual,
        batch[
            "loss_mask"
        ],
        PSEUDO_HUBER_DELTA_V,
    )


data_loss_gradient = jax.jit(
    jax.grad(
        data_loss_only,
        argnums=0,
    )
)

sensitivity_rng = np.random.default_rng(
    RANDOM_SEED
    +
    260_000
)

sensitivity_rows: list[
    np.ndarray
] = []

for _ in range(
    SENSITIVITY_BATCH_COUNT
):
    sampled_row_groups = sensitivity_rng.choice(
        training_row_group_indices,
        size=min(
            TRAINING_BATCH_TRAJECTORIES,
            TRAIN_ROW_GROUP_COUNT,
        ),
        replace=(
            TRAIN_ROW_GROUP_COUNT
            <
            TRAINING_BATCH_TRAJECTORIES
        ),
    )

    batch = batch_to_jax(
        load_padded_trajectory_batch(
            partition="train",
            row_group_indices=(
                sampled_row_groups
            ),
        )
    )

    gradient = np.asarray(
        data_loss_gradient(
            jnp.asarray(
                FROZEN_REDUCED_COEFFICIENTS,
                dtype=jnp.float64,
            ),
            batch,
        ),
        dtype=np.float64,
    )

    sensitivity_rows.append(
        gradient[
            support_indices
        ]
    )

sensitivity_matrix = np.vstack(
    sensitivity_rows
)

centered_sensitivity_matrix = (
    sensitivity_matrix
    -
    sensitivity_matrix.mean(
        axis=0,
        keepdims=True,
    )
)

sensitivity_singular_values = np.linalg.svd(
    centered_sensitivity_matrix,
    compute_uv=False,
)

if sensitivity_singular_values.size == 0:
    practical_rank = 0
    condition_number = np.inf
else:
    rank_threshold = (
        max(
            centered_sensitivity_matrix.shape
        )
        *
        np.finfo(
            np.float64
        ).eps
        *
        sensitivity_singular_values[
            0
        ]
    )

    practical_rank = int(
        (
            sensitivity_singular_values
            >
            rank_threshold
        ).sum()
    )

    positive_singular_values = (
        sensitivity_singular_values[
            sensitivity_singular_values
            >
            rank_threshold
        ]
    )

    condition_number = (
        float(
            positive_singular_values[
                0
            ]
            /
            positive_singular_values[
                -1
            ]
        )
        if positive_singular_values.size
        >
        1
        else 1.0
        if positive_singular_values.size
        ==
        1
        else np.inf
    )

NOTEBOOK_12_SENSITIVITY_SINGULAR_VALUES_DF = pd.DataFrame(
    {
        "singular_value_index": np.arange(
            sensitivity_singular_values.size,
            dtype=np.int64,
        ),
        "singular_value": (
            sensitivity_singular_values
        ),
    }
)

NOTEBOOK_12_SENSITIVITY_SUMMARY_DF = pd.DataFrame(
    [
        {
            "active_reduced_coefficient_count": int(
                support_indices.size
            ),
            "sensitivity_batch_count": int(
                SENSITIVITY_BATCH_COUNT
            ),
            "practical_rank": int(
                practical_rank
            ),
            "condition_number": float(
                condition_number
            ),
            "rank_fraction": float(
                practical_rank
                /
                max(
                    support_indices.size,
                    1,
                )
            ),
        }
    ]
)

NOTEBOOK_12_RESIDUAL_CORRELATION_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_final_residual_correlations.parquet"
)

NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_physical_and_stability_diagnostics.parquet"
)

NOTEBOOK_12_SENSITIVITY_SINGULAR_VALUES_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_sensitivity_singular_values.parquet"
)

NOTEBOOK_12_SENSITIVITY_SUMMARY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_sensitivity_summary.parquet"
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_RESIDUAL_CORRELATION_DF,
    NOTEBOOK_12_RESIDUAL_CORRELATION_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF,
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_SENSITIVITY_SINGULAR_VALUES_DF,
    NOTEBOOK_12_SENSITIVITY_SINGULAR_VALUES_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_SENSITIVITY_SUMMARY_DF,
    NOTEBOOK_12_SENSITIVITY_SUMMARY_PATH,
)

print("=" * 110)
print("FINAL DIAGNOSTICS")
print("=" * 110)
print(
    f"Maximum test residual correlation   : "
    f"{NOTEBOOK_12_TEST_RESIDUAL_CORRELATION_DF['absolute_residual_correlation'].max():.6f}"
)
print(
    f"Maximum strict residual correlation : "
    f"{NOTEBOOK_12_STRICT_TEST_RESIDUAL_CORRELATION_DF['absolute_residual_correlation'].max():.6f}"
)
print(
    f"Active support practical rank        : "
    f"{practical_rank}/{support_indices.size}"
)
print("=" * 110)

display(
    NOTEBOOK_12_RESIDUAL_CORRELATION_DF
)

display(
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF
)

display(
    NOTEBOOK_12_SENSITIVITY_SUMMARY_DF
)


FINAL DIAGNOSTICS
Maximum test residual correlation   : 0.763612
Maximum strict residual correlation : 0.904732
Active support practical rank        : 5/5


,predictor,sample_count,residual_correlation,absolute_residual_correlation,partition
0,current_discharge_a,4917214,0.182820,0.182820,test
1,soc_for_ocv,4917214,-0.763612,0.763612,test
2,temperature_c,4917214,-0.069790,0.069790,test
3,elapsed_time_s,4917214,-0.238870,0.238870,test
4,current_discharge_a,35321,0.675903,0.675903,strict_test
5,soc_for_ocv,35321,-0.022880,0.022880,strict_test
6,temperature_c,35321,-0.904732,0.904732,strict_test
7,elapsed_time_s,35321,0.713053,0.713053,strict_test


,partition,minimum_R0_ohm,maximum_R0_ohm,minimum_R1_ohm,maximum_R1_ohm,minimum_R2_ohm,maximum_R2_ohm,minimum_tau1_s,minimum_tau_gap_s,maximum_tau2_s,maximum_alpha1,maximum_alpha2,positive_resistances,ordered_positive_time_constants,recursive_stability
0,train,0.000011,0.000011,0.000039,0.000039,0.00008,0.00008,37.156048,182.238615,219.394663,1.0,1.0,True,True,True
1,calibration,0.000011,0.000011,0.000039,0.000039,0.00008,0.00008,37.156048,182.238615,219.394663,1.0,1.0,True,True,True
2,test,0.000011,0.000011,0.000039,0.000039,0.00008,0.00008,37.156048,182.238615,219.394663,1.0,1.0,True,True,True
3,strict_test,0.000011,0.000011,0.000039,0.000039,0.00008,0.00008,37.156048,182.238615,219.394663,1.0,1.0,True,True,True


,active_reduced_coefficient_count,sensitivity_batch_count,practical_rank,condition_number,rank_fraction
0,5,20,5,871.773559,1.0


In [44]:
# ======================================================================================
# CELL 27 — TEST THE HYPOTHESES AND FORM THE FINAL SCIENTIFIC DECISION
# ======================================================================================

baseline_test_rmse_v = np.nan
baseline_strict_test_rmse_v = np.nan
baseline_test_bias_v = np.nan

if FIXED_ECM_TEST_HEADLINE_METRICS_PATH.is_file():
    fixed_headline_df = pd.read_parquet(
        FIXED_ECM_TEST_HEADLINE_METRICS_PATH,
        engine="pyarrow",
    )

    required_fixed_columns = {
        "evaluation_partition",
        "model_label",
        "pooled_rmse_v",
        "pooled_bias_v",
    }

    if required_fixed_columns.issubset(
        fixed_headline_df.columns
    ):
        primary_baseline_df = (
            fixed_headline_df.loc[
                fixed_headline_df[
                    "model_label"
                ].astype(str).eq(
                    "PRIMARY_FIXED_BASELINE"
                )
            ]
            .copy()
        )

        def select_fixed_baseline_row(
            partition: str,
        ) -> pd.Series | None:
            partition_df = (
                primary_baseline_df.loc[
                    primary_baseline_df[
                        "evaluation_partition"
                    ].astype(str).eq(
                        partition
                    )
                ]
                .copy()
            )

            if partition_df.empty:
                return None

            if "chemistry_group" in partition_df.columns:
                all_chemistry_mask = (
                    partition_df[
                        "chemistry_group"
                    ].astype(str).str.upper().isin(
                        (
                            "ALL",
                            "GLOBAL",
                            "ALL_CHEMISTRIES",
                        )
                    )
                )

                if all_chemistry_mask.any():
                    partition_df = partition_df.loc[
                        all_chemistry_mask
                    ]

            if "soc_eligibility_group" in partition_df.columns:
                preferred_order = {
                    "ALL": 0,
                    "FIT_ELIGIBLE": 1,
                    "EVALUATION_ONLY": 2,
                }

                partition_df[
                    "_eligibility_rank"
                ] = (
                    partition_df[
                        "soc_eligibility_group"
                    ]
                    .astype(str)
                    .str.upper()
                    .map(
                        preferred_order
                    )
                    .fillna(
                        99
                    )
                )

            else:
                partition_df[
                    "_eligibility_rank"
                ] = 0

            sample_column = (
                "evaluation_sample_count"
                if "evaluation_sample_count"
                in
                partition_df.columns
                else None
            )

            sort_columns = [
                "_eligibility_rank"
            ]

            ascending = [
                True
            ]

            if sample_column is not None:
                sort_columns.append(
                    sample_column
                )

                ascending.append(
                    False
                )

            partition_df = partition_df.sort_values(
                sort_columns,
                ascending=ascending,
                kind="mergesort",
            )

            return partition_df.iloc[
                0
            ]

        fixed_test_row = select_fixed_baseline_row(
            "test"
        )

        fixed_strict_row = select_fixed_baseline_row(
            "strict_test"
        )

        if fixed_test_row is not None:
            baseline_test_rmse_v = float(
                fixed_test_row[
                    "pooled_rmse_v"
                ]
            )

            baseline_test_bias_v = float(
                fixed_test_row[
                    "pooled_bias_v"
                ]
            )

        if fixed_strict_row is not None:
            baseline_strict_test_rmse_v = float(
                fixed_strict_row[
                    "pooled_rmse_v"
                ]
            )

lpv_test_rmse_v = float(
    TEST_FINAL_SUMMARY[
        "pooled_rmse_v"
    ]
)

lpv_strict_test_rmse_v = float(
    STRICT_TEST_FINAL_SUMMARY[
        "pooled_rmse_v"
    ]
)

lpv_test_bias_v = float(
    TEST_FINAL_SUMMARY[
        "pooled_bias_v"
    ]
)

test_relative_improvement = (
    float(
        (
            baseline_test_rmse_v
            -
            lpv_test_rmse_v
        )
        /
        baseline_test_rmse_v
    )
    if np.isfinite(
        baseline_test_rmse_v
    )
    and
    baseline_test_rmse_v
    >
    0.0
    else np.nan
)

strict_relative_improvement = (
    float(
        (
            baseline_strict_test_rmse_v
            -
            lpv_strict_test_rmse_v
        )
        /
        baseline_strict_test_rmse_v
    )
    if np.isfinite(
        baseline_strict_test_rmse_v
    )
    and
    baseline_strict_test_rmse_v
    >
    0.0
    else np.nan
)

global_candidate_mask = (
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF[
        "candidate_id"
    ].isin(
        (
            "M1_GLOBAL_DENSE_LPV",
            "M2_GLOBAL_SPARSE_LPV",
        )
    )
)

hierarchical_candidate_mask = (
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF[
        "candidate_id"
    ].isin(
        (
            "M3_HIERARCHICAL_DENSE_LPV",
            "M4_HIERARCHICAL_SPARSE_LPV",
        )
    )
)

best_global_calibration_rmse = float(
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF.loc[
        global_candidate_mask,
        "trajectory_weighted_rmse_v",
    ].min()
)

best_hierarchical_calibration_rmse = float(
    NOTEBOOK_12_CALIBRATION_JOB_METRICS_DF.loc[
        hierarchical_candidate_mask,
        "trajectory_weighted_rmse_v",
    ].min()
)

hierarchical_relative_improvement = float(
    (
        best_global_calibration_rmse
        -
        best_hierarchical_calibration_rmse
    )
    /
    best_global_calibration_rmse
)

selected_is_sparse = bool(
    MODEL_CANDIDATE_SPECIFICATIONS[
        SELECTED_CANDIDATE_ID
    ][
        "sparse"
    ]
)

selected_support_summary = (
    coefficient_support_summary(
        FROZEN_REDUCED_COEFFICIENTS,
        candidate_id=(
            SELECTED_CANDIDATE_ID
        ),
    )
)

dense_hierarchical_allowed_count = int(
    CANDIDATE_ACTIVE_MASKS[
        "M3_HIERARCHICAL_DENSE_LPV"
    ].sum()
)

physical_pass = bool(
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
        "positive_resistances"
    ].all()
    and
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
        "ordered_positive_time_constants"
    ].all()
)

stability_pass = bool(
    NOTEBOOK_12_PHYSICAL_DIAGNOSTICS_DF[
        "recursive_stability"
    ].all()
)

leakage_safe_pass = bool(
    MODEL_SELECTION_FROZEN
    and
    ORDINARY_TEST_ACCESS_PERMITTED
    and
    STRICT_TEST_ACCESS_PERMITTED
)

hypothesis_records: list[
    dict[str, Any]
] = [
    {
        "hypothesis": "H1_LPV_ADEQUACY",
        "criterion": (
            "Frozen LPV ordinary-test pooled RMSE is lower than "
            "the Notebook 11 primary fixed baseline."
        ),
        "observed_value": (
            test_relative_improvement
        ),
        "reference_value": 0.0,
        "passed": bool(
            np.isfinite(
                test_relative_improvement
            )
            and
            test_relative_improvement
            >
            0.0
        ),
        "details": (
            f"LPV test RMSE={lpv_test_rmse_v:.8f} V; "
            f"fixed baseline={baseline_test_rmse_v:.8f} V"
        ),
    },
    {
        "hypothesis": "H2_HIERARCHICAL_TRANSFER",
        "criterion": (
            "The best hierarchical calibration model improves on "
            "the best global-only LPV calibration model."
        ),
        "observed_value": (
            hierarchical_relative_improvement
        ),
        "reference_value": 0.0,
        "passed": bool(
            hierarchical_relative_improvement
            >
            0.0
        ),
        "details": (
            f"best global={best_global_calibration_rmse:.8f} V; "
            f"best hierarchical={best_hierarchical_calibration_rmse:.8f} V"
        ),
    },
    {
        "hypothesis": "H3_SPARSE_SUFFICIENCY",
        "criterion": (
            "A sparse selected model remains within the calibration "
            "complexity tolerance and uses fewer reduced coefficients "
            "than the dense hierarchical candidate."
        ),
        "observed_value": int(
            selected_support_summary[
                "nonzero_coefficient_count"
            ]
        ),
        "reference_value": int(
            dense_hierarchical_allowed_count
        ),
        "passed": bool(
            selected_is_sparse
            and
            selected_support_summary[
                "nonzero_coefficient_count"
            ]
            <
            dense_hierarchical_allowed_count
        ),
        "details": (
            f"selected candidate={SELECTED_CANDIDATE_ID}; "
            f"active coefficients="
            f"{selected_support_summary['nonzero_coefficient_count']}; "
            f"dense hierarchical allowed={dense_hierarchical_allowed_count}"
        ),
    },
    {
        "hypothesis": "H4_PHYSICAL_CONSISTENCY",
        "criterion": (
            "All evaluated resistances and time constants are positive "
            "and tau2 remains greater than tau1."
        ),
        "observed_value": (
            physical_pass
        ),
        "reference_value": True,
        "passed": (
            physical_pass
        ),
        "details": (
            "Physical parameter envelopes were checked on every evaluated trajectory."
        ),
    },
    {
        "hypothesis": "H5_RECURSIVE_STABILITY",
        "criterion": (
            "All evaluated recursive factors satisfy alpha1 <= 1 and alpha2 <= 1."
        ),
        "observed_value": (
            stability_pass
        ),
        "reference_value": True,
        "passed": (
            stability_pass
        ),
        "details": (
            "The exact zero-order-hold factors were audited on every evaluated trajectory."
        ),
    },
    {
        "hypothesis": "H6_LEAKAGE_SAFETY",
        "criterion": (
            "The model, support, and regularization were frozen before "
            "ordinary-test or strict-test sample access."
        ),
        "observed_value": (
            leakage_safe_pass
        ),
        "reference_value": True,
        "passed": (
            leakage_safe_pass
        ),
        "details": (
            "The quarantine contract records test_data_used_for_selection=False "
            "and strict_test_data_used_for_selection=False."
        ),
    },
]

NOTEBOOK_12_HYPOTHESIS_DECISION_DF = pd.DataFrame(
    hypothesis_records
)

hard_requirements_pass = bool(
    NOTEBOOK_12_HYPOTHESIS_DECISION_DF.loc[
        NOTEBOOK_12_HYPOTHESIS_DECISION_DF[
            "hypothesis"
        ].isin(
            (
                "H4_PHYSICAL_CONSISTENCY",
                "H5_RECURSIVE_STABILITY",
                "H6_LEAKAGE_SAFETY",
            )
        ),
        "passed",
    ].all()
)

predictive_hypotheses_passed = int(
    NOTEBOOK_12_HYPOTHESIS_DECISION_DF.loc[
        NOTEBOOK_12_HYPOTHESIS_DECISION_DF[
            "hypothesis"
        ].isin(
            (
                "H1_LPV_ADEQUACY",
                "H2_HIERARCHICAL_TRANSFER",
                "H3_SPARSE_SUFFICIENCY",
            )
        ),
        "passed",
    ].sum()
)

if not hard_requirements_pass:
    NOTEBOOK_12_FINAL_DECISION = "REJECT"

elif predictive_hypotheses_passed == 3:
    NOTEBOOK_12_FINAL_DECISION = "ACCEPT"

elif predictive_hypotheses_passed >= 1:
    NOTEBOOK_12_FINAL_DECISION = (
        "ACCEPT_WITH_CAUTION"
    )

else:
    NOTEBOOK_12_FINAL_DECISION = "REJECT"

NOTEBOOK_12_FINAL_DECISION_SUMMARY_DF = pd.DataFrame(
    [
        {
            "selected_candidate_id": (
                SELECTED_CANDIDATE_ID
            ),
            "selected_job_id": (
                SELECTED_JOB_ID
            ),
            "selected_regularization_lambda": (
                SELECTED_REGULARIZATION_LAMBDA
            ),
            "test_pooled_rmse_v": (
                lpv_test_rmse_v
            ),
            "strict_test_pooled_rmse_v": (
                lpv_strict_test_rmse_v
            ),
            "test_pooled_bias_v": (
                lpv_test_bias_v
            ),
            "fixed_baseline_test_rmse_v": (
                baseline_test_rmse_v
            ),
            "fixed_baseline_strict_test_rmse_v": (
                baseline_strict_test_rmse_v
            ),
            "test_relative_improvement": (
                test_relative_improvement
            ),
            "strict_test_relative_improvement": (
                strict_relative_improvement
            ),
            "maximum_test_residual_correlation": float(
                NOTEBOOK_12_TEST_RESIDUAL_CORRELATION_DF[
                    "absolute_residual_correlation"
                ].max()
            ),
            "active_group_count": int(
                len(
                    SELECTED_ACTIVE_GROUP_IDS
                )
            ),
            "active_reduced_coefficient_count": int(
                SELECTED_SUPPORT_MASK.sum()
            ),
            "practical_sensitivity_rank": int(
                practical_rank
            ),
            "physical_requirements_passed": (
                physical_pass
            ),
            "recursive_stability_passed": (
                stability_pass
            ),
            "leakage_safety_passed": (
                leakage_safe_pass
            ),
            "final_decision": (
                NOTEBOOK_12_FINAL_DECISION
            ),
        }
    ]
)

NOTEBOOK_12_HYPOTHESIS_DECISION_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_hypothesis_decisions.parquet"
)

NOTEBOOK_12_FINAL_DECISION_SUMMARY_PATH = (
    NOTEBOOK_12_TABLE_DIR
    /
    "lpv_final_decision_summary.parquet"
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_HYPOTHESIS_DECISION_DF,
    NOTEBOOK_12_HYPOTHESIS_DECISION_PATH,
)

write_dataframe_parquet_atomic(
    NOTEBOOK_12_FINAL_DECISION_SUMMARY_DF,
    NOTEBOOK_12_FINAL_DECISION_SUMMARY_PATH,
)

print("=" * 110)
print("NOTEBOOK 12 SCIENTIFIC DECISION")
print("=" * 110)
print(
    f"Selected candidate        : "
    f"{SELECTED_CANDIDATE_ID}"
)
print(
    f"Ordinary-test RMSE        : "
    f"{lpv_test_rmse_v:.8f} V"
)
print(
    f"Strict-test RMSE          : "
    f"{lpv_strict_test_rmse_v:.8f} V"
)
print(
    f"Fixed test RMSE           : "
    f"{baseline_test_rmse_v:.8f} V"
)
print(
    f"Relative test improvement : "
    f"{test_relative_improvement:.4%}"
)
print(
    f"Final decision            : "
    f"{NOTEBOOK_12_FINAL_DECISION}"
)
print("=" * 110)

display(
    NOTEBOOK_12_HYPOTHESIS_DECISION_DF
)

display(
    NOTEBOOK_12_FINAL_DECISION_SUMMARY_DF
)


NOTEBOOK 12 SCIENTIFIC DECISION
Selected candidate        : M0_FIXED_2RC
Ordinary-test RMSE        : 0.43214877 V
Strict-test RMSE          : 0.60761881 V
Fixed test RMSE           : 0.51159970 V
Relative test improvement : 15.5299%
Final decision            : ACCEPT_WITH_CAUTION


,hypothesis,criterion,observed_value,reference_value,passed,details
0,H1_LPV_ADEQUACY,Frozen LPV ordinary-test pooled RMSE is lower ...,0.155299,0.0,True,LPV test RMSE=0.43214877 V; fixed baseline=0.5...
1,H2_HIERARCHICAL_TRANSFER,The best hierarchical calibration model improv...,-0.000021,0.0,False,best global=0.29105713 V; best hierarchical=0....
2,H3_SPARSE_SUFFICIENCY,A sparse selected model remains within the cal...,5,240,False,selected candidate=M0_FIXED_2RC; active coeffi...
3,H4_PHYSICAL_CONSISTENCY,All evaluated resistances and time constants a...,True,True,True,Physical parameter envelopes were checked on e...
4,H5_RECURSIVE_STABILITY,All evaluated recursive factors satisfy alpha1...,True,True,True,The exact zero-order-hold factors were audited...
5,H6_LEAKAGE_SAFETY,"The model, support, and regularization were fr...",True,True,True,The quarantine contract records test_data_used...


,selected_candidate_id,selected_job_id,selected_regularization_lambda,test_pooled_rmse_v,strict_test_pooled_rmse_v,test_pooled_bias_v,fixed_baseline_test_rmse_v,fixed_baseline_strict_test_rmse_v,test_relative_improvement,strict_test_relative_improvement,maximum_test_residual_correlation,active_group_count,active_reduced_coefficient_count,practical_sensitivity_rank,physical_requirements_passed,recursive_stability_passed,leakage_safety_passed,final_decision
0,M0_FIXED_2RC,M0_FIXED_2RC__lambda_0p000e00,0.0,0.432149,0.607619,0.167656,0.5116,0.629889,0.155299,0.035356,0.763612,5,5,5,True,True,True,ACCEPT_WITH_CAUTION
